# Sentinel-3 OLCI L1B → L2Gen → Global Lake CI / CIcyano / NDCI / MPH / FAI
## Storage-efficient, scene-deduplicated, resumable Colab workflow (v2.6.8 native-first CyAN comparison)

This notebook preserves the validated **Sentinel-3 OLCI → SeaDAS/OCSSW L2Gen → Rayleigh-corrected rhos → CI / CIcyano / NDCI → HydroLAKES** science from the uploaded v1.4.2 notebook, while replacing its persistent full-scene storage pattern with a global production architecture.



### v2.6.8 comparison correction

- NASA CyAN comparison now selects the **nearest same-lake native OLCI observation first**, before any QA filtering.
- The chosen native pixel is then evaluated with the selected QA basis. If it fails QA, the comparison location remains no-data; the code does **not** substitute a farther clear pixel.
- **Bloom-aware QA is the default comparison basis.**
- Generic `CLDICE`-excluded screening remains available only as a sensitivity test, not as a CyAN-equivalent mask.
- Pair CSV/NetCDF outputs now record the nearest native coordinate, QA validity, `CLDICE`, bloom-rescue status, and matching distance for auditability.

### v2.6.7 production decision

- **Native compact OLCI lake pixels remain the quantitative source of truth.**
- Production mapping returns to the efficient branch; OCSSW `l2bin/l3mapgen` area-weighting is **not required** for routine processing and is reserved for optional validation experiments.
- Gradio and SNAP/GIS products use a **same-lake nearest-native remap** on the exact requested nominal grid. A target cell can only use a source observation with the same `Hylak_id`.
- The nearest native source is selected **before** QA is inspected, so a cloudy/invalid source remains no-data rather than being replaced by a farther clear pixel.
- There is **no value interpolation, averaging, seam filling, or center-only binning** in the mapped product.
- The primary science QA remains the bloom-aware cloud-recovery mask. Generic `CLDICE`-excluded screening is retained only as a sensitivity diagnostic and is not described as CyAN-equivalent.
- The S3A/S3B daily CIcyano maximum composite is intentionally deferred until this efficient mapping branch is validated.

### v2.6.5 corrections
- Exact requested live-map resolution: 300 m no longer silently becomes ~500 m; large full-scene previews fail explicitly instead of coarsening.
- Added active planning-region (State/Province/bbox/5°) map scope while keeping full-scene archive/export.
- Scene lake dropdowns now come from actual compact scene pixels only.
- Added clickable coordinate-spectrum map with target and matched native pixels.
- CyAN comparison now reports high-CI bias diagnostics and emphasizes that Merged-S3-CYAN is a daily S3A/S3B per-pixel maximum, while the local product is one scene.
- Replaced center-only sinusoidal share gridding with nominal-resolution EPSG:4326 nearest-native remapping for direct SNAP/GIS geolocation and removal of artificial row/stripe gaps.

### v2.6.2 privacy / planning / export changes

- **No Google Drive mount or permission request.** All working and output files live under `/content` in the current Colab runtime. Download the scene share ZIPs and registry CSV before the runtime is deleted.
- Scene discovery can be organized by **country + first-level administrative region (ADM1)** such as state, province, region, department, governorate, etc., using geoBoundaries `gbOpen` boundaries. Bounding-box and 5° grid discovery remain available.
- Administrative regions are **discovery/work-tracking shards only**. A Sentinel-3 product is still deduplicated by scene ID + science configuration, so a frame crossing two states is processed once, not twice.


- **Minimal share ZIP:** exactly three files: one all-science SNAP NetCDF, one all-index per-lake statistics CSV, and one small metadata JSON.
- **SNAP raster seam repair:** only narrow one-cell regular-grid binning cracks are repaired; there is no broad interpolation across missing-data or QA gaps.
- **Mask transparency:** metadata records the actual QA flags. The default `bloom_retaining` profile is not a conventional cloud mask; it retains `CLDICE`/`HILT` unless a stricter profile is used.

### Production design

1. Catalogue OLCI EFR L1 products and intersect their **scene footprints with HydroLAKES before download**.
2. Use the immutable **CDSE product ID + processing-configuration hash** as the registry key, so overlapping date/bbox searches do not process the same scene/config twice.
3. Prefer **CDSE S3 direct staging** of the `.SEN3` tree. OData ZIP remains an automatic fallback.
4. Keep L1B and native L2Gen products only in local Colab scratch.
5. Use **adaptive L2Gen geographic windows** around lake clusters when worthwhile; otherwise process the scene once.
6. Persist only a compressed **derived lake-pixel point NetCDF**, per-scene lake statistics, manifests, and the registry.
7. Delete successful L1/L2 scratch automatically.
8. Explore the resulting archive through a global Gradio queue, lake time series, Esri-basemap maps, corrected Web-Mercator scene overlays, portable hover HTML exports, native orbit/frame archive indexing, and on-demand share bundles.

> **Important terminology:** the persistent compact NetCDF is **not an official OBPG Level-2 swath product**. L2Gen's native Level-2 file is created transiently and retains standard swath dimensions. The persistent file is a derived lake-pixel archive containing the L2Gen outputs and flags needed for the retained HydroLAKES pixels.

### v2.6 consolidation

- Production indices are **CI, CIcyano, NDCI, MPH, and OLCI-adapted FAI/AFAI**. FLH and OLH are intentionally removed from the production branch.
- Live/HTML maps are rendered on a true **EPSG:3857 Web-Mercator grid** so the raster aligns with Esri/OSM basemaps. SNAP/GIS exports remain CF-compliant EPSG:4326 rasters.
- Compact outputs are physically organized by Sentinel-3 **platform / relative orbit / frame / date**, while a 5-degree grid is used only as a search/index crosswalk.
- Download bundles are created on demand only from files that currently exist.



## v2.6.1 changes

- Fixed live Gradio map rendering by serving the generated Folium HTML through Gradio's same-origin file endpoint instead of a large `srcdoc` iframe.
- Restored editable latitude/longitude multi-point spectra while retaining the selected-lake mean/median spectrum.
- Restored an optional NASA OBPG Merged-S3-CYAN same-day comparison using compact native lake pixels.
- Fixed `NETCDF4_CLASSIC` QA-mask compatibility: gridded QA masks are stored as signed byte (`i1`, fill `-1`) rather than unsupported unsigned byte (`u1`).
- Share bundles now always generate one consolidated SNAP/GIS CF NetCDF containing **CI, CIcyano, NDCI, MPH, OLCI-adapted FAI/AFAI, and every required `rhos` band**. This export is independent of which map layer was viewed.
- Share bundles also include one multi-layer portable HTML with true color + all production indices and hover readout of all index values.


## 0. Install Python dependencies


In [1]:
import os
import subprocess
import sys

PYTHON_PACKAGES = (
    "numpy", "pandas", "xarray", "netCDF4", "h5netcdf", "scipy",
    "matplotlib", "requests", "tqdm", "pyresample", "pyproj",
    "shapely", "geopandas", "pyogrio", "rasterio", "cartopy",
    "plotly>=6.0", "pillow", "xyzservices",
    "gradio>=6.0,<7", "fastapi>=0.115", "uvicorn>=0.30",
    "rio-tiler>=7,<9", "mercantile>=1.2",
    "boto3>=1.34", "pyarrow>=15", "folium>=0.17", "pycountry>=24",
)

try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

print(f"Using existing Python environment: {sys.executable}. Colab={IS_COLAB}")


Using existing Python environment: /home/myid/mc16405/miniconda3/envs/s3olci-batch/bin/python. Colab=False


## 1. Install NASA OCSSW/L2Gen in this runtime

The cell queries the installer for the newest operational `V` tag instead of freezing a version. It skips installation when a complete OLCI-capable installation already exists in `/content/ocssw`.

The validation follows the current installer layout:
`share/olci/s3a` and `share/olci/s3b`. If an earlier notebook already downloaded OCSSW but
failed only on its old path check, rerunning this corrected cell reuses that
installation instead of downloading it again.

Allow roughly 15–40 minutes depending on Colab and NASA server throughput. Keep the browser tab open while the installer runs.


In [2]:
# Robust OCSSW / L2Gen installer for Colab
# v2.6 production installer: fresh install + pinned stable tag + fallback + visible logs

import os
import re
import shutil
import subprocess
import urllib.request
from pathlib import Path

OCSSWROOT = Path('/home/myid/mc16405/cyanoTrend/software/ocssw')
OCSSW_INSTALLER_DIR = Path('/home/myid/mc16405/cyanolake-web/ocssw-installer')
RUN_OCSSW_INSTALL = False

# NASA SeaDAS support explicitly recommended a fresh V2026.3 installation
# during 2026 troubleshooting. Pinning also improves reproducibility.
# Change to None if you explicitly want newest-first behavior.
PREFERRED_OCSSW_TAG = 'V2026.3'

# If the preferred tag fails, try this many newest operational V-tags.
FALLBACK_TAG_COUNT = 3


def operational_tag_key(tag: str) -> tuple[int, int]:
    m = re.fullmatch(r'V(\d{4})\.(\d+)', str(tag).strip())
    if not m:
        raise ValueError(tag)
    return int(m.group(1)), int(m.group(2))


def _tail(text: str, n: int = 80) -> str:
    lines = (text or '').splitlines()
    return '\n'.join(lines[-n:])


def _valid_ocssw_install() -> bool:
    required = (
        OCSSWROOT / 'bin' / 'l2gen',
        OCSSWROOT / 'share' / 'olci',
        OCSSWROOT / 'share' / 'olci' / 's3a',
        OCSSWROOT / 'share' / 'olci' / 's3b',
    )
    return all(p.exists() for p in required)


def install_ocssw_for_olci() -> str:
    if _valid_ocssw_install():
        print(f'Reusing complete OCSSW installation at {OCSSWROOT}')
        return 'existing installation'

    if not RUN_OCSSW_INSTALL:
        raise RuntimeError(
            'RUN_OCSSW_INSTALL=False but a complete OCSSW installation is absent.'
        )

    # Make sure a previous failed install cannot contaminate this run.
    if OCSSWROOT.exists():
        print('Removing incomplete/partial OCSSW installation ...')
        shutil.rmtree(OCSSWROOT, ignore_errors=True)

    OCSSW_INSTALLER_DIR.mkdir(parents=True, exist_ok=True)

    downloads = {
        'install_ocssw': 'https://oceandata.sci.gsfc.nasa.gov/manifest/install_ocssw',
        'manifest.py': 'https://oceandata.sci.gsfc.nasa.gov/manifest/manifest.py',
    }
    for filename, url in downloads.items():
        destination = OCSSW_INSTALLER_DIR / filename
        print(f'Downloading {filename} ...')
        urllib.request.urlretrieve(url, destination)

    installer = OCSSW_INSTALLER_DIR / 'install_ocssw'
    installer.chmod(0o755)

    tags_proc = subprocess.run(
        [str(installer), '--list_tags'],
        cwd=OCSSW_INSTALLER_DIR,
        capture_output=True,
        text=True,
        check=False,
    )
    if tags_proc.returncode != 0:
        raise RuntimeError(
            'Could not query NASA OCSSW tags.\n\nSTDOUT:\n'
            + _tail(tags_proc.stdout)
            + '\n\nSTDERR:\n'
            + _tail(tags_proc.stderr)
        )

    operational_tags = sorted(
        {
            line.strip()
            for line in tags_proc.stdout.splitlines()
            if re.fullmatch(r'V\d{4}\.\d+', line.strip())
        },
        key=operational_tag_key,
        reverse=True,
    )
    if not operational_tags:
        raise RuntimeError('NASA installer returned no operational OCSSW V tags.')

    candidates = []
    if PREFERRED_OCSSW_TAG:
        if PREFERRED_OCSSW_TAG in operational_tags:
            candidates.append(PREFERRED_OCSSW_TAG)
        else:
            print(
                f'Preferred tag {PREFERRED_OCSSW_TAG} is not listed; '
                'falling back to current operational tags.'
            )

    for tag in operational_tags[:FALLBACK_TAG_COUNT]:
        if tag not in candidates:
            candidates.append(tag)

    print('OCSSW install candidates:', ', '.join(candidates))

    failures = []
    env = os.environ.copy()
    env['OCSSWROOT'] = str(OCSSWROOT)

    for tag in candidates:
        # Each tag gets a genuinely clean installation directory.
        shutil.rmtree(OCSSWROOT, ignore_errors=True)
        OCSSWROOT.parent.mkdir(parents=True, exist_ok=True)

        cmd = [
            str(installer),
            '--install_dir', str(OCSSWROOT),
            '--tag', tag,
            '--seadas',
            '--olcis3a',
            '--olcis3b',
            '--verbose',
        ]

        print(f'\nInstalling OCSSW {tag} for Sentinel-3A/B OLCI ...')
        print('Command:', ' '.join(cmd))

        proc = subprocess.run(
            cmd,
            cwd=OCSSW_INSTALLER_DIR,
            env=env,
            capture_output=True,
            text=True,
            check=False,
        )

        # Keep a complete log for troubleshooting.
        log_path = OCSSW_INSTALLER_DIR / f'install_{tag}.log'
        log_path.write_text(
            'COMMAND:\n' + ' '.join(cmd)
            + '\n\nSTDOUT:\n' + (proc.stdout or '')
            + '\n\nSTDERR:\n' + (proc.stderr or ''),
            encoding='utf-8',
        )

        if proc.returncode == 0 and _valid_ocssw_install():
            os.environ['OCSSWROOT'] = str(OCSSWROOT)
            version = subprocess.run(
                [str(OCSSWROOT / 'bin' / 'l2gen'), '--version'],
                capture_output=True,
                text=True,
                check=False,
                env=env,
            )
            print(f'OCSSW installation successful: {tag}')
            if version.stdout.strip() or version.stderr.strip():
                print('l2gen version:', (version.stdout or version.stderr).strip())
            print('Installer log:', log_path)
            return tag

        failure_summary = (
            f'{tag}: exit={proc.returncode}\n'
            f'STDOUT tail:\n{_tail(proc.stdout)}\n'
            f'STDERR tail:\n{_tail(proc.stderr)}\n'
            f'Full log: {log_path}'
        )
        failures.append(failure_summary)
        print('\nInstallation attempt failed:')
        print(failure_summary)

    raise RuntimeError(
        'All OCSSW installation attempts failed. The detailed NASA installer '
        'output is shown above and saved under /content/ocssw-installer/.\n\n'
        + '\n\n'.join(failures)
    )


OCSSW_TAG = install_ocssw_for_olci()
os.environ['OCSSWROOT'] = str(OCSSWROOT)

disk = shutil.disk_usage('/home/myid/mc16405/cyanolake-web')
print(f'OCSSW ready: {OCSSW_TAG}')
print(f'Free Colab disk: {disk.free / 1024**3:.1f} GiB')


Reusing complete OCSSW installation at /home/myid/mc16405/cyanoTrend/software/ocssw
OCSSW ready: existing installation
Free Colab disk: 734.0 GiB


## 2. Configure NASA Earthdata Login for `getanc`

This is separate from the Copernicus Data Space login used to download Sentinel-3. The password prompt is hidden, and the temporary `.netrc` disappears when the Colab runtime is deleted.

If you deliberately want a quick climatology-only test, set `CONFIGURE_EARTHDATA_NOW=False`, then set `GET_ANCILLARY=False` in the configuration cell. For scientific processing, keep both enabled.


In [4]:
from getpass import getpass
from pathlib import Path
import os

CONFIGURE_EARTHDATA_NOW = True


def configure_earthdata_login() -> Path:
    username = input("NASA Earthdata username: ").strip()
    password = getpass("NASA Earthdata password (hidden): ")
    if not username or not password:
        raise ValueError("Earthdata username and password are required")
    if any(character.isspace() for character in username + password):
        raise ValueError("The .netrc format does not support whitespace in these credentials")
    netrc_path = Path.home() / ".netrc"
    netrc_path.write_text(
        "machine urs.earthdata.nasa.gov\n"
        f"  login {username}\n"
        f"  password {password}\n"
        "machine oceandata.sci.gsfc.nasa.gov\n"
        f"  login {username}\n"
        f"  password {password}\n",
        encoding="utf-8",
    )
    netrc_path.chmod(0o600)
    print("Temporary Earthdata credentials configured for getanc.")
    return netrc_path


if CONFIGURE_EARTHDATA_NOW:
    configure_earthdata_login()
else:
    print("Earthdata setup skipped. Set GET_ANCILLARY=False below or configure .netrc before L2Gen.")


NASA Earthdata username:  mc16405
NASA Earthdata password (hidden):  ········


Temporary Earthdata credentials configured for getanc.


## 3. Configure the processing workspace

The default profile is deliberately bloom-retaining: only `NAVFAIL` is excluded after L2Gen, the `rhos865` bright screen is off, and no shoreline dilation is applied. The app can change these choices and rebuild the derived NetCDF without rerunning atmospheric correction.


In [5]:
from pathlib import Path
import os

# ---------- Colab processing configuration ----------
PROJECT_ROOT = Path("/home/myid/mc16405/cyanolake-web/s3_olci_l2gen").resolve()
OCSSWROOT = Path("/home/myid/mc16405/cyanoTrend/software/ocssw").resolve()

# Default: full-swath Rayleigh rhos for CI/CIcyano and rhos-based NDCI.
L2_MODE = "full_swath_rhos"  # "full_swath_rhos" or "standard_dual"
NDCI_SOURCE = "rhos"          # use "Rrs" only with standard_dual

# False is safer for the first Colab smoke test. The dashboard bbox then crops
# L2Gen processing. Switch the dashboard checkbox on for a full-scene run.
PROCESS_FULL_SCENE = False
GET_ANCILLARY = True
ALLOW_CLIMATOLOGY_FALLBACK = False
SAVE_FULL_REFLECTANCE_SPECTRA = False

# Hashing multi-gigabyte files costs extra Colab time. Enable for archival runs.
COMPUTE_SHA256 = False

# CyAN algorithm detection limit. The standard detection rule is strictly > 0.0001.
CYAN_CI_DETECTION_LIMIT = 1.0e-4
CI_DETECTION_LIMIT = CYAN_CI_DETECTION_LIMIT
MASK_FLAG_CHOICES = (
    "ATMFAIL", "NAVFAIL", "NAVWARN", "HILT", "CLDICE", "STRAYLIGHT",
    "HIGLINT", "HISATZEN", "HISOLZEN",
)
MASK_PRESETS = {
    "bloom_retaining": {
        "label": "Bloom-retaining inland (recommended)",
        "qa_flags": ("NAVFAIL",),
        "land_adjacency_pixels": 0,
        "note": "Excludes only navigation failure. HILT and CLDICE are retained because bright blooms can trigger them; inspect flags and reason maps before publication.",
    },
    "balanced_inland": {
        "label": "Balanced inland (recommended)",
        "qa_flags": ("ATMFAIL", "NAVFAIL", "HILT", "CLDICE", "STRAYLIGHT"),
        "land_adjacency_pixels": 0,
        "note": "Excludes hard failures, saturation, cloud/ice, and straylight; no automatic shoreline dilation.",
    },
    "relaxed_inland": {
        "label": "Relaxed inland (exploratory)",
        "qa_flags": ("NAVFAIL", "HILT", "CLDICE"),
        "land_adjacency_pixels": 0,
        "note": "Retains ATMFAIL, warning, glint, geometry, and straylight pixels for sensitivity analysis.",
    },
    "diagnostic": {
        "label": "Diagnostic finite-water only",
        "qa_flags": (),
        "land_adjacency_pixels": 0,
        "note": "Troubleshooting only: applies LAND/geolocation and finite-input tests but no additional QA exclusions.",
    },
    "conservative": {
        "label": "Conservative",
        "qa_flags": MASK_FLAG_CHOICES,
        "land_adjacency_pixels": 1,
        "note": "Original strict mask: all listed QA flags plus a one-pixel shoreline buffer.",
    },
}
DEFAULT_MASK_PROFILE = "bloom_retaining"
QA_EXCLUDE_FLAGS = MASK_PRESETS[DEFAULT_MASK_PROFILE]["qa_flags"]
LAND_ADJACENCY_PIXELS = MASK_PRESETS[DEFAULT_MASK_PROFILE]["land_adjacency_pixels"]
NETCDF_DEFLATE = 4
ROW_BLOCK_SIZE = 256
DEFAULT_WATER_DEFINITION = "union"
DEFAULT_SHORE_BUFFER_M = 0.0
DEFAULT_BRIGHT_SCREEN = False
DEFAULT_BRIGHT_RHOS865 = 0.08
DEFAULT_MAP_RESOLUTION_M = 300.0
MAP_MAX_CELLS = 60_000_000
INTERACTIVE_MAX_CELLS = 60_000_000

RAW_L1_DIR = PROJECT_ROOT / "raw_l1"
STAGING_DIR = PROJECT_ROOT / "staging"
L2_DIR = PROJECT_ROOT / "l2"
DERIVED_DIR = PROJECT_ROOT / "derived"
EXPORT_DIR = PROJECT_ROOT / "exports"
PAR_DIR = PROJECT_ROOT / "par"
LOG_DIR = PROJECT_ROOT / "logs"
HYDRO_ROOT = PROJECT_ROOT / "hydrolakes"

for directory in (
    RAW_L1_DIR, STAGING_DIR, L2_DIR, DERIVED_DIR, EXPORT_DIR, PAR_DIR, LOG_DIR, HYDRO_ROOT
):
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"OCSSW root   : {OCSSWROOT}")
print(f"L2 mode      : {L2_MODE}")
print(f"NDCI source  : {NDCI_SOURCE}")
print(f"Full scene   : {PROCESS_FULL_SCENE} (change later in dashboard)")


Project root : /home/myid/mc16405/cyanolake-web/s3_olci_l2gen
OCSSW root   : /home/myid/mc16405/cyanoTrend/software/ocssw
L2 mode      : full_swath_rhos
NDCI source  : rhos
Full scene   : False (change later in dashboard)


## 3. Activate OCSSW and verify the runtime

This imports variables from `OCSSW_bash.env` into the Python kernel and locates `l2gen` and `getanc`. Do not continue unless both paths are printed.


In [6]:
from __future__ import annotations

import datetime as dt
import getpass
import hashlib
import json
import math
import platform
import re
import shlex
import shutil
import subprocess
import sys
import time
import warnings
import zipfile
from typing import Any, Iterable

import matplotlib.pyplot as plt
import netCDF4
import numpy as np
import pandas as pd
import requests
import xarray as xr
from scipy.ndimage import binary_dilation
from tqdm.auto import tqdm

try:
    import ipywidgets as widgets
    from IPython.display import HTML, clear_output, display
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False

warnings.filterwarnings("ignore", category=RuntimeWarning)


def read_bash_environment(script: Path) -> dict[str, str]:
    """Return a sourced Bash environment without mutating this Python process."""
    environment = os.environ.copy()
    script = Path(script)
    if not script.is_file():
        return environment
    command = f"source {shlex.quote(str(script))} >/dev/null 2>&1; env -0"
    result = subprocess.run(
        ["bash", "-lc", command], capture_output=True, check=True,
        env=environment,
    )
    for item in result.stdout.split(b"\0"):
        if b"=" not in item:
            continue
        key, value = item.split(b"=", 1)
        environment[key.decode(errors="ignore")] = value.decode(errors="ignore")
    return environment


# OCSSW ships its own older PROJ/GDAL libraries and database.  They are correct
# for l2gen/getanc, but exposing them through os.environ makes Colab's newer
# Rasterio/PyProj read an incompatible proj.db.  Keep the complete OCSSW
# environment only for OCSSW subprocesses.
OCSSW_SUBPROCESS_ENV = read_bash_environment(OCSSWROOT / "OCSSW_bash.env")
OCSSW_SUBPROCESS_ENV["OCSSWROOT"] = str(OCSSWROOT)
os.environ["OCSSWROOT"] = str(OCSSWROOT)
for variable in ("PROJ_LIB", "PROJ_DATA", "GDAL_DATA"):
    value = os.environ.get(variable, "")
    if value and str(OCSSWROOT) in value:
        os.environ.pop(variable, None)
for variable in ("LD_LIBRARY_PATH", "PYTHONPATH", "GDAL_DRIVER_PATH"):
    value = os.environ.get(variable, "")
    if value and str(OCSSWROOT) in value:
        retained = [part for part in value.split(os.pathsep) if str(OCSSWROOT) not in part]
        if retained:
            os.environ[variable] = os.pathsep.join(retained)
        else:
            os.environ.pop(variable, None)

print("OCSSW environment isolated from Python geospatial libraries.")

# Force the Python stack to Rasterio's own modern PROJ database.  Merely
# deleting PROJ_LIB is insufficient in some Colab sessions because PROJ may
# already have cached the OCSSW search path.
import importlib.util as _importlib_util
import sqlite3 as _sqlite3


def _proj_database_layout(database: Path) -> tuple[int, int]:
    with _sqlite3.connect(database) as connection:
        rows = dict(connection.execute(
            "SELECT key, value FROM metadata "
            "WHERE key IN ('DATABASE.LAYOUT.VERSION.MAJOR', "
            "'DATABASE.LAYOUT.VERSION.MINOR')"
        ).fetchall())
    return int(rows['DATABASE.LAYOUT.VERSION.MAJOR']), int(rows['DATABASE.LAYOUT.VERSION.MINOR'])


_rasterio_spec = _importlib_util.find_spec("rasterio")
if _rasterio_spec is None or _rasterio_spec.origin is None:
    raise RuntimeError("Rasterio is not installed; rerun the dependency cell")
PYTHON_PROJ_DATA_DIR = Path(_rasterio_spec.origin).resolve().parent / "proj_data"
PYTHON_PROJ_DATABASE = PYTHON_PROJ_DATA_DIR / "proj.db"
if not PYTHON_PROJ_DATABASE.is_file():
    raise RuntimeError(f"Rasterio's packaged proj.db was not found: {PYTHON_PROJ_DATABASE}")
_python_proj_layout = _proj_database_layout(PYTHON_PROJ_DATABASE)
if _python_proj_layout < (1, 6):
    raise RuntimeError(
        f"Rasterio's packaged proj.db is unexpectedly old: layout {_python_proj_layout}"
    )
os.environ["PROJ_DATA"] = str(PYTHON_PROJ_DATA_DIR)
os.environ["PROJ_LIB"] = str(PYTHON_PROJ_DATA_DIR)  # compatibility with older PROJ clients
print(
    f"Python PROJ database forced to: {PYTHON_PROJ_DATABASE} "
    f"(layout {_python_proj_layout[0]}.{_python_proj_layout[1]})"
)


def find_ocssw_tool(name: str) -> Path | None:
    candidates = [
        OCSSWROOT / "bin" / name,
        OCSSWROOT / "scripts" / name,
        OCSSWROOT / "scripts" / f"{name}.py",
    ]
    found = shutil.which(name) or shutil.which(f"{name}.py")
    if found:
        candidates.insert(0, Path(found))
    return next((path for path in candidates if path.is_file()), None)


L2GEN_BIN = find_ocssw_tool("l2gen")
GETANC_BIN = find_ocssw_tool("getanc")


def command_version(path: Path | None) -> str:
    if path is None:
        return "not found"
    for option in ("--version", "version"):
        result = subprocess.run(
            [str(path), option], capture_output=True, text=True,
            env=OCSSW_SUBPROCESS_ENV,
        )
        text = (result.stdout + "\n" + result.stderr).strip()
        if text:
            return text.splitlines()[0]
    return str(path)


print(f"OS            : {platform.system()} {platform.machine()}")
print(f"Python        : {sys.version.split()[0]}")
print(f"netCDF4       : {netCDF4.__version__}")
print(f"l2gen         : {command_version(L2GEN_BIN)}")
print(f"getanc        : {GETANC_BIN or 'not found'}")
if L2GEN_BIN is None:
    print("\nOCSSW is not active. Complete the setup guide, restart Jupyter, and rerun this cell.")


OCSSW environment isolated from Python geospatial libraries.
Python PROJ database forced to: /home/myid/mc16405/miniconda3/envs/s3olci-batch/lib/python3.11/site-packages/rasterio/proj_data/proj.db (layout 1.6)
OS            : Linux x86_64
Python        : 3.11.16
netCDF4       : 1.7.4
l2gen         : msl12 10.3.0-V2026.4 (Jul 28 2026 00:32:36)
getanc        : /home/myid/mc16405/cyanoTrend/software/ocssw/bin/getanc


In [7]:
def utc_now() -> str:
    return dt.datetime.now(dt.timezone.utc).isoformat()


def human_size(size: int | float) -> str:
    value = float(size)
    for unit in ("B", "KiB", "MiB", "GiB", "TiB"):
        if value < 1024 or unit == "TiB":
            return f"{value:.2f} {unit}"
        value /= 1024


def sha256_file(path: Path, chunk_bytes: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_bytes), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_json(path: Path, payload: dict[str, Any]) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
    temporary.replace(path)
    return path


def run_logged(
    command: list[str], log_path: Path, timeout: int | None = None,
    cwd: Path | None = None,
) -> subprocess.CompletedProcess:
    """Run a command, write a complete log, and raise with a useful tail."""
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    started = time.time()
    result = subprocess.run(
        [str(item) for item in command],
        cwd=str(cwd) if cwd else None,
        capture_output=True,
        text=True,
        timeout=timeout,
        env=OCSSW_SUBPROCESS_ENV.copy(),
    )
    elapsed = time.time() - started
    header = (
        f"started_utc={utc_now()}\n"
        f"elapsed_seconds={elapsed:.3f}\n"
        f"return_code={result.returncode}\n"
        f"command={shlex.join([str(item) for item in command])}\n\n"
    )
    log_path.write_text(
        header + "===== STDOUT =====\n" + (result.stdout or "")
        + "\n===== STDERR =====\n" + (result.stderr or ""),
        encoding="utf-8",
    )
    if result.returncode != 0:
        combined = ((result.stdout or "") + "\n" + (result.stderr or "")).strip()
        raise RuntimeError(
            f"Command failed with return code {result.returncode}. "
            f"Log: {log_path}\n\n{combined[-4000:]}"
        )
    return result


def require_ocssw(require_getanc: bool = False) -> None:
    if L2GEN_BIN is None:
        raise RuntimeError("l2gen was not found. Complete the local OCSSW setup first.")
    if require_getanc and GETANC_BIN is None:
        raise RuntimeError("getanc was not found in the active OCSSW installation.")


## 4. Search and preserve Sentinel-3 OLCI Level-1

CDSE catalogue search is public. Downloading requires a free Copernicus Data Space account. Credentials entered in the dashboard remain in kernel memory and are not written to disk. `getanc` uses a separate NASA Earthdata Login configured in your local `.netrc`; the setup guide shows the secure manual step.


In [8]:
CDSE_CATALOGUE_URL = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"
CDSE_TOKEN_URL = (
    "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/"
    "protocol/openid-connect/token"
)
CDSE_DOWNLOAD_ROOT = "https://download.dataspace.copernicus.eu/odata/v1"


class CDSEAuth:
    def __init__(self) -> None:
        self.access_token: str | None = None
        self.refresh_token: str | None = None
        self.expires_at = 0.0

    def login(self, username: str, password: str) -> str:
        response = requests.post(
            CDSE_TOKEN_URL,
            data={
                "client_id": "cdse-public",
                "grant_type": "password",
                "username": username,
                "password": password,
            },
            timeout=60,
        )
        response.raise_for_status()
        payload = response.json()
        self.access_token = payload["access_token"]
        self.refresh_token = payload.get("refresh_token")
        self.expires_at = time.time() + int(payload.get("expires_in", 600)) - 60
        return self.access_token

    def token(self, username: str, password: str) -> str:
        if self.access_token and time.time() < self.expires_at:
            return self.access_token
        return self.login(username, password)


CDSE_AUTH = CDSEAuth()


def normalize_bbox(bbox: Iterable[float]) -> tuple[float, float, float, float]:
    west, south, east, north = map(float, bbox)
    if not (-180 <= west < east <= 180 and -90 <= south < north <= 90):
        raise ValueError("bbox must be (west, south, east, north) in valid longitude/latitude ranges")
    return west, south, east, north


def search_olci_l1(
    start_date: str, end_date: str,
    bbox: tuple[float, float, float, float],
    max_results: int = 30,
) -> list[dict[str, Any]]:
    """Search CDSE for OLCI EFR Level-1 products intersecting bbox."""
    west, south, east, north = normalize_bbox(bbox)
    polygon = (
        f"POLYGON(({west} {south},{east} {south},{east} {north},"
        f"{west} {north},{west} {south}))"
    )
    start = pd.Timestamp(start_date).strftime("%Y-%m-%d")
    end_exclusive = (pd.Timestamp(end_date) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    filters = [
        "Collection/Name eq 'SENTINEL-3'",
        (
            "Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'productType' "
            "and att/OData.CSC.StringAttribute/Value eq 'OL_1_EFR___')"
        ),
        f"ContentDate/Start ge {start}T00:00:00.000Z",
        f"ContentDate/Start lt {end_exclusive}T00:00:00.000Z",
        f"OData.CSC.Intersects(area=geography'SRID=4326;{polygon}')",
    ]
    response = requests.get(
        CDSE_CATALOGUE_URL,
        params={
            "$filter": " and ".join(filters),
            "$orderby": "ContentDate/Start desc",
            "$top": int(max_results),
            "$expand": "Attributes",
        },
        timeout=90,
    )
    response.raise_for_status()
    rows = []
    for item in response.json().get("value", []):
        name = str(item.get("Name", ""))
        rows.append({
            "id": item["Id"],
            "name": name.removesuffix(".SEN3"),
            "satellite": name.split("_", 1)[0] if name else "S3",
            "start": item.get("ContentDate", {}).get("Start", ""),
            "end": item.get("ContentDate", {}).get("End", ""),
            "size_bytes": int(item.get("ContentLength") or 0),
            "online": bool(item.get("Online", True)),
        })
    return rows


In [9]:
def safe_extract_zip(zip_path: Path, destination: Path) -> Path:
    """Extract a ZIP after rejecting path traversal and validate one .SEN3 product."""
    zip_path = Path(zip_path)
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        bad_member = archive.testzip()
        if bad_member:
            raise RuntimeError(f"ZIP CRC validation failed at {bad_member}")
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if target != destination and destination not in target.parents:
                raise RuntimeError(f"Unsafe ZIP member: {member.filename}")
        archive.extractall(destination)

    products = sorted({
        path.parent for path in destination.rglob("xfdumanifest.xml")
        if path.parent.name.endswith(".SEN3")
    })
    if len(products) != 1:
        raise RuntimeError(f"Expected one extracted .SEN3 product, found {len(products)}")
    return products[0]


def validate_sen3(path: Path) -> Path:
    path = Path(path).expanduser().resolve()
    manifest = path / "xfdumanifest.xml"
    if not path.is_dir() or not manifest.is_file():
        raise FileNotFoundError(f"Not a valid OLCI .SEN3 directory: {path}")
    radiance_files = list(path.glob("Oa*_radiance.nc"))
    if len(radiance_files) < 20:
        raise RuntimeError(f"Incomplete OLCI product: found only {len(radiance_files)} radiance files")
    return path


def download_cdse_scene(
    scene: dict[str, Any], username: str, password: str,
) -> tuple[Path, Path]:
    """Resume a CDSE ZIP download, preserve it, and stage its .SEN3 product."""
    product_name = str(scene["name"]).removesuffix(".SEN3")
    zip_path = RAW_L1_DIR / f"{product_name}.zip"
    partial = zip_path.with_suffix(".zip.part")
    stage_root = STAGING_DIR / product_name

    existing_products = list(stage_root.rglob("xfdumanifest.xml")) if stage_root.exists() else []
    if zip_path.is_file():
        with zipfile.ZipFile(zip_path) as archive:
            bad_member = archive.testzip()
            if bad_member:
                raise RuntimeError(f"Existing ZIP failed CRC at {bad_member}: {zip_path}")
        if existing_products:
            return zip_path, validate_sen3(existing_products[0].parent)
        if stage_root.exists():
            shutil.rmtree(stage_root)
        return zip_path, validate_sen3(safe_extract_zip(zip_path, stage_root))

    token = CDSE_AUTH.token(username, password)
    headers = {"Authorization": f"Bearer {token}"}
    mode = "wb"
    if partial.exists() and partial.stat().st_size:
        headers["Range"] = f"bytes={partial.stat().st_size}-"
        mode = "ab"

    url = f"{CDSE_DOWNLOAD_ROOT}/Products({scene['id']})/$value"
    with requests.get(url, headers=headers, stream=True, timeout=(60, 900)) as response:
        if response.status_code == 200 and mode == "ab":
            mode = "wb"  # server ignored Range; restart safely
        response.raise_for_status()
        initial = partial.stat().st_size if mode == "ab" and partial.exists() else 0
        remaining = int(response.headers.get("content-length") or 0)
        with partial.open(mode) as stream, tqdm(
            total=initial + remaining if remaining else None,
            initial=initial,
            unit="B", unit_scale=True, desc=product_name[:32],
        ) as progress:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    stream.write(chunk)
                    progress.update(len(chunk))

    partial.replace(zip_path)
    with zipfile.ZipFile(zip_path) as archive:
        bad_member = archive.testzip()
        if bad_member:
            raise RuntimeError(f"Downloaded ZIP failed CRC at {bad_member}")

    if stage_root.exists():
        shutil.rmtree(stage_root)
    sen3_path = safe_extract_zip(zip_path, stage_root)
    validate_sen3(sen3_path)

    archive_manifest = {
        "created_utc": utc_now(),
        "scene": scene,
        "raw_zip": str(zip_path),
        "raw_zip_size_bytes": zip_path.stat().st_size,
        "raw_zip_sha256": sha256_file(zip_path) if COMPUTE_SHA256 else None,
        "staged_sen3": str(sen3_path),
    }
    write_json(EXPORT_DIR / product_name / "l1_archive_manifest.json", archive_manifest)
    return zip_path, sen3_path


def stage_existing_input(path: str | Path) -> tuple[Path | None, Path]:
    """Use an existing .SEN3 directory or stage an existing CDSE ZIP."""
    path = Path(path).expanduser().resolve()
    if path.is_dir():
        return None, validate_sen3(path)
    if path.suffix.lower() == ".zip" and path.is_file():
        product_name = path.stem
        stage_root = STAGING_DIR / product_name
        manifests = list(stage_root.rglob("xfdumanifest.xml")) if stage_root.exists() else []
        if manifests:
            sen3 = validate_sen3(manifests[0].parent)
        else:
            if stage_root.exists():
                shutil.rmtree(stage_root)
            sen3 = safe_extract_zip(path, stage_root)
        return path, sen3
    raise FileNotFoundError("Existing input must be a CDSE ZIP or an extracted .SEN3 directory")


## 5. L2Gen scientific foundation

These functions preserve the original validated L2Gen/Rayleigh-correction implementation. In the global-efficient section later in the notebook, `l2_products()` is intentionally overridden so production runs request only the core `rhos` bands required for CI/NDCI by default. Full reflectance spectra remain an opt-in diagnostic mode because they increase temporary L2 size.


In [10]:
CORE_RHOS_WAVELENGTHS = (620, 665, 681, 709)
SPECTRAL_RHOS_WAVELENGTHS = (
    400, 412, 443, 490, 510, 560, 620, 665, 674, 681,
    709, 754, 779, 865, 885, 900,
)
AQUATIC_RRS_WAVELENGTHS = (400, 412, 443, 490, 510, 560, 620, 665, 674, 681, 709)
TRUE_COLOR_PRODUCTS = ("rhot_443", "rhot_560", "rhot_665")


def l2_products(mode: str, full_spectrum: bool = True) -> tuple[list[str], set[str]]:
    mode = mode.lower()
    if mode in {"full_swath_rhos", "rayleigh_ci"}:
        mode = "full_swath_rhos"
    if mode not in {"full_swath_rhos", "standard_dual"}:
        raise ValueError("mode must be 'full_swath_rhos' or 'standard_dual'")
    waves = SPECTRAL_RHOS_WAVELENGTHS if full_spectrum else CORE_RHOS_WAVELENGTHS
    products = [f"rhos_{wave}" for wave in waves]
    required = {f"rhos_{wave}" for wave in CORE_RHOS_WAVELENGTHS}
    products.extend(TRUE_COLOR_PRODUCTS)
    if mode == "standard_dual":
        rrs_waves = AQUATIC_RRS_WAVELENGTHS if full_spectrum else (665, 709)
        products.extend(f"Rrs_{wave}" for wave in rrs_waves)
        required.update({"Rrs_665", "Rrs_709"})
    return list(dict.fromkeys(products)), required


def write_par_file(path: Path, parameters: dict[str, Any]) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    lines = []
    for key, value in parameters.items():
        if value is None:
            continue
        if isinstance(value, bool):
            value = int(value)
        lines.append(f"{key}={value}")
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return path


def product_basename(sen3_path: Path) -> str:
    return Path(sen3_path).name.removesuffix(".SEN3")


def olci_acquisition_interval(sen3_path: Path) -> tuple[str, str]:
    """Return ISO start/stop times from a standard Sentinel-3 product name."""
    name = product_basename(Path(sen3_path))
    stamps = re.findall(r"\d{8}T\d{6}", name)
    if len(stamps) < 2:
        raise ValueError(
            "Could not parse the acquisition start/stop times from the OLCI "
            f"product name: {name}"
        )
    start_dt = dt.datetime.strptime(stamps[0], "%Y%m%dT%H%M%S")
    stop_dt = dt.datetime.strptime(stamps[1], "%Y%m%dT%H%M%S")
    duration = (stop_dt - start_dt).total_seconds()
    if duration <= 0 or duration > 2 * 3600:
        raise ValueError(
            f"Implausible OLCI acquisition interval ({duration:g} seconds) in {name}"
        )
    return (
        start_dt.strftime("%Y-%m-%dT%H:%M:%S"),
        stop_dt.strftime("%Y-%m-%dT%H:%M:%S"),
    )


def validate_ancillary_par(path: Path) -> dict[str, Any]:
    """Require a non-empty getanc par file with usable MET/OZONE inputs."""
    path = Path(path)
    if not path.is_file() or path.stat().st_size == 0:
        raise RuntimeError("getanc returned no usable ancillary parameter file")
    parameters: dict[str, str] = {}
    for raw_line in path.read_text(encoding="utf-8", errors="replace").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        parameters[key.strip()] = value.strip()
    required = ("met1", "met2", "met3", "ozone1", "ozone2", "ozone3")
    missing_keys = [key for key in required if not parameters.get(key)]
    if missing_keys:
        raise RuntimeError(
            "ancillary parameter file lacks required entries: " + ", ".join(missing_keys)
        )
    missing_files = []
    for key, value in parameters.items():
        expanded = Path(os.path.expandvars(value)).expanduser()
        if not expanded.is_file():
            missing_files.append(f"{key}={expanded}")
    if missing_files:
        raise RuntimeError(
            "ancillary parameter file references missing downloads: "
            + "; ".join(missing_files[:12])
        )
    return {
        "path": str(path),
        "size_bytes": path.stat().st_size,
        "parameter_keys": sorted(parameters),
        "referenced_file_count": len(parameters),
    }


def run_getanc(sen3_path: Path, allow_fallback: bool = False) -> Path | None:
    """Download time-specific ancillary data and return its .anc parameter file."""
    if not GET_ANCILLARY:
        return None
    require_ocssw(require_getanc=True)
    sen3_path = validate_sen3(sen3_path)
    name = product_basename(sen3_path)
    anc_path = PAR_DIR / f"{name}.anc"
    if anc_path.is_file() and anc_path.stat().st_size:
        try:
            validate_ancillary_par(anc_path)
            return anc_path
        except RuntimeError:
            anc_path.unlink(missing_ok=True)

    start_time, stop_time = olci_acquisition_interval(sen3_path)
    ancdb_path = PAR_DIR / "ancillary_data.db"
    attempts = [
        (
            "mission-neutral explicit-time query",
            [
                str(GETANC_BIN), "--start", start_time, "--stop", stop_time,
                "--ofile", str(anc_path), "--ancdb", str(ancdb_path),
                "--noprint", "--verbose",
            ],
        ),
        (
            "standard OLCI manifest query",
            [
                str(GETANC_BIN), str(sen3_path / "xfdumanifest.xml"),
                "--ofile", str(anc_path), "--ancdb", str(ancdb_path),
                "--noprint", "--verbose", "--refreshDB",
            ],
        ),
    ]
    errors: list[str] = []
    for number, (label, command) in enumerate(attempts, start=1):
        log_path = LOG_DIR / f"{name}_getanc_attempt{number}.log"
        try:
            result = run_logged(command, log_path, timeout=1800, cwd=PAR_DIR)
            try:
                validation = validate_ancillary_par(anc_path)
            except RuntimeError as validation_error:
                output_tail = (
                    ((result.stdout or "") + "\n" + (result.stderr or "")).strip()
                    or "getanc produced no console output"
                )[-3000:]
                raise RuntimeError(
                    f"{label}: getanc exited 0 but its result was unusable: "
                    f"{validation_error}. Log: {log_path}\n{output_tail}"
                ) from validation_error
            write_json(
                EXPORT_DIR / name / "ancillary_summary.json",
                {
                    "created_utc": utc_now(),
                    "query_method": label,
                    "acquisition_start": start_time,
                    "acquisition_stop": stop_time,
                    **validation,
                },
            )
            print(f"Ancillary data ready via {label}: {anc_path}")
            return anc_path
        except Exception as error:
            errors.append(f"Attempt {number} ({label})\n{error}")
            anc_path.unlink(missing_ok=True)
    if allow_fallback:
        print(
            "WARNING: getanc failed after both OLCI strategies; L2Gen will use "
            "its climatological ancillary defaults because fallback was explicitly enabled."
        )
        for error in errors:
            print("\n" + error[-2500:])
        return None
    raise RuntimeError(
        "No usable time-specific ancillary file was produced, and climatology "
        "fallback is disabled. The detailed attempt logs are under "
        f"{LOG_DIR}. If you deliberately accept climatological defaults for a "
        "test run, set ALLOW_CLIMATOLOGY_FALLBACK=True and rerun the configuration "
        "and dashboard cells.\n\n" + "\n\n".join(errors)
    )


def validate_l2_file(path: Path, required_products: Iterable[str]) -> dict[str, Any]:
    path = Path(path)
    if not path.is_file() or path.stat().st_size == 0:
        raise RuntimeError(f"Missing/empty L2 output: {path}")
    with netCDF4.Dataset(path) as root:
        if "geophysical_data" not in root.groups or "navigation_data" not in root.groups:
            raise RuntimeError("L2 output lacks geophysical_data/navigation_data groups")
        geo = root.groups["geophysical_data"]
        nav = root.groups["navigation_data"]
        flags_present = (
            "l2_flags" in geo.variables
            or "l2_flags" in nav.variables
            or "l2_flags" in root.variables
        )
        if not flags_present:
            raise RuntimeError("L2 output lacks l2_flags required for water/QA masking")
        missing = [name for name in required_products if name not in geo.variables]
        if missing:
            raise RuntimeError(f"L2 output lacks required products: {missing}")
        for coordinate in ("latitude", "longitude"):
            if coordinate not in nav.variables:
                raise RuntimeError(f"L2 output lacks navigation variable {coordinate}")
        shape = nav.variables["latitude"].shape
        if nav.variables["longitude"].shape != shape:
            raise RuntimeError("L2 latitude/longitude shapes differ")
        provenance_attributes = {}
        for attribute in (
            "product_name", "instrument", "platform", "processing_version",
            "software_name", "software_version", "processing_time",
            "processing_control", "history",
        ):
            if attribute in root.ncattrs():
                provenance_attributes[attribute] = str(root.getncattr(attribute))
        return {
            "path": str(path),
            "shape": tuple(map(int, shape)),
            "products": sorted(geo.variables),
            "size_bytes": path.stat().st_size,
            "source_global_attributes": provenance_attributes,
        }


def run_l2gen(
    sen3_path: Path,
    mode: str = L2_MODE,
    process_full_scene: bool = PROCESS_FULL_SCENE,
    bbox: tuple[float, float, float, float] | None = None,
    full_spectrum: bool = SAVE_FULL_REFLECTANCE_SPECTRA,
) -> tuple[Path, dict[str, Any]]:
    """Run one OLCI scene through L2Gen, with optional-product retry."""
    require_ocssw(require_getanc=GET_ANCILLARY)
    sen3_path = validate_sen3(sen3_path)
    mode = mode.lower()
    name = product_basename(sen3_path)
    suffix = "FULLSWATH_RHOS" if mode in {"full_swath_rhos", "rayleigh_ci"} else "FULLSWATH_RHOS_RRS"
    processing_bbox = None
    if process_full_scene:
        scope_tag = "FULL"
    else:
        if bbox is None:
            raise ValueError("bbox is required when process_full_scene=False")
        processing_bbox = normalize_bbox(bbox)
        bbox_text = ",".join(f"{value:.6f}" for value in processing_bbox)
        scope_tag = "BBOX_" + hashlib.sha1(bbox_text.encode("utf-8")).hexdigest()[:10]
    spectrum_tag = "SPEC" if full_spectrum else "CORE"
    output = L2_DIR / f"{name}_L2GEN_{suffix}_{scope_tag}_{spectrum_tag}.nc"
    products, required = l2_products(mode, full_spectrum)

    if output.exists():
        try:
            info = validate_l2_file(output, required)
            info.update({
                "mode": mode,
                "processing_scope": "full_scene" if process_full_scene else "bbox",
                "processing_bbox": processing_bbox,
                "full_reflectance_spectra_requested": bool(full_spectrum),
                "current_l2gen_version": command_version(L2GEN_BIN),
                "reused_existing": True,
            })
            print(f"Using validated existing L2 file: {output}")
            return output, info
        except Exception:
            stamp = dt.datetime.now().strftime("%Y%m%dT%H%M%S")
            incomplete = output.with_name(
                f"{output.stem}.incompatible_{stamp}{output.suffix}"
            )
            output.replace(incomplete)
            print(f"Moved incompatible prior output to {incomplete}")

    anc_path = run_getanc(sen3_path, allow_fallback=ALLOW_CLIMATOLOGY_FALLBACK)
    run_tag = f"{mode}_{scope_tag}_{spectrum_tag}".lower()
    par_path = PAR_DIR / f"{name}_l2gen_{run_tag}.par"
    log_path = LOG_DIR / f"{name}_l2gen_{run_tag}.log"

    while True:
        parameters: dict[str, Any] = {
            "ifile": sen3_path / "xfdumanifest.xml",
            "ofile": output,
            "l2prod": " ".join(products),
            # Preserve calculated output rather than allowing L2Gen's output masks
            # to erase it. LAND and every other condition remain encoded in l2_flags.
            "proc_ocean": 2 if mode in {"full_swath_rhos", "rayleigh_ci"} else 1,
            "proc_land": 1,
            "maskland": 0,
            "maskcloud": 0,
            "maskglint": 0,
            "masksunzen": 0,
            "masksatzen": 0,
            "maskhilt": 0,
            "maskstlight": 0,
            "proc_uncertainty": 0,
            "gas_opt": 15,       # O3 + CO2 + NO2 + H2O; includes water vapor
            "deflate": int(NETCDF_DEFLATE),
        }
        if mode in {"full_swath_rhos", "rayleigh_ci"}:
            parameters.update({"aer_opt": -99, "brdf_opt": 0})
        if not process_full_scene:
            west, south, east, north = processing_bbox
            parameters.update({"west": west, "south": south, "east": east, "north": north})
        write_par_file(par_path, parameters)

        command = [str(L2GEN_BIN)]
        if anc_path is not None:
            command.append(f"par={anc_path}")
        command.append(f"par={par_path}")
        try:
            run_logged(command, log_path, timeout=4 * 3600)
            break
        except RuntimeError as error:
            text = str(error)
            match = re.search(r"product\s+([A-Za-z0-9_]+)\s+not found", text, re.IGNORECASE)
            if not match:
                output.unlink(missing_ok=True)
                raise
            unavailable = match.group(1)
            if unavailable in required or unavailable not in products:
                output.unlink(missing_ok=True)
                raise RuntimeError(f"Required L2Gen product unavailable: {unavailable}") from error
            products.remove(unavailable)
            output.unlink(missing_ok=True)
            print(f"Optional product {unavailable} is unavailable; retrying without it.")

    info = validate_l2_file(output, required)
    info.update({
        "mode": mode,
        "l2gen_version": command_version(L2GEN_BIN),
        "processing_scope": "full_scene" if process_full_scene else "bbox",
        "processing_bbox": processing_bbox,
        "full_reflectance_spectra_requested": bool(full_spectrum),
        "reused_existing": False,
        "internal_output_masks_disabled": True,
        "rhos_scope": "complete geolocated swath where L2Gen computes finite values; LAND retained in l2_flags",
        "products_requested_final": products,
        "parameter_file": str(par_path),
        "ancillary_file": str(anc_path) if anc_path else None,
        "log": str(log_path),
    })
    print(f"L2 complete: {output} ({human_size(output.stat().st_size)})")
    return output, info


## 6. Post-L2 water definition, QA, and index semantics

All analysis masks are applied to the saved L2 scene. Water can be defined by inverse `LAND`, HydroLAKES polygons, their union or intersection, or HydroLAKES with automatic inverse-`LAND` fallback. QA flags, LAND dilation, metric shoreline erosion, and the optional `rhos_865` bright-pixel screen are independent controls.

The default bloom-retaining preset excludes only `NAVFAIL`. It deliberately retains `HILT`, `CLDICE`, `ATMFAIL`, `STRAYLIGHT`, glint and geometry warnings because intense or bright blooms can trigger some of those conditions. This is a diagnostic starting point; the reason bands and mask report show exactly what a stricter publication mask would remove.

Finite negative core-band `rhos` values are counted rather than rejected by an arbitrary positivity threshold. `CI` candidates are retained after the SS681 gate, while standard `CI_cyano` detections require `CI>0.0001`; valid non-detections are fill/`NaN`. Boolean candidate and detection masks preserve the gate outcomes.

The standard gates are SS681<0, SS665>0, and CIcyano>0.0001. They are explicit controls, separate from QA masking. Relaxing a gate is a sensitivity experiment rather than the standard CyAN-compatible detection product; map `CI`, `SS_665`, and `CI_exclusion_reason` before changing it.


In [11]:
PRODUCT_PATTERN = re.compile(r"^(?P<prefix>[A-Za-z]+)_(?P<wavelength>\d+(?:\.\d+)?)$")
DERIVED_FILL = np.float32(-32767.0)


def to_float(values: Any) -> np.ndarray:
    return np.ma.asarray(values).filled(np.nan).astype(np.float32, copy=False)


def product_wavelength(name: str) -> float:
    match = PRODUCT_PATTERN.match(name)
    if not match:
        raise ValueError(f"Cannot parse wavelength from product name: {name}")
    return float(match.group("wavelength"))


def nearest_product(
    variables: Iterable[str], prefix: str, target: float, tolerance_nm: float = 12.0
) -> str:
    candidates = []
    for name in variables:
        if not name.startswith(prefix + "_"):
            continue
        try:
            wavelength = product_wavelength(name)
        except ValueError:
            continue
        candidates.append((abs(wavelength - target), name))
    if not candidates or min(candidates)[0] > tolerance_nm:
        raise KeyError(f"No {prefix} product within {tolerance_nm} nm of {target}")
    return min(candidates)[1]


def decode_flag_metadata(variable: netCDF4.Variable) -> dict[str, np.uint32]:
    meanings = str(getattr(variable, "flag_meanings", "")).replace(",", " ").split()
    masks = np.asarray(getattr(variable, "flag_masks", []), dtype=np.uint32).ravel()
    if len(meanings) != len(masks):
        raise RuntimeError("l2_flags flag_meanings and flag_masks metadata do not align")
    return {meaning.upper(): np.uint32(mask) for meaning, mask in zip(meanings, masks)}


def flag_is_set(flags: np.ndarray, flag_map: dict[str, np.uint32], name: str) -> np.ndarray:
    mask = flag_map.get(name.upper())
    if mask is None:
        return np.zeros(flags.shape, dtype=bool)
    return (flags.astype(np.uint32, copy=False) & mask) != 0


def locate_l2_flags(root: netCDF4.Dataset) -> netCDF4.Variable:
    for group_name in ("geophysical_data", "navigation_data"):
        group = root.groups.get(group_name)
        if group is not None and "l2_flags" in group.variables:
            return group.variables["l2_flags"]
    if "l2_flags" in root.variables:
        return root.variables["l2_flags"]
    raise KeyError("l2_flags is absent from the L2 file")


In [12]:
def ci_family(
    r620: np.ndarray,
    r665: np.ndarray,
    r681: np.ndarray,
    r709: np.ndarray,
    wavelengths: tuple[float, float, float, float] = (620.0, 665.0, 681.0, 709.0),
    detection_limit: float = CI_DETECTION_LIMIT,
) -> dict[str, np.ndarray]:
    """Return spectral shapes plus detection-only CI and CIcyano arrays."""
    r620, r665, r681, r709 = [
        np.asarray(array, dtype=np.float32) for array in (r620, r665, r681, r709)
    ]
    w620, w665, w681, w709 = map(float, wavelengths)
    finite = np.isfinite(r620) & np.isfinite(r665) & np.isfinite(r681) & np.isfinite(r709)

    ss681 = r681 - r665 - (r709 - r665) * ((w681 - w665) / (w709 - w665))
    ss665 = r665 - r620 - (r681 - r620) * ((w665 - w620) / (w681 - w620))
    ss681 = np.where(finite, ss681, np.nan).astype(np.float32)
    ss665 = np.where(finite, ss665, np.nan).astype(np.float32)

    ci_candidate = finite & (ss681 < 0)
    ci = np.where(ci_candidate, -ss681, np.nan).astype(np.float32)
    cyano_gate = (
        ci_candidate & (ss665 > 0)
        & (np.nan_to_num(ci, nan=-np.inf) > float(detection_limit))
    )
    ci_cyano = np.where(cyano_gate, ci, np.nan).astype(np.float32)
    return {
        "SS_681": ss681,
        "SS_665": ss665,
        "CI": ci,
        "CI_cyano": ci_cyano,
        "finite": finite,
        "ci_candidate": ci_candidate,
        "ci_detection": cyano_gate,
    }


def ndci_index(red: np.ndarray, red_edge: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    red = np.asarray(red, dtype=np.float32)
    red_edge = np.asarray(red_edge, dtype=np.float32)
    denominator = red_edge + red
    valid = np.isfinite(red) & np.isfinite(red_edge) & (np.abs(denominator) > 1e-8)
    result = np.full(red.shape, np.nan, dtype=np.float32)
    np.divide(red_edge - red, denominator, out=result, where=valid)
    return result, valid


In [13]:
def diagnose_l2_mask(
    l2_path: Path,
    ndci_source: str = NDCI_SOURCE,
    detection_limit: float = CI_DETECTION_LIMIT,
    qa_flags: Iterable[str] = QA_EXCLUDE_FLAGS,
    land_adjacency_pixels: int = LAND_ADJACENCY_PIXELS,
) -> dict[str, Any]:
    """Report every post-L2 mask stage without creating a derived product."""
    l2_path = Path(l2_path).expanduser().resolve()
    qa_flags = tuple(dict.fromkeys(str(name).upper() for name in qa_flags))
    land_adjacency_pixels = int(land_adjacency_pixels)
    if land_adjacency_pixels < 0:
        raise ValueError("land_adjacency_pixels must be non-negative")
    source_prefix = "rhos" if ndci_source.strip().lower() == "rhos" else "Rrs"
    if ndci_source.strip().lower() not in {"rhos", "rrs"}:
        raise ValueError("ndci_source must be 'rhos' or 'Rrs'")

    with netCDF4.Dataset(l2_path) as src:
        geo = src.groups["geophysical_data"]
        nav = src.groups["navigation_data"]
        variables = list(geo.variables)
        rhos_names = {
            target: nearest_product(variables, "rhos", target)
            for target in CORE_RHOS_WAVELENGTHS
        }
        ndci_names = {
            target: nearest_product(variables, source_prefix, target)
            for target in (665, 709)
        }
        actual_wavelengths = tuple(
            product_wavelength(rhos_names[target])
            for target in CORE_RHOS_WAVELENGTHS
        )

        latitude = to_float(nav.variables["latitude"][:])
        longitude = to_float(nav.variables["longitude"][:])
        flags_var = locate_l2_flags(src)
        flags_masked = np.ma.asarray(flags_var[:])
        flags_valid = ~np.ma.getmaskarray(flags_masked)
        flags = flags_masked.filled(0).astype(np.uint32)
        flag_map = decode_flag_metadata(flags_var)
        if "LAND" not in flag_map:
            raise RuntimeError("l2_flags metadata lacks the LAND bit")

        geolocated = (
            np.isfinite(latitude) & np.isfinite(longitude)
            & (np.abs(latitude) <= 90) & (np.abs(longitude) <= 180)
            & flags_valid
        )
        land = flag_is_set(flags, flag_map, "LAND")
        water = geolocated & ~land
        qa_excluded = np.zeros(water.shape, dtype=bool)
        for name in qa_flags:
            qa_excluded |= flag_is_set(flags, flag_map, name)
        if land_adjacency_pixels:
            adjacent = (
                binary_dilation(land, iterations=land_adjacency_pixels) & ~land
            )
        else:
            adjacent = np.zeros(water.shape, dtype=bool)
        qa_excluded |= adjacent
        retained_water = water & ~qa_excluded

        water_count = int(water.sum())
        requested = set(qa_flags)
        flag_details = {}
        for name in sorted(flag_map):
            count = int((water & flag_is_set(flags, flag_map, name)).sum())
            flag_details[name] = {
                "selected_for_exclusion": name in requested,
                "water_pixel_count": count,
                "percent_of_water": (
                    100.0 * count / water_count if water_count else None
                ),
            }

        counters = {
            "ci_finite_before_qa": 0,
            "ci_finite_after_qa": 0,
            "ci_candidates_before_qa": 0,
            "ci_candidates_after_qa": 0,
            "ci_detections_before_qa": 0,
            "ci_detections_after_qa": 0,
            "ndci_valid_before_qa": 0,
            "ndci_valid_after_qa": 0,
        }
        for start in range(0, water.shape[0], ROW_BLOCK_SIZE):
            stop = min(water.shape[0], start + ROW_BLOCK_SIZE)
            rs = slice(start, stop)
            arrays = {
                target: to_float(geo.variables[name][rs, :])
                for target, name in rhos_names.items()
            }
            core_finite = np.logical_and.reduce(
                [np.isfinite(values) for values in arrays.values()]
            )
            family = ci_family(
                arrays[620], arrays[665], arrays[681], arrays[709],
                wavelengths=actual_wavelengths,
                detection_limit=detection_limit,
            )
            ci_before = water[rs, :] & core_finite
            ci_after = retained_water[rs, :] & core_finite
            counters["ci_finite_before_qa"] += int(ci_before.sum())
            counters["ci_finite_after_qa"] += int(ci_after.sum())
            counters["ci_candidates_before_qa"] += int(
                (ci_before & family["ci_candidate"]).sum()
            )
            counters["ci_candidates_after_qa"] += int(
                (ci_after & family["ci_candidate"]).sum()
            )
            counters["ci_detections_before_qa"] += int(
                (ci_before & family["ci_detection"]).sum()
            )
            counters["ci_detections_after_qa"] += int(
                (ci_after & family["ci_detection"]).sum()
            )

            nd_red = to_float(geo.variables[ndci_names[665]][rs, :])
            nd_edge = to_float(geo.variables[ndci_names[709]][rs, :])
            _, ndci_finite = ndci_index(nd_red, nd_edge)
            counters["ndci_valid_before_qa"] += int(
                (water[rs, :] & ndci_finite).sum()
            )
            counters["ndci_valid_after_qa"] += int(
                (retained_water[rs, :] & ndci_finite).sum()
            )

    stages = {
        "geolocated_pixels": int(geolocated.sum()),
        "water_pixels": water_count,
        "qa_or_adjacency_excluded_water": int((water & qa_excluded).sum()),
        "retained_water_pixels": int(retained_water.sum()),
        **counters,
    }
    messages = []
    if stages["water_pixels"] == 0:
        messages.append(
            "No geolocated pixels survive the L2 LAND test. Post-L2 QA tuning "
            "cannot recover pixels classified as LAND; inspect the L2 land mask."
        )
    elif stages["ci_finite_before_qa"] == 0:
        messages.append(
            "Water pixels exist, but the four required rhos bands contain no "
            "jointly finite pixels. Those values were not produced by L2Gen and "
            "cannot be restored by relaxing the post-L2 QA mask."
        )
    elif stages["ci_finite_after_qa"] == 0:
        messages.append(
            "Finite CI inputs exist before QA, but the selected post-L2 mask "
            "removes all of them. Choose a less restrictive preset and inspect "
            "the per-flag percentages."
        )
    elif stages["ci_detections_after_qa"] == 0:
        messages.append(
            "Usable CI inputs remain after QA, but none pass SS(681)<0, SS(665)>0, "
            f"and CI>{float(detection_limit):g}. An all-NaN CI_cyano layer here is an algorithmic "
            "non-detection, not complete masking."
        )
    else:
        messages.append(
            "The selected mask retains finite CI inputs and CIcyano detections."
        )
    if stages["ndci_valid_before_qa"] and not stages["ndci_valid_after_qa"]:
        messages.append(
            "NDCI inputs exist before QA but are completely removed by the "
            "selected mask."
        )

    return {
        "created_utc": utc_now(),
        "source_l2": str(l2_path),
        "ndci_source": source_prefix,
        "ci_detection_limit": float(detection_limit),
        "qa_flags_requested": list(qa_flags),
        "qa_flags_missing": [name for name in qa_flags if name not in flag_map],
        "land_adjacency_pixels": land_adjacency_pixels,
        "core_band_mapping": rhos_names,
        "ndci_band_mapping": ndci_names,
        "stages": stages,
        "flag_details": flag_details,
        "messages": messages,
    }


def mask_diagnostic_tables(
    report: dict[str, Any],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Return compact stage and per-flag tables for notebook display."""
    labels = {
        "geolocated_pixels": "Geolocated L2 pixels",
        "water_pixels": "Water pixels (LAND not set)",
        "qa_or_adjacency_excluded_water": "Water removed by selected QA/buffer",
        "retained_water_pixels": "Water retained after QA/buffer",
        "ci_finite_before_qa": "Finite 4-band CI inputs before QA",
        "ci_finite_after_qa": "Finite 4-band CI inputs after QA",
        "ci_candidates_after_qa": "CI candidates: SS(681) < 0",
        "ci_detections_after_qa": f"CIcyano detections: SS(665)>0 and CI>{float(report.get('ci_detection_limit', 0)):g}",
        "ndci_valid_before_qa": "Valid NDCI inputs before QA",
        "ndci_valid_after_qa": "Valid NDCI inputs after QA",
    }
    water = report["stages"].get("water_pixels", 0)
    stage_rows = []
    for key, label in labels.items():
        count = int(report["stages"].get(key, 0))
        stage_rows.append({
            "stage": label,
            "pixels": count,
            "percent_of_water": (
                np.nan if key == "geolocated_pixels"
                else (100.0 * count / water if water else np.nan)
            ),
        })
    flag_rows = []
    for name, details in report["flag_details"].items():
        if details["selected_for_exclusion"] or details["water_pixel_count"]:
            flag_rows.append({
                "flag": name,
                "excluded_by_current_mask": bool(details["selected_for_exclusion"]),
                "flagged_water_pixels": int(details["water_pixel_count"]),
                "percent_of_water": details["percent_of_water"],
            })
    return pd.DataFrame(stage_rows), pd.DataFrame(flag_rows)


def _create_float_variable(
    dst: netCDF4.Dataset, name: str, dimensions: tuple[str, ...],
    chunks: tuple[int, ...], attributes: dict[str, Any],
) -> netCDF4.Variable:
    variable = dst.createVariable(
        name, "f4", dimensions, zlib=True, complevel=NETCDF_DEFLATE,
        shuffle=True, chunksizes=chunks, fill_value=DERIVED_FILL,
    )
    variable.setncatts(attributes)
    return variable


def validate_derived_file(path: Path) -> dict[str, Any]:
    required = (
        "longitude", "latitude", "water_mask", "valid_water_mask",
        "CI", "CI_cyano", "NDCI", "ci_valid_mask",
        "ci_candidate_mask", "ci_detection_mask", "ndci_valid_mask",
    )
    with netCDF4.Dataset(path) as dataset:
        missing = [name for name in required if name not in dataset.variables]
        if missing:
            raise RuntimeError(f"Derived file lacks variables: {missing}")
        shape = dataset.variables["latitude"].shape
        if any(dataset.variables[name].shape != shape for name in required):
            raise RuntimeError("Derived variable shapes differ")
        stride = max(1, int(math.sqrt(max(1, np.prod(shape) / 250_000))))
        ci_valid = np.ma.asarray(dataset.variables["ci_valid_mask"][::stride, ::stride]).filled(0) == 1
        ci_candidate = np.ma.asarray(
            dataset.variables["ci_candidate_mask"][::stride, ::stride]
        ).filled(0) == 1
        ci_detected = np.ma.asarray(
            dataset.variables["ci_detection_mask"][::stride, ::stride]
        ).filled(0) == 1
        ndci_valid = np.ma.asarray(dataset.variables["ndci_valid_mask"][::stride, ::stride]).filled(0) == 1
        ci = to_float(dataset.variables["CI"][::stride, ::stride])
        cyano = to_float(dataset.variables["CI_cyano"][::stride, ::stride])
        ndci = to_float(dataset.variables["NDCI"][::stride, ::stride])
        ci_finite = np.isfinite(ci)
        cyano_finite = np.isfinite(cyano)
        if not np.array_equal(ci_finite, ci_candidate):
            raise RuntimeError("Finite CI pixels do not match ci_candidate_mask")
        if not np.array_equal(cyano_finite, ci_detected):
            raise RuntimeError("Finite CI_cyano pixels do not match ci_detection_mask")
        if np.any(ci_candidate & ~ci_valid) or np.any(ci_detected & ~ci_candidate):
            raise RuntimeError("CI masks are not nested within CI-input-valid water")
        if ndci_valid.any() and not np.all(np.isfinite(ndci[ndci_valid])):
            raise RuntimeError("NDCI is not finite on sampled NDCI-valid water pixels")
        return {
            "path": str(path),
            "shape": tuple(map(int, shape)),
            "size_bytes": Path(path).stat().st_size,
            "sample_ci_valid_water": int(ci_valid.sum()),
            "sample_ci_candidates": int(ci_candidate.sum()),
            "sample_ci_cyano_detections": int(ci_detected.sum()),
            "sample_ndci_valid_water": int(ndci_valid.sum()),
        }


def build_indices_netcdf(
    l2_path: Path,
    ndci_source: str = NDCI_SOURCE,
    detection_limit: float = CI_DETECTION_LIMIT,
    qa_flags: Iterable[str] = QA_EXCLUDE_FLAGS,
    land_adjacency_pixels: int = LAND_ADJACENCY_PIXELS,
    mask_profile: str | None = DEFAULT_MASK_PROFILE,
    destination: Path | None = None,
    rows_per_block: int = ROW_BLOCK_SIZE,
) -> tuple[Path, dict[str, Any]]:
    """Create a compressed native-swath water-only CI/CIcyano/NDCI NetCDF."""
    l2_path = Path(l2_path).expanduser().resolve()
    ndci_source = ndci_source.strip()
    if ndci_source.lower() not in {"rhos", "rrs"}:
        raise ValueError("ndci_source must be 'rhos' or 'Rrs'")
    source_prefix = "rhos" if ndci_source.lower() == "rhos" else "Rrs"
    qa_flags = tuple(dict.fromkeys(str(name).upper() for name in qa_flags))
    land_adjacency_pixels = int(land_adjacency_pixels)
    if land_adjacency_pixels < 0:
        raise ValueError("land_adjacency_pixels must be non-negative")
    destination = Path(destination or (
        DERIVED_DIR / f"{l2_path.stem}_CI_CIcyano_NDCI_water.nc"
    ))
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".tmp")
    temporary.unlink(missing_ok=True)

    summary: dict[str, Any] = {
        "created_utc": utc_now(),
        "source_l2": str(l2_path),
        "ndci_source": source_prefix,
        "ci_detection_limit": float(detection_limit),
        "mask_profile": mask_profile or "custom",
        "qa_flags_requested": list(qa_flags),
        "land_adjacency_pixels": land_adjacency_pixels,
        "valid_water_pixels": 0,
        "ci_valid_pixels": 0,
        "ci_candidates": 0,
        "ci_detections": 0,
        "ndci_valid_pixels": 0,
        "water_pixels": 0,
        "qa_excluded_pixels": 0,
        "negative_core_rhos_pixels": 0,
    }

    with netCDF4.Dataset(l2_path) as src:
        geo = src.groups["geophysical_data"]
        nav = src.groups["navigation_data"]
        variables = list(geo.variables)
        rhos_names = {
            target: nearest_product(variables, "rhos", target)
            for target in CORE_RHOS_WAVELENGTHS
        }
        ndci_names = {
            target: nearest_product(variables, source_prefix, target)
            for target in (665, 709)
        }
        actual_wavelengths = tuple(product_wavelength(rhos_names[target]) for target in CORE_RHOS_WAVELENGTHS)

        lat_var, lon_var = nav.variables["latitude"], nav.variables["longitude"]
        shape = lat_var.shape
        dimensions = lat_var.dimensions
        if len(shape) != 2 or lon_var.shape != shape:
            raise RuntimeError("Expected matching two-dimensional L2 latitude/longitude arrays")

        latitude = to_float(lat_var[:])
        longitude = to_float(lon_var[:])
        flags_var = locate_l2_flags(src)
        flags_masked = np.ma.asarray(flags_var[:])
        flags_valid = ~np.ma.getmaskarray(flags_masked)
        flags = flags_masked.filled(0).astype(np.uint32)
        flag_map = decode_flag_metadata(flags_var)
        if "LAND" not in flag_map:
            raise RuntimeError("l2_flags metadata lacks the LAND bit required for a water-only product")
        summary["l2_flag_map"] = {key: int(value) for key, value in flag_map.items()}
        summary["qa_flags_available"] = sorted(flag_map)

        geolocated = (
            np.isfinite(latitude) & np.isfinite(longitude)
            & (np.abs(latitude) <= 90) & (np.abs(longitude) <= 180)
            & flags_valid
        )
        land = flag_is_set(flags, flag_map, "LAND")
        water = geolocated & ~land
        water_count = int(water.sum())
        summary["qa_flag_pixel_counts_water"] = {
            name: int((water & flag_is_set(flags, flag_map, name)).sum())
            for name in sorted(flag_map)
        }
        summary["qa_flag_percentages_water"] = {
            name: (100.0 * count / water_count if water_count else None)
            for name, count in summary["qa_flag_pixel_counts_water"].items()
        }
        qa_excluded = np.zeros(shape, dtype=bool)
        missing_qa_flags = []
        for name in qa_flags:
            if name.upper() not in flag_map:
                missing_qa_flags.append(name.upper())
            qa_excluded |= flag_is_set(flags, flag_map, name)
        if int(land_adjacency_pixels) > 0:
            adjacent = binary_dilation(land, iterations=int(land_adjacency_pixels)) & ~land
        else:
            adjacent = np.zeros(shape, dtype=bool)
        qa_excluded |= adjacent
        base_valid_water = water & ~qa_excluded
        summary["qa_flags_missing"] = missing_qa_flags
        summary["water_pixels"] = water_count
        summary["valid_water_pixels"] = int(base_valid_water.sum())
        summary["qa_excluded_pixels"] = int((water & qa_excluded).sum())

        with netCDF4.Dataset(temporary, "w", format="NETCDF4") as dst:
            for dimension in dimensions:
                src_dimension = src.dimensions[dimension]
                dst.createDimension(
                    dimension, None if src_dimension.isunlimited() else len(src_dimension)
                )
            dst.setncatts({
                "Conventions": "CF-1.8 ACDD-1.3",
                "title": "L2Gen-derived water-only OLCI CI, CIcyano, and NDCI",
                "source_l2": l2_path.name,
                "created_utc": utc_now(),
                "reflectance_for_CI": "L2Gen rhos; Rayleigh-subtracted surface reflectance; dimensionless",
                "ndci_reflectance_source": source_prefix,
                "algorithm_scope": "SS681 plus SS665 gates and strict CI>0.0001 detection limit; research reconstruction, not operational NASA CyAN V6",
                "non_detection_semantics": "CI and CI_cyano valid non-detections are encoded as NetCDF fill/NaN; ci_candidate_mask and ci_detection_mask distinguish them from invalid/non-water pixels; the published equations conventionally encode non-detections as 0",
                "negative_reflectance_policy": "No arbitrary positivity threshold; negative core-band count retained as QA",
                "mask_profile": mask_profile or "custom",
                "qa_excluded_flags": " ".join(str(item).upper() for item in qa_flags),
                "qa_missing_flags": " ".join(missing_qa_flags),
                "land_adjacency_pixels": int(land_adjacency_pixels),
                "ci_detection_limit": float(detection_limit),
                "ci_detection_operator": ">",
                "core_band_mapping_json": json.dumps(rhos_names),
                "ndci_band_mapping_json": json.dumps(ndci_names),
            })
            chunks = (min(rows_per_block, shape[0]), min(512, shape[1]))

            lon_out = _create_float_variable(dst, "longitude", dimensions, chunks, {
                "standard_name": "longitude", "long_name": "longitude",
                "units": "degrees_east", "valid_min": -180.0, "valid_max": 180.0,
            })
            lat_out = _create_float_variable(dst, "latitude", dimensions, chunks, {
                "standard_name": "latitude", "long_name": "latitude",
                "units": "degrees_north", "valid_min": -90.0, "valid_max": 90.0,
            })
            flags_out = dst.createVariable(
                "l2_flags", "u4", dimensions, zlib=True, complevel=NETCDF_DEFLATE,
                shuffle=True, chunksizes=chunks,
            )
            flags_out.setncatts({
                "long_name": "source L2Gen Level-2 flags",
                "flag_masks": np.asarray(list(flag_map.values()), dtype=np.uint32),
                "flag_meanings": " ".join(flag_map.keys()),
                "coordinates": "latitude longitude",
            })

            def mask_variable(name: str, long_name: str):
                variable = dst.createVariable(
                    name, "u1", dimensions, zlib=True, complevel=NETCDF_DEFLATE,
                    shuffle=True, chunksizes=chunks, fill_value=np.uint8(255),
                )
                variable.setncatts({
                    "long_name": long_name,
                    "flag_values": np.array([0, 1], dtype=np.uint8),
                    "flag_meanings": "false true",
                    "valid_range": np.array([0, 1], dtype=np.uint8),
                    "coordinates": "latitude longitude",
                })
                return variable

            water_out = mask_variable("water_mask", "L2Gen non-land geolocated pixel")
            valid_out = mask_variable(
                "valid_water_mask",
                "geolocated non-land pixel retained after configured QA and land-adjacency tests",
            )
            qa_out = mask_variable("qa_excluded_mask", "water pixel excluded by configured L2 flags or land adjacency")
            adjacent_out = mask_variable("land_adjacency_mask", "non-land pixel within configured pixel distance of L2 land")
            ci_valid_out = mask_variable("ci_valid_mask", "valid-water pixel with finite 620, 665, 681, and 709 nm rhos")
            ci_candidate_out = mask_variable("ci_candidate_mask", "negative-SS681 CI candidate within CI-input-valid water")
            ci_detect_out = mask_variable("ci_detection_mask", "CIcyano detection within valid water where SS681<0, SS665>0, and CI>detection_limit")
            ndci_valid_out = mask_variable("ndci_valid_mask", "finite NDCI observation within valid water")
            negative_out = dst.createVariable(
                "negative_core_rhos_count", "u1", dimensions, zlib=True,
                complevel=NETCDF_DEFLATE, shuffle=True, chunksizes=chunks,
                fill_value=np.uint8(255),
            )
            negative_out.setncatts({
                "long_name": "number of negative rhos values among 620, 665, 681, and 709 nm",
                "valid_range": np.array([0, 4], dtype=np.uint8),
                "coordinates": "latitude longitude",
            })

            ss681_out = _create_float_variable(dst, "SS_681", dimensions, chunks, {
                "long_name": "spectral shape centered near 681 nm", "units": "1",
                "coordinates": "latitude longitude",
            })
            ss665_out = _create_float_variable(dst, "SS_665", dimensions, chunks, {
                "long_name": "spectral shape centered near 665 nm", "units": "1",
                "coordinates": "latitude longitude",
            })
            ci_out = _create_float_variable(dst, "CI", dimensions, chunks, {
                "long_name": "detected-only cyanobacteria index before the SS665 exclusion gate", "units": "1",
                "coordinates": "latitude longitude",
                "algorithm": "-SS_681 where SS_681<0; valid non-detections encoded as fill/NaN",
                "published_non_detection_value": "0",
            })
            cyano_out = _create_float_variable(dst, "CI_cyano", dimensions, chunks, {
                "long_name": "detected-only two-spectral-shape cyanobacteria index", "units": "1",
                "coordinates": "latitude longitude",
                "algorithm": "CI where SS_681<0, SS_665>0, and CI>detection_limit; standard detection_limit=0.0001; valid non-detections encoded as fill/NaN",
                "published_non_detection_value": "0",
                "status": "research reconstruction; not operational NASA CyAN V6",
            })
            ndci_out = _create_float_variable(dst, "NDCI", dimensions, chunks, {
                "long_name": "normalized difference chlorophyll index", "units": "1",
                "coordinates": "latitude longitude",
                "formula": f"({source_prefix}_709-{source_prefix}_665)/({source_prefix}_709+{source_prefix}_665)",
                "negative_input_policy": "retained when denominator is finite and nonzero; inspect QA masks",
            })

            for start in tqdm(range(0, shape[0], rows_per_block), desc="Derived NetCDF row blocks"):
                stop = min(shape[0], start + rows_per_block)
                rs = slice(start, stop)
                lon_out[rs, :] = np.where(geolocated[rs, :], longitude[rs, :], DERIVED_FILL)
                lat_out[rs, :] = np.where(geolocated[rs, :], latitude[rs, :], DERIVED_FILL)
                flags_out[rs, :] = flags[rs, :]

                arrays = {
                    target: to_float(geo.variables[name][rs, :])
                    for target, name in rhos_names.items()
                }
                core_finite = np.logical_and.reduce([np.isfinite(values) for values in arrays.values()])
                ci_valid = base_valid_water[rs, :] & core_finite
                family = ci_family(
                    arrays[620], arrays[665], arrays[681], arrays[709],
                    wavelengths=actual_wavelengths,
                    detection_limit=detection_limit,
                )
                ci_candidate = ci_valid & family["ci_candidate"]
                negative_count = sum((values < 0).astype(np.uint8) for values in arrays.values())

                nd_red = to_float(geo.variables[ndci_names[665]][rs, :])
                nd_edge = to_float(geo.variables[ndci_names[709]][rs, :])
                ndci, ndci_finite = ndci_index(nd_red, nd_edge)
                ndci_valid = base_valid_water[rs, :] & ndci_finite
                detected = ci_valid & family["ci_detection"]

                water_out[rs, :] = water[rs, :].astype(np.uint8)
                valid_out[rs, :] = base_valid_water[rs, :].astype(np.uint8)
                qa_out[rs, :] = (water[rs, :] & qa_excluded[rs, :]).astype(np.uint8)
                adjacent_out[rs, :] = adjacent[rs, :].astype(np.uint8)
                ci_valid_out[rs, :] = ci_valid.astype(np.uint8)
                ci_candidate_out[rs, :] = np.where(
                    ci_valid, ci_candidate.astype(np.uint8), np.uint8(255)
                )
                ci_detect_out[rs, :] = np.where(ci_valid, detected.astype(np.uint8), np.uint8(255))
                ndci_valid_out[rs, :] = ndci_valid.astype(np.uint8)
                negative_out[rs, :] = np.where(ci_valid, negative_count, np.uint8(255))

                ss681_out[rs, :] = np.where(ci_valid, family["SS_681"], DERIVED_FILL)
                ss665_out[rs, :] = np.where(ci_valid, family["SS_665"], DERIVED_FILL)
                ci_out[rs, :] = np.where(ci_candidate, family["CI"], DERIVED_FILL)
                cyano_out[rs, :] = np.where(detected, family["CI_cyano"], DERIVED_FILL)
                ndci_out[rs, :] = np.where(ndci_valid, ndci, DERIVED_FILL)

                summary["ci_valid_pixels"] += int(ci_valid.sum())
                summary["ci_candidates"] += int(ci_candidate.sum())
                summary["ci_detections"] += int(detected.sum())
                summary["ndci_valid_pixels"] += int(ndci_valid.sum())
                summary["negative_core_rhos_pixels"] += int((ci_valid & (negative_count > 0)).sum())

    temporary.replace(destination)
    validation = validate_derived_file(destination)
    summary.update(validation)
    summary["size_human"] = human_size(destination.stat().st_size)
    summary["ci_detection_fraction_ci_valid"] = (
        summary["ci_detections"] / summary["ci_valid_pixels"]
        if summary["ci_valid_pixels"] else None
    )
    summary["ci_candidate_fraction_ci_valid"] = (
        summary["ci_candidates"] / summary["ci_valid_pixels"]
        if summary["ci_valid_pixels"] else None
    )
    summary["negative_core_rhos_fraction_ci_valid"] = (
        summary["negative_core_rhos_pixels"] / summary["ci_valid_pixels"]
        if summary["ci_valid_pixels"] else None
    )
    write_json(EXPORT_DIR / l2_path.stem / "derived_index_summary.json", summary)
    print(f"Derived product: {destination} ({summary['size_human']})")
    return destination, summary


## 7. Non-network algorithm checks

Run this before processing a real scene. The tests verify a positive two-stage
CIcyano detection, `NaN` encoding for a valid non-detection, the independent
SS665 exclusion, retention of finite negative reflectance inputs, and stable
NDCI denominator handling.


In [14]:
_detect = ci_family(
    np.array([0.010]), np.array([0.030]),
    np.array([0.020]), np.array([0.040]),
)
assert _detect["SS_681"][0] < 0
assert _detect["SS_665"][0] > 0
assert _detect["CI"][0] > 0
assert np.isclose(_detect["CI_cyano"][0], _detect["CI"][0])
assert _detect["ci_candidate"][0] and _detect["ci_detection"][0]

_no_ci = ci_family(
    np.array([0.010]), np.array([0.015]),
    np.array([0.030]), np.array([0.040]),
)
assert _no_ci["SS_681"][0] > 0
assert np.isnan(_no_ci["CI"][0])
assert np.isnan(_no_ci["CI_cyano"][0])
assert not _no_ci["ci_candidate"][0] and not _no_ci["ci_detection"][0]

_noncyano = ci_family(
    np.array([0.040]), np.array([0.020]),
    np.array([0.015]), np.array([0.040]),
)
assert _noncyano["SS_681"][0] < 0 and _noncyano["CI"][0] > 0
assert _noncyano["SS_665"][0] <= 0 and np.isnan(_noncyano["CI_cyano"][0])
assert _noncyano["ci_candidate"][0] and not _noncyano["ci_detection"][0]

_below_limit = ci_family(
    np.array([0.010]), np.array([0.030]),
    np.array([0.03355]), np.array([0.040]),
)
assert _below_limit["SS_681"][0] < 0 and _below_limit["SS_665"][0] > 0
assert 0 < _below_limit["CI"][0] < CI_DETECTION_LIMIT
assert np.isnan(_below_limit["CI_cyano"][0]) and not _below_limit["ci_detection"][0]

_boundary_source = ci_family(
    np.array([0.010]), np.array([0.030]),
    np.array([0.03353636]), np.array([0.040]), detection_limit=0.0,
)
_boundary_value = float(_boundary_source["CI"][0])
_at_boundary = ci_family(
    np.array([0.010]), np.array([0.030]),
    np.array([0.03353636]), np.array([0.040]), detection_limit=_boundary_value,
)
assert np.isclose(_at_boundary["CI"][0], _boundary_value)
assert np.isnan(_at_boundary["CI_cyano"][0]) and not _at_boundary["ci_detection"][0]

_negative = ci_family(
    np.array([-0.005]), np.array([0.005]),
    np.array([-0.002]), np.array([0.010]),
)
assert _negative["finite"][0]

_ndci, _ndci_valid = ndci_index(
    np.array([0.02, -0.01, 0.01]),
    np.array([0.04, 0.02, -0.01]),
)
assert _ndci_valid.tolist() == [True, True, False]
assert np.isclose(_ndci[0], 1 / 3)

print("Core CI/CIcyano/NDCI checks passed, including strict CIcyano > 0.0001 semantics.")


Core CI/CIcyano/NDCI checks passed, including strict CIcyano > 0.0001 semantics.


## 8. Maps, publication exports, and pixel spectra

The native OLCI swath is curvilinear. v5.3 resamples it once, directly to a regular EPSG:4326 latitude/longitude grid. It does not ask Cartopy to reproject a previously resampled UTM image, which had produced the large block artifacts seen in v5.2.

- Native-swath arrays remain the authoritative data for index calculation, masks, lake statistics, and NetCDF export.
- The regular grid is used only for visualisation and georeferenced GeoTIFF output.
- Static maps use Cartopy geographic axes, longitude/latitude labels, HydroLAKES overlays, and optional geographic context. Their safety cap is 32 million display cells.
- Interactive maps are standalone Folium/Leaflet HTML with basemap switching, coordinate readout, click coordinates, measurement, fullscreen, HydroLAKES, and a 3-million-cell browser-preview cap.
- Any preview coarsening changes only the display raster, never the native NetCDF, index calculation, or lake statistics.
- CI/CIcyano/NDCI maps distinguish valid algorithm non-detections (grey) from selected post-mask exclusions (dark grey/black); coloured pixels are finite retrieved values.
- True colour uses L2Gen `rhot` when present, so land, clouds, open water, and bloom texture appear as a conventional satellite image. It is labelled pseudo true colour when `rhos` must be used instead.
- PNG output defaults to 400 dpi, PDF retains vector text/boundaries with raster imagery, and GeoTIFFs record CRS, transform, nodata, resampling, and effective resolution.


In [15]:
def bbox_slices(
    latitude: np.ndarray, longitude: np.ndarray,
    bbox: tuple[float, float, float, float] | None,
    margin: int = 4,
) -> tuple[slice, slice]:
    if bbox is None:
        return slice(None), slice(None)
    west, south, east, north = normalize_bbox(bbox)
    inside = (
        np.isfinite(latitude) & np.isfinite(longitude)
        & (longitude >= west) & (longitude <= east)
        & (latitude >= south) & (latitude <= north)
    )
    rows, columns = np.where(inside)
    if not rows.size:
        raise ValueError("No geolocated pixels intersect the requested bbox")
    return (
        slice(max(0, rows.min() - margin), min(latitude.shape[0], rows.max() + margin + 1)),
        slice(max(0, columns.min() - margin), min(latitude.shape[1], columns.max() + margin + 1)),
    )


def preview_stride(shape: tuple[int, int], max_points: int = 350_000) -> int:
    return max(1, int(math.ceil(math.sqrt(np.prod(shape) / max_points))))


def robust_limits(values: np.ndarray, default: tuple[float, float]) -> tuple[float, float]:
    finite = values[np.isfinite(values)]
    if not finite.size:
        return default
    low, high = np.nanpercentile(finite, [2, 98])
    if not np.isfinite(low) or not np.isfinite(high) or low == high:
        return default
    return float(low), float(high)


def load_true_color(l2_path: Path, row_slice: slice, col_slice: slice, stride: int):
    with netCDF4.Dataset(l2_path) as root:
        geo = root.groups["geophysical_data"]
        nav = root.groups["navigation_data"]
        try:
            names = [nearest_product(geo.variables, "rhot", target, 15) for target in (665, 560, 443)]
            title = "L2Gen input TOA true color"
        except KeyError:
            # rhot is a convenience product and can be dropped if a particular
            # L2Gen build rejects it.  The required rhos bands still provide a
            # useful (but explicitly labelled) Rayleigh-corrected RGB preview.
            names = [nearest_product(geo.variables, "rhos", target, 15) for target in (665, 560, 443)]
            title = "L2Gen Rayleigh-corrected pseudo true color"
        rs = slice(row_slice.start, row_slice.stop, stride)
        cs = slice(col_slice.start, col_slice.stop, stride)
        latitude = to_float(nav.variables["latitude"][rs, cs])
        longitude = to_float(nav.variables["longitude"][rs, cs])
        rgb = np.dstack([to_float(geo.variables[name][rs, cs]) for name in names])
    finite = np.all(np.isfinite(rgb), axis=2)
    stretched = np.zeros_like(rgb, dtype=np.float32)
    for channel in range(3):
        values = rgb[..., channel]
        sample = values[finite]
        low, high = (0.0, 0.12) if not sample.size else np.nanpercentile(sample, [1, 99])
        if not np.isfinite(high) or high <= low:
            low, high = 0.0, 0.12
        stretched[..., channel] = np.clip((values - low) / (high - low), 0, 1)
    stretched = np.power(stretched, 0.8)
    return latitude, longitude, stretched, finite, title


LAYER_STYLE = {
    "CI": ("magma", (0.0, 0.003), "CI (dimensionless)"),
    "CI_cyano": ("viridis", (0.0001, 0.003), r"CI$_{cyano}$ detection (>0.0001)"),
    "NDCI": ("RdYlGn", (-0.25, 0.5), "NDCI (dimensionless)"),
    "SS_681": ("RdBu_r", (-0.003, 0.003), "SS(681)"),
    "SS_665": ("RdBu_r", (-0.003, 0.003), "SS(665)"),
    "negative_core_rhos_count": ("plasma", (0, 4), "Negative core rhos band count"),
}
MASK_LAYER_STYLE = {
    "water_mask": ("Blues", "Water mask (0/1)"),
    "valid_water_mask": ("Greens", "Retained water after QA (0/1)"),
    "qa_excluded_mask": ("Reds", "QA/buffer-excluded water (0/1)"),
    "land_adjacency_mask": ("Oranges", "Land-adjacency mask (0/1)"),
    "ci_valid_mask": ("Greens", "Finite CI inputs after QA (0/1)"),
    "ci_candidate_mask": ("Purples", "SS(681) CI candidate mask (0/1)"),
    "ci_detection_mask": ("viridis", "CIcyano detection mask (0/1)"),
    "ndci_valid_mask": ("Greens", "Valid NDCI mask (0/1)"),
}


def render_layer(
    ax: plt.Axes, derived_path: Path, l2_path: Path,
    layer: str, bbox: tuple[float, float, float, float] | None = None,
    max_points: int = 350_000,
    vmin: float | None = None, vmax: float | None = None,
) -> Any:
    with netCDF4.Dataset(derived_path) as derived:
        latitude_full = to_float(derived.variables["latitude"][:])
        longitude_full = to_float(derived.variables["longitude"][:])
        row_slice, col_slice = bbox_slices(latitude_full, longitude_full, bbox)
        sub_shape = latitude_full[row_slice, col_slice].shape
        stride = preview_stride(sub_shape, max_points)
        rs = slice(row_slice.start, row_slice.stop, stride)
        cs = slice(col_slice.start, col_slice.stop, stride)
        latitude = latitude_full[rs, cs]
        longitude = longitude_full[rs, cs]

        if layer == "true_color":
            latitude, longitude, rgb, valid, title = load_true_color(
                l2_path, row_slice, col_slice, stride
            )
            artist = ax.scatter(
                longitude[valid], latitude[valid], c=rgb[valid], s=2.2,
                marker="s", linewidths=0, rasterized=True,
            )
            ax.set_title(title)
        else:
            if layer not in derived.variables:
                raise KeyError(f"Layer not in derived NetCDF: {layer}")
            values = to_float(derived.variables[layer][rs, cs])
            if layer in MASK_LAYER_STYLE:
                valid = np.isfinite(values) & np.isfinite(latitude) & np.isfinite(longitude)
                cmap, label = MASK_LAYER_STYLE[layer]
                lower = 0.0 if vmin is None else float(vmin)
                upper = 1.0 if vmax is None else float(vmax)
            else:
                valid_water = np.ma.asarray(
                    derived.variables["valid_water_mask"][rs, cs]
                ).filled(0) == 1
                valid = valid_water & np.isfinite(values)
                cmap, defaults, label = LAYER_STYLE[layer]
                auto_min, auto_max = robust_limits(values[valid], defaults)
                lower = auto_min if vmin is None else float(vmin)
                upper = auto_max if vmax is None else float(vmax)
            artist = ax.scatter(
                longitude[valid], latitude[valid], c=values[valid], s=2.2,
                marker="s", linewidths=0, cmap=cmap, vmin=lower, vmax=upper,
                rasterized=True,
            )
            ax.set_title(layer.replace("_", " "))
            plt.colorbar(artist, ax=ax, fraction=0.046, pad=0.03, label=label)

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_aspect("equal", adjustable="box")
    ax.grid(alpha=0.16, linewidth=0.5)
    return artist


def plot_layer(
    derived_path: Path, l2_path: Path, layer: str = "CI_cyano",
    bbox: tuple[float, float, float, float] | None = None,
    vmin: float | None = None, vmax: float | None = None,
    figsize: tuple[float, float] = (10, 8),
) -> plt.Figure:
    figure, axis = plt.subplots(figsize=figsize, constrained_layout=True)
    render_layer(axis, derived_path, l2_path, layer, bbox, vmin=vmin, vmax=vmax)
    return figure


In [16]:
def export_map_bundle(
    derived_path: Path, l2_path: Path,
    bbox: tuple[float, float, float, float] | None = None,
    dpi: int = 300,
) -> dict[str, Path]:
    """Save true-color/detection-only CI/CIcyano/NDCI figures."""
    scene_dir = EXPORT_DIR / Path(l2_path).stem / "figures"
    scene_dir.mkdir(parents=True, exist_ok=True)
    outputs: dict[str, Path] = {}
    layers = ("true_color", "CI", "CI_cyano", "NDCI")
    for layer in layers:
        figure = plot_layer(derived_path, l2_path, layer=layer, bbox=bbox)
        png = scene_dir / f"{Path(l2_path).stem}_{layer}.png"
        pdf = png.with_suffix(".pdf")
        figure.savefig(png, dpi=dpi, bbox_inches="tight", pad_inches=0.06)
        figure.savefig(pdf, dpi=dpi, bbox_inches="tight", pad_inches=0.06)
        plt.close(figure)
        outputs[f"{layer}_png"] = png
        outputs[f"{layer}_pdf"] = pdf

    combined, axes = plt.subplots(2, 2, figsize=(15, 12), constrained_layout=True)
    for axis, layer in zip(axes.ravel(), layers):
        render_layer(axis, derived_path, l2_path, layer, bbox)
    combined_png = scene_dir / f"{Path(l2_path).stem}_four_panel.png"
    combined_pdf = combined_png.with_suffix(".pdf")
    combined.savefig(combined_png, dpi=dpi, bbox_inches="tight", pad_inches=0.06)
    combined.savefig(combined_pdf, dpi=dpi, bbox_inches="tight", pad_inches=0.06)
    plt.close(combined)
    outputs.update({"combined_png": combined_png, "combined_pdf": combined_pdf})
    print(f"Saved {len(outputs)} figure files under {scene_dir}")
    return outputs


def nearest_valid_pixel(
    derived_path: Path, target_lat: float, target_lon: float,
) -> tuple[int, int, float, float]:
    with netCDF4.Dataset(derived_path) as derived:
        latitude = to_float(derived.variables["latitude"][:])
        longitude = to_float(derived.variables["longitude"][:])
        valid = np.ma.asarray(derived.variables["ci_valid_mask"][:]).filled(0) == 1
    scale = max(math.cos(math.radians(float(target_lat))), 0.1)
    distance = (latitude - target_lat) ** 2 + ((longitude - target_lon) * scale) ** 2
    distance[~valid | ~np.isfinite(distance)] = np.inf
    if not np.isfinite(distance).any():
        raise RuntimeError("No CI-valid water pixel is available")
    row, column = np.unravel_index(np.argmin(distance), distance.shape)
    return int(row), int(column), float(latitude[row, column]), float(longitude[row, column])


def export_pixel_spectrum(
    l2_path: Path, derived_path: Path,
    target_lat: float, target_lon: float,
) -> tuple[Path, Path, pd.DataFrame]:
    row, column, pixel_lat, pixel_lon = nearest_valid_pixel(
        derived_path, target_lat, target_lon
    )
    with netCDF4.Dataset(l2_path) as l2:
        geo = l2.groups["geophysical_data"]
        records_by_wavelength: dict[float, dict[str, Any]] = {}
        for prefix in ("rhos", "Rrs"):
            names = []
            for name in geo.variables:
                if not name.startswith(prefix + "_"):
                    continue
                try:
                    wavelength = product_wavelength(name)
                except ValueError:
                    continue
                names.append((wavelength, name))
            for wavelength, name in sorted(names):
                record = records_by_wavelength.setdefault(wavelength, {
                "target_latitude": float(target_lat),
                "target_longitude": float(target_lon),
                "pixel_latitude": pixel_lat,
                "pixel_longitude": pixel_lon,
                "row": row, "column": column,
                "wavelength_nm": wavelength,
                })
                record[f"{prefix}_product"] = name
                record[prefix] = float(to_float(geo.variables[name][row, column]))
    table = pd.DataFrame(
        [records_by_wavelength[key] for key in sorted(records_by_wavelength)]
    )
    scene_dir = EXPORT_DIR / Path(l2_path).stem / "spectra"
    scene_dir.mkdir(parents=True, exist_ok=True)
    stem = f"spectrum_{pixel_lat:.5f}_{pixel_lon:.5f}".replace("-", "m")
    csv_path = scene_dir / f"{stem}.csv"
    png_path = scene_dir / f"{stem}.png"
    table.to_csv(csv_path, index=False)

    panels = [name for name in ("rhos", "Rrs") if name in table and table[name].notna().any()]
    figure, axes = plt.subplots(
        len(panels), 1, figsize=(9, 4.5 * len(panels)),
        sharex=True, constrained_layout=True, squeeze=False,
    )
    for axis, prefix in zip(axes.ravel(), panels):
        axis.plot(table["wavelength_nm"], table[prefix], "o-", lw=1.5, ms=4)
        for wavelength in (620, 665, 681, 709):
            axis.axvline(wavelength, color="0.65", lw=0.7, ls="--")
        axis.axhline(0, color="0.35", lw=0.8)
        axis.set_ylabel(
            "L2Gen rhos (dimensionless)" if prefix == "rhos" else r"L2Gen Rrs (sr$^{-1}$)"
        )
        axis.grid(alpha=0.2)
    axes[-1, 0].set_xlabel("Wavelength (nm)")
    figure.suptitle(f"Nearest CI-valid water pixel: {pixel_lat:.5f}, {pixel_lon:.5f}")
    figure.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(figure)
    print(f"Spectrum CSV: {csv_path}")
    return csv_path, png_path, table


In [17]:
def write_scene_manifest(
    scene: dict[str, Any] | None,
    raw_zip: Path | None,
    sen3_path: Path,
    l2_path: Path,
    derived_path: Path,
    l2_info: dict[str, Any],
    derived_info: dict[str, Any],
) -> Path:
    scene_name = product_basename(sen3_path)
    payload = {
        "created_utc": utc_now(),
        "scene_catalogue_record": scene,
        "files": {
            "raw_l1_zip": str(raw_zip) if raw_zip else None,
            "staged_sen3": str(sen3_path),
            "l2gen_netcdf": str(l2_path),
            "derived_indices_netcdf": str(derived_path),
        },
        "sizes_bytes": {
            "raw_l1_zip": raw_zip.stat().st_size if raw_zip and raw_zip.exists() else None,
            "l2gen_netcdf": l2_path.stat().st_size,
            "derived_indices_netcdf": derived_path.stat().st_size,
        },
        "checksums_sha256": {
            "raw_l1_zip": sha256_file(raw_zip) if COMPUTE_SHA256 and raw_zip and raw_zip.exists() else None,
            "l2gen_netcdf": sha256_file(l2_path) if COMPUTE_SHA256 else None,
            "derived_indices_netcdf": sha256_file(derived_path) if COMPUTE_SHA256 else None,
        },
        "configuration": {
            "l2_mode": l2_info.get("mode"),
            "ndci_source": derived_info.get("ndci_source"),
            "process_full_scene": l2_info.get("processing_scope") == "full_scene",
            "processing_bbox": l2_info.get("processing_bbox"),
            "full_reflectance_spectra_requested": l2_info.get("full_reflectance_spectra_requested"),
            "get_ancillary": GET_ANCILLARY,
            "allow_climatology_fallback": ALLOW_CLIMATOLOGY_FALLBACK,
            "ci_detection_limit": derived_info.get("ci_detection_limit"),
            "mask_profile": derived_info.get("mask_profile"),
            "qa_exclude_flags": derived_info.get("qa_flags_requested"),
            "land_adjacency_pixels": derived_info.get("land_adjacency_pixels"),
        },
        "l2gen": l2_info,
        "derived_summary": derived_info,
    }
    path = EXPORT_DIR / scene_name / "processing_manifest.json"
    return write_json(path, payload)


## HydroLAKES, post-L2 water masks, and regular-grid publication products

The cells below adapt the HydroLAKES workflow from the ACOLITE dashboard to the saved L2Gen scene. HydroLAKES can define lake water independently of the L2 `LAND` bit, with an optional inward shore buffer in metres. The bright-pixel screen is **off by default** so intense surface scums are not automatically rejected. When enabled, it is a transparent post-L2 test on `rhos_865`; the original native L2 values remain available.

Publication maps use nearest-neighbour swath-to-grid resampling at an explicit target resolution. Nearest neighbour avoids inventing new spectral extrema, while a common projected grid yields square map pixels, reliable overlays, scale bars, GeoTIFF georeferencing, and clean PDF/PNG output. Statistics are always calculated on the native swath, not on the display raster.


In [18]:
import geopandas as gpd
import pyogrio
import shapely
from shapely import box, contains_xy, union_all
from pyproj import CRS, Transformer
from pyresample import geometry as pr_geometry
from pyresample import kd_tree as pr_kd_tree
import rasterio
from rasterio.transform import from_bounds
import matplotlib.colors as mcolors
import matplotlib.patheffects as path_effects
from matplotlib.patches import Patch
import plotly.graph_objects as go
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import folium
from folium import plugins as folium_plugins
from branca.colormap import LinearColormap
from PIL import Image
from IPython.display import IFrame

# Override any path cached by an earlier import in this Colab kernel.
from pyproj import datadir as _pyproj_datadir
import rasterio._env as _rasterio_environment
_pyproj_datadir.set_data_dir(str(PYTHON_PROJ_DATA_DIR))
_rasterio_environment.set_proj_data_search_path(str(PYTHON_PROJ_DATA_DIR))

# Fail early, before map creation, if the repair did not take effect.
try:
    _pyproj_epsg4326 = CRS.from_epsg(4326)
    _rasterio_epsg4326 = rasterio.crs.CRS.from_epsg(4326)
except Exception as error:
    raise RuntimeError(
        "Python PROJ/Rasterio could not be redirected away from OCSSW's proj.db. "
        "Use Runtime > Disconnect and delete runtime, then run this v1.4 notebook."
    ) from error
print(
    "Python PROJ/Rasterio sanity check passed (EPSG:4326); "
    f"active database: {PYTHON_PROJ_DATABASE}"
)

HYDROLAKES_URL = "https://data.hydrosheds.org/file/hydrolakes/HydroLAKES_polys_v10_shp.zip"
US_STATES_URL = "https://www2.census.gov/geo/tiger/GENZ2025/shp/cb_2025_us_state_5m.zip"

# Extend the base validator with a sampled, explicit rhos-over-land audit.  Keep
# the original callable in a stable global so rerunning this cell is harmless.
if "_BASE_VALIDATE_L2_FILE_V5" not in globals():
    _BASE_VALIDATE_L2_FILE_V5 = validate_l2_file


def validate_l2_file(path: Path, required_products: Iterable[str]) -> dict[str, Any]:
    info = _BASE_VALIDATE_L2_FILE_V5(path, required_products)
    with netCDF4.Dataset(path) as root:
        geo = root.groups["geophysical_data"]
        nav = root.groups["navigation_data"]
        rhos_name = nearest_product(geo.variables, "rhos", 681)
        shape = nav.variables["latitude"].shape
        stride = max(1, int(math.sqrt(max(1, np.prod(shape) / 350_000))))
        rhos = to_float(geo.variables[rhos_name][::stride, ::stride])
        flags_var = locate_l2_flags(root)
        flags = np.ma.asarray(flags_var[::stride, ::stride]).filled(0).astype(np.uint32)
        flag_map = decode_flag_metadata(flags_var)
        land = flag_is_set(flags, flag_map, "LAND")
        finite = np.isfinite(rhos)
        info["sampled_rhos_coverage"] = {
            "product": rhos_name, "stride": stride,
            "finite_all_pixels": int(finite.sum()),
            "sampled_all_pixels": int(finite.size),
            "finite_fraction_all": float(finite.mean()),
            "finite_land_pixels": int((finite & land).sum()),
            "sampled_land_pixels": int(land.sum()),
            "finite_fraction_land": float((finite & land).sum() / land.sum()) if land.any() else None,
            "finite_nonland_pixels": int((finite & ~land).sum()),
            "sampled_nonland_pixels": int((~land).sum()),
            "finite_fraction_nonland": float((finite & ~land).sum() / (~land).sum()) if (~land).any() else None,
        }
    return info


def download_stream(url: str, destination: Path) -> Path:
    """Resumable streaming download used only when the user requests HydroLAKES."""
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    part = destination.with_suffix(destination.suffix + ".part")
    start = part.stat().st_size if part.exists() else 0
    headers = {"Range": f"bytes={start}-"} if start else {}
    response = requests.get(url, headers=headers, stream=True, timeout=(60, 900))
    if response.status_code == 416 and part.exists() and zipfile.is_zipfile(part):
        os.replace(part, destination)
        return destination
    if start and response.status_code != 206:
        start = 0
    response.raise_for_status()
    mode = "ab" if start and response.status_code == 206 else "wb"
    total = int(response.headers.get("Content-Length", 0)) + (start if mode == "ab" else 0)
    with part.open(mode) as handle, tqdm(
        total=total or None, initial=start if mode == "ab" else 0,
        unit="B", unit_scale=True, desc=destination.name,
    ) as bar:
        for chunk in response.iter_content(8 * 1024 * 1024):
            if chunk:
                handle.write(chunk)
                bar.update(len(chunk))
    os.replace(part, destination)
    return destination


def _zip_vector_path(zip_path: Path, suffix: str = ".shp") -> str:
    with zipfile.ZipFile(zip_path) as archive:
        candidates = [name for name in archive.namelist() if name.lower().endswith(suffix)]
    if len(candidates) != 1:
        raise RuntimeError(f"Expected one {suffix} in {zip_path.name}; found {len(candidates)}")
    return f"/vsizip/{zip_path.resolve()}/{candidates[0]}"


def _normalize_hydrolakes(frame: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    aliases = {column.lower(): column for column in frame.columns}
    required = {"hylak_id", "lake_area", "geometry"}
    missing = sorted(required - set(aliases))
    if missing:
        raise ValueError(f"HydroLAKES file lacks columns: {missing}")
    rename = {aliases["hylak_id"]: "Hylak_id", aliases["lake_area"]: "Lake_area"}
    for optional, canonical in (
        ("lake_name", "Lake_name"), ("country", "Country"),
        ("lake_type", "Lake_type"), ("continent", "Continent"),
    ):
        if optional in aliases:
            rename[aliases[optional]] = canonical
    frame = frame.rename(columns=rename).copy()
    frame["Hylak_id"] = pd.to_numeric(frame["Hylak_id"], errors="raise").astype("int64")
    frame["Lake_area"] = pd.to_numeric(frame["Lake_area"], errors="coerce")
    frame = frame.set_crs(4326) if frame.crs is None else frame.to_crs(4326)
    frame = frame[frame.geometry.notna() & ~frame.geometry.is_empty].copy()
    try:
        frame.geometry = frame.geometry.make_valid()
    except AttributeError:
        frame.geometry = frame.geometry.buffer(0)
    return frame


def prepare_hydrolakes(
    min_area_km2: float = 10.0,
    region: str = "USA",
    delete_global_archive: bool = True,
) -> tuple[gpd.GeoDataFrame, Path]:
    """Download/cache an official HydroLAKES subset only on explicit request."""
    region = str(region).upper()
    region_slug = "USA" if region == "USA" else "WORLD"
    cache_path = HYDRO_ROOT / f"HydroLAKES_{region_slug}_FULLPOLY_v14_ge{min_area_km2:g}km2.gpkg"
    if cache_path.exists():
        return _normalize_hydrolakes(gpd.read_file(cache_path)), cache_path
    archive = HYDRO_ROOT / "HydroLAKES_polys_v10_shp.zip"
    if not archive.exists():
        print("Downloading the official global HydroLAKES archive (large; cached temporarily).")
        download_stream(HYDROLAKES_URL, archive)
    columns = ["Hylak_id", "Lake_name", "Country", "Continent", "Lake_type", "Lake_area"]
    lakes = pyogrio.read_dataframe(
        _zip_vector_path(archive), columns=columns,
        where=f"Lake_area >= {float(min_area_km2)}",
    )
    lakes = _normalize_hydrolakes(gpd.GeoDataFrame(lakes, geometry="geometry", crs=4326))
    if region == "USA":
        states_zip = HYDRO_ROOT / "cb_2025_us_state_5m.zip"
        if not states_zip.exists():
            download_stream(US_STATES_URL, states_zip)
        states = gpd.read_file(f"zip://{states_zip}").to_crs(4326)
        usa = union_all(states.geometry.values)
        # Region is a catalogue filter, not a geometry clip.  Retaining the
        # complete HydroLAKES polygon is essential for transboundary lakes such
        # as Erie, Ontario, Superior, Huron, and Champlain.
        lakes = lakes[lakes.intersects(usa)].copy()
        lakes = lakes[~lakes.geometry.is_empty].copy()
        lakes["boundary_scope"] = "full polygon; intersects USA"
        lakes["analysis_area_km2"] = lakes.to_crs(6933).geometry.area / 1e6
    else:
        lakes["boundary_scope"] = "full polygon; worldwide catalogue"
        lakes["analysis_area_km2"] = lakes.to_crs(6933).geometry.area / 1e6
    lakes.to_file(cache_path, driver="GPKG", layer="hydrolakes", index=False)
    if delete_global_archive:
        archive.unlink(missing_ok=True)
    print(f"Cached {len(lakes):,} lakes: {cache_path}")
    return lakes.reset_index(drop=True), cache_path


def load_hydrolakes_file(path: str | Path, min_area_km2: float = 0.0) -> tuple[gpd.GeoDataFrame, Path]:
    path = Path(path).expanduser().resolve()
    frame = gpd.read_file(f"zip://{path}" if path.suffix.lower() == ".zip" else path)
    frame = _normalize_hydrolakes(frame)
    frame = frame[frame["Lake_area"] >= float(min_area_km2)].copy()
    cache = HYDRO_ROOT / f"uploaded_HydroLAKES_ge{min_area_km2:g}km2.gpkg"
    frame.to_file(cache, driver="GPKG", layer="hydrolakes", index=False)
    return frame.reset_index(drop=True), cache


def save_uploaded_vector(upload_widget) -> Path:
    value = upload_widget.value
    if not value:
        raise ValueError("Choose a GeoPackage, GeoJSON, or zipped shapefile first")
    if isinstance(value, dict):
        item = next(iter(value.values()))
        name = item.get("metadata", {}).get("name") or item.get("name") or "hydrolakes_upload"
    else:
        item = value[0]
        name = item.get("name", "hydrolakes_upload")
    content = item["content"]
    if hasattr(content, "tobytes"):
        content = content.tobytes()
    destination = HYDRO_ROOT / Path(name).name
    destination.write_bytes(bytes(content))
    return destination


def scene_geolocation(path: Path) -> tuple[np.ndarray, np.ndarray]:
    with netCDF4.Dataset(path) as root:
        if "navigation_data" in root.groups:
            nav = root.groups["navigation_data"]
            return to_float(nav.variables["latitude"][:]), to_float(nav.variables["longitude"][:])
        return to_float(root.variables["latitude"][:]), to_float(root.variables["longitude"][:])


def subset_lakes_to_scene(path: Path, lakes: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    latitude, longitude = scene_geolocation(path)
    stride = max(1, int(math.sqrt(max(1, latitude.size / 350_000))))
    lat = latitude[::stride, ::stride]
    lon = longitude[::stride, ::stride]
    valid = np.isfinite(lon) & np.isfinite(lat) & (np.abs(lon) <= 180) & (np.abs(lat) <= 90)
    if not valid.any():
        raise RuntimeError("Scene has no valid geolocation sample")
    footprint = box(float(lon[valid].min()), float(lat[valid].min()),
                    float(lon[valid].max()), float(lat[valid].max()))
    return lakes[lakes.intersects(footprint)].copy().reset_index(drop=True)


def erode_lake_geometries(lakes: gpd.GeoDataFrame, distance_m: float) -> gpd.GeoDataFrame:
    result = lakes.copy()
    if distance_m <= 0 or result.empty:
        return result
    local_crs = result.estimate_utm_crs() or "EPSG:6933"
    projected = result.to_crs(local_crs)
    projected.geometry = projected.geometry.buffer(-float(distance_m))
    projected = projected[~projected.geometry.is_empty].copy()
    return projected.to_crs(4326)


def hydrolakes_point_mask(
    latitude: np.ndarray, longitude: np.ndarray,
    lakes: gpd.GeoDataFrame | None, shore_buffer_m: float = 0.0,
) -> np.ndarray | None:
    if lakes is None or lakes.empty:
        return None
    buffered = erode_lake_geometries(lakes, float(shore_buffer_m))
    if buffered.empty:
        return np.zeros(latitude.shape, dtype=bool)
    geometry = union_all(buffered.geometry.values)
    geolocated = np.isfinite(latitude) & np.isfinite(longitude)
    result = np.zeros(latitude.shape, dtype=bool)
    result[geolocated] = contains_xy(geometry, longitude[geolocated], latitude[geolocated])
    return result


def selected_lake_bbox(lake: gpd.GeoDataFrame | None, pad_fraction: float = 0.12):
    if lake is None or lake.empty:
        return None
    west, south, east, north = map(float, lake.total_bounds)
    dx, dy = max(east - west, 0.01), max(north - south, 0.01)
    return (west - dx * pad_fraction, south - dy * pad_fraction,
            east + dx * pad_fraction, north + dy * pad_fraction)


Python PROJ/Rasterio sanity check passed (EPSG:4326); active database: /home/myid/mc16405/miniconda3/envs/s3olci-batch/lib/python3.11/site-packages/rasterio/proj_data/proj.db


In [19]:
def ci_family(
    r620: np.ndarray, r665: np.ndarray, r681: np.ndarray, r709: np.ndarray,
    wavelengths: tuple[float, float, float, float] = (620.0, 665.0, 681.0, 709.0),
    detection_limit: float = CI_DETECTION_LIMIT, ss681_max: float = 0.0, ss665_min: float = 0.0,
) -> dict[str, np.ndarray]:
    """Two-stage CI family with explicit, tunable spectral gates."""
    r620, r665, r681, r709 = [
        np.asarray(array, dtype=np.float32) for array in (r620, r665, r681, r709)
    ]
    w620, w665, w681, w709 = map(float, wavelengths)
    finite = np.isfinite(r620) & np.isfinite(r665) & np.isfinite(r681) & np.isfinite(r709)
    ss681 = r681 - r665 - (r709-r665) * ((w681-w665)/(w709-w665))
    ss665 = r665 - r620 - (r681-r620) * ((w665-w620)/(w681-w620))
    ss681 = np.where(finite, ss681, np.nan).astype(np.float32)
    ss665 = np.where(finite, ss665, np.nan).astype(np.float32)
    candidate = finite & (ss681 < float(ss681_max))
    ci = np.where(candidate, -ss681, np.nan).astype(np.float32)
    detection = candidate & (ss665 > float(ss665_min)) & (ci > float(detection_limit))
    return {"SS_681": ss681, "SS_665": ss665, "CI": ci,
            "CI_cyano": np.where(detection, ci, np.nan).astype(np.float32),
            "finite": finite, "ci_candidate": candidate, "ci_detection": detection}


def _scene_mask_context(
    l2_path: Path,
    qa_flags: Iterable[str],
    land_adjacency_pixels: int,
    water_definition: str,
    lakes: gpd.GeoDataFrame | None,
    shore_buffer_m: float,
    bright_screen: bool,
    bright_rhos865: float,
) -> dict[str, Any]:
    """Build masks without changing the native L2Gen file."""
    water_definition = str(water_definition).lower()
    with netCDF4.Dataset(l2_path) as src:
        geo = src.groups["geophysical_data"]
        nav = src.groups["navigation_data"]
        latitude = to_float(nav.variables["latitude"][:])
        longitude = to_float(nav.variables["longitude"][:])
        flags_var = locate_l2_flags(src)
        flags_ma = np.ma.asarray(flags_var[:])
        flags_valid = ~np.ma.getmaskarray(flags_ma)
        flags = flags_ma.filled(0).astype(np.uint32)
        flag_map = decode_flag_metadata(flags_var)
        if "LAND" not in flag_map:
            raise RuntimeError("l2_flags metadata lacks LAND")
        geolocated = (
            np.isfinite(latitude) & np.isfinite(longitude)
            & (np.abs(latitude) <= 90) & (np.abs(longitude) <= 180)
            & flags_valid
        )
        land = flag_is_set(flags, flag_map, "LAND")
        l2_water = geolocated & ~land
        hydro_water = hydrolakes_point_mask(latitude, longitude, lakes, shore_buffer_m)

        if water_definition == "l2_land":
            water = l2_water
            resolved_water_definition = "inverse L2 LAND flag"
        elif water_definition == "hydrolakes":
            if hydro_water is None:
                raise ValueError("HydroLAKES water was requested but no scene lakes are loaded")
            water = geolocated & hydro_water
            resolved_water_definition = "HydroLAKES polygons"
        elif water_definition == "intersection":
            if hydro_water is None:
                raise ValueError("Intersection water was requested but no scene lakes are loaded")
            water = l2_water & hydro_water
            resolved_water_definition = "HydroLAKES ∩ inverse L2 LAND"
        elif water_definition == "union":
            if hydro_water is None:
                water = l2_water
                resolved_water_definition = "inverse L2 LAND flag (HydroLAKES not loaded)"
            else:
                water = l2_water | (geolocated & hydro_water)
                resolved_water_definition = "HydroLAKES ∪ inverse L2 LAND"
        elif water_definition == "hydrolakes_if_loaded":
            if hydro_water is None:
                water = l2_water
                resolved_water_definition = "inverse L2 LAND flag (HydroLAKES not loaded)"
            else:
                water = geolocated & hydro_water
                resolved_water_definition = "HydroLAKES polygons"
        else:
            raise ValueError(
                "water_definition must be l2_land, hydrolakes, intersection, union, "
                "or hydrolakes_if_loaded"
            )

        requested = tuple(dict.fromkeys(str(name).upper() for name in qa_flags))
        flag_excluded = np.zeros(latitude.shape, dtype=bool)
        missing_flags = []
        for name in requested:
            if name not in flag_map:
                missing_flags.append(name)
            flag_excluded |= flag_is_set(flags, flag_map, name)
        if int(land_adjacency_pixels) > 0:
            adjacent = binary_dilation(land, iterations=int(land_adjacency_pixels)) & ~land
        else:
            adjacent = np.zeros(latitude.shape, dtype=bool)

        bright = np.zeros(latitude.shape, dtype=bool)
        bright_name = None
        if bright_screen:
            try:
                bright_name = nearest_product(geo.variables, "rhos", 865, tolerance_nm=25)
                rhos865 = to_float(geo.variables[bright_name][:])
                bright = np.isfinite(rhos865) & (rhos865 > float(bright_rhos865))
            except KeyError as error:
                raise RuntimeError(
                    "Bright-pixel screen requested, but no rhos band near 865 nm exists"
                ) from error
        qa_excluded = flag_excluded | adjacent | bright
        valid_water = water & ~qa_excluded
        return {
            "latitude": latitude, "longitude": longitude,
            "geolocated": geolocated, "flags": flags, "flag_map": flag_map,
            "land": land, "l2_water": l2_water, "hydrolakes_water": hydro_water,
            "water": water, "valid_water": valid_water,
            "flag_excluded": flag_excluded, "adjacent": adjacent,
            "bright": bright, "qa_excluded": qa_excluded,
            "requested_flags": requested, "missing_flags": missing_flags,
            "water_definition_requested": water_definition,
            "water_definition_resolved": resolved_water_definition,
            "shore_buffer_m": float(shore_buffer_m),
            "bright_screen": bool(bright_screen),
            "bright_rhos865": float(bright_rhos865),
            "bright_product": bright_name,
        }


def diagnose_l2_mask(
    l2_path: Path,
    ndci_source: str = NDCI_SOURCE,
    detection_limit: float = CI_DETECTION_LIMIT,
    ss681_max: float = 0.0,
    ss665_min: float = 0.0,
    qa_flags: Iterable[str] = QA_EXCLUDE_FLAGS,
    land_adjacency_pixels: int = LAND_ADJACENCY_PIXELS,
    water_definition: str = DEFAULT_WATER_DEFINITION,
    lakes: gpd.GeoDataFrame | None = None,
    shore_buffer_m: float = DEFAULT_SHORE_BUFFER_M,
    bright_screen: bool = DEFAULT_BRIGHT_SCREEN,
    bright_rhos865: float = DEFAULT_BRIGHT_RHOS865,
) -> dict[str, Any]:
    """Audit exactly where pixels are lost, entirely after L2Gen."""
    context = _scene_mask_context(
        Path(l2_path), qa_flags, land_adjacency_pixels, water_definition,
        lakes, shore_buffer_m, bright_screen, bright_rhos865,
    )
    with netCDF4.Dataset(l2_path) as src:
        geo = src.groups["geophysical_data"]
        names = {
            wave: nearest_product(geo.variables, "rhos", wave)
            for wave in CORE_RHOS_WAVELENGTHS
        }
        core = {wave: to_float(geo.variables[name][:]) for wave, name in names.items()}
        finite_core = np.logical_and.reduce([np.isfinite(array) for array in core.values()])
        family = ci_family(
            core[620], core[665], core[681], core[709],
            wavelengths=tuple(product_wavelength(names[w]) for w in CORE_RHOS_WAVELENGTHS),
            detection_limit=detection_limit, ss681_max=ss681_max, ss665_min=ss665_min,
        )
        prefix = "rhos" if ndci_source.lower() == "rhos" else "Rrs"
        red_name = nearest_product(geo.variables, prefix, 665)
        edge_name = nearest_product(geo.variables, prefix, 709)
        _, ndci_finite = ndci_index(
            to_float(geo.variables[red_name][:]), to_float(geo.variables[edge_name][:])
        )
    water = context["water"]
    retained = context["valid_water"]
    ci_valid = retained & finite_core
    stages = {
        "total_pixels": int(water.size),
        "geolocated_pixels": int(context["geolocated"].sum()),
        "water_pixels": int(water.sum()),
        "after_selected_flags": int((water & ~context["flag_excluded"]).sum()),
        "after_land_adjacency": int((water & ~context["flag_excluded"] & ~context["adjacent"]).sum()),
        "valid_water_pixels": int(retained.sum()),
        "ci_finite_before_qa": int((water & finite_core).sum()),
        "ci_finite_all_geolocated": int((context["geolocated"] & finite_core).sum()),
        "ci_candidates_all_geolocated": int((context["geolocated"] & family["ci_candidate"]).sum()),
        "ci_detections_all_geolocated": int((context["geolocated"] & family["ci_detection"]).sum()),
        "ci_finite_after_qa": int(ci_valid.sum()),
        "ci_candidates_after_qa": int((ci_valid & family["ci_candidate"]).sum()),
        "ci_detections_after_qa": int((ci_valid & family["ci_detection"]).sum()),
        "ndci_valid_before_qa": int((water & ndci_finite).sum()),
        "ndci_valid_all_geolocated": int((context["geolocated"] & ndci_finite).sum()),
        "ndci_valid_after_qa": int((retained & ndci_finite).sum()),
    }
    water_count = max(stages["water_pixels"], 1)
    details = {}
    for name in sorted(context["flag_map"]):
        flagged = water & flag_is_set(context["flags"], context["flag_map"], name)
        details[name] = {
            "selected_for_exclusion": name in context["requested_flags"],
            "water_pixel_count": int(flagged.sum()),
            "percent_of_water": 100.0 * float(flagged.sum()) / water_count,
        }
    messages = [
        f"Water definition resolved to: {context['water_definition_resolved']}.",
        "The native L2 NetCDF was not changed; every exclusion shown here is post-L2.",
    ]
    if bright_screen:
        messages.append(
            f"Bright screen removed {int((water & context['bright']).sum()):,} water pixels "
            f"using {context['bright_product']} > {bright_rhos865:g}. Disable it when testing surface scums."
        )
    if stages["ci_finite_before_qa"] == 0:
        messages.append("No selected-water pixels have four finite CI rhos inputs; inspect L2 rhos coverage.")
    elif stages["ci_finite_after_qa"] == 0:
        messages.append("Finite CI inputs exist before QA, so the selected post-mask is the cause of total loss.")
    ci_candidates = stages["ci_candidates_after_qa"]
    ci_detections = stages["ci_detections_after_qa"]
    ss665_rejections = int((ci_valid & family["ci_candidate"] & ~(family["SS_665"] > float(ss665_min))).sum())
    threshold_rejections = int((ci_valid & family["ci_candidate"]
        & (family["SS_665"] > float(ss665_min))
        & ~(family["CI"] > float(detection_limit))).sum())
    if ci_candidates and not ci_detections:
        messages.append(
            f"The SS681 test found {ci_candidates:,} CI candidate pixels, but none passed both "
            f"SS665>{float(ss665_min):g} and CI>{float(detection_limit):g}. SS665 rejected "
            f"{ss665_rejections:,}; the CI detection limit rejected {threshold_rejections:,}. "
            "This is an algorithmic non-detection, not a QA/water-mask loss; inspect CI, SS_665 and CI_exclusion_reason."
        )
    elif ci_candidates and ss665_rejections:
        messages.append(
            f"The SS665>{float(ss665_min):g} confirmation gate rejected {ss665_rejections:,} of "
            f"{ci_candidates:,} retained CI candidates."
        )
    finite_shape = ci_valid & family["finite"]
    def shape_quantiles(array):
        values = array[finite_shape & np.isfinite(array)]
        return ({str(q): float(np.nanpercentile(values, q)) for q in (1, 5, 25, 50, 75, 95, 99)}
                if values.size else {})
    return {
        "created_utc": utc_now(), "source_l2": str(l2_path),
        "water_definition_requested": water_definition,
        "water_definition_resolved": context["water_definition_resolved"],
        "shore_buffer_m": float(shore_buffer_m),
        "bright_screen": bool(bright_screen),
        "bright_rhos865": float(bright_rhos865),
        "qa_flags_requested": list(context["requested_flags"]),
        "qa_flags_missing": context["missing_flags"],
        "land_adjacency_pixels": int(land_adjacency_pixels),
        "spectral_gate_settings": {"ss681_max": float(ss681_max),
                                   "ss665_min": float(ss665_min),
                                   "ci_detection_limit": float(detection_limit)},
        "spectral_shape_audit": {
            "SS681_quantiles_retained_finite": shape_quantiles(family["SS_681"]),
            "SS665_quantiles_retained_finite": shape_quantiles(family["SS_665"]),
            "ss665_rejected_ci_candidates": ss665_rejections,
            "ci_threshold_rejected_candidates": threshold_rejections,
        },
        "stages": stages, "flag_details": details, "messages": messages,
        "rhos_product_mapping": names,
        "ndci_product_mapping": {665: red_name, 709: edge_name},
        "post_mask_counts": {
            "selected_flag": int((water & context["flag_excluded"]).sum()),
            "land_adjacency": int((water & context["adjacent"]).sum()),
            "bright": int((water & context["bright"]).sum()),
        },
    }


def build_indices_netcdf(
    l2_path: Path,
    ndci_source: str = NDCI_SOURCE,
    detection_limit: float = CI_DETECTION_LIMIT,
    ss681_max: float = 0.0,
    ss665_min: float = 0.0,
    qa_flags: Iterable[str] = QA_EXCLUDE_FLAGS,
    land_adjacency_pixels: int = LAND_ADJACENCY_PIXELS,
    mask_profile: str | None = DEFAULT_MASK_PROFILE,
    destination: Path | None = None,
    rows_per_block: int = ROW_BLOCK_SIZE,
    water_definition: str = DEFAULT_WATER_DEFINITION,
    lakes: gpd.GeoDataFrame | None = None,
    shore_buffer_m: float = DEFAULT_SHORE_BUFFER_M,
    bright_screen: bool = DEFAULT_BRIGHT_SCREEN,
    bright_rhos865: float = DEFAULT_BRIGHT_RHOS865,
) -> tuple[Path, dict[str, Any]]:
    """Create a native-swath, selected-water index cube using post-L2 masks."""
    l2_path = Path(l2_path).expanduser().resolve()
    prefix = "rhos" if ndci_source.lower() == "rhos" else "Rrs"
    destination = Path(destination or (DERIVED_DIR / f"{l2_path.stem}_CI_CIcyano_NDCI_water.nc"))
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".tmp")
    temporary.unlink(missing_ok=True)
    context = _scene_mask_context(
        l2_path, qa_flags, land_adjacency_pixels, water_definition, lakes,
        shore_buffer_m, bright_screen, bright_rhos865,
    )
    summary: dict[str, Any] = {
        "created_utc": utc_now(), "source_l2": str(l2_path),
        "ndci_source": prefix, "ci_detection_limit": float(detection_limit),
        "ci_detection_operator": ">",
        "ss681_max": float(ss681_max), "ss665_min": float(ss665_min),
        "mask_profile": mask_profile or "custom",
        "water_definition_requested": water_definition,
        "water_definition_resolved": context["water_definition_resolved"],
        "shore_buffer_m": float(shore_buffer_m),
        "qa_flags_requested": list(context["requested_flags"]),
        "qa_flags_missing": context["missing_flags"],
        "land_adjacency_pixels": int(land_adjacency_pixels),
        "bright_screen": bool(bright_screen),
        "bright_rhos865_threshold": float(bright_rhos865),
        "bright_rhos865_product": context["bright_product"],
        "water_pixels": int(context["water"].sum()),
        "valid_water_pixels": int(context["valid_water"].sum()),
        "qa_excluded_pixels": int((context["water"] & context["qa_excluded"]).sum()),
        "bright_excluded_pixels": int((context["water"] & context["bright"]).sum()),
        "ci_valid_pixels": 0, "ci_candidates": 0, "ci_detections": 0,
        "ndci_valid_pixels": 0, "negative_core_rhos_pixels": 0,
    }
    with netCDF4.Dataset(l2_path) as src:
        geo = src.groups["geophysical_data"]
        nav = src.groups["navigation_data"]
        variables = list(geo.variables)
        rhos_names = {wave: nearest_product(variables, "rhos", wave) for wave in CORE_RHOS_WAVELENGTHS}
        ndci_names = {wave: nearest_product(variables, prefix, wave) for wave in (665, 709)}
        actual_waves = tuple(product_wavelength(rhos_names[w]) for w in CORE_RHOS_WAVELENGTHS)
        lat_var = nav.variables["latitude"]
        shape, dimensions = lat_var.shape, lat_var.dimensions
        chunks = (min(rows_per_block, shape[0]), min(512, shape[1]))
        with netCDF4.Dataset(temporary, "w", format="NETCDF4") as dst:
            for dimension in dimensions:
                src_dim = src.dimensions[dimension]
                dst.createDimension(dimension, None if src_dim.isunlimited() else len(src_dim))
            dst.setncatts({
                "Conventions": "CF-1.8 ACDD-1.3",
                "title": "L2Gen-derived OLCI lake-water CI, CIcyano, and NDCI",
                "source_l2": l2_path.name, "created_utc": utc_now(),
                "reflectance_for_CI": "L2Gen rhos; Rayleigh-corrected surface reflectance; dimensionless",
                "ndci_reflectance_source": prefix,
                "algorithm_scope": "SS681 plus SS665 gates and strict CI>0.0001 detection limit; research reconstruction, not operational NASA CyAN",
                "ss681_candidate_max": float(ss681_max),
                "ss665_confirmation_min": float(ss665_min),
                "ci_detection_limit": float(detection_limit),
                "ci_detection_operator": ">",
                "non_detection_semantics": "CI and CI_cyano valid non-detections are fill/NaN; candidate and detection masks record presence/absence",
                "water_definition": context["water_definition_resolved"],
                "shore_buffer_m": float(shore_buffer_m),
                "mask_profile": mask_profile or "custom",
                "qa_excluded_flags": " ".join(context["requested_flags"]),
                "land_adjacency_pixels": int(land_adjacency_pixels),
                "bright_screen_enabled": int(bool(bright_screen)),
                "bright_rhos865_threshold": float(bright_rhos865),
                "l2_internal_output_masks": "disabled during v5 L2Gen call; conditions retained in l2_flags",
                "core_band_mapping_json": json.dumps(rhos_names),
                "ndci_band_mapping_json": json.dumps(ndci_names),
            })
            lon_out = _create_float_variable(dst, "longitude", dimensions, chunks, {
                "standard_name": "longitude", "units": "degrees_east", "coordinates": "latitude longitude",
            })
            lat_out = _create_float_variable(dst, "latitude", dimensions, chunks, {
                "standard_name": "latitude", "units": "degrees_north", "coordinates": "latitude longitude",
            })
            flags_out = dst.createVariable("l2_flags", "u4", dimensions, zlib=True,
                                           complevel=NETCDF_DEFLATE, shuffle=True, chunksizes=chunks)
            flags_out.setncatts({
                "long_name": "source L2Gen flags", "flag_masks": np.asarray(list(context["flag_map"].values()), dtype=np.uint32),
                "flag_meanings": " ".join(context["flag_map"].keys()), "coordinates": "latitude longitude",
            })

            def mask_var(name: str, long_name: str):
                variable = dst.createVariable(name, "u1", dimensions, zlib=True,
                    complevel=NETCDF_DEFLATE, shuffle=True, chunksizes=chunks,
                    fill_value=np.uint8(255))
                variable.setncatts({"long_name": long_name,
                    "flag_values": np.array([0, 1], dtype=np.uint8),
                    "flag_meanings": "false true", "coordinates": "latitude longitude"})
                return variable

            water_out = mask_var("water_mask", "water under the selected post-L2 definition")
            l2_water_out = mask_var("l2_water_mask", "geolocated inverse-LAND L2 mask")
            hydro_out = mask_var("hydrolakes_water_mask", "HydroLAKES polygon water after shore buffer")
            valid_out = mask_var("valid_water_mask", "selected water retained after post-L2 QA")
            qa_out = mask_var("qa_excluded_mask", "selected water excluded by post-L2 tests")
            adjacent_out = mask_var("land_adjacency_mask", "non-land pixels in the configured LAND dilation")
            bright_out = mask_var("bright_pixel_mask", "pixels exceeding the optional rhos865 threshold")
            ci_valid_out = mask_var("ci_valid_mask", "retained water with four finite CI rhos inputs")
            ci_candidate_out = mask_var("ci_candidate_mask", "negative-SS681 CI candidate")
            ci_detect_out = mask_var("ci_detection_mask", "CIcyano detection after SS681, SS665, and CI detection-limit gates")
            ndci_valid_out = mask_var("ndci_valid_mask", "retained water with finite NDCI")

            def reason_var(name: str, long_name: str, meanings: str):
                variable = dst.createVariable(name, "u1", dimensions, zlib=True,
                    complevel=NETCDF_DEFLATE, shuffle=True, chunksizes=chunks,
                    fill_value=np.uint8(255))
                variable.setncatts({"long_name": long_name,
                    "flag_values": np.arange(len(meanings.split())),
                    "flag_meanings": meanings, "coordinates": "latitude longitude"})
                return variable

            ci_reason_out = reason_var(
                "CI_exclusion_reason", "exclusive reason for CIcyano display outcome",
                "not_geolocated outside_selected_water post_mask_excluded nonfinite_core_rhos "
                "SS681_gate_failed SS665_gate_failed below_CI_threshold detected",
            )
            ndci_reason_out = reason_var(
                "NDCI_exclusion_reason", "exclusive reason for NDCI display outcome",
                "not_geolocated outside_selected_water post_mask_excluded red_nonfinite "
                "red_edge_nonfinite denominator_invalid valid",
            )
            negative_out = dst.createVariable("negative_core_rhos_count", "u1", dimensions,
                zlib=True, complevel=NETCDF_DEFLATE, shuffle=True, chunksizes=chunks,
                fill_value=np.uint8(255))
            negative_out.setncatts({"long_name": "negative values among rhos 620/665/681/709",
                                    "valid_range": np.array([0, 4], dtype=np.uint8)})
            ss681_out = _create_float_variable(dst, "SS_681", dimensions, chunks,
                {"long_name": "spectral shape near 681 nm", "units": "1"})
            ss665_out = _create_float_variable(dst, "SS_665", dimensions, chunks,
                {"long_name": "spectral shape near 665 nm", "units": "1"})
            ci_out = _create_float_variable(dst, "CI", dimensions, chunks,
                {"long_name": "detected-only CI before SS665 gate", "units": "1",
                 "algorithm": f"-SS_681 where SS_681<{float(ss681_max):g}; non-detections are fill/NaN"})
            cyano_out = _create_float_variable(dst, "CI_cyano", dimensions, chunks,
                {"long_name": "detected-only two-stage cyanobacteria index", "units": "1",
                 "algorithm": f"CI where SS_681<{float(ss681_max):g}, SS_665>{float(ss665_min):g}, and CI>threshold; standard threshold=0.0001; non-detections are fill/NaN"})
            ci_pre_out = _create_float_variable(dst, "CI_pre_mask", dimensions, chunks,
                {"long_name": "CI before the selected water and QA masks", "units": "1",
                 "algorithm": "-SS_681 where finite and SS_681<0; all geolocated pixels"})
            cyano_pre_out = _create_float_variable(dst, "CI_cyano_pre_mask", dimensions, chunks,
                {"long_name": "CIcyano before the selected water and QA masks", "units": "1",
                 "algorithm": "two-stage spectral gates on all finite geolocated pixels"})
            ndci_out = _create_float_variable(dst, "NDCI", dimensions, chunks,
                {"long_name": "normalized difference chlorophyll index", "units": "1",
                 "formula": f"({prefix}_709-{prefix}_665)/({prefix}_709+{prefix}_665)"})
            ndci_pre_out = _create_float_variable(dst, "NDCI_pre_mask", dimensions, chunks,
                {"long_name": "NDCI before the selected water and QA masks", "units": "1",
                 "formula": f"({prefix}_709-{prefix}_665)/({prefix}_709+{prefix}_665)"})

            geolocated = context["geolocated"]
            lon_out[:] = np.where(geolocated, context["longitude"], DERIVED_FILL)
            lat_out[:] = np.where(geolocated, context["latitude"], DERIVED_FILL)
            flags_out[:] = context["flags"]
            water_out[:] = context["water"].astype(np.uint8)
            l2_water_out[:] = context["l2_water"].astype(np.uint8)
            if context["hydrolakes_water"] is None:
                hydro_out[:] = np.full(shape, np.uint8(255), dtype=np.uint8)
            else:
                hydro_out[:] = context["hydrolakes_water"].astype(np.uint8)
            valid_out[:] = context["valid_water"].astype(np.uint8)
            qa_out[:] = (context["water"] & context["qa_excluded"]).astype(np.uint8)
            adjacent_out[:] = context["adjacent"].astype(np.uint8)
            bright_out[:] = context["bright"].astype(np.uint8)

            for start in tqdm(range(0, shape[0], rows_per_block), desc="v5 derived row blocks"):
                stop = min(shape[0], start + rows_per_block)
                rs = slice(start, stop)
                arrays = {wave: to_float(geo.variables[name][rs, :]) for wave, name in rhos_names.items()}
                family = ci_family(arrays[620], arrays[665], arrays[681], arrays[709],
                                   wavelengths=actual_waves, detection_limit=detection_limit,
                                   ss681_max=ss681_max, ss665_min=ss665_min)
                ci_valid = context["valid_water"][rs, :] & family["finite"]
                candidate = ci_valid & family["ci_candidate"]
                detected = ci_valid & family["ci_detection"]
                negative_count = sum((array < 0).astype(np.uint8) for array in arrays.values())
                red = to_float(geo.variables[ndci_names[665]][rs, :])
                edge = to_float(geo.variables[ndci_names[709]][rs, :])
                ndci, ndci_finite = ndci_index(red, edge)
                ndci_valid = context["valid_water"][rs, :] & ndci_finite
                local_geo = context["geolocated"][rs, :]
                local_water = context["water"][rs, :]
                local_qa = context["qa_excluded"][rs, :]
                ci_reason = np.zeros(ci_valid.shape, dtype=np.uint8)
                ci_reason[local_geo & ~local_water] = 1
                ci_reason[local_water & local_qa] = 2
                eligible_ci = local_water & ~local_qa
                ci_reason[eligible_ci & ~family["finite"]] = 3
                ci_reason[eligible_ci & family["finite"] & ~(family["SS_681"] < float(ss681_max))] = 4
                ci_reason[eligible_ci & family["ci_candidate"] & ~(family["SS_665"] > float(ss665_min))] = 5
                ci_reason[eligible_ci & family["ci_candidate"] & (family["SS_665"] > float(ss665_min))
                          & ~(family["CI"] > float(detection_limit))] = 6
                ci_reason[eligible_ci & family["ci_detection"]] = 7
                ndci_reason = np.zeros(ndci_valid.shape, dtype=np.uint8)
                ndci_reason[local_geo & ~local_water] = 1
                ndci_reason[local_water & local_qa] = 2
                eligible_ndci = local_water & ~local_qa
                ndci_reason[eligible_ndci & ~np.isfinite(red)] = 3
                ndci_reason[eligible_ndci & np.isfinite(red) & ~np.isfinite(edge)] = 4
                ndci_reason[eligible_ndci & np.isfinite(red) & np.isfinite(edge)
                            & ~(np.abs(red + edge) > 1e-8)] = 5
                ndci_reason[eligible_ndci & ndci_finite] = 6
                ci_valid_out[rs, :] = ci_valid.astype(np.uint8)
                ci_candidate_out[rs, :] = np.where(ci_valid, candidate.astype(np.uint8), np.uint8(255))
                ci_detect_out[rs, :] = np.where(ci_valid, detected.astype(np.uint8), np.uint8(255))
                ndci_valid_out[rs, :] = ndci_valid.astype(np.uint8)
                ci_reason_out[rs, :] = ci_reason
                ndci_reason_out[rs, :] = ndci_reason
                negative_out[rs, :] = np.where(ci_valid, negative_count, np.uint8(255))
                ss681_out[rs, :] = np.where(ci_valid, family["SS_681"], DERIVED_FILL)
                ss665_out[rs, :] = np.where(ci_valid, family["SS_665"], DERIVED_FILL)
                ci_out[rs, :] = np.where(candidate, family["CI"], DERIVED_FILL)
                cyano_out[rs, :] = np.where(detected, family["CI_cyano"], DERIVED_FILL)
                ci_pre_out[rs, :] = np.where(local_geo & family["ci_candidate"], family["CI"], DERIVED_FILL)
                cyano_pre_out[rs, :] = np.where(local_geo & family["ci_detection"], family["CI_cyano"], DERIVED_FILL)
                ndci_out[rs, :] = np.where(ndci_valid, ndci, DERIVED_FILL)
                ndci_pre_out[rs, :] = np.where(local_geo & ndci_finite, ndci, DERIVED_FILL)
                summary["ci_valid_pixels"] += int(ci_valid.sum())
                summary["ci_candidates"] += int(candidate.sum())
                summary["ci_detections"] += int(detected.sum())
                summary["ndci_valid_pixels"] += int(ndci_valid.sum())
                summary["negative_core_rhos_pixels"] += int((ci_valid & (negative_count > 0)).sum())
    temporary.replace(destination)
    summary.update(validate_derived_file(destination))
    summary["size_human"] = human_size(destination.stat().st_size)
    summary["ci_detection_fraction_ci_valid"] = (
        summary["ci_detections"] / summary["ci_valid_pixels"] if summary["ci_valid_pixels"] else None
    )
    write_json(EXPORT_DIR / l2_path.stem / "derived_index_summary.json", summary)
    print(f"Derived product: {destination} ({summary['size_human']})")
    return destination, summary


In [20]:
# Accessible publication defaults; users retain explicit vmin/vmax control.
LAYER_STYLE.update({
    "CI": ("magma", (0.0, 0.003), "CI (dimensionless)"),
    "CI_cyano": ("cividis", (0.0001, 0.003), r"CI$_{cyano}$ detection (>0.0001)"),
    "NDCI": ("BrBG", (-0.25, 0.5), "NDCI (dimensionless)"),
})


def _target_crs(longitude: np.ndarray, latitude: np.ndarray) -> CRS:
    valid = np.isfinite(longitude) & np.isfinite(latitude)
    if not valid.any():
        raise RuntimeError("No finite geolocation is available for map projection")
    center_lon = float(np.nanmedian(longitude[valid]))
    center_lat = float(np.nanmedian(latitude[valid]))
    zone = int(np.clip(math.floor((center_lon + 180.0) / 6.0) + 1, 1, 60))
    epsg = (32600 if center_lat >= 0 else 32700) + zone
    return CRS.from_epsg(epsg)


def regular_area_definition(
    longitude: np.ndarray, latitude: np.ndarray,
    resolution_m: float = DEFAULT_MAP_RESOLUTION_M,
    max_cells: int = MAP_MAX_CELLS,
) -> tuple[pr_geometry.AreaDefinition, dict[str, Any]]:
    """Create a north-up square-pixel UTM grid and coarsen only for safety."""
    resolution_m = float(resolution_m)
    if resolution_m <= 0:
        raise ValueError("Map resolution must be positive")
    valid = (
        np.isfinite(longitude) & np.isfinite(latitude)
        & (np.abs(longitude) <= 180) & (np.abs(latitude) <= 90)
    )
    if not valid.any():
        raise RuntimeError("No geolocated source pixels are available")
    crs = _target_crs(longitude[valid], latitude[valid])
    transformer = Transformer.from_crs(4326, crs, always_xy=True)
    x, y = transformer.transform(longitude[valid], latitude[valid])
    xmin, xmax = float(np.nanmin(x)), float(np.nanmax(x))
    ymin, ymax = float(np.nanmin(y)), float(np.nanmax(y))
    pad = max(resolution_m, 0.002 * max(xmax - xmin, ymax - ymin))
    xmin, xmax, ymin, ymax = xmin - pad, xmax + pad, ymin - pad, ymax + pad
    width = max(1, int(math.ceil((xmax - xmin) / resolution_m)))
    height = max(1, int(math.ceil((ymax - ymin) / resolution_m)))
    effective = resolution_m
    if width * height > int(max_cells):
        factor = math.sqrt((width * height) / int(max_cells))
        effective *= factor
        width = max(1, int(math.ceil((xmax - xmin) / effective)))
        height = max(1, int(math.ceil((ymax - ymin) / effective)))
        print(
            f"Requested map grid exceeded {max_cells:,} cells; effective resolution "
            f"was increased to {effective:.1f} m for this display/export."
        )
    extent = (xmin, ymin, xmax, ymax)
    area = pr_geometry.AreaDefinition(
        "olci_regular_grid", "OLCI regular projected grid", "olci_utm",
        crs, width, height, extent,
    )
    return area, {
        "crs": crs, "crs_wkt": crs.to_wkt(), "epsg": crs.to_epsg(),
        "extent": extent, "width": width, "height": height,
        "requested_resolution_m": resolution_m,
        "effective_resolution_m": effective,
    }


def _resample_nearest(
    longitude: np.ndarray, latitude: np.ndarray, values: np.ndarray,
    area: pr_geometry.AreaDefinition, resolution_m: float,
) -> np.ndarray:
    geolocated = (
        np.isfinite(longitude) & np.isfinite(latitude)
        & (np.abs(longitude) <= 180) & (np.abs(latitude) <= 90)
    )
    swath = pr_geometry.SwathDefinition(
        lons=np.ma.array(longitude, mask=~geolocated),
        lats=np.ma.array(latitude, mask=~geolocated),
    )
    data = np.asarray(values)
    if data.shape[:2] != longitude.shape:
        raise ValueError(f"Data shape {data.shape} does not match geolocation {longitude.shape}")
    radius = max(600.0, 2.25 * float(resolution_m))
    if data.ndim == 2:
        return np.asarray(pr_kd_tree.resample_nearest(
            swath, data, area, radius_of_influence=radius,
            fill_value=np.nan, epsilon=0.2,
        ), dtype=np.float32)
    channels = [
        pr_kd_tree.resample_nearest(
            swath, data[..., band], area, radius_of_influence=radius,
            fill_value=np.nan, epsilon=0.2,
        )
        for band in range(data.shape[2])
    ]
    return np.dstack(channels).astype(np.float32)


def _rgb_stretch(rgb: np.ndarray) -> np.ndarray:
    output = np.zeros_like(rgb, dtype=np.float32)
    valid = np.all(np.isfinite(rgb), axis=2)
    for band in range(3):
        sample = rgb[..., band][valid]
        if sample.size:
            low, high = np.nanpercentile(sample, [1.0, 99.5])
        else:
            low, high = 0.0, 0.15
        if not np.isfinite(high) or high <= low:
            low, high = 0.0, 0.15
        output[..., band] = np.clip((rgb[..., band] - low) / (high - low), 0, 1)
    # A restrained gamma improves visual interpretation without altering source data.
    output = np.power(output, 0.82)
    output[~valid] = np.nan
    return output


def regular_scene_raster(
    derived_path: Path, l2_path: Path, layer: str,
    bbox: tuple[float, float, float, float] | None = None,
    resolution_m: float = DEFAULT_MAP_RESOLUTION_M,
) -> dict[str, Any]:
    """Resample a native swath layer to a regular projected display grid."""
    with netCDF4.Dataset(derived_path) as derived:
        lat_full = to_float(derived.variables["latitude"][:])
        lon_full = to_float(derived.variables["longitude"][:])
        row_slice, col_slice = bbox_slices(lat_full, lon_full, bbox, margin=8)
        latitude = lat_full[row_slice, col_slice]
        longitude = lon_full[row_slice, col_slice]
        if layer != "true_color":
            if layer not in derived.variables:
                raise KeyError(f"Layer not in derived NetCDF: {layer}")
            values = to_float(derived.variables[layer][row_slice, col_slice])
    area, metadata = regular_area_definition(longitude, latitude, resolution_m)
    if layer == "true_color":
        with netCDF4.Dataset(l2_path) as root:
            geo = root.groups["geophysical_data"]
            try:
                names = [nearest_product(geo.variables, "rhot", wave, 20) for wave in (665, 560, 443)]
                title = "Sentinel-3 OLCI top-of-atmosphere true colour"
                rgb_source = "rhot"
            except KeyError:
                names = [nearest_product(geo.variables, "rhos", wave, 20) for wave in (665, 560, 443)]
                title = "Sentinel-3 OLCI Rayleigh-corrected pseudo true colour"
                rgb_source = "rhos"
            values = np.dstack([
                to_float(geo.variables[name][row_slice, col_slice]) for name in names
            ])
        raster = _rgb_stretch(_resample_nearest(
            longitude, latitude, values, area, metadata["effective_resolution_m"]
        ))
    else:
        raster = _resample_nearest(
            longitude, latitude, values, area, metadata["effective_resolution_m"]
        )
        title, rgb_source, names = layer.replace("_", " "), None, None
    return {
        "layer": layer, "data": raster, "area": area, **metadata,
        "title": title, "rgb_source": rgb_source, "source_products": names,
        "bbox": bbox,
    }


def _project_lakes(lakes: gpd.GeoDataFrame | None, crs: CRS):
    if lakes is None or lakes.empty:
        return None
    return lakes.to_crs(crs)


def _add_scale_and_north(axis: plt.Axes, grid: dict[str, Any]) -> None:
    xmin, ymin, xmax, ymax = grid["extent"]
    width = xmax - xmin
    candidate = width * 0.20
    power = 10 ** math.floor(math.log10(max(candidate, 1)))
    scale = max(power, round(candidate / power) * power)
    x0, y0 = xmin + width * 0.06, ymin + (ymax - ymin) * 0.065
    axis.plot([x0, x0 + scale], [y0, y0], color="white", lw=5, solid_capstyle="butt",
              path_effects=[path_effects.Stroke(linewidth=7, foreground="black"), path_effects.Normal()])
    axis.text(x0 + scale / 2, y0 + (ymax - ymin) * 0.025, f"{scale/1000:g} km",
              ha="center", va="bottom", color="white", fontsize=9, weight="bold",
              path_effects=[path_effects.withStroke(linewidth=2.5, foreground="black")])
    axis.annotate("N", xy=(0.94, 0.91), xytext=(0.94, 0.79), xycoords="axes fraction",
                  ha="center", va="center", color="white", fontsize=11, weight="bold",
                  arrowprops=dict(facecolor="white", edgecolor="black", width=2.5, headwidth=9),
                  path_effects=[path_effects.withStroke(linewidth=2, foreground="black")])


def render_layer(
    ax: plt.Axes, derived_path: Path, l2_path: Path,
    layer: str, bbox: tuple[float, float, float, float] | None = None,
    max_points: int = 350_000, vmin: float | None = None,
    vmax: float | None = None, resolution_m: float = DEFAULT_MAP_RESOLUTION_M,
    lakes: gpd.GeoDataFrame | None = None, show_hydrolakes: bool = True,
) -> Any:
    """Draw a regular image raster; never scatter or pcolormesh the swath."""
    grid = regular_scene_raster(derived_path, l2_path, layer, bbox, resolution_m)
    data, extent = grid["data"], grid["extent"]
    image_extent = (extent[0], extent[2], extent[1], extent[3])
    if layer == "true_color":
        display_data = np.nan_to_num(data, nan=0.92)
        artist = ax.imshow(display_data, extent=image_extent, origin="upper",
                           interpolation="nearest", rasterized=True)
        ax.set_title(grid["title"], weight="bold")
    else:
        if layer in MASK_LAYER_STYLE or layer in {
            "l2_water_mask", "hydrolakes_water_mask", "bright_pixel_mask"
        }:
            cmap, label = MASK_LAYER_STYLE.get(layer, ("viridis", layer.replace("_", " ")))
            lower, upper = (0.0 if vmin is None else float(vmin),
                            1.0 if vmax is None else float(vmax))
        else:
            cmap, defaults, label = LAYER_STYLE[layer]
            lower, upper = robust_limits(data, defaults)
            lower = lower if vmin is None else float(vmin)
            upper = upper if vmax is None else float(vmax)
        cmap_obj = plt.get_cmap(cmap).copy()
        cmap_obj.set_bad((0.96, 0.96, 0.96, 0.0))
        artist = ax.imshow(data, extent=image_extent, origin="upper", interpolation="nearest",
                           cmap=cmap_obj, vmin=lower, vmax=upper, rasterized=True)
        ax.set_title(layer.replace("_", " "), weight="bold")
        plt.colorbar(artist, ax=ax, fraction=0.046, pad=0.025, label=label)
    projected = _project_lakes(lakes, grid["crs"])
    if show_hydrolakes and projected is not None:
        projected.boundary.plot(ax=ax, color="#00ffff", linewidth=0.8, alpha=0.95, zorder=5)
    ax.set_xlim(extent[0], extent[2]); ax.set_ylim(extent[1], extent[3])
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel(f"Easting (m; EPSG:{grid['epsg']})")
    ax.set_ylabel("Northing (m)")
    ax.grid(color="white", alpha=0.20, linewidth=0.5)
    _add_scale_and_north(ax, grid)
    ax.text(0.995, 0.006,
            f"nearest-neighbour • {grid['effective_resolution_m']:.0f} m grid",
            transform=ax.transAxes, ha="right", va="bottom", fontsize=7,
            color="white", path_effects=[path_effects.withStroke(linewidth=2, foreground="black")])
    ax._olci_grid = grid
    return artist


def plot_layer(
    derived_path: Path, l2_path: Path, layer: str = "CI_cyano",
    bbox: tuple[float, float, float, float] | None = None,
    vmin: float | None = None, vmax: float | None = None,
    figsize: tuple[float, float] = (10, 8),
    resolution_m: float = DEFAULT_MAP_RESOLUTION_M,
    lakes: gpd.GeoDataFrame | None = None,
) -> plt.Figure:
    figure, axis = plt.subplots(figsize=figsize, constrained_layout=True)
    render_layer(axis, derived_path, l2_path, layer, bbox, vmin=vmin, vmax=vmax,
                 resolution_m=resolution_m, lakes=lakes)
    return figure


def interactive_regular_map(
    derived_path: Path, l2_path: Path, layer: str = "true_color",
    bbox: tuple[float, float, float, float] | None = None,
    resolution_m: float = DEFAULT_MAP_RESOLUTION_M,
    lakes: gpd.GeoDataFrame | None = None,
) -> go.Figure:
    grid = regular_scene_raster(derived_path, l2_path, layer, bbox, resolution_m)
    xmin, ymin, xmax, ymax = grid["extent"]
    dx = (xmax - xmin) / grid["width"]
    dy = (ymax - ymin) / grid["height"]
    if layer == "true_color":
        image = np.uint8(np.clip(np.nan_to_num(grid["data"], nan=0.92), 0, 1) * 255)
        trace = go.Image(z=image, x0=xmin + dx / 2, dx=dx, y0=ymax - dy / 2, dy=-dy,
                         name="OLCI regular raster")
        figure = go.Figure(trace)
    else:
        data = grid["data"]
        x = xmin + dx * (np.arange(grid["width"]) + 0.5)
        y = ymax - dy * (np.arange(grid["height"]) + 0.5)
        if layer in LAYER_STYLE:
            _, defaults, label = LAYER_STYLE[layer]
            low, high = robust_limits(data, defaults)
        else:
            label, low, high = layer.replace("_", " "), 0, 1
        figure = go.Figure(go.Heatmap(
            z=data, x=x, y=y, colorscale="Viridis", zmin=low, zmax=high,
            colorbar=dict(title=label), hoverongaps=False,
        ))
    projected = _project_lakes(lakes, grid["crs"])
    if projected is not None:
        for geometry in projected.geometry:
            boundaries = [geometry] if geometry.geom_type == "LineString" else list(geometry.boundary.geoms) if hasattr(geometry.boundary, "geoms") else [geometry.boundary]
            for boundary in boundaries:
                xx, yy = boundary.xy
                figure.add_trace(go.Scatter(x=np.asarray(xx), y=np.asarray(yy), mode="lines",
                    line=dict(color="#00ffff", width=1.3), name="HydroLAKES",
                    hoverinfo="skip", showlegend=False))
    figure.update_layout(
        title=f"{layer.replace('_', ' ')} — regular {grid['effective_resolution_m']:.0f} m grid",
        template="plotly_white", height=700, margin=dict(l=55, r=20, t=55, b=50),
        dragmode="pan", uirevision="olci-v5-3",
    )
    figure.update_xaxes(title=f"Easting (m; EPSG:{grid['epsg']})", constrain="domain")
    figure.update_yaxes(title="Northing (m)", scaleanchor="x", scaleratio=1,
                        autorange=False, range=[ymin, ymax])
    return figure


def write_regular_geotiff(grid: dict[str, Any], path: Path) -> Path:
    """Write a cloud-optimized GeoTIFF when the rasterio COG driver is available."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    data = grid["data"]
    if data.ndim == 3:
        array = np.moveaxis(np.uint8(np.clip(np.nan_to_num(data, nan=0), 0, 1) * 255), -1, 0)
        dtype, nodata, count = "uint8", 0, array.shape[0]
    else:
        array = np.where(np.isfinite(data), data, DERIVED_FILL).astype(np.float32)[None, ...]
        dtype, nodata, count = "float32", float(DERIVED_FILL), 1
    transform = from_bounds(*grid["extent"], grid["width"], grid["height"])
    profile = dict(driver="COG", height=grid["height"], width=grid["width"], count=count,
                   dtype=dtype, crs=grid["crs"], transform=transform, nodata=nodata,
                   compress="DEFLATE", blocksize=512, overview_resampling="nearest")
    try:
        with rasterio.open(path, "w", **profile) as destination:
            destination.write(array)
            destination.update_tags(
                source_layer=grid["layer"], resampling="nearest",
                effective_resolution_m=grid["effective_resolution_m"],
            )
    except Exception:
        path.unlink(missing_ok=True)
        profile.update(driver="GTiff", tiled=True, blockxsize=512, blockysize=512)
        profile.pop("blocksize", None); profile.pop("overview_resampling", None)
        with rasterio.open(path, "w", **profile) as destination:
            destination.write(array)
            levels = [level for level in (2, 4, 8, 16) if min(grid["width"], grid["height"]) // level >= 128]
            if levels:
                destination.build_overviews(levels, rasterio.enums.Resampling.nearest)
    return path


def export_map_bundle(
    derived_path: Path, l2_path: Path,
    bbox: tuple[float, float, float, float] | None = None,
    dpi: int = 400, resolution_m: float = DEFAULT_MAP_RESOLUTION_M,
    lakes: gpd.GeoDataFrame | None = None, export_cogs: bool = True,
) -> dict[str, Path]:
    scene_dir = EXPORT_DIR / Path(l2_path).stem / "publication_maps"
    scene_dir.mkdir(parents=True, exist_ok=True)
    outputs: dict[str, Path] = {}
    layers = ("true_color", "CI", "CI_cyano", "NDCI")
    grids = {}
    for layer in layers:
        figure = plot_layer(derived_path, l2_path, layer, bbox,
                            resolution_m=resolution_m, lakes=lakes)
        png = scene_dir / f"{Path(l2_path).stem}_{layer}_{int(resolution_m)}m.png"
        pdf = png.with_suffix(".pdf")
        figure.savefig(png, dpi=int(dpi), facecolor="white")
        figure.savefig(pdf, dpi=int(dpi), facecolor="white")
        plt.close(figure)
        outputs[f"{layer}_png"] = png; outputs[f"{layer}_pdf"] = pdf
        if export_cogs:
            grid = regular_scene_raster(derived_path, l2_path, layer, bbox, resolution_m)
            grids[layer] = grid
            tif = write_regular_geotiff(grid, png.with_suffix(".tif"))
            outputs[f"{layer}_geotiff"] = tif
    combined, axes = plt.subplots(2, 2, figsize=(15.5, 12.5), constrained_layout=False,
                                  subplot_kw={"projection": ccrs.PlateCarree()})
    for axis, layer in zip(axes.ravel(), layers):
        render_layer(axis, derived_path, l2_path, layer, bbox,
                     resolution_m=resolution_m, lakes=lakes)
    combined.subplots_adjust(left=0.07, right=0.94, bottom=0.06, top=0.95,
                             wspace=0.22, hspace=0.20)
    combined_png = scene_dir / f"{Path(l2_path).stem}_four_panel_{int(resolution_m)}m.png"
    combined_pdf = combined_png.with_suffix(".pdf")
    combined.savefig(combined_png, dpi=int(dpi), facecolor="white")
    combined.savefig(combined_pdf, dpi=int(dpi), facecolor="white")
    plt.close(combined)
    outputs.update({"combined_png": combined_png, "combined_pdf": combined_pdf})
    metadata = {
        "created_utc": utc_now(), "source_l2": str(l2_path), "source_derived": str(derived_path),
        "requested_resolution_m": float(resolution_m), "resampling": "nearest-neighbour",
        "bbox": bbox, "dpi": int(dpi), "hydrolakes_overlay": bool(lakes is not None and not lakes.empty),
        "native_swath_statistics": True, "regular_grid_used_only_for_maps_and_GeoTIFF": True,
    }
    metadata_path = write_json(scene_dir / "publication_map_metadata.json", metadata)
    outputs["metadata_json"] = metadata_path
    print(f"Saved {len(outputs)} publication files under {scene_dir}")
    return outputs


def analyze_lakes(
    derived_path: Path, lakes: gpd.GeoDataFrame,
    shore_buffer_m: float = 0.0,
) -> pd.DataFrame:
    """Native-swath per-lake statistics; map resampling never enters these values."""
    if lakes is None or lakes.empty:
        raise ValueError("No scene HydroLAKES polygons are loaded")
    analysis_lakes = erode_lake_geometries(lakes, shore_buffer_m)
    with netCDF4.Dataset(derived_path) as ds:
        lat = to_float(ds.variables["latitude"][:]); lon = to_float(ds.variables["longitude"][:])
        retained = np.ma.asarray(ds.variables["valid_water_mask"][:]).filled(0) == 1
        ci_valid = np.ma.asarray(ds.variables["ci_valid_mask"][:]).filled(0) == 1
        detected = np.ma.asarray(ds.variables["ci_detection_mask"][:]).filled(0) == 1
        bright = np.ma.asarray(ds.variables["bright_pixel_mask"][:]).filled(0) == 1 if "bright_pixel_mask" in ds.variables else np.zeros(lat.shape, bool)
        ci = to_float(ds.variables["CI"][:]); cyano = to_float(ds.variables["CI_cyano"][:])
        ndci = to_float(ds.variables["NDCI"][:])
    records = []
    for _, lake in tqdm(analysis_lakes.iterrows(), total=len(analysis_lakes), desc="Lake statistics"):
        west, south, east, north = lake.geometry.bounds
        window = (lon >= west) & (lon <= east) & (lat >= south) & (lat <= north)
        rows, columns = np.where(window)
        if not rows.size:
            polygon = np.zeros(lat.shape, dtype=bool)
        else:
            rs = slice(rows.min(), rows.max() + 1); cs = slice(columns.min(), columns.max() + 1)
            local = contains_xy(lake.geometry, lon[rs, cs], lat[rs, cs])
            polygon = np.zeros(lat.shape, dtype=bool); polygon[rs, cs] = local
        kept = polygon & retained
        valid_ci = polygon & ci_valid
        hits = polygon & detected
        ci_values = ci[polygon & np.isfinite(ci)]
        cyano_values = cyano[hits & np.isfinite(cyano)]
        ndci_values = ndci[kept & np.isfinite(ndci)]
        def percentile(values, q):
            return float(np.nanpercentile(values, q)) if values.size else np.nan
        record = {
            "Hylak_id": int(lake["Hylak_id"]),
            "Lake_name": lake.get("Lake_name", "") or "",
            "Lake_area_km2": float(lake.get("Lake_area", np.nan)),
            "native_polygon_pixels": int(polygon.sum()),
            "retained_pixels": int(kept.sum()),
            "retained_percent": 100.0 * kept.sum() / polygon.sum() if polygon.any() else np.nan,
            "bright_screen_pixels": int((polygon & bright).sum()),
            "ci_input_valid_pixels": int(valid_ci.sum()),
            "ci_candidate_pixels": int((polygon & np.isfinite(ci)).sum()),
            "ci_cyano_detection_pixels": int(hits.sum()),
            "ci_cyano_detection_fraction_of_ci_valid": hits.sum() / valid_ci.sum() if valid_ci.any() else np.nan,
            "ci_cyano_detection_fraction_of_retained": hits.sum() / kept.sum() if kept.any() else np.nan,
            "CI_median_candidates": percentile(ci_values, 50),
            "CI_p10_candidates": percentile(ci_values, 10),
            "CI_p90_candidates": percentile(ci_values, 90),
            "CIcyano_median_detections": percentile(cyano_values, 50),
            "CIcyano_p10_detections": percentile(cyano_values, 10),
            "CIcyano_p90_detections": percentile(cyano_values, 90),
            "NDCI_median_retained": percentile(ndci_values, 50),
            "NDCI_p10_retained": percentile(ndci_values, 10),
            "NDCI_p90_retained": percentile(ndci_values, 90),
        }
        records.append(record)
    return pd.DataFrame(records).sort_values(["Lake_area_km2", "Hylak_id"], ascending=[False, True])


def export_lake_statistics(derived_path: Path, table: pd.DataFrame) -> Path:
    destination = EXPORT_DIR / Path(derived_path).stem / "hydrolakes_native_swath_statistics.csv"
    destination.parent.mkdir(parents=True, exist_ok=True)
    table.to_csv(destination, index=False)
    print(f"Lake statistics: {destination}")
    return destination

# v5.3 map engine: native swath -> regular geographic raster, Cartopy publication axes,
# and a standalone Folium/Leaflet latitude-longitude viewer.
LAYER_STYLE.update({
    "CI_pre_mask": ("magma", (0.0, 0.003), "CI before water/QA mask"),
    "CI_cyano_pre_mask": ("cividis", (0.0001, 0.003), r"CI$_{cyano}$ detection (>0.0001) before water/QA mask"),
    "NDCI_pre_mask": ("BrBG", (-0.25, 0.5), "NDCI before water/QA mask"),
    "rhos_620": ("viridis", (-0.02, 0.12), r"$\rho_s$(620)"),
    "rhos_665": ("viridis", (-0.02, 0.12), r"$\rho_s$(665)"),
    "rhos_681": ("viridis", (-0.02, 0.12), r"$\rho_s$(681)"),
    "rhos_709": ("viridis", (-0.02, 0.12), r"$\rho_s$(709)"),
    "rhos_865": ("viridis", (-0.02, 0.12), r"$\rho_s$(865)"),
    "CI_exclusion_reason": ("turbo", (0, 7), "CIcyano outcome reason (code)"),
    "NDCI_exclusion_reason": ("turbo", (0, 6), "NDCI outcome reason (code)"),
})


def _native_scene_layer(derived_path, l2_path, layer, bbox=None):
    with netCDF4.Dataset(derived_path) as derived:
        lat_full = to_float(derived.variables["latitude"][:])
        lon_full = to_float(derived.variables["longitude"][:])
        row_slice, col_slice = bbox_slices(lat_full, lon_full, bbox, margin=8)
        latitude = lat_full[row_slice, col_slice]
        longitude = lon_full[row_slice, col_slice]
        if layer not in {"true_color"} and not layer.startswith("rhos_"):
            if layer not in derived.variables:
                raise KeyError(f"Layer not in derived NetCDF: {layer}")
            return latitude, longitude, to_float(derived.variables[layer][row_slice, col_slice]), None
    with netCDF4.Dataset(l2_path) as root:
        geo = root.groups["geophysical_data"]
        if layer.startswith("rhos_"):
            wave = int(layer.split("_")[1])
            name = nearest_product(geo.variables, "rhos", wave, 25)
            values = to_float(geo.variables[name][row_slice, col_slice])
            return latitude, longitude, values, [name]
        try:
            names = [nearest_product(geo.variables, "rhot", wave, 20) for wave in (665, 560, 443)]
            source = "rhot"
        except KeyError:
            names = [nearest_product(geo.variables, "rhos", wave, 20) for wave in (665, 560, 443)]
            source = "rhos"
        values = np.dstack([to_float(geo.variables[name][row_slice, col_slice]) for name in names])
        return latitude, longitude, values, {"products": names, "source": source}


def regular_scene_raster(derived_path, l2_path, layer, bbox=None,
                         resolution_m=DEFAULT_MAP_RESOLUTION_M, max_cells=MAP_MAX_CELLS):
    latitude, longitude, values, source = _native_scene_layer(derived_path, l2_path, layer, bbox)
    area, metadata = regular_area_definition(longitude, latitude, resolution_m, max_cells=max_cells)
    raster = _resample_nearest(longitude, latitude, values, area, metadata["effective_resolution_m"])
    if layer == "true_color":
        raster = _rgb_stretch(raster)
        title = ("Sentinel-3 OLCI top-of-atmosphere true colour" if source["source"] == "rhot"
                 else "Sentinel-3 OLCI Rayleigh-corrected pseudo true colour")
    else:
        title = layer.replace("_", " ")
    return {"layer": layer, "data": raster, "area": area, **metadata,
            "title": title, "source_products": source, "bbox": bbox}


def _grid_lonlat_bounds(grid):
    xmin, ymin, xmax, ymax = grid["extent"]
    transformer = Transformer.from_crs(grid["crs"], 4326, always_xy=True)
    xs = np.array([xmin, xmax, xmax, xmin]); ys = np.array([ymin, ymin, ymax, ymax])
    lon, lat = transformer.transform(xs, ys)
    return float(np.min(lon)), float(np.min(lat)), float(np.max(lon)), float(np.max(lat))


def render_layer(ax, derived_path, l2_path, layer, bbox=None, max_points=350_000,
                 vmin=None, vmax=None, resolution_m=DEFAULT_MAP_RESOLUTION_M,
                 lakes=None, show_hydrolakes=True, show_context=False):
    """Geographic publication raster without a second Cartopy reprojection."""
    grid = regular_geographic_raster(
        derived_path, l2_path, layer, bbox, resolution_m, max_cells=MAP_MAX_CELLS
    )
    data = grid["data"]
    west, south, east, north = grid["extent"]
    kwargs = dict(extent=(west, east, south, north), origin="upper",
                  transform=ccrs.PlateCarree(), interpolation="nearest", rasterized=True)
    title = layer.replace("_", " ")
    if layer == "true_color":
        artist = ax.imshow(np.nan_to_num(data, nan=0.92), **kwargs)
        title = ("Sentinel-3 OLCI top-of-atmosphere true colour"
                 if grid["source_products"]["source"] == "rhot"
                 else "Sentinel-3 OLCI Rayleigh-corrected pseudo true colour")
    else:
        if layer in MASK_LAYER_STYLE or layer in {"l2_water_mask", "hydrolakes_water_mask",
                                                  "bright_pixel_mask"}:
            cmap, label = MASK_LAYER_STYLE.get(layer, ("viridis", layer.replace("_", " ")))
            lower, upper = 0 if vmin is None else float(vmin), 1 if vmax is None else float(vmax)
        else:
            cmap, defaults, label = LAYER_STYLE[layer]
            lower, upper = robust_limits(data, defaults)
            lower = lower if vmin is None else float(vmin)
            upper = upper if vmax is None else float(vmax)
        cmap_obj = plt.get_cmap(cmap).copy(); cmap_obj.set_bad((0.96, 0.96, 0.96, 0))
        if layer in {"CI", "CI_cyano", "NDCI"}:
            def companion(name):
                lat, lon, values, _ = _native_scene_layer(derived_path, l2_path, name, bbox)
                return _resample_nearest(lon, lat, values, grid["area"],
                                         grid["effective_resolution_m"])
            water = companion("water_mask") >= 0.5
            retained = companion("valid_water_mask") >= 0.5
            finite = np.isfinite(data)
            rgba = np.zeros(data.shape + (4,), dtype=np.float32)
            rgba[water & ~retained] = mcolors.to_rgba("#252525", 0.92)
            rgba[retained & ~finite] = mcolors.to_rgba("#c7c7c7", 0.92)
            rgba[finite] = cmap_obj(mcolors.Normalize(lower, upper, clip=True)(data[finite]))
            artist = ax.imshow(rgba, **kwargs)
            scalar = plt.cm.ScalarMappable(norm=mcolors.Normalize(lower, upper), cmap=cmap_obj)
            scalar.set_array([])
            plt.colorbar(scalar, ax=ax, fraction=0.043, pad=0.04, label=label)
            ax.legend(handles=[
                Patch(facecolor="#c7c7c7", label="Valid water: algorithm non-detection"),
                Patch(facecolor="#252525", label="Excluded by selected post-mask"),
            ], loc="lower left", fontsize=7, framealpha=0.88)
            title = f"{title} — detections coloured; non-detections grey"
        else:
            artist = ax.imshow(data, cmap=cmap_obj, vmin=lower, vmax=upper, **kwargs)
            plt.colorbar(artist, ax=ax, fraction=0.043, pad=0.04, label=label)
    ax.set_extent((west, east, south, north), crs=ccrs.PlateCarree())
    if show_context:
        try:
            ax.add_feature(cfeature.LAKES.with_scale("10m"), facecolor="none", edgecolor="#607d8b", linewidth=0.45)
            ax.add_feature(cfeature.COASTLINE.with_scale("10m"), edgecolor="#37474f", linewidth=0.55)
            ax.add_feature(cfeature.BORDERS.with_scale("10m"), edgecolor="#546e7a", linewidth=0.45)
            ax.add_feature(cfeature.STATES.with_scale("10m"), edgecolor="#78909c", linewidth=0.35)
        except Exception as error:
            print(f"Context boundaries unavailable: {error}")
    if show_hydrolakes and lakes is not None and not lakes.empty:
        for geometry in lakes.to_crs(4326).geometry:
            ax.add_geometries([geometry], ccrs.PlateCarree(), facecolor="none",
                              edgecolor="#00bcd4", linewidth=0.7, zorder=6)
    gridlines = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, linewidth=0.45,
                            color="#455a64", alpha=0.5, linestyle="--")
    gridlines.top_labels = False; gridlines.right_labels = False
    gridlines.xformatter = LONGITUDE_FORMATTER; gridlines.yformatter = LATITUDE_FORMATTER
    gridlines.xlabel_style = {"size": 8}; gridlines.ylabel_style = {"size": 8}
    ax.set_title(title, weight="bold", pad=10)
    ax.annotate("N", xy=(0.94, 0.93), xytext=(0.94, 0.81), xycoords="axes fraction",
                ha="center", weight="bold", arrowprops=dict(facecolor="white", edgecolor="black", width=2))
    ax.text(0.995, 0.006,
            f"nearest-neighbour • {grid['width']}×{grid['height']} cells • "
            f"{grid['effective_resolution_m']:.0f} m display grid",
            transform=ax.transAxes, ha="right", va="bottom", fontsize=7,
            color="white", path_effects=[path_effects.withStroke(linewidth=2, foreground="black")])
    ax._olci_grid = grid
    return artist


def plot_layer(derived_path, l2_path, layer="CI_cyano", bbox=None, vmin=None, vmax=None,
               figsize=(10, 8), resolution_m=DEFAULT_MAP_RESOLUTION_M, lakes=None,
               show_context=False):
    figure = plt.figure(figsize=figsize, constrained_layout=False)
    axis = figure.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
    render_layer(axis, derived_path, l2_path, layer, bbox, vmin=vmin, vmax=vmax,
                 resolution_m=resolution_m, lakes=lakes, show_context=show_context)
    figure.subplots_adjust(left=0.10, right=0.90, bottom=0.09, top=0.91)
    return figure


def regular_geographic_raster(derived_path, l2_path, layer="true_color", bbox=None,
                              resolution_m=DEFAULT_MAP_RESOLUTION_M,
                              max_cells=MAP_MAX_CELLS):
    """Regular EPSG:4326 preview for browser maps; native science arrays are unchanged."""
    latitude, longitude, values, source = _native_scene_layer(derived_path, l2_path, layer, bbox)
    valid = np.isfinite(latitude) & np.isfinite(longitude)
    west, east = float(np.nanmin(longitude[valid])), float(np.nanmax(longitude[valid]))
    south, north = float(np.nanmin(latitude[valid])), float(np.nanmax(latitude[valid]))
    center_lat = 0.5 * (south + north)
    dy = float(resolution_m) / 111_320.0
    dx = float(resolution_m) / max(111_320.0 * math.cos(math.radians(center_lat)), 1.0)
    width = max(1, int(math.ceil((east - west) / dx)))
    height = max(1, int(math.ceil((north - south) / dy)))
    effective = float(resolution_m)
    if width * height > int(max_cells):
        factor = math.sqrt(width * height / int(max_cells))
        effective *= factor; dx *= factor; dy *= factor
        width = max(1, int(math.ceil((east - west) / dx)))
        height = max(1, int(math.ceil((north - south) / dy)))
        print(f"Display grid was coarsened to {effective:.1f} m ({width*height:,} cells); "
              "native NetCDF values are unchanged.")
    area = pr_geometry.AreaDefinition("olci_leaflet", "OLCI geographic preview", "latlon",
                                      CRS.from_epsg(4326), width, height, (west, south, east, north))
    data = _resample_nearest(longitude, latitude, values, area, effective)
    if layer == "true_color": data = _rgb_stretch(data)
    return {"layer": layer, "data": data, "area": area, "crs": CRS.from_epsg(4326),
            "epsg": 4326, "extent": (west, south, east, north), "width": width,
            "height": height, "requested_resolution_m": float(resolution_m),
            "effective_resolution_m": effective, "bbox": bbox, "source_products": source}


def _folium_rgba(data, layer, vmin=None, vmax=None, water=None, retained=None):
    if layer == "true_color":
        rgb = np.uint8(np.clip(np.nan_to_num(data, nan=0), 0, 1) * 255)
        alpha = np.uint8(np.all(np.isfinite(data), axis=2) * 255)
        return np.dstack([rgb, alpha]), None
    if layer in MASK_LAYER_STYLE or layer in {"l2_water_mask", "hydrolakes_water_mask",
                                              "bright_pixel_mask"}:
        cmap_name, label, low, high = "viridis", layer.replace("_", " "), 0, 1
    else:
        cmap_name, defaults, label = LAYER_STYLE[layer]
        low, high = robust_limits(data, defaults)
    low = low if vmin is None else float(vmin); high = high if vmax is None else float(vmax)
    norm = mcolors.Normalize(low, high, clip=True)
    rgba = np.uint8(plt.get_cmap(cmap_name)(norm(np.nan_to_num(data, nan=low))) * 255)
    rgba[..., 3] = np.uint8(np.isfinite(data) * 220)
    if layer in {"CI", "CI_cyano", "NDCI"} and water is not None and retained is not None:
        rgba[water & ~retained] = np.array([37, 37, 37, 235], dtype=np.uint8)
        rgba[retained & ~np.isfinite(data)] = np.array([199, 199, 199, 235], dtype=np.uint8)
    colors = [mcolors.to_hex(plt.get_cmap(cmap_name)(value)) for value in np.linspace(0, 1, 7)]
    legend = LinearColormap(colors, vmin=low, vmax=high, caption=label)
    return rgba, legend


def interactive_regular_map(derived_path, l2_path, layer="true_color", bbox=None,
                            resolution_m=DEFAULT_MAP_RESOLUTION_M, lakes=None,
                            vmin=None, vmax=None, destination=None):
    """Return a standalone Leaflet map with true longitude/latitude interaction."""
    grid = regular_geographic_raster(
        derived_path, l2_path, layer, bbox, resolution_m,
        max_cells=INTERACTIVE_MAX_CELLS,
    )
    west, south, east, north = grid["extent"]
    water = retained = None
    if layer in {"CI", "CI_cyano", "NDCI"}:
        def companion(name):
            lat, lon, values, _ = _native_scene_layer(derived_path, l2_path, name, bbox)
            return _resample_nearest(lon, lat, values, grid["area"],
                                     grid["effective_resolution_m"])
        water = companion("water_mask") >= 0.5
        retained = companion("valid_water_mask") >= 0.5
    rgba, legend = _folium_rgba(grid["data"], layer, vmin, vmax, water, retained)
    map_object = folium.Map(location=[(south+north)/2, (west+east)/2], zoom_start=8,
                            tiles=None, control_scale=True, prefer_canvas=True)
    folium.TileLayer("CartoDB positron", name="Light basemap", control=True).add_to(map_object)
    folium.TileLayer("OpenStreetMap", name="OpenStreetMap", control=True).add_to(map_object)
    folium.raster_layers.ImageOverlay(rgba, bounds=[[south, west], [north, east]],
        name=f"OLCI {layer}", opacity=1.0, interactive=True, cross_origin=False,
        zindex=3).add_to(map_object)
    if lakes is not None and not lakes.empty:
        folium.GeoJson(lakes.to_crs(4326).__geo_interface__, name="HydroLAKES",
                       style_function=lambda feature: {"color": "#00bcd4", "weight": 1.2,
                                                        "fillOpacity": 0}).add_to(map_object)
    if legend is not None: legend.add_to(map_object)
    folium_plugins.Fullscreen(position="topright").add_to(map_object)
    folium_plugins.MousePosition(position="bottomright", separator=" | ",
                                 prefix="Lat / Lon", num_digits=5).add_to(map_object)
    folium.LatLngPopup().add_to(map_object)
    folium_plugins.MeasureControl(position="topleft", primary_length_unit="kilometers").add_to(map_object)
    folium.LayerControl(collapsed=False).add_to(map_object)
    map_object.fit_bounds([[south, west], [north, east]])
    if destination is None:
        destination = EXPORT_DIR / Path(l2_path).stem / "interactive_maps" / f"{layer}_leaflet.html"
    destination = Path(destination); destination.parent.mkdir(parents=True, exist_ok=True)
    map_object.save(str(destination)); map_object._olci_html_path = destination
    print(f"Interactive latitude/longitude map: {destination}")
    return map_object


## 9. Global-efficient architecture (preferred production path)

The earlier cells remain the scientific foundation: OCSSW/L2Gen, CI-family equations, QA flags, HydroLAKES geometry handling, and native-swath utilities. The cells below change **how scenes are discovered, staged, retained, resumed, and explored**.

### Why this is scene-centric rather than State/country-centric

Administrative partitions are poor processing units for OLCI because one swath can cross many States, countries, or fixed geographic tiles. Processing independently by those units can atmospheric-correct the same scene repeatedly. Here, the **scene is the expensive processing unit**. A bbox can still be used to shard catalogue searches, but the registry deduplicates product IDs.

### Storage lifecycle

`CDSE catalogue → scene/lake intersection → local L1B scratch → L2Gen lake window(s) → compact lake-pixel NetCDF + Parquet lake stats → delete L1B/L2 scratch`

### Why Dask is not used as a replacement for L1B staging

Dask can be useful later for aggregating the compact NetCDF/Parquet archive, but it does not replace L2Gen's documented Level-1 `ifile` input model. This notebook therefore minimizes **peak local storage and retained storage**, rather than using an unsupported remote-chunk input path for L2Gen.

### What is retained

For each HydroLAKES pixel the compact NetCDF stores geolocation, `Hylak_id`, raw `l2_flags`, the four core Rayleigh-corrected `rhos` bands (620/665/681/709), mask states, SS(681), SS(665), CI, CIcyano, and NDCI. Non-lake pixels are **omitted rather than stored as millions of NaNs**. Masked derived values use NetCDF `_FillValue` and decode as `NaN`.


### 9.1 Persistent/scratch configuration and lean L2 products


In [21]:
# Efficient global Sentinel-3 OLCI lake processing additions.
# This code is designed to run after the scientific/helper cells in v1.4.2.

import base64
import contextlib
import concurrent.futures as cf
import io
import sqlite3
from collections import defaultdict
from pathlib import Path
from shapely.geometry import shape as shapely_shape, box as shapely_box
from shapely.ops import unary_union

import boto3
import pyarrow as pa
import pyarrow.parquet as pq

# -----------------------------------------------------------------------------
# v2 global / storage-efficient configuration
# -----------------------------------------------------------------------------
# v2.6 intentionally requests NO Google Drive access.
# Everything below is runtime-local under /content. Colab deletes these files when
# the runtime is discarded, so download share ZIPs / registry exports you want to keep.
USE_GOOGLE_DRIVE_FOR_PERSISTENT = False  # retained only as a compatibility flag
PERSISTENT_ROOT = Path("/home/myid/mc16405/cyanolake-web/s3_olci_global").resolve()
SCRATCH_ROOT = Path("/home/myid/mc16405/cyanolake-web/s3_olci_scratch").resolve()
# Persistent products: small registry, lake-only pixels, stats, logs, exports.
PROJECT_ROOT = PERSISTENT_ROOT
EXPORT_DIR = PERSISTENT_ROOT / "exports"
PAR_DIR = PERSISTENT_ROOT / "par"
LOG_DIR = PERSISTENT_ROOT / "logs"
HYDRO_ROOT = PERSISTENT_ROOT / "hydrolakes"
COMPACT_DIR = PERSISTENT_ROOT / "lake_only"
COMPACT_NC_DIR = COMPACT_DIR / "netcdf"
STATS_SCENE_DIR = COMPACT_DIR / "stats_by_scene"
CATALOG_DIR = PERSISTENT_ROOT / "catalog"
HTML_DIR = PERSISTENT_ROOT / "html_maps"
ADMIN_ROOT = PERSISTENT_ROOT / "admin_boundaries"
REGISTRY_DB = PERSISTENT_ROOT / "scene_registry.sqlite"
TARGET_LAKES_GPKG = CATALOG_DIR / "target_hydrolakes.gpkg"

# Ephemeral products: at most one staged L1 product + one/few L2 windows at a time.
RAW_L1_DIR = SCRATCH_ROOT / "raw_l1_fallback"
STAGING_DIR = SCRATCH_ROOT / "staging"
L2_DIR = SCRATCH_ROOT / "l2"
DERIVED_DIR = SCRATCH_ROOT / "legacy_dense_derived"

for _d in (
    PERSISTENT_ROOT, SCRATCH_ROOT, EXPORT_DIR, PAR_DIR, LOG_DIR, HYDRO_ROOT,
    COMPACT_DIR, COMPACT_NC_DIR, STATS_SCENE_DIR, CATALOG_DIR, HTML_DIR, ADMIN_ROOT,
    RAW_L1_DIR, STAGING_DIR, L2_DIR, DERIVED_DIR,
):
    _d.mkdir(parents=True, exist_ok=True)

# Production defaults intentionally minimize local disk and retained products.
DOWNLOAD_METHOD = "auto"              # auto -> CDSE S3 direct, then ZIP fallback
KEEP_STAGED_L1 = False                 # delete .SEN3 after successful extraction
KEEP_RAW_ZIP_FALLBACK = False          # delete ZIP if fallback route was used
KEEP_L2_INTERMEDIATE = False           # delete temporary L2Gen NetCDF after lake extraction
KEEP_FAILED_SCRATCH = True             # useful for debugging a failed scene
SAVE_FULL_REFLECTANCE_SPECTRA = False  # core bands only for CI/NDCI
SAVE_TRUE_COLOR_L2 = False             # temporary L2 size reduction
SAVE_RHOS865_FOR_BRIGHT_SCREEN = False # set True only if bright-rhos865 screening is enabled
SAVE_COMPACT_LAKE_NETCDF = True
SAVE_PIXEL_PARQUET = False             # optional duplicate point-store; stats remain Parquet
COMPACT_DEFLATE = 6
COMPACT_CHUNK_OBS = 65536

# Scene-local processing. Nearby lake boxes are merged. If lake windows cover too
# much of a scene or fragment into too many groups, process the full scene once.
PROCESSING_STRATEGY = "adaptive_lake_windows"  # or "full_scene"
L2_WINDOW_MARGIN_KM = 3.0
MAX_L2_WINDOWS_PER_SCENE = 6
FULL_SCENE_FRACTION_THRESHOLD = 0.35

# Lake-only science defaults.
#
# v2.6.3 production default is intentionally CyAN-comparable for quantitative work:
# NASA/OBPG CyAN Level-3 binning documents LAND, CLDICE, and HISATZEN as the
# Level-2 exclusion flags. LAND is implemented here through REQUIRE_L2_NONLAND=True;
# CLDICE and HISATZEN are applied from l2_flags after L2Gen so the original flags
# remain auditable in the temporary L2 file.
MASK_PRESETS["cyan_comparable"] = {
    "label": "CyAN-comparable quantitative (LAND + CLDICE + HISATZEN)",
    "qa_flags": ("CLDICE", "HISATZEN"),
    "land_adjacency_pixels": 0,
    "note": (
        "Closest documented post-L2 flag configuration to NASA/OBPG CyAN Level-3 "
        "binning. LAND is enforced separately through REQUIRE_L2_NONLAND=True."
    ),
}
REQUIRE_L2_NONLAND = True
DEFAULT_WATER_DEFINITION = "hydrolakes"
DEFAULT_MASK_PROFILE = "cyan_comparable"
QA_EXCLUDE_FLAGS = MASK_PRESETS[DEFAULT_MASK_PROFILE]["qa_flags"]
LAND_ADJACENCY_PIXELS = MASK_PRESETS[DEFAULT_MASK_PROFILE]["land_adjacency_pixels"]
DEFAULT_SHORE_BUFFER_M = 0.0
DEFAULT_BRIGHT_SCREEN = False
DEFAULT_BRIGHT_RHOS865 = 0.08

# Keep L2Gen output lean. This overrides the earlier helper at runtime.
def l2_products(mode: str, full_spectrum: bool = False) -> tuple[list[str], set[str]]:
    mode = mode.lower()
    if mode in {"full_swath_rhos", "rayleigh_ci"}:
        mode = "full_swath_rhos"
    if mode not in {"full_swath_rhos", "standard_dual"}:
        raise ValueError("mode must be 'full_swath_rhos' or 'standard_dual'")
    waves = SPECTRAL_RHOS_WAVELENGTHS if full_spectrum else CORE_RHOS_WAVELENGTHS
    products = [f"rhos_{wave}" for wave in waves]
    required = {f"rhos_{wave}" for wave in CORE_RHOS_WAVELENGTHS}
    if SAVE_RHOS865_FOR_BRIGHT_SCREEN and "rhos_865" not in products:
        products.append("rhos_865")
    if SAVE_TRUE_COLOR_L2:
        products.extend(TRUE_COLOR_PRODUCTS)
    if mode == "standard_dual":
        rrs_waves = AQUATIC_RRS_WAVELENGTHS if full_spectrum else (665, 709)
        products.extend(f"Rrs_{wave}" for wave in rrs_waves)
        required.update({"Rrs_665", "Rrs_709"})
    return list(dict.fromkeys(products)), required

print("Efficient global architecture enabled")
print("Runtime-local output root:", PERSISTENT_ROOT)
print("Scratch root   :", SCRATCH_ROOT)
print("Default L2     : core rhos only; full spectra =", SAVE_FULL_REFLECTANCE_SPECTRA)
print("Cleanup        : staged L1 =", KEEP_STAGED_L1, " L2 =", KEEP_L2_INTERMEDIATE)



Efficient global architecture enabled
Runtime-local output root: /home/myid/mc16405/cyanolake-web/s3_olci_global
Scratch root   : /home/myid/mc16405/cyanolake-web/s3_olci_scratch
Default L2     : core rhos only; full spectra = False
Cleanup        : staged L1 = False  L2 = False


### 9.2 Scene registry and reproducibility hash


In [22]:
# -----------------------------------------------------------------------------
# Registry / reproducibility
# -----------------------------------------------------------------------------
def _registry_connection():
    conn = sqlite3.connect(REGISTRY_DB, timeout=60)
    # v2.6 uses runtime-local /content storage only; WAL is appropriate here.
    conn.execute("PRAGMA journal_mode=WAL")
    conn.execute("PRAGMA synchronous=NORMAL")
    conn.execute(
        """
        CREATE TABLE IF NOT EXISTS scenes (
            scene_id TEXT NOT NULL,
            config_hash TEXT NOT NULL,
            scene_name TEXT,
            acquisition_start TEXT,
            acquisition_end TEXT,
            s3_path TEXT,
            size_bytes INTEGER,
            candidate_lakes INTEGER,
            status TEXT NOT NULL DEFAULT 'queued',
            attempts INTEGER NOT NULL DEFAULT 0,
            processing_strategy TEXT,
            window_count INTEGER,
            download_method TEXT,
            compact_bytes INTEGER,
            stats_rows INTEGER,
            last_error TEXT,
            updated_utc TEXT,
            scene_json TEXT,
            PRIMARY KEY(scene_id, config_hash)
        )
        """
    )
    conn.commit()
    return conn


def processing_config_hash(min_area_km2: float, region: str = "WORLD") -> str:
    payload = {
        "l2_mode": L2_MODE,
        "ndci_source": NDCI_SOURCE,
        "core_rhos": list(CORE_RHOS_WAVELENGTHS),
        "full_spectrum": bool(SAVE_FULL_REFLECTANCE_SPECTRA),
        "true_color_l2": bool(SAVE_TRUE_COLOR_L2),
        "qa_flags": list(QA_EXCLUDE_FLAGS),
        "land_adjacency_pixels": int(LAND_ADJACENCY_PIXELS),
        "require_l2_nonland": bool(REQUIRE_L2_NONLAND),
        "detection_limit": float(CI_DETECTION_LIMIT),
        "bright_screen": bool(DEFAULT_BRIGHT_SCREEN),
        "bright_rhos865": float(DEFAULT_BRIGHT_RHOS865),
        "shore_buffer_m": float(DEFAULT_SHORE_BUFFER_M),
        "processing_strategy": PROCESSING_STRATEGY,
        "window_margin_km": float(L2_WINDOW_MARGIN_KM),
        "max_windows": int(MAX_L2_WINDOWS_PER_SCENE),
        "full_scene_fraction": float(FULL_SCENE_FRACTION_THRESHOLD),
        "min_lake_area_km2": float(min_area_km2),
        "region": str(region).upper(),
        "schema": "global-efficient-v2.5",
    }
    return hashlib.sha1(json.dumps(payload, sort_keys=True).encode()).hexdigest()[:16]


def registry_upsert_scene(scene: dict[str, Any], config_hash: str, candidate_lakes: int) -> None:
    with _registry_connection() as conn:
        conn.execute(
            """
            INSERT INTO scenes (
                scene_id, config_hash, scene_name, acquisition_start, acquisition_end,
                s3_path, size_bytes, candidate_lakes, status, updated_utc, scene_json
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, 'queued', ?, ?)
            ON CONFLICT(scene_id, config_hash) DO UPDATE SET
                scene_name=excluded.scene_name,
                acquisition_start=excluded.acquisition_start,
                acquisition_end=excluded.acquisition_end,
                s3_path=excluded.s3_path,
                size_bytes=excluded.size_bytes,
                candidate_lakes=excluded.candidate_lakes,
                updated_utc=excluded.updated_utc,
                scene_json=excluded.scene_json
            """,
            (
                str(scene["id"]), config_hash, scene.get("name"), scene.get("start"),
                scene.get("end"), scene.get("s3_path"), int(scene.get("size_bytes") or 0),
                int(candidate_lakes), utc_now(), json.dumps(scene, default=str),
            ),
        )
        conn.commit()


def registry_update(scene_id: str, config_hash: str, **fields) -> None:
    if not fields:
        return
    fields["updated_utc"] = utc_now()
    keys = list(fields)
    sql = ", ".join(f"{k}=?" for k in keys)
    values = [fields[k] for k in keys] + [scene_id, config_hash]
    with _registry_connection() as conn:
        conn.execute(f"UPDATE scenes SET {sql} WHERE scene_id=? AND config_hash=?", values)
        conn.commit()


def registry_table(config_hash: str | None = None) -> pd.DataFrame:
    with _registry_connection() as conn:
        if config_hash:
            return pd.read_sql_query(
                "SELECT * FROM scenes WHERE config_hash=? ORDER BY acquisition_start", conn,
                params=(config_hash,),
            )
        return pd.read_sql_query("SELECT * FROM scenes ORDER BY acquisition_start", conn)


def registry_export_csv() -> Path:
    out = EXPORT_DIR / "scene_registry.csv"
    registry_table().to_csv(out, index=False)
    return out



### 9.3 Global catalogue discovery and HydroLAKES intersection


In [23]:
# -----------------------------------------------------------------------------
# Catalogue discovery with local HydroLAKES intersection
# -----------------------------------------------------------------------------
def _odata_scene(item: dict[str, Any]) -> dict[str, Any]:
    name = str(item.get("Name", "")).removesuffix(".SEN3")
    content = item.get("ContentDate") or {}
    return {
        "id": str(item["Id"]),
        "name": name,
        "satellite": name.split("_", 1)[0] if name else "S3",
        "start": content.get("Start", ""),
        "end": content.get("End", ""),
        "size_bytes": int(item.get("ContentLength") or 0),
        "online": bool(item.get("Online", True)),
        "s3_path": item.get("S3Path"),
        "geofootprint": item.get("GeoFootprint"),
    }


def search_olci_l1_catalog(
    start_date: str,
    end_date: str,
    bbox: tuple[float, float, float, float] | None = None,
    max_products: int | None = None,
    page_size: int = 1000,
) -> list[dict[str, Any]]:
    """Catalogue OLCI EFR L1 scenes once; global search uses time only."""
    start = pd.Timestamp(start_date).strftime("%Y-%m-%d")
    end_exclusive = (pd.Timestamp(end_date) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    filters = [
        "Collection/Name eq 'SENTINEL-3'",
        (
            "Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'productType' "
            "and att/OData.CSC.StringAttribute/Value eq 'OL_1_EFR___')"
        ),
        f"ContentDate/Start ge {start}T00:00:00.000Z",
        f"ContentDate/Start lt {end_exclusive}T00:00:00.000Z",
    ]
    if bbox is not None:
        west, south, east, north = normalize_bbox(bbox)
        polygon = (
            f"POLYGON(({west} {south},{east} {south},{east} {north},"
            f"{west} {north},{west} {south}))"
        )
        filters.append(f"OData.CSC.Intersects(area=geography'SRID=4326;{polygon}')")
    params = {
        "$filter": " and ".join(filters),
        "$orderby": "ContentDate/Start asc",
        "$top": min(int(page_size), 1000),
        "$select": "Id,Name,S3Path,GeoFootprint,ContentLength,ContentDate,Online",
    }
    rows: list[dict[str, Any]] = []
    url = CDSE_CATALOGUE_URL
    while url:
        response = requests.get(url, params=params if url == CDSE_CATALOGUE_URL else None, timeout=120)
        response.raise_for_status()
        payload = response.json()
        for item in payload.get("value", []):
            rows.append(_odata_scene(item))
            if max_products and len(rows) >= int(max_products):
                return rows[: int(max_products)]
        next_url = payload.get("@odata.nextLink")
        if next_url and str(next_url).startswith("/"):
            next_url = "https://catalogue.dataspace.copernicus.eu" + str(next_url)
        url = next_url
        params = None
    return rows


def scene_geometry(scene: dict[str, Any]):
    payload = scene.get("geofootprint")
    if not payload:
        return None
    try:
        geom = shapely_shape(payload)
        if not geom.is_valid:
            geom = geom.buffer(0)
        return geom
    except Exception:
        return None


def candidate_lakes_for_scene(scene: dict[str, Any], lakes: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    geom = scene_geometry(scene)
    if geom is None or lakes is None or lakes.empty:
        return lakes.iloc[0:0].copy()
    try:
        idx = lakes.sindex.query(geom, predicate="intersects")
        candidates = lakes.iloc[np.asarray(idx, dtype=int)].copy()
    except Exception:
        candidates = lakes[lakes.intersects(geom)].copy()
    if candidates.empty:
        return candidates
    return candidates[candidates.intersects(geom)].copy().reset_index(drop=True)


def save_target_lakes(lakes: gpd.GeoDataFrame) -> Path:
    TARGET_LAKES_GPKG.unlink(missing_ok=True)
    lakes.to_file(TARGET_LAKES_GPKG, driver="GPKG", layer="hydrolakes", index=False)
    return TARGET_LAKES_GPKG


def plan_catalogue(
    start_date: str,
    end_date: str,
    lakes: gpd.GeoDataFrame,
    min_area_km2: float,
    region: str = "WORLD",
    bbox: tuple[float, float, float, float] | None = None,
    max_products: int | None = None,
) -> tuple[pd.DataFrame, str]:
    if lakes is None or lakes.empty:
        raise ValueError("Load HydroLAKES before planning scenes")
    save_target_lakes(lakes)
    config_hash = processing_config_hash(min_area_km2, region)
    catalog = search_olci_l1_catalog(start_date, end_date, bbox=bbox, max_products=max_products)
    kept = []
    for scene in tqdm(catalog, desc="Intersecting scene footprints with HydroLAKES"):
        subset = candidate_lakes_for_scene(scene, lakes)
        if subset.empty:
            continue
        scene = dict(scene)
        scene["candidate_lakes"] = int(len(subset))
        registry_upsert_scene(scene, config_hash, len(subset))
        kept.append(scene)
    table = registry_table(config_hash)
    return table, config_hash



### 9.4 Direct CDSE S3 staging with ZIP fallback


In [24]:
# -----------------------------------------------------------------------------
# CDSE direct S3 staging: avoids keeping both ZIP + extracted SEN3
# -----------------------------------------------------------------------------
CDSE_S3_ENDPOINT = "https://eodata.dataspace.copernicus.eu"
CDSE_S3_KEYS_URL = "https://s3-keys-manager.cloudferro.com/api/user/credentials"
_S3_CLIENT_CACHE = {}


def temporary_s3_client(username: str, password: str):
    # One credential/client per user per runtime is enough for a batch; avoid
    # calling the key manager for every scene.
    cached = _S3_CLIENT_CACHE.get(str(username))
    if cached and (time.time() - cached[0]) < 45 * 60:
        return cached[1]
    token = CDSE_AUTH.token(username, password)
    response = requests.post(
        CDSE_S3_KEYS_URL,
        headers={"Authorization": f"Bearer {token}"},
        timeout=90,
    )
    response.raise_for_status()
    data = response.json()
    access = data.get("access_id") or data.get("access_key") or data.get("access")
    secret = data.get("secret") or data.get("secret_key")
    if not access or not secret:
        raise RuntimeError("CDSE S3 credential service did not return access_id/secret")
    # The CDSE documentation notes that newly created temporary S3 credentials
    # can need a few seconds to propagate. This happens once per runtime/user.
    time.sleep(5)
    client = boto3.client(
        "s3", endpoint_url=CDSE_S3_ENDPOINT,
        aws_access_key_id=access, aws_secret_access_key=secret,
        region_name="default",
    )
    _S3_CLIENT_CACHE[str(username)] = (time.time(), client, access)
    return client


def release_temporary_s3_credentials(username: str, password: str) -> None:
    """Best-effort deletion of a temporary S3 key created by this runtime."""
    cached = _S3_CLIENT_CACHE.pop(str(username), None)
    if not cached or len(cached) < 3:
        return
    access_id = cached[2]
    try:
        token = CDSE_AUTH.token(username, password)
        requests.delete(
            f"{CDSE_S3_KEYS_URL}/access_id/{access_id}",
            headers={"Authorization": f"Bearer {token}"}, timeout=60,
        )
    except Exception as exc:
        print(f"Warning: temporary S3 key cleanup failed: {exc}")


def _split_s3_path(s3_path: str) -> tuple[str, str]:
    text = str(s3_path or "").strip()
    if not text:
        raise ValueError("Catalogue record has no S3Path")
    parts = text.lstrip("/").split("/", 1)
    if len(parts) != 2:
        raise ValueError(f"Unexpected S3Path: {s3_path}")
    return parts[0], parts[1].rstrip("/") + "/"


def download_scene_s3_direct(
    scene: dict[str, Any], username: str, password: str,
    workers: int = 6,
) -> Path:
    """Download the SEN3 object tree directly. Existing same-size files are reused."""
    client = temporary_s3_client(username, password)
    bucket, prefix = _split_s3_path(scene.get("s3_path"))
    target = STAGING_DIR / f"{scene['name']}.SEN3"
    target.mkdir(parents=True, exist_ok=True)
    paginator = client.get_paginator("list_objects_v2")
    objects = []
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        objects.extend(page.get("Contents", []))
    objects = [obj for obj in objects if not str(obj["Key"]).endswith("/")]
    if not objects:
        raise FileNotFoundError(f"No S3 objects found under {scene.get('s3_path')}")

    def fetch(obj):
        key = str(obj["Key"])
        relative = key[len(prefix):]
        destination = target / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        expected = int(obj.get("Size") or 0)
        if destination.exists() and destination.stat().st_size == expected:
            return expected, False
        client.download_file(bucket, key, str(destination))
        return expected, True

    downloaded = 0
    with cf.ThreadPoolExecutor(max_workers=max(1, int(workers))) as executor:
        futures = [executor.submit(fetch, obj) for obj in objects]
        for future in tqdm(cf.as_completed(futures), total=len(futures), desc=f"S3 {scene['name'][:24]}"):
            size, changed = future.result()
            if changed:
                downloaded += size
    validate_sen3(target)
    print(f"S3 direct stage ready: {target} (downloaded {human_size(downloaded)})")
    return target


def stage_scene_efficient(
    scene: dict[str, Any], username: str, password: str,
    method: str = DOWNLOAD_METHOD,
) -> tuple[Path | None, Path, str]:
    method = str(method).lower()
    if method not in {"auto", "s3", "zip"}:
        raise ValueError("download method must be auto, s3, or zip")
    if method in {"auto", "s3"}:
        try:
            return None, download_scene_s3_direct(scene, username, password), "s3_direct"
        except Exception as exc:
            if method == "s3":
                raise
            print(f"S3 direct staging failed; using ZIP fallback: {exc}")
    raw_zip, sen3 = download_cdse_scene(scene, username, password)
    return raw_zip, sen3, "odata_zip_fallback"



### 9.5 Adaptive lake-window planner


In [25]:
# -----------------------------------------------------------------------------
# Adaptive scene window planner
# -----------------------------------------------------------------------------
def _expanded_lonlat_box(bounds, margin_km: float):
    west, south, east, north = map(float, bounds)
    lat0 = float(np.clip((south + north) / 2.0, -85.0, 85.0))
    dlat = float(margin_km) / 111.32
    dlon = float(margin_km) / max(111.32 * math.cos(math.radians(lat0)), 1.0)
    return shapely_box(max(-180, west-dlon), max(-90, south-dlat),
                       min(180, east+dlon), min(90, north+dlat))


def plan_scene_windows(
    scene: dict[str, Any], scene_lakes: gpd.GeoDataFrame,
    strategy: str = PROCESSING_STRATEGY,
    margin_km: float = L2_WINDOW_MARGIN_KM,
    max_windows: int = MAX_L2_WINDOWS_PER_SCENE,
    full_scene_fraction: float = FULL_SCENE_FRACTION_THRESHOLD,
) -> list[tuple[float, float, float, float] | None]:
    if str(strategy).lower() == "full_scene":
        return [None]
    geom = scene_geometry(scene)
    if geom is None or scene_lakes is None or scene_lakes.empty:
        return [None]
    minx, miny, maxx, maxy = geom.bounds
    if maxx - minx > 170:  # dateline geometry: prefer safe full scene
        return [None]
    boxes = [_expanded_lonlat_box(g.bounds, margin_km) for g in scene_lakes.geometry]
    merged = unary_union(boxes)
    parts = list(merged.geoms) if hasattr(merged, "geoms") else [merged]
    windows = []
    for part in parts:
        clipped = part.intersection(geom)
        if clipped.is_empty:
            continue
        west, south, east, north = clipped.bounds
        if west < east and south < north:
            windows.append((float(west), float(south), float(east), float(north)))
    if not windows or len(windows) > int(max_windows):
        return [None]
    scene_area = max((maxx-minx)*(maxy-miny), 1e-9)
    window_area = sum(max(0, e-w)*max(0, n-s) for w, s, e, n in windows)
    if window_area / scene_area >= float(full_scene_fraction):
        return [None]
    return sorted(windows, key=lambda b: (b[1], b[0]))


def window_id(window) -> str:
    if window is None:
        return "FULL"
    text = ",".join(f"{v:.6f}" for v in window)
    return "W_" + hashlib.sha1(text.encode()).hexdigest()[:10]


def lakes_for_window(lakes: gpd.GeoDataFrame, window) -> gpd.GeoDataFrame:
    if window is None:
        return lakes.copy().reset_index(drop=True)
    geom = shapely_box(*window)
    return lakes[lakes.intersects(geom)].copy().reset_index(drop=True)



### 9.6 Compact derived lake-pixel archive

This is the main persistent storage change. Instead of another dense 2-D swath whose non-lake cells are mostly fill/NaN, the notebook stores only pixels that fall inside the selected HydroLAKES polygons. NetCDF4 DEFLATE + shuffle compression is applied to the 1-D observation dimension. v2.6 uses only the local Colab runtime; no Google Drive mount or account permission is requested. Download shareable scene ZIPs before ending the runtime if you want to keep them.


In [26]:
# -----------------------------------------------------------------------------
# Compact lake-only derived point archive
# -----------------------------------------------------------------------------
def _open_compact_writer(path: Path, metadata: dict[str, Any]):
    path.parent.mkdir(parents=True, exist_ok=True)
    # Runtime-local atomic write. No Google Drive / FUSE branch in v2.6.
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.unlink(missing_ok=True)
    ds = netCDF4.Dataset(tmp, "w", format="NETCDF4")
    ds.createDimension("obs", None)
    for key, value in metadata.items():
        if value is not None:
            try:
                ds.setncattr(key, value)
            except Exception:
                ds.setncattr(key, str(value))
    chunk = (COMPACT_CHUNK_OBS,)
    vars_ = {}
    vars_["Hylak_id"] = ds.createVariable("Hylak_id", "i8", ("obs",), zlib=True,
                                           complevel=COMPACT_DEFLATE, shuffle=True, chunksizes=chunk)
    vars_["source_row"] = ds.createVariable("source_row", "i4", ("obs",), zlib=True,
                                              complevel=COMPACT_DEFLATE, shuffle=True, chunksizes=chunk)
    vars_["source_col"] = ds.createVariable("source_col", "i4", ("obs",), zlib=True,
                                              complevel=COMPACT_DEFLATE, shuffle=True, chunksizes=chunk)
    vars_["l2_flags"] = ds.createVariable("l2_flags", "u4", ("obs",), zlib=True,
                                            complevel=COMPACT_DEFLATE, shuffle=True, chunksizes=chunk)
    for name in ("valid_water_mask", "ci_valid_mask", "ci_candidate_mask", "ci_detection_mask", "ndci_valid_mask"):
        vars_[name] = ds.createVariable(name, "u1", ("obs",), zlib=True,
                                        complevel=COMPACT_DEFLATE, shuffle=True, chunksizes=chunk)
    for name in (
        "latitude", "longitude", "rhos_620", "rhos_665", "rhos_681", "rhos_709",
        "SS_681", "SS_665", "CI", "CI_cyano", "NDCI",
    ):
        vars_[name] = ds.createVariable(
            name, "f4", ("obs",), zlib=True, complevel=COMPACT_DEFLATE,
            shuffle=True, chunksizes=chunk, fill_value=DERIVED_FILL,
        )
    return ds, vars_, tmp


def _append_float(var, start: int, values):
    a = np.asarray(values, dtype=np.float32)
    var[start:start+len(a)] = np.ma.masked_invalid(a)


def _membership_block(latitude, longitude, lakes: gpd.GeoDataFrame) -> np.ndarray:
    """Assign HydroLAKES IDs to one row block, using the GeoPandas spatial index first."""
    ids = np.zeros(latitude.shape, dtype=np.int64)
    if lakes is None or lakes.empty:
        return ids
    finite = np.isfinite(latitude) & np.isfinite(longitude)
    if not finite.any():
        return ids
    bwest, beast = float(np.nanmin(longitude[finite])), float(np.nanmax(longitude[finite]))
    bsouth, bnorth = float(np.nanmin(latitude[finite])), float(np.nanmax(latitude[finite]))
    block_geom = shapely_box(bwest, bsouth, beast, bnorth)
    try:
        hit_idx = lakes.sindex.query(block_geom, predicate="intersects")
        block_lakes = lakes.iloc[np.asarray(hit_idx, dtype=int)]
    except Exception:
        block_lakes = lakes
    for _, lake in block_lakes.iterrows():
        west, south, east, north = lake.geometry.bounds
        if east < bwest or west > beast or north < bsouth or south > bnorth:
            continue
        rough = finite & (longitude >= west) & (longitude <= east) & (latitude >= south) & (latitude <= north)
        rr, cc = np.where(rough)
        if not rr.size:
            continue
        rs = slice(rr.min(), rr.max()+1); cs = slice(cc.min(), cc.max()+1)
        inside = contains_xy(lake.geometry, longitude[rs, cs], latitude[rs, cs])
        local = ids[rs, cs]
        # HydroLAKES polygons should not overlap materially; first assignment wins.
        local[(local == 0) & inside] = int(lake["Hylak_id"])
        ids[rs, cs] = local
    return ids


def extract_lake_only_netcdf(
    l2_path: Path,
    scene: dict[str, Any],
    lakes: gpd.GeoDataFrame,
    output_path: Path,
    window=None,
    qa_flags: Iterable[str] = QA_EXCLUDE_FLAGS,
    land_adjacency_pixels: int = LAND_ADJACENCY_PIXELS,
    require_l2_nonland: bool = REQUIRE_L2_NONLAND,
    detection_limit: float = CI_DETECTION_LIMIT,
    ss681_max: float = 0.0,
    ss665_min: float = 0.0,
    bright_screen: bool = DEFAULT_BRIGHT_SCREEN,
    bright_rhos865: float = DEFAULT_BRIGHT_RHOS865,
    rows_per_block: int = ROW_BLOCK_SIZE,
    config_hash: str | None = None,
    shore_buffer_m: float = DEFAULT_SHORE_BUFFER_M,
) -> tuple[Path | None, dict[str, Any]]:
    """
    Write only HydroLAKES pixels as a 1-D point collection.
    Non-lake pixels do not exist on disk. Masked derived values use _FillValue and
    decode to NaN in xarray/netCDF4 masked reads.
    """
    output_path = Path(output_path)
    # Preserve the uploaded notebook's HydroLAKES shoreline-buffer semantics.
    # The default is 0 m; a positive value erodes polygons before pixel extraction.
    analysis_lakes = erode_lake_geometries(lakes, float(shore_buffer_m)) if float(shore_buffer_m) > 0 else lakes
    if analysis_lakes is None or analysis_lakes.empty:
        return None, {"nobs": 0, "missing_flags": [], "reason": "no lake area after shoreline buffer"}
    with netCDF4.Dataset(l2_path) as src:
        geo = src.groups["geophysical_data"]
        nav = src.groups["navigation_data"]
        flags_var = locate_l2_flags(src)
        flag_map = decode_flag_metadata(flags_var)
        if "LAND" not in flag_map:
            raise RuntimeError("L2 l2_flags lacks LAND")
        rhos_names = {w: nearest_product(geo.variables, "rhos", w) for w in CORE_RHOS_WAVELENGTHS}
        actual_waves = tuple(product_wavelength(rhos_names[w]) for w in CORE_RHOS_WAVELENGTHS)
        ndci_prefix = "rhos" if NDCI_SOURCE.lower() == "rhos" else "Rrs"
        ndci_names = {665: nearest_product(geo.variables, ndci_prefix, 665),
                      709: nearest_product(geo.variables, ndci_prefix, 709)}
        bright_name = None
        if bright_screen:
            bright_name = nearest_product(geo.variables, "rhos", 865, tolerance_nm=25)

        shape_2d = nav.variables["latitude"].shape
        metadata = {
            "title": "Sentinel-3 OLCI derived lake-pixel point collection",
            "schema": "global-efficient-v2.5",
            "processing_config_hash": str(config_hash or ""),
            "scene_id": str(scene.get("id", "")),
            "scene_name": str(scene.get("name", "")),
            "acquisition_start": str(scene.get("start", "")),
            "acquisition_end": str(scene.get("end", "")),
            "window_id": window_id(window),
            "window_bbox": "FULL" if window is None else ",".join(map(str, window)),
            "source_l2": str(l2_path.name),
            "l2gen_version": command_version(L2GEN_BIN),
            "qa_exclude_flags": ",".join(map(str, qa_flags)),
            "l2_flag_map_json": json.dumps({k: int(v) for k, v in flag_map.items()}, sort_keys=True),
            "require_l2_nonland": int(bool(require_l2_nonland)),
            "shore_buffer_m": float(shore_buffer_m),
            "ci_detection_limit": float(detection_limit),
            "ss681_max": float(ss681_max),
            "ss665_min": float(ss665_min),
            "non_lake_storage": "omitted entirely; not dense NaN cells",
            "masked_value_semantics": "_FillValue decodes as NaN",
        }
        ds, out, tmp = _open_compact_writer(output_path, metadata)
        nobs = 0
        requested_flags = tuple(dict.fromkeys(str(x).upper() for x in qa_flags))
        missing_flags = [x for x in requested_flags if x not in flag_map]
        try:
            for start in tqdm(range(0, shape_2d[0], int(rows_per_block)), desc=f"Lake pixels {window_id(window)}"):
                stop = min(shape_2d[0], start + int(rows_per_block))
                rs = slice(start, stop)
                lat = to_float(nav.variables["latitude"][rs, :])
                lon = to_float(nav.variables["longitude"][rs, :])
                lake_ids = _membership_block(lat, lon, analysis_lakes)
                inside = lake_ids > 0
                if not inside.any():
                    continue
                flags_ma = np.ma.asarray(flags_var[rs, :])
                flags_valid = ~np.ma.getmaskarray(flags_ma)
                flags = flags_ma.filled(0).astype(np.uint32)
                geolocated = np.isfinite(lat) & np.isfinite(lon) & flags_valid
                land = flag_is_set(flags, flag_map, "LAND")
                base = inside & geolocated
                if require_l2_nonland:
                    base &= ~land
                excluded = np.zeros(base.shape, dtype=bool)
                for fname in requested_flags:
                    if fname in flag_map:
                        excluded |= flag_is_set(flags, flag_map, fname)
                if int(land_adjacency_pixels) > 0:
                    # Block-local dilation. For production shoreline sensitivity work,
                    # keep this at 0 or rerun compact masks later from l2_flags.
                    excluded |= binary_dilation(land, iterations=int(land_adjacency_pixels)) & ~land
                arrays = {w: to_float(geo.variables[rhos_names[w]][rs, :]) for w in CORE_RHOS_WAVELENGTHS}
                if bright_screen:
                    r865 = to_float(geo.variables[bright_name][rs, :])
                    excluded |= np.isfinite(r865) & (r865 > float(bright_rhos865))
                valid_water = base & ~excluded
                family = ci_family(
                    arrays[620], arrays[665], arrays[681], arrays[709],
                    wavelengths=actual_waves, detection_limit=detection_limit,
                    ss681_max=ss681_max, ss665_min=ss665_min,
                )
                ci_valid = valid_water & family["finite"]
                candidate = ci_valid & family["ci_candidate"]
                detected = ci_valid & family["ci_detection"]
                red = to_float(geo.variables[ndci_names[665]][rs, :])
                edge = to_float(geo.variables[ndci_names[709]][rs, :])
                ndci, ndci_finite = ndci_index(red, edge)
                ndci_valid = valid_water & ndci_finite

                # Keep every HydroLAKES polygon pixel, including QA-excluded pixels,
                # so future mask sensitivity tests can use rhos + l2_flags without L1.
                take = inside & geolocated
                rr, cc = np.where(take)
                if not rr.size:
                    continue
                count = len(rr)
                sl = slice(nobs, nobs + count)
                out["Hylak_id"][sl] = lake_ids[rr, cc]
                out["source_row"][sl] = rr.astype(np.int32) + int(start)
                out["source_col"][sl] = cc.astype(np.int32)
                out["l2_flags"][sl] = flags[rr, cc]
                out["valid_water_mask"][sl] = valid_water[rr, cc].astype(np.uint8)
                out["ci_valid_mask"][sl] = ci_valid[rr, cc].astype(np.uint8)
                out["ci_candidate_mask"][sl] = candidate[rr, cc].astype(np.uint8)
                out["ci_detection_mask"][sl] = detected[rr, cc].astype(np.uint8)
                out["ndci_valid_mask"][sl] = ndci_valid[rr, cc].astype(np.uint8)
                _append_float(out["latitude"], nobs, lat[rr, cc])
                _append_float(out["longitude"], nobs, lon[rr, cc])
                for w in CORE_RHOS_WAVELENGTHS:
                    _append_float(out[f"rhos_{w}"], nobs, arrays[w][rr, cc])
                _append_float(out["SS_681"], nobs, np.where(ci_valid, family["SS_681"], np.nan)[rr, cc])
                _append_float(out["SS_665"], nobs, np.where(ci_valid, family["SS_665"], np.nan)[rr, cc])
                _append_float(out["CI"], nobs, np.where(candidate, family["CI"], np.nan)[rr, cc])
                _append_float(out["CI_cyano"], nobs, np.where(detected, family["CI_cyano"], np.nan)[rr, cc])
                _append_float(out["NDCI"], nobs, np.where(ndci_valid, ndci, np.nan)[rr, cc])
                nobs += count
        finally:
            ds.close()
        if nobs == 0:
            tmp.unlink(missing_ok=True)
            return None, {"nobs": 0, "missing_flags": missing_flags}
        if tmp.parent == output_path.parent:
            tmp.replace(output_path)
        else:
            # Atomic/cross-filesystem-safe finalization for runtime-local outputs.
            transfer = output_path.with_suffix(output_path.suffix + ".copying")
            transfer.unlink(missing_ok=True)
            shutil.copy2(tmp, transfer)
            transfer.replace(output_path)
            tmp.unlink(missing_ok=True)
    info = {
        "nobs": int(nobs),
        "size_bytes": int(output_path.stat().st_size),
        "size_human": human_size(output_path.stat().st_size),
        "missing_flags": missing_flags,
        "window_id": window_id(window),
    }
    return output_path, info


def compact_stats(nc_path: Path, lake_metadata: gpd.GeoDataFrame | None = None) -> pd.DataFrame:
    with netCDF4.Dataset(nc_path) as ds:
        frame = pd.DataFrame({
            "Hylak_id": np.asarray(ds.variables["Hylak_id"][:], dtype=np.int64),
            "valid": np.asarray(ds.variables["valid_water_mask"][:]) == 1,
            "ci_valid": np.asarray(ds.variables["ci_valid_mask"][:]) == 1,
            "candidate": np.asarray(ds.variables["ci_candidate_mask"][:]) == 1,
            "detected": np.asarray(ds.variables["ci_detection_mask"][:]) == 1,
            "CI": to_float(ds.variables["CI"][:]),
            "CI_cyano": to_float(ds.variables["CI_cyano"][:]),
            "NDCI": to_float(ds.variables["NDCI"][:]),
        })
        attrs = {k: getattr(ds, k, None) for k in ("scene_id", "scene_name", "acquisition_start", "window_id")}
    meta = {}
    if lake_metadata is not None and not lake_metadata.empty:
        for _, row in lake_metadata.iterrows():
            meta[int(row["Hylak_id"])] = {
                "Lake_name": row.get("Lake_name", "") or "",
                "Lake_area_km2": float(row.get("Lake_area", np.nan)),
                "Country": row.get("Country", "") if "Country" in row else "",
                "Continent": row.get("Continent", "") if "Continent" in row else "",
            }
    records = []
    def q(series, p):
        a = pd.to_numeric(series, errors="coerce").to_numpy(float)
        a = a[np.isfinite(a)]
        return float(np.nanpercentile(a, p)) if a.size else np.nan
    for lake_id, group in frame.groupby("Hylak_id", sort=False):
        m = meta.get(int(lake_id), {})
        ci = group.loc[group["candidate"], "CI"]
        cy = group.loc[group["detected"], "CI_cyano"]
        nd = group.loc[group["valid"], "NDCI"]
        valid_n = int(group["valid"].sum())
        ci_valid_n = int(group["ci_valid"].sum())
        detected_n = int(group["detected"].sum())
        records.append({
            **attrs,
            "Hylak_id": int(lake_id), **m,
            "native_polygon_pixels": int(len(group)),
            "retained_pixels": valid_n,
            "retained_percent": 100.0 * valid_n / len(group) if len(group) else np.nan,
            "ci_input_valid_pixels": ci_valid_n,
            "ci_candidate_pixels": int(group["candidate"].sum()),
            "ci_cyano_detection_pixels": detected_n,
            "ci_cyano_detection_fraction_of_ci_valid": detected_n / ci_valid_n if ci_valid_n else np.nan,
            "ci_cyano_detection_fraction_of_retained": detected_n / valid_n if valid_n else np.nan,
            "CI_mean_candidates": float(np.nanmean(ci)) if np.isfinite(ci).any() else np.nan,
            "CI_median_candidates": q(ci, 50),
            "CI_p10_candidates": q(ci, 10),
            "CI_p90_candidates": q(ci, 90),
            "CI_max_candidates": float(np.nanmax(ci)) if np.isfinite(ci).any() else np.nan,
            "CIcyano_mean_detections": float(np.nanmean(cy)) if np.isfinite(cy).any() else np.nan,
            "CIcyano_median_detections": q(cy, 50),
            "CIcyano_p90_detections": q(cy, 90),
            "NDCI_mean_retained": float(np.nanmean(nd)) if np.isfinite(nd).any() else np.nan,
            "NDCI_median_retained": q(nd, 50),
            "NDCI_p90_retained": q(nd, 90),
            "compact_netcdf": str(nc_path),
        })
    return pd.DataFrame(records)



### 9.7 Scene processor, cleanup, and resumable batch execution


In [27]:
# -----------------------------------------------------------------------------
# Scene processor and cleanup
# -----------------------------------------------------------------------------
def _scene_output_dir(scene: dict[str, Any]) -> Path:
    stamp = pd.Timestamp(scene.get("start") or "1970-01-01")
    return COMPACT_NC_DIR / f"year={stamp.year:04d}" / f"month={stamp.month:02d}" / str(scene["id"])


def _cleanup_scene_scratch(raw_zip: Path | None, sen3: Path | None, l2_paths: list[Path], success=True):
    if not success and KEEP_FAILED_SCRATCH:
        return
    if not KEEP_L2_INTERMEDIATE:
        for p in l2_paths:
            Path(p).unlink(missing_ok=True)
    if sen3 is not None and not KEEP_STAGED_L1:
        shutil.rmtree(Path(sen3), ignore_errors=True)
        # remove empty product staging parent if present
        parent = Path(sen3).parent
        if parent != STAGING_DIR and parent.exists():
            with contextlib.suppress(OSError):
                parent.rmdir()
    if raw_zip is not None and not KEEP_RAW_ZIP_FALLBACK:
        Path(raw_zip).unlink(missing_ok=True)


def process_scene_once(
    scene: dict[str, Any], config_hash: str,
    lakes_all: gpd.GeoDataFrame,
    username: str, password: str,
    download_method: str = DOWNLOAD_METHOD,
    strategy: str = PROCESSING_STRATEGY,
) -> pd.DataFrame:
    scene_id = str(scene["id"])
    existing = registry_table(config_hash)
    row = existing[existing["scene_id"] == scene_id]
    if not row.empty and row.iloc[0]["status"] == "done":
        stats_path = STATS_SCENE_DIR / f"{scene_id}_{config_hash}.parquet"
        if stats_path.exists():
            print("Already complete; reusing", scene_id)
            return pd.read_parquet(stats_path)
    scene_lakes = candidate_lakes_for_scene(scene, lakes_all)
    if scene_lakes.empty:
        registry_update(scene_id, config_hash, status="skipped_no_lakes", candidate_lakes=0)
        return pd.DataFrame()
    registry_update(scene_id, config_hash, status="running", attempts=int(row.iloc[0]["attempts"] if not row.empty else 0)+1)
    raw_zip = None; sen3 = None; l2_paths: list[Path] = []
    success = False
    try:
        raw_zip, sen3, used_method = stage_scene_efficient(scene, username, password, download_method)
        windows = plan_scene_windows(scene, scene_lakes, strategy=strategy)
        registry_update(scene_id, config_hash, processing_strategy=("full_scene" if windows == [None] else "adaptive_lake_windows"),
                        window_count=len(windows), download_method=used_method)
        all_stats = []
        total_compact = 0
        window_provenance = []
        out_dir = _scene_output_dir(scene)
        out_dir.mkdir(parents=True, exist_ok=True)
        for window in windows:
            win_id = window_id(window)
            win_lakes = lakes_for_window(scene_lakes, window)
            compact_path = out_dir / f"{scene_id}_{win_id}_{config_hash}_lakepixels.nc"
            if compact_path.exists():
                print("Reusing completed compact window", compact_path.name)
                stats = compact_stats(compact_path, win_lakes)
                all_stats.append(stats)
                total_compact += compact_path.stat().st_size
                continue
            l2_path, l2_info = run_l2gen(
                sen3, mode=L2_MODE,
                process_full_scene=(window is None), bbox=window,
                full_spectrum=False,
            )
            l2_paths.append(Path(l2_path))
            window_provenance.append({
                "window_id": win_id,
                "bbox": None if window is None else list(window),
                "l2gen": {k: v for k, v in l2_info.items() if k not in {"variables"}},
            })
            compact, info = extract_lake_only_netcdf(
                l2_path, scene, win_lakes, compact_path, window=window,
                config_hash=config_hash, shore_buffer_m=DEFAULT_SHORE_BUFFER_M,
            )
            if compact is not None:
                stats = compact_stats(compact, win_lakes)
                all_stats.append(stats)
                total_compact += compact.stat().st_size
            if not KEEP_L2_INTERMEDIATE:
                Path(l2_path).unlink(missing_ok=True)
        stats_table = pd.concat(all_stats, ignore_index=True) if all_stats else pd.DataFrame()
        if not stats_table.empty:
            stats_table = stats_table.drop_duplicates(["scene_id", "window_id", "Hylak_id"])
        STATS_SCENE_DIR.mkdir(parents=True, exist_ok=True)
        stats_path = STATS_SCENE_DIR / f"{scene_id}_{config_hash}.parquet"
        stats_table.to_parquet(stats_path, index=False)
        csv_path = stats_path.with_suffix(".csv")
        stats_table.to_csv(csv_path, index=False)
        manifest = {
            "schema": "global-efficient-v2.5",
            "created_utc": utc_now(),
            "scene": scene,
            "config_hash": config_hash,
            "candidate_lakes": int(len(scene_lakes)),
            "windows": ["FULL" if w is None else list(w) for w in windows],
            "window_provenance": window_provenance,
            "compact_files": sorted(str(p) for p in out_dir.glob("*.nc")),
            "stats_parquet": str(stats_path),
            "download_method": used_method,
            "cleanup": {
                "keep_staged_l1": KEEP_STAGED_L1,
                "keep_l2_intermediate": KEEP_L2_INTERMEDIATE,
                "keep_raw_zip_fallback": KEEP_RAW_ZIP_FALLBACK,
            },
        }
        write_json(EXPORT_DIR / scene_id / f"manifest_{config_hash}.json", manifest)
        registry_update(scene_id, config_hash, status="done", compact_bytes=int(total_compact),
                        stats_rows=int(len(stats_table)), last_error=None)
        with contextlib.suppress(Exception):
            registry_export_csv()
        success = True
        return stats_table
    except Exception as exc:
        registry_update(scene_id, config_hash, status="failed", last_error=f"{type(exc).__name__}: {exc}")
        with contextlib.suppress(Exception):
            registry_export_csv()
        raise
    finally:
        _cleanup_scene_scratch(raw_zip, sen3, l2_paths, success=success)


def load_scene_from_registry(scene_id: str, config_hash: str) -> dict[str, Any]:
    table = registry_table(config_hash)
    row = table[table.scene_id.astype(str) == str(scene_id)]
    if row.empty:
        raise KeyError(scene_id)
    return json.loads(row.iloc[0]["scene_json"])


def process_next_queued(
    config_hash: str, lakes: gpd.GeoDataFrame,
    username: str, password: str,
    n_scenes: int = 1,
    download_method: str = DOWNLOAD_METHOD,
    retry_failed: bool = False,
    scene_ids: list[str] | None = None,
) -> pd.DataFrame:
    table = registry_table(config_hash)
    if scene_ids:
        wanted = {str(x) for x in scene_ids}
        table = table[table.scene_id.astype(str).isin(wanted)].copy()
    statuses = ["queued"] + (["failed"] if retry_failed else [])
    queue = table[table.status.isin(statuses)].head(int(n_scenes))
    outputs = []
    try:
        for _, row in queue.iterrows():
            scene = json.loads(row.scene_json)
            try:
                result = process_scene_once(scene, config_hash, lakes, username, password, download_method)
                if not result.empty:
                    outputs.append(result)
            except Exception as exc:
                print(f"Scene failed {row.scene_name}: {exc}")
    finally:
        # CDSE's official S3 example deletes temporary keys after use. Do the same
        # once per batch; the key is reused across all scenes within this batch.
        with contextlib.suppress(Exception):
            release_temporary_s3_credentials(username, password)
    return pd.concat(outputs, ignore_index=True) if outputs else pd.DataFrame()


def all_stats(config_hash: str | None = None) -> pd.DataFrame:
    files = sorted(STATS_SCENE_DIR.glob("*.parquet"))
    if config_hash:
        files = [p for p in files if p.stem.endswith("_" + config_hash)]
    frames = []
    for p in files:
        with contextlib.suppress(Exception):
            frames.append(pd.read_parquet(p))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def storage_summary() -> pd.DataFrame:
    def tree_size(path: Path):
        if not path.exists(): return 0
        return sum(p.stat().st_size for p in path.rglob("*") if p.is_file())
    rows = []
    for label, path in (
        ("Runtime lake-only outputs", COMPACT_DIR),
        ("Runtime logs/exports", EXPORT_DIR),
        ("Scratch staged L1", STAGING_DIR),
        ("Scratch temporary L2", L2_DIR),
        ("Scratch fallback ZIP", RAW_L1_DIR),
        ("Scratch compact-file temp", SCRATCH_ROOT / "compact_tmp"),
    ):
        size = tree_size(path)
        rows.append({"category": label, "path": str(path), "bytes": size, "size": human_size(size)})
    return pd.DataFrame(rows)


def clear_scratch() -> str:
    for path in (STAGING_DIR, L2_DIR, RAW_L1_DIR, DERIVED_DIR, SCRATCH_ROOT / "compact_tmp"):
        shutil.rmtree(path, ignore_errors=True)
        path.mkdir(parents=True, exist_ok=True)
    return f"Scratch cleared: {SCRATCH_ROOT}"


## 9.8 v2.6 spectral/index + archive enhancements

Production keeps the lean bands needed for **true color, CI/CIcyano, NDCI, MPH, and OLCI-adapted FAI/AFAI**. FLH and OLH are deliberately excluded from this branch. The persistent compact product still contains only HydroLAKES pixels.

New compact files are organized physically by Sentinel-3 **platform / relative orbit / frame / year / month / day**, while optional 5° grid IDs are catalogue cross-references only and never cause duplicate L2Gen processing.


In [28]:

# =============================================================================
# v2.6 SCIENCE + ARCHIVE ENHANCEMENTS
# =============================================================================
# Lean production band set: enough for CI/CIcyano/NDCI + RGB + MPH/AFAI.
GLOBAL_ANALYSIS_RHOS_WAVELENGTHS = (490, 560, 620, 665, 681, 709, 754, 865, 885)
TRUE_COLOR_RHOS_WAVELENGTHS = (665, 560, 490)  # R, G, B
ADDITIONAL_INDEX_NAMES = ("MPH", "FAI")


def _spectral_line_height(center, left, right, wl_center, wl_left, wl_right):
    center = np.asarray(center, dtype=float)
    left = np.asarray(left, dtype=float)
    right = np.asarray(right, dtype=float)
    baseline = left + (right - left) * ((float(wl_center) - float(wl_left)) / (float(wl_right) - float(wl_left)))
    return center - baseline


def additional_olci_indices(rhos: dict[int, np.ndarray], wavelengths: dict[int, float] | None = None):
    """Compute production OLCI bloom indices from Rayleigh-corrected rhos.

    MPH: maximum peak height among ~681/~709/~754 nm above a common 665--~885 nm baseline.
    FAI: OLCI-adapted floating-algae index (AFAI form), 754 nm above the 665--865 nm baseline.
         This is not the original Hu SWIR-based FAI because OLCI has no suitable SWIR band.
    """
    need = (665, 681, 709, 754, 865, 885)
    missing = [w for w in need if w not in rhos]
    if missing:
        raise KeyError(f"Missing rhos bands required for MPH/FAI: {missing}")
    wl = {w: float((wavelengths or {}).get(w, w)) for w in need}
    finite = {w: np.isfinite(rhos[w]) for w in need}

    fai = _spectral_line_height(rhos[754], rhos[665], rhos[865], wl[754], wl[665], wl[865])
    fai_valid = finite[754] & finite[665] & finite[865]

    mph_candidates = []
    mph_valid_components = []
    for peak in (681, 709, 754):
        h = _spectral_line_height(rhos[peak], rhos[665], rhos[885], wl[peak], wl[665], wl[885])
        v = finite[peak] & finite[665] & finite[885]
        mph_candidates.append(np.where(v, h, np.nan))
        mph_valid_components.append(v)
    stack = np.stack(mph_candidates, axis=0)
    with np.errstate(all="ignore"):
        mph = np.nanmax(stack, axis=0)
    mph_valid = np.logical_or.reduce(mph_valid_components) & np.isfinite(mph)
    peak_waves = np.asarray([wl[681], wl[709], wl[754]], dtype=np.float32)
    safe = np.where(np.isfinite(stack), stack, -np.inf)
    peak_idx = np.argmax(safe, axis=0)
    mph_peak_nm = peak_waves[peak_idx]
    mph_peak_nm[~mph_valid] = np.nan

    return {
        "MPH": mph, "MPH_valid": mph_valid, "MPH_peak_nm": mph_peak_nm,
        "FAI": fai, "FAI_valid": fai_valid,
    }


# Override the lean L2 product request. Full-spectrum mode still retains the original option.
# IMPORTANT: NASA/OBPG names the Sentinel-3 OLCI ~885 nm rhos product `rhos_884`.
# We keep 885 as the nominal scientific wavelength key used by MPH, but translate
# it only at the L2Gen product-request boundary. Downstream nearest_product(..., 885)
# resolves rhos_884 and records the actual wavelength from the product name.
OLCI_L2GEN_RHOS_NAME_OVERRIDES = {885: 884}

def _l2gen_rhos_product_name(wave: int) -> str:
    return f"rhos_{OLCI_L2GEN_RHOS_NAME_OVERRIDES.get(int(wave), int(wave))}"

def l2_products(mode: str, full_spectrum: bool = False) -> tuple[list[str], set[str]]:
    mode = mode.lower()
    if mode in {"full_swath_rhos", "rayleigh_ci"}:
        mode = "full_swath_rhos"
    if mode not in {"full_swath_rhos", "standard_dual"}:
        raise ValueError("mode must be 'full_swath_rhos' or 'standard_dual'")
    waves = SPECTRAL_RHOS_WAVELENGTHS if full_spectrum else GLOBAL_ANALYSIS_RHOS_WAVELENGTHS
    products = [_l2gen_rhos_product_name(wave) for wave in waves]
    required = {_l2gen_rhos_product_name(wave) for wave in GLOBAL_ANALYSIS_RHOS_WAVELENGTHS}
    if mode == "standard_dual":
        rrs_waves = AQUATIC_RRS_WAVELENGTHS if full_spectrum else (665, 709)
        products.extend(f"Rrs_{wave}" for wave in rrs_waves)
        required.update({"Rrs_665", "Rrs_709"})
    return list(dict.fromkeys(products)), required


def processing_config_hash(min_area_km2: float, region: str = "WORLD") -> str:
    payload = {
        "l2_mode": L2_MODE,
        "ndci_source": NDCI_SOURCE,
        "analysis_rhos": list(GLOBAL_ANALYSIS_RHOS_WAVELENGTHS),
        "true_color_rhos": list(TRUE_COLOR_RHOS_WAVELENGTHS),
        "indices": ["CI", "CI_cyano", "NDCI", *ADDITIONAL_INDEX_NAMES],
        "full_spectrum": bool(SAVE_FULL_REFLECTANCE_SPECTRA),
        "qa_flags": list(QA_EXCLUDE_FLAGS),
        "land_adjacency_pixels": int(LAND_ADJACENCY_PIXELS),
        "require_l2_nonland": bool(REQUIRE_L2_NONLAND),
        "detection_limit": float(CI_DETECTION_LIMIT),
        "bright_screen": bool(DEFAULT_BRIGHT_SCREEN),
        "bright_rhos865": float(DEFAULT_BRIGHT_RHOS865),
        "shore_buffer_m": float(DEFAULT_SHORE_BUFFER_M),
        "processing_strategy": PROCESSING_STRATEGY,
        "window_margin_km": float(L2_WINDOW_MARGIN_KM),
        "max_windows": int(MAX_L2_WINDOWS_PER_SCENE),
        "full_scene_fraction": float(FULL_SCENE_FRACTION_THRESHOLD),
        "min_lake_area_km2": float(min_area_km2),
        "region": str(region).upper(),
        "schema": "global-efficient-v2.4",
    }
    return hashlib.sha1(json.dumps(payload, sort_keys=True).encode()).hexdigest()[:16]


# Sentinel-3 native frame identity: the closest analogue to a path/row catalogue key.
S3_FRAME_RE = re.compile(
    r"^(S3[AB])_OL_1_EFR____(\d{8}T\d{6})_(\d{8}T\d{6})_(\d{8}T\d{6})_(\d{4})_(\d{3})_(\d{3})_(\d{4})_"
)

def parse_s3_frame_metadata(name: str) -> dict[str, Any]:
    n = Path(str(name)).name.removesuffix(".SEN3").removesuffix(".zip")
    m = S3_FRAME_RE.match(n)
    if not m:
        raise ValueError(f"Cannot parse Sentinel-3 OLCI frame fields from {n}")
    platform, start, stop, created, duration, cycle, relative_orbit, frame = m.groups()
    return {
        "scene_name": n, "platform": platform, "sensing_start": start,
        "sensing_stop": stop, "creation_time": created, "duration_s": int(duration),
        "cycle": int(cycle), "relative_orbit": int(relative_orbit), "frame": int(frame),
        "native_frame_id": f"{platform}_R{int(relative_orbit):03d}_F{int(frame):04d}",
    }

def _scene_output_dir(scene: dict[str, Any]) -> Path:
    """Physical persistent layout: platform / relative orbit / frame / date / scene ID."""
    stamp = pd.Timestamp(scene.get("start") or "1970-01-01")
    try:
        meta = parse_s3_frame_metadata(scene.get("name", ""))
        return (COMPACT_NC_DIR / f"platform={meta['platform']}" /
                f"relative_orbit={meta['relative_orbit']:03d}" /
                f"frame={meta['frame']:04d}" / f"year={stamp.year:04d}" /
                f"month={stamp.month:02d}" / f"day={stamp.day:02d}" / str(scene["id"]))
    except Exception:
        return COMPACT_NC_DIR / "platform=UNKNOWN" / f"year={stamp.year:04d}" / f"month={stamp.month:02d}" / str(scene["id"])


def _open_compact_writer_v22(path: Path, metadata: dict[str, Any]):
    path.parent.mkdir(parents=True, exist_ok=True)
    # Runtime-local atomic write; v2.6 never mounts Google Drive.
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.unlink(missing_ok=True)
    ds = netCDF4.Dataset(tmp, "w", format="NETCDF4")
    ds.createDimension("obs", None)
    for key, value in metadata.items():
        if value is not None:
            try: ds.setncattr(key, value)
            except Exception: ds.setncattr(key, str(value))
    chunk = (COMPACT_CHUNK_OBS,)
    out = {}
    out["Hylak_id"] = ds.createVariable("Hylak_id", "i8", ("obs",), zlib=True, complevel=COMPACT_DEFLATE, shuffle=True, chunksizes=chunk)
    out["source_row"] = ds.createVariable("source_row", "i4", ("obs",), zlib=True, complevel=COMPACT_DEFLATE, shuffle=True, chunksizes=chunk)
    out["source_col"] = ds.createVariable("source_col", "i4", ("obs",), zlib=True, complevel=COMPACT_DEFLATE, shuffle=True, chunksizes=chunk)
    out["l2_flags"] = ds.createVariable("l2_flags", "u4", ("obs",), zlib=True, complevel=COMPACT_DEFLATE, shuffle=True, chunksizes=chunk)
    for name in (
        "valid_water_mask", "ci_valid_mask", "ci_candidate_mask", "ci_detection_mask", "ndci_valid_mask",
        "mph_valid_mask", "fai_valid_mask", 
    ):
        out[name] = ds.createVariable(name, "u1", ("obs",), zlib=True, complevel=COMPACT_DEFLATE, shuffle=True, chunksizes=chunk)
    float_names = ["latitude", "longitude"] + [f"rhos_{w}" for w in GLOBAL_ANALYSIS_RHOS_WAVELENGTHS] + [
        "SS_681", "SS_665", "CI", "CI_cyano", "NDCI", "MPH", "MPH_peak_nm", "FAI"
    ]
    for name in float_names:
        out[name] = ds.createVariable(name, "f4", ("obs",), zlib=True, complevel=COMPACT_DEFLATE, shuffle=True, chunksizes=chunk, fill_value=DERIVED_FILL)
    out["latitude"].setncatts({"standard_name":"latitude", "units":"degrees_north"})
    out["longitude"].setncatts({"standard_name":"longitude", "units":"degrees_east"})
    for w in GLOBAL_ANALYSIS_RHOS_WAVELENGTHS:
        out[f"rhos_{w}"].setncatts({"long_name":f"Rayleigh-corrected OLCI reflectance near {w} nm", "units":"1"})
    out["MPH"].setncatts({"long_name":"Maximum Peak Height reflectance index", "units":"1", "formula":"max line height at 681/709/754 nm above 665-885 nm baseline"})
    out["MPH_peak_nm"].setncatts({"long_name":"wavelength of MPH maximum", "units":"nm"})
    out["FAI"].setncatts({"long_name":"OLCI-adapted Floating Algae Index (AFAI form)", "units":"1", "formula":"rhos754 - linear baseline(rhos665,rhos865)", "note":"Not the original SWIR-based Hu FAI; OLCI lacks the required SWIR band."})
    return ds, out, tmp


def extract_lake_only_netcdf(
    l2_path: Path, scene: dict[str, Any], lakes: gpd.GeoDataFrame, output_path: Path,
    window=None, qa_flags: Iterable[str] = QA_EXCLUDE_FLAGS,
    land_adjacency_pixels: int = LAND_ADJACENCY_PIXELS,
    require_l2_nonland: bool = REQUIRE_L2_NONLAND,
    detection_limit: float = CI_DETECTION_LIMIT, ss681_max: float = 0.0, ss665_min: float = 0.0,
    bright_screen: bool = DEFAULT_BRIGHT_SCREEN, bright_rhos865: float = DEFAULT_BRIGHT_RHOS865,
    rows_per_block: int = ROW_BLOCK_SIZE, config_hash: str | None = None,
    shore_buffer_m: float = DEFAULT_SHORE_BUFFER_M,
) -> tuple[Path | None, dict[str, Any]]:
    """Persist only HydroLAKES pixels, with RGB bands and CI/CIcyano/NDCI/MPH/FAI."""
    output_path = Path(output_path)
    analysis_lakes = erode_lake_geometries(lakes, float(shore_buffer_m)) if float(shore_buffer_m) > 0 else lakes
    if analysis_lakes is None or analysis_lakes.empty:
        return None, {"nobs":0, "missing_flags":[], "reason":"no lake area after shoreline buffer"}
    with netCDF4.Dataset(l2_path) as src:
        geo = src.groups["geophysical_data"]; nav = src.groups["navigation_data"]
        flags_var = locate_l2_flags(src); flag_map = decode_flag_metadata(flags_var)
        if "LAND" not in flag_map: raise RuntimeError("L2 l2_flags lacks LAND")
        rhos_names = {w: nearest_product(geo.variables, "rhos", w, tolerance_nm=30) for w in GLOBAL_ANALYSIS_RHOS_WAVELENGTHS}
        actual_waves = {w: product_wavelength(rhos_names[w]) for w in GLOBAL_ANALYSIS_RHOS_WAVELENGTHS}
        ndci_prefix = "rhos" if NDCI_SOURCE.lower() == "rhos" else "Rrs"
        ndci_names = {665: nearest_product(geo.variables, ndci_prefix, 665), 709: nearest_product(geo.variables, ndci_prefix, 709)}
        shape_2d = nav.variables["latitude"].shape
        metadata = {
            "Conventions":"CF-1.8 ACDD-1.3", "title":"Sentinel-3 OLCI derived lake-pixel point collection",
            "schema":"global-efficient-v2.5", "processing_config_hash":str(config_hash or ""),
            "scene_id":str(scene.get("id","")), "scene_name":str(scene.get("name","")),
            "acquisition_start":str(scene.get("start","")), "acquisition_end":str(scene.get("end","")),
            "window_id":window_id(window), "window_bbox":"FULL" if window is None else ",".join(map(str,window)),
            "source_l2":str(Path(l2_path).name), "l2gen_version":command_version(L2GEN_BIN),
            "qa_exclude_flags":",".join(map(str,qa_flags)), "l2_flag_map_json":json.dumps({k:int(v) for k,v in flag_map.items()},sort_keys=True),
            "require_l2_nonland":int(bool(require_l2_nonland)), "shore_buffer_m":float(shore_buffer_m),
            "ci_detection_limit":float(detection_limit), "ss681_max":float(ss681_max), "ss665_min":float(ss665_min),
            "non_lake_storage":"omitted entirely; not dense NaN cells", "masked_value_semantics":"_FillValue decodes as NaN",
            "true_color_bands":"rhos665,rhos560,rhos490", "additional_indices":"MPH, OLCI-adapted FAI/AFAI",
            "l2gen_nominal_band_mapping":"nominal 885 nm -> L2Gen rhos_884",
        }
        ds, out, tmp = _open_compact_writer_v22(output_path, metadata)
        nobs=0; requested_flags=tuple(dict.fromkeys(str(x).upper() for x in qa_flags)); missing_flags=[x for x in requested_flags if x not in flag_map]
        try:
            for start in tqdm(range(0, shape_2d[0], int(rows_per_block)), desc=f"Lake pixels {window_id(window)}"):
                stop=min(shape_2d[0], start+int(rows_per_block)); rs=slice(start,stop)
                lat=to_float(nav.variables["latitude"][rs,:]); lon=to_float(nav.variables["longitude"][rs,:])
                lake_ids=_membership_block(lat,lon,analysis_lakes); inside=lake_ids>0
                if not inside.any(): continue
                flags_ma=np.ma.asarray(flags_var[rs,:]); flags_valid=~np.ma.getmaskarray(flags_ma); flags=flags_ma.filled(0).astype(np.uint32)
                geolocated=np.isfinite(lat)&np.isfinite(lon)&flags_valid; land=flag_is_set(flags,flag_map,"LAND")
                base=inside&geolocated
                if require_l2_nonland: base &= ~land
                excluded=np.zeros(base.shape,dtype=bool)
                for fname in requested_flags:
                    if fname in flag_map: excluded |= flag_is_set(flags,flag_map,fname)
                if int(land_adjacency_pixels)>0: excluded |= binary_dilation(land,iterations=int(land_adjacency_pixels)) & ~land
                arrays={w:to_float(geo.variables[rhos_names[w]][rs,:]) for w in GLOBAL_ANALYSIS_RHOS_WAVELENGTHS}
                if bright_screen: excluded |= np.isfinite(arrays[865]) & (arrays[865] > float(bright_rhos865))
                valid_water=base & ~excluded
                family=ci_family(arrays[620],arrays[665],arrays[681],arrays[709], wavelengths=tuple(actual_waves[w] for w in CORE_RHOS_WAVELENGTHS), detection_limit=detection_limit, ss681_max=ss681_max, ss665_min=ss665_min)
                ci_valid=valid_water & family["finite"]; candidate=ci_valid & family["ci_candidate"]; detected=ci_valid & family["ci_detection"]
                red=to_float(geo.variables[ndci_names[665]][rs,:]); edge=to_float(geo.variables[ndci_names[709]][rs,:]); ndci,ndci_finite=ndci_index(red,edge); ndci_valid=valid_water&ndci_finite
                extra=additional_olci_indices(arrays,actual_waves)
                extra_valid={k: valid_water & extra[f"{k}_valid"] for k in ADDITIONAL_INDEX_NAMES}
                take=inside&geolocated; rr,cc=np.where(take)
                if not rr.size: continue
                count=len(rr); sl=slice(nobs,nobs+count)
                out["Hylak_id"][sl]=lake_ids[rr,cc]; out["source_row"][sl]=rr.astype(np.int32)+int(start); out["source_col"][sl]=cc.astype(np.int32); out["l2_flags"][sl]=flags[rr,cc]
                for name,mask in {
                    "valid_water_mask":valid_water, "ci_valid_mask":ci_valid, "ci_candidate_mask":candidate, "ci_detection_mask":detected, "ndci_valid_mask":ndci_valid,
                    "mph_valid_mask":extra_valid["MPH"], "fai_valid_mask":extra_valid["FAI"],
                }.items(): out[name][sl]=mask[rr,cc].astype(np.uint8)
                _append_float(out["latitude"],nobs,lat[rr,cc]); _append_float(out["longitude"],nobs,lon[rr,cc])
                for w in GLOBAL_ANALYSIS_RHOS_WAVELENGTHS: _append_float(out[f"rhos_{w}"],nobs,arrays[w][rr,cc])
                _append_float(out["SS_681"],nobs,np.where(ci_valid,family["SS_681"],np.nan)[rr,cc]); _append_float(out["SS_665"],nobs,np.where(ci_valid,family["SS_665"],np.nan)[rr,cc])
                _append_float(out["CI"],nobs,np.where(candidate,family["CI"],np.nan)[rr,cc]); _append_float(out["CI_cyano"],nobs,np.where(detected,family["CI_cyano"],np.nan)[rr,cc]); _append_float(out["NDCI"],nobs,np.where(ndci_valid,ndci,np.nan)[rr,cc])
                _append_float(out["MPH"],nobs,np.where(extra_valid["MPH"],extra["MPH"],np.nan)[rr,cc]); _append_float(out["MPH_peak_nm"],nobs,np.where(extra_valid["MPH"],extra["MPH_peak_nm"],np.nan)[rr,cc])
                _append_float(out["FAI"],nobs,np.where(extra_valid["FAI"],extra["FAI"],np.nan)[rr,cc])
                nobs += count
        finally: ds.close()
        if nobs==0:
            tmp.unlink(missing_ok=True); return None,{"nobs":0,"missing_flags":missing_flags}
        if tmp.parent==output_path.parent: tmp.replace(output_path)
        else:
            transfer=output_path.with_suffix(output_path.suffix+".copying"); transfer.unlink(missing_ok=True); shutil.copy2(tmp,transfer); transfer.replace(output_path); tmp.unlink(missing_ok=True)
    return output_path,{"nobs":int(nobs),"size_bytes":int(output_path.stat().st_size),"size_human":human_size(output_path.stat().st_size),"missing_flags":missing_flags,"window_id":window_id(window)}


def compact_stats(nc_path: Path, lake_metadata: gpd.GeoDataFrame | None = None) -> pd.DataFrame:
    with netCDF4.Dataset(nc_path) as ds:
        data={
            "Hylak_id":np.asarray(ds.variables["Hylak_id"][:],dtype=np.int64),
            "valid":np.asarray(ds.variables["valid_water_mask"][:])==1,
            "ci_valid":np.asarray(ds.variables["ci_valid_mask"][:])==1,
            "candidate":np.asarray(ds.variables["ci_candidate_mask"][:])==1,
            "detected":np.asarray(ds.variables["ci_detection_mask"][:])==1,
        }
        for name in ("CI","CI_cyano","NDCI","MPH","FAI"):
            data[name]=to_float(ds.variables[name][:]) if name in ds.variables else np.full(len(data["Hylak_id"]),np.nan)
        frame=pd.DataFrame(data); attrs={k:getattr(ds,k,None) for k in ("scene_id","scene_name","acquisition_start","window_id")}
    meta={}
    if lake_metadata is not None and not lake_metadata.empty:
        for _,row in lake_metadata.iterrows(): meta[int(row["Hylak_id"])]={"Lake_name":row.get("Lake_name","") or "","Lake_area_km2":float(row.get("Lake_area",np.nan)),"Country":row.get("Country","") if "Country" in row else "","Continent":row.get("Continent","") if "Continent" in row else ""}
    def vals(series):
        a=pd.to_numeric(series,errors="coerce").to_numpy(float); return a[np.isfinite(a)]
    def summary(prefix,a):
        if not len(a): return {f"{prefix}_mean_retained":np.nan,f"{prefix}_median_retained":np.nan,f"{prefix}_p10_retained":np.nan,f"{prefix}_p90_retained":np.nan,f"{prefix}_max_retained":np.nan}
        return {f"{prefix}_mean_retained":float(np.nanmean(a)),f"{prefix}_median_retained":float(np.nanmedian(a)),f"{prefix}_p10_retained":float(np.nanpercentile(a,10)),f"{prefix}_p90_retained":float(np.nanpercentile(a,90)),f"{prefix}_max_retained":float(np.nanmax(a))}
    records=[]
    for lake_id,g in frame.groupby("Hylak_id",sort=False):
        ci=vals(g.loc[g["candidate"],"CI"]); cy=vals(g.loc[g["detected"],"CI_cyano"]); nd=vals(g.loc[g["valid"],"NDCI"])
        valid_n=int(g["valid"].sum()); ci_valid_n=int(g["ci_valid"].sum()); detected_n=int(g["detected"].sum())
        rec={**attrs,"Hylak_id":int(lake_id),**meta.get(int(lake_id),{}),"native_polygon_pixels":int(len(g)),"retained_pixels":valid_n,"retained_percent":100.0*valid_n/len(g) if len(g) else np.nan,"ci_input_valid_pixels":ci_valid_n,"ci_candidate_pixels":int(g["candidate"].sum()),"ci_cyano_detection_pixels":detected_n,"ci_cyano_detection_fraction_of_ci_valid":detected_n/ci_valid_n if ci_valid_n else np.nan,"ci_cyano_detection_fraction_of_retained":detected_n/valid_n if valid_n else np.nan,"CI_mean_candidates":float(np.nanmean(ci)) if len(ci) else np.nan,"CI_median_candidates":float(np.nanmedian(ci)) if len(ci) else np.nan,"CI_p10_candidates":float(np.nanpercentile(ci,10)) if len(ci) else np.nan,"CI_p90_candidates":float(np.nanpercentile(ci,90)) if len(ci) else np.nan,"CI_max_candidates":float(np.nanmax(ci)) if len(ci) else np.nan,"CIcyano_mean_detections":float(np.nanmean(cy)) if len(cy) else np.nan,"CIcyano_median_detections":float(np.nanmedian(cy)) if len(cy) else np.nan,"CIcyano_p90_detections":float(np.nanpercentile(cy,90)) if len(cy) else np.nan,"NDCI_mean_retained":float(np.nanmean(nd)) if len(nd) else np.nan,"NDCI_median_retained":float(np.nanmedian(nd)) if len(nd) else np.nan,"NDCI_p90_retained":float(np.nanpercentile(nd,90)) if len(nd) else np.nan,"compact_netcdf":str(nc_path)}
        for k in ADDITIONAL_INDEX_NAMES: rec.update(summary(k,vals(g.loc[g["valid"],k])))
        records.append(rec)
    return pd.DataFrame(records)


print("v2.3 enhancements active:")
print("  lean rhos bands:", GLOBAL_ANALYSIS_RHOS_WAVELENGTHS)
print("  true color RGB :", TRUE_COLOR_RHOS_WAVELENGTHS)
print("  extra indices  :", ADDITIONAL_INDEX_NAMES)


v2.3 enhancements active:
  lean rhos bands: (490, 560, 620, 665, 681, 709, 754, 865, 885)
  true color RGB : (665, 560, 490)
  extra indices  : ('MPH', 'FAI')


## 10. Global Gradio dashboard v2.6

This is the recommended production interface. It combines scene/all-lakes visualization, true color, CI/CIcyano/NDCI/MPH/FAI map layers, corrected Web-Mercator basemap alignment, scalar legends, hoverable portable HTML, SNAP/GIS CF NetCDF export, lake spectra/statistics, and a clear native orbit/frame global archive catalogue.


In [29]:
# Gradio dashboard v2.6.1: scene/all-lakes + one-lake maps, RGB, legends, hover HTML, SNAP-friendly gridded NetCDF.
import base64
import html as html_lib
import io
import json
import math
import re
import contextlib
import tempfile
import zipfile
import datetime as dt
import netrc as netrc_module
from pathlib import Path
from urllib.parse import urlparse, quote

import folium
from folium import plugins as folium_plugins
import gradio as gr
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import netCDF4
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from PIL import Image
from pyproj import Transformer
from scipy.spatial import cKDTree
from scipy import stats as scipy_stats
import rasterio
import requests
import xarray as xr
import pycountry
import geopandas as gpd
from shapely.geometry import box as shapely_box

GLOBAL_STATE = {"lakes": None, "config_hash": None, "min_area_km2": 10.0, "region": "WORLD",
                "planned_scene_ids": [], "planning_shard": "", "planning_geometry": None}
ADM1_STATE = {"country_iso3": None, "gdf": None, "lookup": {}, "metadata": {}}
COUNTRY_CHOICES = [(f"{c.name} ({c.alpha_3})", c.alpha_3) for c in sorted(pycountry.countries, key=lambda c: c.name)]
GEOB_API = "https://www.geoboundaries.org/api/current/gbOpen/{iso3}/ADM1/"
BASEMAPS = ["Esri Light Gray","Esri Dark Gray","Esri World Imagery","OpenStreetMap","None"]
SCALAR_METRICS = ["CI","CI_cyano","NDCI","MPH","FAI"] + [f"rhos_{w}" for w in GLOBAL_ANALYSIS_RHOS_WAVELENGTHS]
MAP_METRICS = ["true_color", *SCALAR_METRICS]
PLOT_METRICS = [
    "ci_cyano_detection_fraction_of_ci_valid","ci_cyano_detection_fraction_of_retained",
    "CI_mean_candidates","CI_median_candidates","CIcyano_mean_detections","CIcyano_median_detections",
    "NDCI_mean_retained","NDCI_median_retained","MPH_mean_retained","FAI_mean_retained","retained_percent",
]
CMAPS=["viridis","plasma","inferno","magma","cividis","turbo","BrBG","Spectral","RdYlBu_r","YlGnBu"]
DISPLAY_MAX_CELLS=8_000_000
SCIENCE_EXPORT_MAX_CELLS=25_000_000
SCIENCE_EXPORT_DEFAULT_RESOLUTION_M=300
SCIENCE_EXPORT_RHOS=[f"rhos_{w}" for w in GLOBAL_ANALYSIS_RHOS_WAVELENGTHS]
SCIENCE_EXPORT_INDICES=["CI","CI_cyano","NDCI","MPH","FAI"]
SCIENCE_EXPORT_VARIABLES=[*SCIENCE_EXPORT_RHOS,*SCIENCE_EXPORT_INDICES]
CYAN_STATE={"source_path":None,"source_date":None}


def _app_status(title, detail="", ok=True):
    color="#157347" if ok else "#b42318"
    return f"<div style='padding:10px 12px;border-left:5px solid {color};background:#f8fafc;border-radius:6px'><b style='color:{color}'>{html_lib.escape(str(title))}</b><br>{html_lib.escape(str(detail)).replace(chr(10),'<br>')}</div>"


def _safe_admin_slug(text):
    text = re.sub(r"[^A-Za-z0-9._-]+", "_", str(text or "").strip())
    return text.strip("_") or "region"


def _adm1_name_column(gdf):
    for name in ("shapeName", "NAME_1", "name", "Name", "NAME", "admin1Name", "ADM1_EN"):
        if name in gdf.columns:
            return name
    # last-resort string column
    for name in gdf.columns:
        if name != gdf.geometry.name and gdf[name].dtype == object:
            return name
    raise ValueError("Could not identify an ADM1 name column in the boundary file")


def _load_geoboundaries_adm1(iso3):
    iso3 = str(iso3 or "").upper().strip()
    if len(iso3) != 3:
        raise ValueError("Choose a country first")
    ADMIN_ROOT.mkdir(parents=True, exist_ok=True)
    meta_path = ADMIN_ROOT / f"geoBoundaries_{iso3}_ADM1_metadata.json"
    geo_path = ADMIN_ROOT / f"geoBoundaries_{iso3}_ADM1_simplified.geojson"
    metadata = None
    if meta_path.exists() and geo_path.exists():
        with contextlib.suppress(Exception):
            metadata = json.loads(meta_path.read_text())
    if metadata is None:
        url = GEOB_API.format(iso3=iso3)
        r = requests.get(url, timeout=90)
        r.raise_for_status()
        metadata = r.json()
        gj = metadata.get("simplifiedGeometryGeoJSON") or metadata.get("gjDownloadURL")
        if not gj:
            raise RuntimeError(f"geoBoundaries returned no ADM1 GeoJSON URL for {iso3}")
        rr = requests.get(gj, timeout=180)
        rr.raise_for_status()
        geo_path.write_bytes(rr.content)
        meta_path.write_text(json.dumps(metadata, indent=2))
    gdf = gpd.read_file(geo_path)
    if gdf.crs is None:
        gdf = gdf.set_crs(4326)
    else:
        gdf = gdf.to_crs(4326)
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy().reset_index(drop=True)
    if gdf.empty:
        raise RuntimeError(f"ADM1 boundary file for {iso3} contains no usable geometries")
    name_col = _adm1_name_column(gdf)
    lookup = {}
    choices = []
    for idx, row in gdf.iterrows():
        name = str(row.get(name_col, f"ADM1 {idx+1}") or f"ADM1 {idx+1}")
        code = ""
        for candidate in ("shapeISO", "HASC_1", "iso_3166_2", "ISO_3166_2"):
            if candidate in gdf.columns and pd.notna(row.get(candidate)):
                code = str(row.get(candidate)); break
        shape_id = str(row.get("shapeID", idx)) if "shapeID" in gdf.columns else str(idx)
        key = f"{idx}:{shape_id}"
        label = f"{name} ({code})" if code and code.lower() != "nan" else name
        lookup[key] = int(idx)
        choices.append((label, key))
    ADM1_STATE.update(country_iso3=iso3, gdf=gdf, lookup=lookup, metadata=metadata)
    return choices, metadata


def load_adm1_callback(country_iso3):
    try:
        choices, meta = _load_geoboundaries_adm1(country_iso3)
        value = choices[0][1] if choices else None
        detail = (
            f"{len(choices)} ADM1 units loaded for {meta.get('boundaryName', country_iso3)}; "
            f"year={meta.get('boundaryYearRepresented','unknown')}; "
            f"source={meta.get('boundarySource','geoBoundaries')}; "
            f"license={meta.get('boundaryLicense','see geoBoundaries metadata')}."
        )
        return gr.Dropdown(choices=choices, value=value), _app_status("State/province list ready", detail)
    except Exception as exc:
        ADM1_STATE.update(country_iso3=None, gdf=None, lookup={}, metadata={})
        return gr.Dropdown(choices=[], value=None), _app_status("ADM1 loading failed", str(exc), False)


def _selected_adm1_geometry(country_iso3, admin_key):
    iso3 = str(country_iso3 or "").upper().strip()
    if ADM1_STATE.get("gdf") is None or ADM1_STATE.get("country_iso3") != iso3:
        _load_geoboundaries_adm1(iso3)
    key = str(admin_key or "")
    if key not in ADM1_STATE.get("lookup", {}):
        raise ValueError("Choose a state/province after loading the country ADM1 list")
    idx = ADM1_STATE["lookup"][key]
    row = ADM1_STATE["gdf"].iloc[idx]
    name_col = _adm1_name_column(ADM1_STATE["gdf"])
    name = str(row.get(name_col, key))
    geom = row.geometry
    if geom is None or geom.is_empty:
        raise ValueError("Selected ADM1 geometry is empty")
    if not geom.is_valid:
        geom = geom.buffer(0)
    return geom, name


def _lakes_intersecting_geometry(lakes, geom):
    if lakes is None or lakes.empty:
        return lakes.iloc[0:0].copy()
    try:
        idx = lakes.sindex.query(geom, predicate="intersects")
        out = lakes.iloc[np.asarray(idx, dtype=int)].copy()
    except Exception:
        out = lakes[lakes.intersects(geom)].copy()
    if out.empty:
        return out
    return out[out.intersects(geom)].copy().reset_index(drop=True)


def _plan_catalogue_shard(start_date, end_date, all_lakes, min_area_km2, config_region,
                          bbox, shard_geometry, shard_label, max_products=None):
    """Plan one discovery shard while preserving scene-level global deduplication.

    Only scenes containing target lakes inside the selected shard are queued. Processing
    still receives the canonical loaded HydroLAKES universe, so the first time a frame is
    processed it can retain all target lakes in that frame. Adjacent states therefore reuse
    the same completed Sentinel-3 scene instead of repeating L2Gen.
    """
    if all_lakes is None or all_lakes.empty:
        raise ValueError("Load HydroLAKES before planning scenes")
    candidate_pool = _lakes_intersecting_geometry(all_lakes, shard_geometry) if shard_geometry is not None else all_lakes
    if candidate_pool.empty:
        raise ValueError(f"No target HydroLAKES polygons intersect planning region: {shard_label}")
    ch = processing_config_hash(float(min_area_km2), str(config_region))
    catalog = search_olci_l1_catalog(start_date, end_date, bbox=bbox, max_products=max_products)
    planned_ids = []
    for scene in tqdm(catalog, desc=f"Planning {shard_label}"):
        sg = scene_geometry(scene)
        if shard_geometry is not None and (sg is None or not sg.intersects(shard_geometry)):
            continue
        subset = candidate_lakes_for_scene(scene, candidate_pool)
        if subset.empty:
            continue
        rec = dict(scene)
        rec["candidate_lakes"] = int(len(subset))
        rec["planning_shard"] = str(shard_label)
        registry_upsert_scene(rec, ch, len(subset))
        planned_ids.append(str(rec["id"]))
    table = registry_table(ch)
    if planned_ids:
        table = table[table.scene_id.astype(str).isin(set(planned_ids))].copy()
    else:
        table = table.iloc[0:0].copy()
    plan_dir = CATALOG_DIR / "plans"
    plan_dir.mkdir(parents=True, exist_ok=True)
    stamp = f"{pd.Timestamp(start_date):%Y%m%d}_{pd.Timestamp(end_date):%Y%m%d}"
    plan_csv = plan_dir / f"plan_{_safe_admin_slug(shard_label)}_{stamp}.csv"
    table.to_csv(plan_csv, index=False)
    return table, ch, planned_ids, plan_csv, int(len(candidate_pool))


def _parse_bbox(text):
    text=str(text or '').strip()
    if not text: return None
    parts=[float(x.strip()) for x in text.split(',')]
    if len(parts)!=4: raise ValueError('BBox must be west,south,east,north')
    return normalize_bbox(parts)


def _load_target_lakes_if_needed():
    if GLOBAL_STATE['lakes'] is not None: return GLOBAL_STATE['lakes']
    if TARGET_LAKES_GPKG.exists():
        GLOBAL_STATE['lakes']=_normalize_hydrolakes(gpd.read_file(TARGET_LAKES_GPKG)); return GLOBAL_STATE['lakes']
    raise RuntimeError('Load/prepare HydroLAKES first')


def load_lakes_callback(region,min_area,existing_path):
    try:
        min_area=float(min_area); existing_path=str(existing_path or '').strip()
        if existing_path:
            lakes,path=load_hydrolakes_file(existing_path,min_area)
        else:
            base_region='USA' if str(region).upper()=='USA' else 'WORLD'
            lakes,path=prepare_hydrolakes(min_area,base_region,delete_global_archive=True)
            if str(region).upper() not in {'WORLD','USA'}:
                if 'Continent' not in lakes.columns: raise ValueError('HydroLAKES subset lacks Continent column')
                target=str(region).replace('_',' ').lower(); lakes=lakes[lakes['Continent'].astype(str).str.lower()==target].copy()
        save_target_lakes(lakes)
        GLOBAL_STATE.update(lakes=lakes,min_area_km2=min_area,region=str(region).upper(),
                            planned_scene_ids=[],planning_shard='',planning_geometry=None)
        preview=lakes[[c for c in ['Hylak_id','Lake_name','Lake_area','Country','Continent'] if c in lakes.columns]].head(50)
        note = ' WORLD is recommended for country/state-by-state global production so one canonical lake universe is used for deduplication.' if str(region).upper()!='WORLD' else ''
        return preview,_app_status('HydroLAKES ready',f'{len(lakes):,} target lakes cached.{note}')
    except Exception as exc: return pd.DataFrame(),_app_status('HydroLAKES failed',str(exc),False)


def _queue_view(table):
    if table is None or table.empty: return pd.DataFrame()
    v=table.copy();
    if 'size_bytes' in v: v['L1_size_GB']=pd.to_numeric(v['size_bytes'],errors='coerce')/(1024**3)
    if 'compact_bytes' in v: v['compact_MB']=pd.to_numeric(v['compact_bytes'],errors='coerce')/(1024**2)
    cols=['scene_name','acquisition_start','candidate_lakes','L1_size_GB','status','attempts','processing_strategy','window_count','download_method','compact_MB','stats_rows','last_error']
    return v[[c for c in cols if c in v.columns]]


def plan_callback(start_date,end_date,planning_mode,country_iso3,admin_key,bbox_text,grid_id,max_products):
    try:
        lakes=_load_target_lakes_if_needed()
        mode=str(planning_mode or 'adm1')
        max_products=None if not max_products or int(max_products)<=0 else int(max_products)
        geom=None; bbox=None; shard='WORLD'
        if mode=='adm1':
            geom,name=_selected_adm1_geometry(country_iso3,admin_key)
            bbox=tuple(map(float,geom.bounds))
            country_name=(ADM1_STATE.get('metadata') or {}).get('boundaryName',str(country_iso3))
            shard=f"ADM1 · {country_name} · {name}"
        elif mode=='bbox':
            bbox=_parse_bbox(bbox_text)
            if bbox is None: raise ValueError('Enter bbox west,south,east,north')
            geom=shapely_box(*bbox); shard=f"BBox · {','.join(f'{x:.4f}' for x in bbox)}"
        elif mode=='grid':
            bbox=grid_id_to_bbox(grid_id)
            geom=shapely_box(*bbox); shard=f"Grid · {str(grid_id).strip().upper()}"
        else:
            raise ValueError(f'Unknown planning mode: {mode}')
        table,ch,ids,plan_csv,n_lakes=_plan_catalogue_shard(
            start_date,end_date,lakes,GLOBAL_STATE['min_area_km2'],GLOBAL_STATE['region'],
            bbox,geom,shard,max_products=max_products)
        GLOBAL_STATE.update(config_hash=ch,planned_scene_ids=ids,planning_shard=shard,planning_geometry=geom)
        total=float(pd.to_numeric(table.get('size_bytes',0),errors='coerce').fillna(0).sum())/(1024**3) if len(table) else 0
        done=int((table.status=='done').sum()) if len(table) else 0
        detail=(f'{len(table):,} scene/config records for {shard}; {n_lakes:,} target lakes in shard; '
                f'{done:,} already complete; L1 catalogue volume ≈ {total:,.1f} GiB; config={ch}. '
                f'Plan snapshot: {plan_csv.name}')
        return _queue_view(table),str(plan_csv),_app_status('Scene queue planned',detail)
    except Exception as exc: return pd.DataFrame(),None,_app_status('Planning failed',str(exc),False)


def _current_plan_table():
    ch=GLOBAL_STATE.get('config_hash')
    if not ch: return pd.DataFrame()
    t=registry_table(ch)
    ids={str(x) for x in (GLOBAL_STATE.get('planned_scene_ids') or [])}
    if ids: t=t[t.scene_id.astype(str).isin(ids)].copy()
    return t


def refresh_queue_callback():
    try:
        t=_current_plan_table(); return _queue_view(t),str(registry_export_csv())
    except Exception: return pd.DataFrame(),None


def run_batch_callback(username,password,n_scenes,method,retry_failed):
    try:
        if not str(username or '').strip() or not str(password or ''): raise ValueError('Enter Copernicus Data Space username and password')
        lakes=_load_target_lakes_if_needed(); ch=GLOBAL_STATE.get('config_hash')
        if not ch: raise RuntimeError('Plan the scene queue first')
        before=_current_plan_table(); todo=before.status.isin(['queued']+(['failed'] if retry_failed else [])).sum() if not before.empty else 0
        if todo==0: return _queue_view(before),_app_status('Nothing queued',f"Current shard complete: {GLOBAL_STATE.get('planning_shard','')}"),storage_summary()
        process_next_queued(ch,lakes,username,password,n_scenes=int(n_scenes),download_method=str(method).lower(),retry_failed=bool(retry_failed),scene_ids=GLOBAL_STATE.get('planned_scene_ids'))
        after=_current_plan_table(); return _queue_view(after),_app_status('Batch finished',f"{GLOBAL_STATE.get('planning_shard','')} · queue status: {after.status.value_counts().to_dict()}"),storage_summary()
    except Exception as exc: return pd.DataFrame(),_app_status('Batch failed',str(exc),False),storage_summary()


def _stats(): return all_stats(GLOBAL_STATE.get('config_hash'))

def _lake_choices():
    s=_stats();
    if s.empty: return []
    cols=['Hylak_id']+(['Lake_name'] if 'Lake_name' in s.columns else []); u=s[cols].drop_duplicates().sort_values('Hylak_id')
    return [(f"{str(r.get('Lake_name','') or 'Unnamed lake')} · Hylak_id {int(r.Hylak_id)}",str(int(r.Hylak_id))) for _,r in u.iterrows()]


def refresh_lakes_callback():
    c=_lake_choices(); return gr.Dropdown(choices=c,value=(c[0][1] if c else None)),_app_status('Lake list refreshed',f'{len(c):,} lakes have processed observations')


def lake_timeseries_callback(lake_id,metric):
    try:
        s=_stats(); lake_id=int(lake_id); t=s[s.Hylak_id==lake_id].copy(); t['time']=pd.to_datetime(t['acquisition_start'],errors='coerce'); t=t.sort_values('time')
        if metric not in t: raise KeyError(metric)
        fig=px.scatter(t,x='time',y=metric,hover_data=[c for c in ['scene_name','window_id','retained_pixels','ci_input_valid_pixels'] if c in t],title=f'Hylak_id {lake_id}: {metric}'); fig.update_traces(mode='lines+markers')
        return fig,t
    except Exception: return None,pd.DataFrame()


def _scene_choices():
    t=registry_table(GLOBAL_STATE.get('config_hash'))
    if t.empty: return []
    t=t[t.status=='done'].sort_values('acquisition_start')
    return [(f"{r.get('acquisition_start','')} · {r.get('scene_name','')}",str(r.scene_id)) for _,r in t.iterrows()]


def refresh_scene_choices_callback():
    c=_scene_choices(); v=c[0][1] if c else None
    return gr.Dropdown(choices=c,value=v),_scene_lake_update(v),_app_status('Scene list refreshed',f'{len(c):,} completed scenes')


def _scene_lake_update(scene_id):
    if not scene_id: return gr.Dropdown(choices=[],value=None)
    s=_stats(); t=s[s.scene_id.astype(str)==str(scene_id)] if not s.empty and 'scene_id' in s else pd.DataFrame()
    if t.empty: return gr.Dropdown(choices=[],value=None)
    c=[]
    for _,r in t[['Hylak_id']+(['Lake_name'] if 'Lake_name' in t else [])].drop_duplicates().sort_values('Hylak_id').iterrows():
        lid=str(int(r.Hylak_id)); c.append((f"{str(r.get('Lake_name','') or 'Unnamed lake')} · {lid}",lid))
    return gr.Dropdown(choices=c,value=(c[0][1] if c else None))


def scene_changed_callback(scene_id): return _scene_lake_update(scene_id)


def _scene_compact_files(scene_id):
    s=_stats(); t=s[s.scene_id.astype(str)==str(scene_id)] if not s.empty else pd.DataFrame()
    files=[]
    for p in t.get('compact_netcdf',pd.Series(dtype=str)).dropna().astype(str).unique():
        q=Path(p)
        if q.exists(): files.append(q)
    if not files:
        # registry/output-tree fallback
        files=sorted(COMPACT_NC_DIR.rglob(f"{scene_id}_*_lakepixels.nc"))
    return files


def _load_scene_pixels(scene_id,scope='scene',lake_id=None):
    frames=[]
    wanted=None if scope=='scene' else int(lake_id)
    names=['latitude','longitude','Hylak_id','valid_water_mask','ci_valid_mask','ci_candidate_mask','ci_detection_mask','MPH_peak_nm',*SCALAR_METRICS]
    for p in _scene_compact_files(scene_id):
        with netCDF4.Dataset(p) as ds:
            ids=np.asarray(ds.variables['Hylak_id'][:],dtype=np.int64); sel=np.ones(len(ids),bool) if wanted is None else ids==wanted
            if not sel.any(): continue
            data={'Hylak_id':ids[sel]}
            for n in names:
                if n=='Hylak_id': continue
                if n in ds.variables: data[n]=to_float(ds.variables[n][:])[sel]
                else: data[n]=np.full(sel.sum(),np.nan)
            frames.append(pd.DataFrame(data))
    if not frames: raise ValueError('No compact lake pixels found for selected scope')
    f=pd.concat(frames,ignore_index=True)
    # windows may overlap around margins; retain one record per native source point/lake approximately.
    f['_latr']=f.latitude.round(6); f['_lonr']=f.longitude.round(6); f=f.drop_duplicates(['Hylak_id','_latr','_lonr']).drop(columns=['_latr','_lonr'])
    return f


_TO_WEBMERC = Transformer.from_crs(4326, 3857, always_xy=True)
_FROM_WEBMERC = Transformer.from_crs(3857, 4326, always_xy=True)

def _grid_geometry_mercator(lat, lon, resolution_m, max_cells=DISPLAY_MAX_CELLS):
    """Exact EPSG:3857 grid for Leaflet/Esri image overlays.

    Bounds are derived from the actual grid edges (nx*resolution, ny*resolution),
    so Leaflet never stretches a raster across inconsistent geographic bounds.
    """
    lat=np.asarray(lat,float); lon=np.asarray(lon,float)
    finite=np.isfinite(lat)&np.isfinite(lon)&(np.abs(lat)<85.0)
    if not finite.any(): raise ValueError('No finite Web-Mercator-compatible geolocation')
    x,y=_TO_WEBMERC.transform(lon[finite],lat[finite]); x=np.asarray(x,float); y=np.asarray(y,float)
    res=float(resolution_m)
    for _ in range(10):
        xmin=math.floor(float(np.nanmin(x))/res)*res-res
        xmax=math.ceil(float(np.nanmax(x))/res)*res+res
        ymin=math.floor(float(np.nanmin(y))/res)*res-res
        ymax=math.ceil(float(np.nanmax(y))/res)*res+res
        nx=max(2,int(round((xmax-xmin)/res))); ny=max(2,int(round((ymax-ymin)/res)))
        xmax=xmin+nx*res; ymax=ymin+ny*res
        if nx*ny<=max_cells: break
        res*=math.sqrt((nx*ny)/max_cells)*1.05
    west,south=_FROM_WEBMERC.transform(xmin,ymin); east,north=_FROM_WEBMERC.transform(xmax,ymax)
    return {'xmin':xmin,'xmax':xmax,'ymin':ymin,'ymax':ymax,'nx':nx,'ny':ny,
            'resolution_m':res,'west':float(west),'east':float(east),'south':float(south),'north':float(north),
            'crs':'EPSG:3857'}

def _regularize_dataframe_mercator(frame, columns, resolution_m):
    lat=frame.latitude.to_numpy(float); lon=frame.longitude.to_numpy(float)
    g=_grid_geometry_mercator(lat,lon,resolution_m)
    x,y=_TO_WEBMERC.transform(lon,lat); x=np.asarray(x,float); y=np.asarray(y,float)
    rr=np.floor((g['ymax']-y)/g['resolution_m']).astype(np.int64)
    cc=np.floor((x-g['xmin'])/g['resolution_m']).astype(np.int64)
    inside=np.isfinite(x)&np.isfinite(y)&(rr>=0)&(rr<g['ny'])&(cc>=0)&(cc<g['nx'])
    flat=rr[inside]*g['nx']+cc[inside]; out={}
    for col in columns:
        vals=frame[col].to_numpy(float)[inside]; good=np.isfinite(vals)
        sums=np.bincount(flat[good],weights=vals[good],minlength=g['ny']*g['nx'])
        counts=np.bincount(flat[good],minlength=g['ny']*g['nx'])
        arr=np.full(g['ny']*g['nx'],np.nan,dtype=np.float32); nz=counts>0
        arr[nz]=(sums[nz]/counts[nz]).astype(np.float32); out[col]=arr.reshape(g['ny'],g['nx'])
    return out,g

def _grid_geometry_geographic(lat,lon,resolution_m,max_cells=DISPLAY_MAX_CELLS):
    """Consistent EPSG:4326 grid used only for SNAP/GIS NetCDF export."""
    lat=np.asarray(lat,float); lon=np.asarray(lon,float); finite=np.isfinite(lat)&np.isfinite(lon)
    if not finite.any(): raise ValueError('No finite geolocation')
    mean_lat=float(np.nanmean(lat[finite])); res=float(resolution_m)
    for _ in range(10):
        dlat=res/111320.0; dlon=res/max(111320.0*np.cos(np.deg2rad(mean_lat)),1000.0)
        west=math.floor(float(np.nanmin(lon[finite]))/dlon)*dlon-dlon
        east=math.ceil(float(np.nanmax(lon[finite]))/dlon)*dlon+dlon
        south=math.floor(float(np.nanmin(lat[finite]))/dlat)*dlat-dlat
        north=math.ceil(float(np.nanmax(lat[finite]))/dlat)*dlat+dlat
        nx=max(2,int(round((east-west)/dlon))); ny=max(2,int(round((north-south)/dlat)))
        east=west+nx*dlon; north=south+ny*dlat
        if nx*ny<=max_cells: break
        res*=math.sqrt((nx*ny)/max_cells)*1.05
    return {'west':west,'east':east,'south':south,'north':north,'dlat':dlat,'dlon':dlon,
            'nx':nx,'ny':ny,'resolution_m':res,'crs':'EPSG:4326'}

def _regularize_dataframe_geographic(frame,columns,resolution_m):
    lat=frame.latitude.to_numpy(float); lon=frame.longitude.to_numpy(float); g=_grid_geometry_geographic(lat,lon,resolution_m)
    rr=np.floor((g['north']-lat)/g['dlat']).astype(np.int64); cc=np.floor((lon-g['west'])/g['dlon']).astype(np.int64)
    inside=np.isfinite(lat)&np.isfinite(lon)&(rr>=0)&(rr<g['ny'])&(cc>=0)&(cc<g['nx'])
    flat=rr[inside]*g['nx']+cc[inside]; out={}
    for col in columns:
        vals=frame[col].to_numpy(float)[inside]; good=np.isfinite(vals)
        sums=np.bincount(flat[good],weights=vals[good],minlength=g['ny']*g['nx'])
        counts=np.bincount(flat[good],minlength=g['ny']*g['nx'])
        arr=np.full(g['ny']*g['nx'],np.nan,dtype=np.float32); nz=counts>0
        arr[nz]=(sums[nz]/counts[nz]).astype(np.float32); out[col]=arr.reshape(g['ny'],g['nx'])
    return out,g

# Backward-compatible alias for any helper outside the map path.
def _regularize_dataframe(frame,columns,resolution_m):
    return _regularize_dataframe_geographic(frame,columns,resolution_m)


def _auto_limits(values,metric):
    v=np.asarray(values,float); v=v[np.isfinite(v)]
    if not v.size: return 0.0,1.0
    if metric=='CI_cyano': v=v[v>0]
    if not v.size: return 0.0,1.0
    lo,hi=np.nanpercentile(v,[2,98]) if v.size>20 else (np.nanmin(v),np.nanmax(v))
    if not np.isfinite(lo) or not np.isfinite(hi) or hi<=lo: lo=float(np.nanmin(v)); hi=float(np.nanmax(v));
    if hi<=lo: hi=lo+1e-6
    return float(lo),float(hi)


def _optional_float(v):
    t=str(v or '').strip().lower(); return None if not t or t=='auto' else float(t)


def _rgba_data_url(grid,metric,cmap_name,reverse,vmin=None,vmax=None):
    vmin0,vmax0=_auto_limits(grid,metric); vmin=vmin0 if vmin is None else float(vmin); vmax=vmax0 if vmax is None else float(vmax)
    if vmax<=vmin: raise ValueError('Color maximum must exceed minimum')
    cm=plt.get_cmap(cmap_name+('_r' if reverse and not cmap_name.endswith('_r') else '')); norm=mcolors.Normalize(vmin=vmin,vmax=vmax,clip=True); rgba=(cm(norm(np.nan_to_num(grid,nan=vmin)))*255).astype(np.uint8); rgba[...,3]=np.where(np.isfinite(grid),230,0).astype(np.uint8)
    im=Image.fromarray(rgba,'RGBA'); b=io.BytesIO(); im.save(b,format='PNG',optimize=True); return 'data:image/png;base64,'+base64.b64encode(b.getvalue()).decode(),vmin,vmax,cm.name


def _truecolor_data_url(grids):
    chans=[]
    for n in ('rhos_665','rhos_560','rhos_490'):
        a=np.asarray(grids[n],float); v=a[np.isfinite(a)]
        if not v.size: chans.append(np.zeros(a.shape,np.float32)); continue
        lo,hi=np.nanpercentile(v,[2,98]); x=np.clip((a-lo)/max(hi-lo,1e-8),0,1); x=np.power(x,0.85); chans.append(x)
    rgb=np.stack(chans,-1); alpha=np.where(np.all(np.isfinite(np.stack([grids['rhos_665'],grids['rhos_560'],grids['rhos_490']],axis=0)),axis=0),255,0).astype(np.uint8); rgba=np.dstack([(np.nan_to_num(rgb)*255).astype(np.uint8),alpha]); im=Image.fromarray(rgba,'RGBA'); b=io.BytesIO(); im.save(b,format='PNG',optimize=True); return 'data:image/png;base64,'+base64.b64encode(b.getvalue()).decode()


def _cmap_gradient_css(cmap_name,reverse=False,n=12):
    cm=plt.get_cmap(cmap_name+('_r' if reverse and not cmap_name.endswith('_r') else '')); stops=[]
    for i in range(n):
        r,g,b,_=cm(i/(n-1)); stops.append(f"rgb({int(r*255)},{int(g*255)},{int(b*255)}) {100*i/(n-1):.1f}%")
    return ','.join(stops)


def _add_basemap(m,basemap,labels=True):
    if basemap=='None': return
    if basemap=='OpenStreetMap': folium.TileLayer('https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png',attr='© OpenStreetMap contributors',name='OpenStreetMap',show=True).add_to(m); return
    if basemap=='Esri World Imagery': folium.TileLayer('https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',attr='Esri',name='Esri World Imagery',show=True).add_to(m); return
    dark=basemap=='Esri Dark Gray'; stem='Canvas/World_Dark_Gray_Base' if dark else 'Canvas/World_Light_Gray_Base'; ref='Canvas/World_Dark_Gray_Reference' if dark else 'Canvas/World_Light_Gray_Reference'
    folium.TileLayer(f'https://server.arcgisonline.com/ArcGIS/rest/services/{stem}/MapServer/tile/{{z}}/{{y}}/{{x}}',attr='Esri',name=basemap,show=True).add_to(m)
    if labels: folium.TileLayer(f'https://server.arcgisonline.com/ArcGIS/rest/services/{ref}/MapServer/tile/{{z}}/{{y}}/{{x}}',attr='Esri',name='Labels',overlay=True,control=True,show=True).add_to(m)


def _lake_boundaries_for_ids(ids):
    try:
        lakes=_load_target_lakes_if_needed(); ids=set(map(int,ids)); return lakes[lakes.Hylak_id.astype(int).isin(ids)].copy()
    except Exception: return gpd.GeoDataFrame()


def _add_scalar_legend(m,metric,cmap_name,reverse,vmin,vmax):
    grad=_cmap_gradient_css(cmap_name,reverse); legend=f"""<div style='position:fixed;bottom:28px;left:24px;z-index:9999;background:rgba(255,255,255,.94);padding:10px 12px;border:1px solid #777;border-radius:5px;font:12px sans-serif;min-width:220px'><b>{html_lib.escape(metric)}</b><div style='height:14px;margin:5px 0 2px;background:linear-gradient(to right,{grad})'></div><div style='display:flex;justify-content:space-between'><span>{vmin:.5g}</span><span>{vmax:.5g}</span></div></div>"""; m.get_root().html.add_child(folium.Element(legend))


def _hover_payload(frame, metric, resolution_m, max_points=180000):
    """Build a compact spatial-hash payload for portable HTML hover.

    The older implementation scanned every retained lake point on every mousemove,
    which became slow for a full OLCI scene.  This version buckets points in a
    lon/lat grid; JavaScript checks only the cursor bucket and its eight neighbors.
    """
    cols = ['latitude','longitude','Hylak_id'] + ([metric] if metric != 'true_color' else ['rhos_665','rhos_560','rhos_490'])
    f = frame[cols].copy()
    f = f[np.isfinite(pd.to_numeric(f['latitude'], errors='coerce')) & np.isfinite(pd.to_numeric(f['longitude'], errors='coerce'))]
    if len(f) > max_points:
        take = np.linspace(0, len(f)-1, max_points, dtype=int)
        f = f.iloc[take]
    # ~0.02 degree bins are much smaller than an OLCI frame but large enough to
    # avoid thousands of tiny JavaScript objects.  The hover-distance test still
    # uses metres, so this bin only accelerates candidate lookup.
    cell = 0.02
    buckets = {}
    for r in f.itertuples(index=False):
        lat = float(r.latitude); lon = float(r.longitude)
        key = f"{math.floor(lon/cell)}:{math.floor(lat/cell)}"
        if metric == 'true_color':
            rec = [round(lat,6), round(lon,6), int(r.Hylak_id),
                   float(getattr(r,'rhos_665')) if np.isfinite(getattr(r,'rhos_665')) else None,
                   float(getattr(r,'rhos_560')) if np.isfinite(getattr(r,'rhos_560')) else None,
                   float(getattr(r,'rhos_490')) if np.isfinite(getattr(r,'rhos_490')) else None]
        else:
            val = getattr(r, metric)
            rec = [round(lat,6), round(lon,6), int(r.Hylak_id), float(val) if np.isfinite(val) else None]
        buckets.setdefault(key, []).append(rec)
    return {'cell': cell, 'buckets': buckets, 'metric': metric}



def _inject_hover_js(m, frame, metric, resolution_m):
    payload = json.dumps(_hover_payload(frame, metric, resolution_m), separators=(',',':'))
    map_name = m.get_name()
    radius = max(float(resolution_m) * 1.8, 450.0)
    js = f"""<script>(function(){{
    var P={payload}; var cell=P.cell, buckets=P.buckets, metric=P.metric; var map={map_name};
    var info=L.control({{position:'topright'}});
    info.onAdd=function(){{this._div=L.DomUtil.create('div','hover-readout');
      this._div.style.cssText='background:rgba(255,255,255,.95);padding:8px 10px;border:1px solid #777;border-radius:4px;font:12px sans-serif;min-width:220px';
      this._div.innerHTML='<b>Hover lake pixels</b><br>Move over a retained lake pixel'; return this._div;}}; info.addTo(map);
    function hav(a,b,c,d){{var R=6371000,p=Math.PI/180,x=(d-b)*p*Math.cos((a+c)*p/2),y=(c-a)*p;return Math.sqrt(x*x+y*y)*R;}}
    map.on('mousemove',function(e){{
      var ix=Math.floor(e.latlng.lng/cell), iy=Math.floor(e.latlng.lat/cell), best=null, bd=1e99;
      for(var dx=-1;dx<=1;dx++) for(var dy=-1;dy<=1;dy++){{
        var arr=buckets[(ix+dx)+':'+(iy+dy)]; if(!arr) continue;
        for(var j=0;j<arr.length;j++){{var p=arr[j],d=hav(e.latlng.lat,e.latlng.lng,p[0],p[1]);if(d<bd){{bd=d;best=p;}}}}
      }}
      if(!best||bd>{radius:.1f}){{info._div.innerHTML='<b>Hover lake pixels</b><br>No nearby retained pixel';return;}}
      var txt='<b>Hylak_id '+best[2]+'</b><br>'+best[0].toFixed(5)+', '+best[1].toFixed(5)+'<br>';
      if(metric==='true_color') txt+='rhos R/G/B: '+best[3]+', '+best[4]+', '+best[5];
      else txt+=metric+': '+(best[3]===null?'NaN':Number(best[3]).toPrecision(6));
      txt+='<br><small>nearest stored native OLCI lake pixel; distance '+Math.round(bd)+' m</small>'; info._div.innerHTML=txt;
    }});
    }})();</script>"""
    m.get_root().html.add_child(folium.Element(js))



def _build_map(scene_id,scope,lake_id,metric,resolution_m,basemap,labels,cmap,reverse,vmin_text,vmax_text,show_lakes=True):
    if not scene_id: raise ValueError('Choose a completed scene')
    scope='scene' if str(scope)=='scene' else 'lake'; frame=_load_scene_pixels(scene_id,scope,lake_id)
    cols=['rhos_665','rhos_560','rhos_490'] if metric=='true_color' else [metric]
    grids,g=_regularize_dataframe_mercator(frame,cols,float(resolution_m))
    bounds=[[g['south'],g['west']],[g['north'],g['east']]]
    m=folium.Map(tiles=None,control_scale=True,prefer_canvas=True)
    _add_basemap(m,basemap,bool(labels))
    if metric=='true_color':
        img=_truecolor_data_url(grids)
        folium.raster_layers.ImageOverlay(img,bounds=bounds,opacity=.95,name='True color rhos 665/560/490').add_to(m)
        vmin=vmax=None
    else:
        img,vmin,vmax,cm=_rgba_data_url(grids[metric],metric,cmap,bool(reverse),_optional_float(vmin_text),_optional_float(vmax_text))
        folium.raster_layers.ImageOverlay(img,bounds=bounds,opacity=.88,name=metric).add_to(m)
        _add_scalar_legend(m,metric,cmap,bool(reverse),vmin,vmax)
    if show_lakes:
        b=_lake_boundaries_for_ids(frame.Hylak_id.unique())
        if not b.empty:
            folium.GeoJson(json.loads(b.to_crs(4326).to_json()),name='HydroLAKES boundaries',
                           style_function=lambda f:{'color':'#00e5ff','weight':1.2,'fillOpacity':0}).add_to(m)
    folium_plugins.Fullscreen(position='topleft').add_to(m)
    folium_plugins.MousePosition(position='bottomright',separator=' | ',prefix='Lat / Lon',num_digits=5).add_to(m)
    folium_plugins.MeasureControl(position='topleft',primary_length_unit='kilometers').add_to(m)
    folium.LayerControl(collapsed=False).add_to(m); m.fit_bounds(bounds)
    _inject_hover_js(m,frame,metric,g['resolution_m'])
    out=HTML_DIR/f"scene_{scene_id}_{scope}_{lake_id or 'ALL'}_{metric}_{int(g['resolution_m'])}m.html"
    m.save(out)
    # Large srcdoc payloads may render blank inside Gradio. Serve the saved Folium HTML
    # through Gradio's authenticated same-origin file endpoint instead.
    map_src=f"/gradio_api/file={quote(str(out), safe='/')}?v={out.stat().st_mtime_ns}"
    iframe=("<iframe style='width:100%;height:740px;border:1px solid #ccd5df;"
            "border-radius:8px;background:white' loading='eager' src='"+map_src+"'></iframe>")
    return iframe,str(out),frame,grids,g

def export_snap_gridded_netcdf(scene_id, scope, lake_id, metric, frame, resolution_m):
    """Write a CF-1.8 EPSG:4326 raster for SNAP/QGIS, independent of the Web-Mercator display grid."""
    SNAP_DIR=EXPORT_DIR/'snap_gridded'; SNAP_DIR.mkdir(parents=True,exist_ok=True)
    names=['rhos_665','rhos_560','rhos_490'] if metric=='true_color' else [metric]
    grids,g=_regularize_dataframe_geographic(frame,names,float(resolution_m))
    out=SNAP_DIR/f"S3_OLCI_{scene_id}_{scope}_{lake_id or 'ALL'}_{metric}_{int(g['resolution_m'])}m_CF.nc"
    lat_desc=g['north']-(np.arange(g['ny'])+.5)*g['dlat']; lon=g['west']+(np.arange(g['nx'])+.5)*g['dlon']
    lat=lat_desc[::-1].astype(np.float64); lon=lon.astype(np.float64)
    from pyproj import CRS
    wkt=CRS.from_epsg(4326).to_wkt()
    with netCDF4.Dataset(out,'w',format='NETCDF4_CLASSIC') as ds:
        ds.createDimension('latitude',len(lat)); ds.createDimension('longitude',len(lon))
        ds.setncatts({'Conventions':'CF-1.8','title':'Sentinel-3 OLCI HydroLAKES gridded export',
                      'scene_id':str(scene_id),'scope':scope,'geospatial_lat_min':float(lat.min()),
                      'geospatial_lat_max':float(lat.max()),'geospatial_lon_min':float(lon.min()),
                      'geospatial_lon_max':float(lon.max()),'geospatial_lat_units':'degrees_north',
                      'geospatial_lon_units':'degrees_east','source_representation':'native OLCI lake pixels resampled to regular EPSG:4326 grid',
                      'note':'SNAP/GIS interoperability raster. Native statistics use the compact native lake-pixel archive.'})
        y=ds.createVariable('latitude','f8',('latitude',)); y[:]=lat; y.setncatts({'standard_name':'latitude','units':'degrees_north','axis':'Y'})
        x=ds.createVariable('longitude','f8',('longitude',)); x[:]=lon; x.setncatts({'standard_name':'longitude','units':'degrees_east','axis':'X'})
        crs=ds.createVariable('crs','i4'); crs.assignValue(0); crs.setncatts({'grid_mapping_name':'latitude_longitude','epsg_code':'EPSG:4326','spatial_ref':wkt,'crs_wkt':wkt,'semi_major_axis':6378137.0,'inverse_flattening':298.257223563})
        for name in names:
            v=ds.createVariable(name,'f4',('latitude','longitude'),zlib=True,complevel=6,shuffle=True,fill_value=np.float32(-9999.0))
            long={'FAI':'OLCI-adapted Floating Algae Index (AFAI form)','MPH':'Maximum Peak Height'}.get(name,name)
            v.setncatts({'coordinates':'latitude longitude','grid_mapping':'crs','units':'1','long_name':long})
            v[:]=np.ma.masked_invalid(np.asarray(grids[name],np.float32)[::-1,:])
        if metric=='true_color': ds.setncattr('true_color_mapping','red=rhos_665; green=rhos_560; blue=rhos_490')
    return str(out)


def _science_grid_setup(frame, requested_resolution_m=SCIENCE_EXPORT_DEFAULT_RESOLUTION_M):
    """Prepare one regular EPSG:4326 grid for a complete SNAP science product."""
    lat=frame.latitude.to_numpy(float); lon=frame.longitude.to_numpy(float)
    g=_grid_geometry_geographic(lat,lon,float(requested_resolution_m),max_cells=SCIENCE_EXPORT_MAX_CELLS)
    rr=np.floor((g['north']-lat)/g['dlat']).astype(np.int64)
    cc=np.floor((lon-g['west'])/g['dlon']).astype(np.int64)
    inside=np.isfinite(lat)&np.isfinite(lon)&(rr>=0)&(rr<g['ny'])&(cc>=0)&(cc<g['nx'])
    flat=rr[inside]*g['nx']+cc[inside]
    return g,inside,flat


def _mean_grid_from_flat(values, inside, flat, ncell):
    values=np.asarray(values,float)[inside]
    good=np.isfinite(values)
    arr=np.full(int(ncell),np.nan,dtype=np.float32)
    if good.any():
        sums=np.bincount(flat[good],weights=values[good],minlength=int(ncell))
        counts=np.bincount(flat[good],minlength=int(ncell))
        nz=counts>0
        arr[nz]=(sums[nz]/counts[nz]).astype(np.float32)
    return arr


def _mask_grid_from_flat(values, inside, flat, ncell):
    values=np.asarray(values,float)[inside]
    arr=np.zeros(int(ncell),dtype=np.uint8)
    good=np.isfinite(values)&(values>0)
    if good.any():
        np.maximum.at(arr,flat[good],np.uint8(1))
    return arr


def export_complete_scene_science_netcdf(scene_id, requested_resolution_m=SCIENCE_EXPORT_DEFAULT_RESOLUTION_M, force=False):
    """Create one self-contained SNAP/GIS NetCDF containing ALL production indices and rhos bands.

    This export is independent of the layer the user visualized.  The compact native lake-pixel
    archive remains the numerical source of truth; this file is a regular-grid interoperability
    product for SNAP/QGIS and sharing.
    """
    if not scene_id:
        raise ValueError('Choose a completed scene')
    frame=_load_scene_pixels(scene_id,'scene',None)
    g,inside,flat=_science_grid_setup(frame,float(requested_resolution_m))
    ncell=int(g['nx']*g['ny'])
    export_dir=EXPORT_DIR/'snap_complete'; export_dir.mkdir(parents=True,exist_ok=True)
    out=export_dir/f"S3_OLCI_{scene_id}_ALL_SCIENCE_{int(round(g['resolution_m']))}m_CF.nc"
    source_files=_scene_compact_files(scene_id)
    newest_source=max((p.stat().st_mtime_ns for p in source_files),default=0)
    if out.exists() and not force and out.stat().st_mtime_ns>=newest_source:
        return str(out),g

    lat_desc=g['north']-(np.arange(g['ny'])+.5)*g['dlat']
    lon=g['west']+(np.arange(g['nx'])+.5)*g['dlon']
    lat=lat_desc[::-1].astype(np.float64); lon=lon.astype(np.float64)
    from pyproj import CRS
    wkt=CRS.from_epsg(4326).to_wkt()
    tmp=out.with_suffix('.nc.tmp'); tmp.unlink(missing_ok=True)
    with netCDF4.Dataset(tmp,'w',format='NETCDF4_CLASSIC') as ds:
        ds.createDimension('latitude',len(lat)); ds.createDimension('longitude',len(lon))
        ds.setncatts({
            'Conventions':'CF-1.8',
            'title':'Sentinel-3 OLCI L2Gen HydroLAKES complete lake-water science product',
            'scene_id':str(scene_id),
            'source_representation':'native OLCI lake pixels stored in compact archives, resampled to regular EPSG:4326 grid',
            'requested_grid_resolution_m':float(requested_resolution_m),
            'actual_approx_grid_resolution_m':float(g['resolution_m']),
            'geospatial_lat_min':float(lat.min()),'geospatial_lat_max':float(lat.max()),
            'geospatial_lon_min':float(lon.min()),'geospatial_lon_max':float(lon.max()),
            'geospatial_lat_units':'degrees_north','geospatial_lon_units':'degrees_east',
            'non_lake_cells':'NetCDF fill/NaN',
            'science_variables':'CI, CI_cyano, NDCI, MPH, OLCI-adapted FAI/AFAI plus required Rayleigh-corrected rhos bands',
            'true_color_mapping':'red=rhos_665; green=rhos_560; blue=rhos_490',
            'nominal_885_mapping':'compact rhos_885 is sourced from canonical L2Gen rhos_884; exported here as rhos_884',
            'CI_cyano_detection_limit':float(CYAN_CI_DETECTION_LIMIT),
            'note':'For exact native-pixel values and lake statistics use the compact archive and CSV/Parquet. This regular grid is provided for SNAP/GIS interoperability.'
        })
        y=ds.createVariable('latitude','f8',('latitude',)); y[:]=lat; y.setncatts({'standard_name':'latitude','units':'degrees_north','axis':'Y'})
        x=ds.createVariable('longitude','f8',('longitude',)); x[:]=lon; x.setncatts({'standard_name':'longitude','units':'degrees_east','axis':'X'})
        crs=ds.createVariable('crs','i4'); crs.assignValue(0); crs.setncatts({'grid_mapping_name':'latitude_longitude','epsg_code':'EPSG:4326','spatial_ref':wkt,'crs_wkt':wkt,'semi_major_axis':6378137.0,'inverse_flattening':298.257223563})
        chunks=(min(512,len(lat)),min(512,len(lon)))
        # Export actual canonical 884 name while reading the nominal compact key rhos_885.
        variable_map=[(name, 'rhos_884' if name=='rhos_885' else name) for name in SCIENCE_EXPORT_VARIABLES]
        for source_name,out_name in variable_map:
            if source_name not in frame.columns:
                continue
            flat_grid=_mean_grid_from_flat(frame[source_name].to_numpy(float),inside,flat,ncell)
            grid=flat_grid.reshape(g['ny'],g['nx'])[::-1,:]
            var=ds.createVariable(out_name,'f4',('latitude','longitude'),zlib=True,complevel=6,shuffle=True,
                                  chunksizes=chunks,fill_value=np.float32(-9999.0))
            long_names={
                'CI':'Cyanobacteria Index candidate magnitude',
                'CI_cyano':'CyAN-compatible cyanobacteria index detections',
                'NDCI':'Normalized Difference Chlorophyll Index',
                'MPH':'Maximum Peak Height',
                'FAI':'OLCI-adapted Floating Algae Index (AFAI form)',
            }
            var.setncatts({'coordinates':'latitude longitude','grid_mapping':'crs','units':'1',
                           'long_name':long_names.get(out_name,f'Rayleigh-corrected OLCI reflectance {out_name}')})
            if out_name=='MPH':
                var.setncattr('formula','maximum line height at approximately 681/709/754 nm above a 665-to-884 nm baseline')
            if out_name=='FAI':
                var.setncattr('formula','rhos_754 minus linear baseline between rhos_665 and rhos_865; OLCI-adapted AFAI form')
            if out_name=='CI_cyano':
                var.setncattr('detection_limit',float(CYAN_CI_DETECTION_LIMIT))
            if source_name=='rhos_885':
                var.setncattr('source_compact_variable','rhos_885 nominal key; canonical L2Gen source rhos_884')
            var[:]=np.ma.masked_invalid(grid)
            del flat_grid,grid
        # Useful QA masks are included but do not replace the exact native masks in compact files.
        for source_name,out_name in [('valid_water_mask','valid_water_mask'),('ci_valid_mask','ci_valid_mask'),
                                     ('ci_candidate_mask','ci_candidate_mask'),('ci_detection_mask','ci_detection_mask')]:
            if source_name not in frame.columns: continue
            flat_grid=_mask_grid_from_flat(frame[source_name].to_numpy(float),inside,flat,ncell)
            grid=flat_grid.reshape(g['ny'],g['nx'])[::-1,:]
            var=ds.createVariable(out_name,'i1',('latitude','longitude'),zlib=True,complevel=6,shuffle=True,chunksizes=chunks,fill_value=np.int8(-1))
            var.setncatts({'coordinates':'latitude longitude','grid_mapping':'crs','long_name':out_name.replace('_',' '),
                           'flag_values':np.array([0,1],dtype=np.int8),'flag_meanings':'false true','valid_range':np.array([0,1],dtype=np.int8)})
            var[:]=grid
            del flat_grid,grid
    tmp.replace(out)
    return str(out),g


def _inject_all_science_hover_js(m, frame, resolution_m):
    cols=['latitude','longitude','Hylak_id','CI','CI_cyano','NDCI','MPH','FAI']
    f=frame[[c for c in cols if c in frame.columns]].copy()
    f=f[np.isfinite(pd.to_numeric(f.latitude,errors='coerce'))&np.isfinite(pd.to_numeric(f.longitude,errors='coerce'))]
    if len(f)>180000:
        f=f.iloc[np.linspace(0,len(f)-1,180000,dtype=int)]
    cell=.02; buckets={}
    for r in f.itertuples(index=False):
        key=f"{math.floor(float(r.longitude)/cell)}:{math.floor(float(r.latitude)/cell)}"
        vals=[]
        for name in ('CI','CI_cyano','NDCI','MPH','FAI'):
            v=getattr(r,name,np.nan); vals.append(float(v) if np.isfinite(v) else None)
        buckets.setdefault(key,[]).append([round(float(r.latitude),6),round(float(r.longitude),6),int(r.Hylak_id),*vals])
    payload=json.dumps({'cell':cell,'buckets':buckets},separators=(',',':'))
    map_name=m.get_name(); radius=max(float(resolution_m)*2.0,650.0)
    js=f"""<script>(function(){{var P={payload},cell=P.cell,buckets=P.buckets,map={map_name};
    var info=L.control({{position:'topright'}});info.onAdd=function(){{this._div=L.DomUtil.create('div','all-science-hover');this._div.style.cssText='background:rgba(255,255,255,.96);padding:8px 10px;border:1px solid #777;border-radius:4px;font:12px sans-serif;min-width:250px';this._div.innerHTML='<b>Hover native lake pixels</b>';return this._div;}};info.addTo(map);
    function hav(a,b,c,d){{var R=6371000,p=Math.PI/180,x=(d-b)*p*Math.cos((a+c)*p/2),y=(c-a)*p;return Math.sqrt(x*x+y*y)*R;}}
    map.on('mousemove',function(e){{var ix=Math.floor(e.latlng.lng/cell),iy=Math.floor(e.latlng.lat/cell),best=null,bd=1e99;for(var dx=-1;dx<=1;dx++)for(var dy=-1;dy<=1;dy++){{var a=buckets[(ix+dx)+':'+(iy+dy)];if(!a)continue;for(var j=0;j<a.length;j++){{var p=a[j],d=hav(e.latlng.lat,e.latlng.lng,p[0],p[1]);if(d<bd){{bd=d;best=p;}}}}}}if(!best||bd>{radius:.1f}){{info._div.innerHTML='<b>Hover native lake pixels</b><br>No nearby retained pixel';return;}}var n=['CI','CIcyano','NDCI','MPH','FAI/AFAI'],txt='<b>Hylak_id '+best[2]+'</b><br>'+best[0].toFixed(5)+', '+best[1].toFixed(5);for(var k=0;k<5;k++)txt+='<br>'+n[k]+': '+(best[3+k]===null?'NaN':Number(best[3+k]).toPrecision(6));txt+='<br><small>nearest stored native OLCI lake pixel · '+Math.round(bd)+' m</small>';info._div.innerHTML=txt;}});}})();</script>"""
    m.get_root().html.add_child(folium.Element(js))


def create_complete_scene_html(scene_id, resolution_m=1000, basemap='Esri Dark Gray', force=False):
    """Portable one-file scene viewer with true color + every production index as selectable layers."""
    if not scene_id: raise ValueError('Choose a completed scene')
    frame=_load_scene_pixels(scene_id,'scene',None)
    grids,g=_regularize_dataframe_mercator(frame,['rhos_665','rhos_560','rhos_490','CI','CI_cyano','NDCI','MPH','FAI'],float(resolution_m))
    out=HTML_DIR/f"scene_{scene_id}_ALL_SCIENCE_{int(round(g['resolution_m']))}m.html"
    if out.exists() and not force:
        return str(out),g
    bounds=[[g['south'],g['west']],[g['north'],g['east']]]
    m=folium.Map(tiles=None,control_scale=True,prefer_canvas=True)
    _add_basemap(m,basemap,True)
    # True color is available but not selected by default; CIcyano is the default science layer.
    tc=_truecolor_data_url({k:grids[k] for k in ('rhos_665','rhos_560','rhos_490')})
    folium.raster_layers.ImageOverlay(tc,bounds=bounds,opacity=.95,name='True color rhos 665/560/490',show=False).add_to(m)
    legend_rows=[]
    palette={'CI':'cividis','CI_cyano':'cividis','NDCI':'BrBG','MPH':'plasma','FAI':'viridis'}
    for metric in ('CI','CI_cyano','NDCI','MPH','FAI'):
        img,vmin,vmax,_=_rgba_data_url(grids[metric],metric,palette[metric],False,None,None)
        folium.raster_layers.ImageOverlay(img,bounds=bounds,opacity=.88,name=metric,show=(metric=='CI_cyano')).add_to(m)
        legend_rows.append(f"<tr><td><b>{html_lib.escape(metric)}</b></td><td>{vmin:.4g}</td><td>{vmax:.4g}</td></tr>")
    b=_lake_boundaries_for_ids(frame.Hylak_id.unique())
    if not b.empty:
        folium.GeoJson(json.loads(b.to_crs(4326).to_json()),name='HydroLAKES boundaries',style_function=lambda f:{'color':'#00e5ff','weight':1.1,'fillOpacity':0}).add_to(m)
    panel="<div style='position:fixed;bottom:28px;left:24px;z-index:9999;background:rgba(255,255,255,.95);padding:8px 10px;border:1px solid #777;border-radius:5px;font:11px sans-serif'><b>Auto display ranges</b><table><tr><th>Layer</th><th>min</th><th>max</th></tr>"+''.join(legend_rows)+"</table><small>Use layer control at upper right.</small></div>"
    m.get_root().html.add_child(folium.Element(panel))
    folium_plugins.Fullscreen(position='topleft').add_to(m); folium_plugins.MousePosition(position='bottomright',separator=' | ',prefix='Lat / Lon',num_digits=5).add_to(m); folium_plugins.MeasureControl(position='topleft',primary_length_unit='kilometers').add_to(m)
    folium.LayerControl(collapsed=False).add_to(m); m.fit_bounds(bounds); _inject_all_science_hover_js(m,frame,g['resolution_m']); m.save(out)
    return str(out),g

def _compact_download(scene_id):
    files=_scene_compact_files(scene_id)
    if not files: return None
    if len(files)==1: return str(files[0])
    d=EXPORT_DIR/'scene_bundles'; d.mkdir(parents=True,exist_ok=True); z=d/f'{scene_id}_compact_windows.zip'
    with zipfile.ZipFile(z,'w',zipfile.ZIP_DEFLATED,allowZip64=True) as arc:
        for p in files: arc.write(p,arcname=p.name)
    return str(z)

def map_callback(scene_id,scope,lake_id,metric,resolution_m,basemap,labels,cmap,reverse,vmin,vmax,show_lakes):
    try:
        iframe,html_path,frame,grids,g=_build_map(scene_id,scope,lake_id,metric,resolution_m,basemap,labels,cmap,reverse,vmin,vmax,show_lakes)
        nc=export_snap_gridded_netcdf(scene_id,'scene' if scope=='scene' else 'lake',lake_id,metric,frame,resolution_m)
        compact=_compact_download(scene_id)
        return iframe,html_path,nc,compact,_app_status('Map ready',f"{len(frame):,} stored lake pixels; {frame.Hylak_id.nunique():,} lakes; Web-Mercator display grid ≈ {g['resolution_m']:.0f} m. Hover reports nearest stored native OLCI lake pixel.")
    except Exception as exc:
        return _app_status('Map failed',str(exc),False),None,None,None,_app_status('Map failed',str(exc),False)

# ---------- Native orbit/frame + 5-degree archive catalogue ----------
GRID_ID_RE=re.compile(r'^G(?P<size>\d{2})_(?P<ns>[NS])(?P<lat>\d{2})_(?P<ew>[EW])(?P<lon>\d{3})$',re.I)
ARCHIVE_INDEX_CSV=CATALOG_DIR/'global_archive_index.csv'

def _grid_label(west,south,size=5):
    return f"G{int(size):02d}_{'N' if south>=0 else 'S'}{abs(int(south)):02d}_{'E' if west>=0 else 'W'}{abs(int(west)):03d}"

def grid_id_to_bbox(grid_id):
    m=GRID_ID_RE.match(str(grid_id or '').strip())
    if not m: raise ValueError('Grid ID example: G05_N40_W085')
    size=int(m.group('size')); south=int(m.group('lat'))*(1 if m.group('ns').upper()=='N' else -1); west=int(m.group('lon'))*(1 if m.group('ew').upper()=='E' else -1)
    return float(west),float(south),float(west+size),float(south+size)

def scene_grid_ids(scene,size=5):
    geom=scene_geometry(scene)
    if geom is None or geom.is_empty: return []
    minx,miny,maxx,maxy=geom.bounds; size=int(size); ids=[]
    w0=math.floor(minx/size)*size; s0=math.floor(miny/size)*size
    w=w0
    while w<maxx:
        s=s0
        while s<maxy:
            if geom.intersects(shapely_box(w,s,w+size,s+size)): ids.append(_grid_label(w,s,size))
            s+=size
        w+=size
    return sorted(set(ids))

def build_archive_index(config_hash=None):
    t=registry_table(config_hash); rows=[]
    if t.empty:
        pd.DataFrame().to_csv(ARCHIVE_INDEX_CSV,index=False); return pd.DataFrame()
    for _,r in t[t.status=='done'].iterrows():
        try: scene=json.loads(r.scene_json) if isinstance(r.scene_json,str) else {}
        except Exception: scene={}
        try: meta=parse_s3_frame_metadata(r.scene_name)
        except Exception: meta={'platform':str(r.scene_name)[:3],'relative_orbit':np.nan,'frame':np.nan,'native_frame_id':''}
        cfg=str(r.config_hash); stats_p=STATS_SCENE_DIR/f'{r.scene_id}_{cfg}.parquet'; stats_c=stats_p.with_suffix('.csv')
        manifest=EXPORT_DIR/str(r.scene_id)/f'manifest_{cfg}.json'; comp=_scene_compact_files(str(r.scene_id))
        rows.append({'scene_id':str(r.scene_id),'scene_name':r.scene_name,'acquisition_start':r.acquisition_start,
                     'platform':meta.get('platform'),'relative_orbit':meta.get('relative_orbit'),'frame':meta.get('frame'),
                     'native_frame_id':meta.get('native_frame_id'),'grid_ids':';'.join(scene_grid_ids(scene,5)),
                     'config_hash':cfg,'candidate_lakes':r.candidate_lakes,'stats_rows':r.stats_rows,
                     'compact_MB':sum(p.stat().st_size for p in comp)/(1024**2) if comp else 0.0,
                     'compact_files':'|'.join(map(str,comp)),'stats_csv':str(stats_c) if stats_c.exists() else '',
                     'stats_parquet':str(stats_p) if stats_p.exists() else '',
                     'manifest':str(manifest) if manifest.exists() else ''})
    out=pd.DataFrame(rows).sort_values(['acquisition_start','scene_name']) if rows else pd.DataFrame()
    ARCHIVE_INDEX_CSV.parent.mkdir(parents=True,exist_ok=True); out.to_csv(ARCHIVE_INDEX_CSV,index=False)
    return out

def archive_refresh_callback(grid_filter='',orbit_filter=''):
    try:
        t=build_archive_index(GLOBAL_STATE.get('config_hash'))
        if str(grid_filter or '').strip(): t=t[t.grid_ids.astype(str).str.contains(str(grid_filter).strip(),case=False,na=False)]
        if str(orbit_filter or '').strip():
            orbit=int(float(orbit_filter)); t=t[pd.to_numeric(t.relative_orbit,errors='coerce')==orbit]
        return t.head(1000),str(ARCHIVE_INDEX_CSV),_app_status('Archive index ready',f'{len(t):,} completed scene records. Grid IDs are search cross-references; scene ID remains the deduplication key.')
    except Exception as exc: return pd.DataFrame(),None,_app_status('Archive index failed',str(exc),False)

def create_scene_share_bundle(scene_id, science_resolution_m=SCIENCE_EXPORT_DEFAULT_RESOLUTION_M):
    """Create a share ZIP that ALWAYS contains all production indices and required rhos bands.

    The all-science SNAP NetCDF and portable multi-layer HTML are created on demand here,
    even when the user never visualized NDCI/MPH/FAI in the map tab.
    """
    if not scene_id: raise ValueError('Choose a completed scene')
    t=registry_table(GLOBAL_STATE.get('config_hash')); r=t[t.scene_id.astype(str)==str(scene_id)]
    if r.empty: raise KeyError(scene_id)
    row=r.iloc[0]; cfg=str(row.config_hash); d=EXPORT_DIR/'scene_bundles'; d.mkdir(parents=True,exist_ok=True)
    complete_nc,g=export_complete_scene_science_netcdf(scene_id,float(science_resolution_m))
    complete_html,hg=create_complete_scene_html(scene_id,1000,'Esri Dark Gray')
    z=d/f"{str(row.scene_name)}_{cfg}_SHARE.zip"
    candidates=[]; candidates.extend(_scene_compact_files(scene_id))
    sp=STATS_SCENE_DIR/f'{scene_id}_{cfg}.parquet'; candidates += [sp,sp.with_suffix('.csv'),EXPORT_DIR/scene_id/f'manifest_{cfg}.json']
    candidates += [Path(complete_nc),Path(complete_html)]
    # Keep any additional user-created map exports and CyAN comparison files too.
    candidates += sorted((EXPORT_DIR/'snap_gridded').glob(f'S3_OLCI_{scene_id}_*.nc')) if (EXPORT_DIR/'snap_gridded').exists() else []
    candidates += sorted(HTML_DIR.glob(f'scene_{scene_id}_*.html')) if HTML_DIR.exists() else []
    cyan_dir=EXPORT_DIR/'cyan_comparison'/str(scene_id)
    candidates += sorted(cyan_dir.glob('*')) if cyan_dir.exists() else []
    # Unique existing files only.
    unique=[]; seen=set()
    for p in candidates:
        p=Path(p)
        if p.exists() and p.is_file() and str(p.resolve()) not in seen:
            seen.add(str(p.resolve())); unique.append(p)
    meta=parse_s3_frame_metadata(row.scene_name)
    readme={
        'scene_id':scene_id,'scene_name':row.scene_name,'config_hash':cfg,**meta,
        'complete_snap_netcdf':Path(complete_nc).name,
        'complete_snap_variables':['rhos_490','rhos_560','rhos_620','rhos_665','rhos_681','rhos_709','rhos_754','rhos_865','rhos_884','CI','CI_cyano','NDCI','MPH','FAI'],
        'complete_snap_actual_resolution_m':float(g['resolution_m']),
        'portable_all_science_html':Path(complete_html).name,
        'note':'Raw L1 and transient dense L2 are intentionally not retained. Compact NetCDFs preserve native lake pixels; the complete CF NetCDF is the SNAP/GIS raster interoperability product.'
    }
    with zipfile.ZipFile(z,'w',zipfile.ZIP_DEFLATED,allowZip64=True) as arc:
        for p in unique: arc.write(p,arcname=p.name)
        arc.writestr('ARCHIVE_README.json',json.dumps(readme,indent=2,default=str))
    return str(z),_app_status('Share bundle created',f"{z.name} · complete SNAP science NetCDF included ({g['resolution_m']:.0f} m approx) · CI/CIcyano/NDCI/MPH/FAI + required rhos bands included regardless of visualization")

def bundle_callback(scene_id,science_resolution_m):
    try: return create_scene_share_bundle(scene_id,science_resolution_m)
    except Exception as exc: return None,_app_status('Bundle failed',str(exc),False)

# ---------- Compact lake spectra / statistics ----------
def scene_spectrum_callback(scene_id,lake_id,statistic='median'):
    try:
        if not scene_id or not lake_id: raise ValueError('Choose a completed scene and lake')
        f=_load_scene_pixels(scene_id,'lake',lake_id); waves=list(GLOBAL_ANALYSIS_RHOS_WAVELENGTHS); rows=[]
        for w in waves:
            a=pd.to_numeric(f[f'rhos_{w}'],errors='coerce').to_numpy(float); a=a[np.isfinite(a)]
            rows.append({'wavelength_nm':w,'mean_rhos':float(np.nanmean(a)) if a.size else np.nan,'median_rhos':float(np.nanmedian(a)) if a.size else np.nan,'std_rhos':float(np.nanstd(a)) if a.size else np.nan,'n':int(a.size)})
        tab=pd.DataFrame(rows); y='median_rhos' if statistic=='median' else 'mean_rhos'
        fig=px.line(tab,x='wavelength_nm',y=y,markers=True,title=f'Hylak_id {lake_id} · {statistic} Rayleigh-corrected rhos')
        stats=_stats(); stats=stats[(stats.scene_id.astype(str)==str(scene_id))&(pd.to_numeric(stats.Hylak_id,errors='coerce')==int(lake_id))] if not stats.empty else pd.DataFrame()
        d=EXPORT_DIR/'spectra'; d.mkdir(parents=True,exist_ok=True); csv=d/f'{scene_id}_Hylak_{int(lake_id)}_spectrum.csv'; tab.to_csv(csv,index=False)
        return fig,tab,stats,str(csv),_app_status('Spectrum ready',f'{len(f):,} retained/native lake-pixel records')
    except Exception as exc: return None,pd.DataFrame(),pd.DataFrame(),None,_app_status('Spectrum failed',str(exc),False)



# ---------- Coordinate-based spectra from compact native lake pixels ----------
def _spectrum_points_frame(points):
    if points is None:
        frame=pd.DataFrame(columns=['label','latitude','longitude'])
    elif isinstance(points,pd.DataFrame): frame=points.copy()
    else: frame=pd.DataFrame(points,columns=['label','latitude','longitude'])
    frame=frame.rename(columns={c:str(c).strip().lower() for c in frame.columns})
    req=['label','latitude','longitude']
    if any(c not in frame.columns for c in req): raise ValueError('Point table requires label, latitude, longitude')
    frame=frame[req].copy(); frame['label']=frame['label'].fillna('').astype(str).str.strip(); frame['latitude']=pd.to_numeric(frame['latitude'],errors='coerce'); frame['longitude']=pd.to_numeric(frame['longitude'],errors='coerce'); frame=frame.dropna(subset=['latitude','longitude']).reset_index(drop=True)
    if frame.empty: raise ValueError('Add at least one latitude/longitude point')
    if len(frame)>20: raise ValueError('Maximum 20 coordinate spectrum points')
    for i in frame.index:
        if not frame.at[i,'label']: frame.at[i,'label']=f'P{i+1}'
    if ((frame.latitude<-90)|(frame.latitude>90)).any() or ((frame.longitude<-180)|(frame.longitude>180)).any(): raise ValueError('Invalid latitude/longitude')
    return frame


def add_spectrum_point_callback(label,latitude,longitude,points):
    try:
        f=pd.DataFrame(columns=['label','latitude','longitude']) if points is None else (points.copy() if isinstance(points,pd.DataFrame) else pd.DataFrame(points,columns=['label','latitude','longitude']))
        row=pd.DataFrame([{'label':str(label or f'P{len(f)+1}'),'latitude':float(latitude),'longitude':float(longitude)}])
        return pd.concat([f,row],ignore_index=True),_app_status('Spectrum point added')
    except Exception as exc: return points,_app_status('Could not add point',str(exc),False)


def clear_spectrum_points_callback():
    return pd.DataFrame(columns=['label','latitude','longitude']),_app_status('Spectrum points cleared')


def _unit_sphere_xyz(lat,lon):
    lat=np.deg2rad(np.asarray(lat,float)); lon=np.deg2rad(np.asarray(lon,float)); c=np.cos(lat)
    return np.column_stack((c*np.cos(lon),c*np.sin(lon),np.sin(lat)))


def coordinate_spectra_callback(scene_id,points,pixel_basis='stored_lake',max_distance_m=2000):
    try:
        if not scene_id: raise ValueError('Choose a completed scene')
        targets=_spectrum_points_frame(points); f=_load_scene_pixels(scene_id,'scene',None)
        mask=np.isfinite(f.latitude)&np.isfinite(f.longitude)
        if pixel_basis=='valid_water' and 'valid_water_mask' in f: mask &= pd.to_numeric(f.valid_water_mask,errors='coerce').fillna(0).to_numpy(float)>0
        if pixel_basis=='ci_valid' and 'ci_valid_mask' in f: mask &= pd.to_numeric(f.ci_valid_mask,errors='coerce').fillna(0).to_numpy(float)>0
        source=f.loc[mask].reset_index(drop=True)
        if source.empty: raise ValueError('No stored pixels satisfy the selected nearest-pixel basis')
        tree=cKDTree(_unit_sphere_xyz(source.latitude,source.longitude)); records=[]
        for _,p in targets.iterrows():
            chord,idx=tree.query(_unit_sphere_xyz([p.latitude],[p.longitude])[0],k=1)
            pix=source.iloc[int(idx)]; angle=2*np.arcsin(min(1.0,float(chord)/2)); dist=6371000.0*angle
            if dist>float(max_distance_m):
                raise ValueError(f"{p.label}: nearest stored lake pixel is {dist:.0f} m away, beyond {float(max_distance_m):.0f} m limit")
            for w in GLOBAL_ANALYSIS_RHOS_WAVELENGTHS:
                col=f'rhos_{w}'; value=float(pix[col]) if col in pix and np.isfinite(pix[col]) else np.nan
                records.append({'point':p.label,'target_latitude':float(p.latitude),'target_longitude':float(p.longitude),'pixel_latitude':float(pix.latitude),'pixel_longitude':float(pix.longitude),'Hylak_id':int(pix.Hylak_id),'match_distance_m':float(dist),'pixel_basis':pixel_basis,'product':('rhos_884' if w==885 else col),'wavelength_nm':884.0 if w==885 else float(w),'value':value})
        tab=pd.DataFrame(records); fig=go.Figure(); symbols=['circle','square','diamond','cross','triangle-up','triangle-down','star','hexagon','x']
        for i,(label,gp) in enumerate(tab.groupby('point',sort=False)):
            finite=gp[np.isfinite(gp.value)]; fig.add_trace(go.Scatter(x=finite.wavelength_nm,y=finite.value,mode='lines+markers',name=str(label),marker={'symbol':symbols[i%len(symbols)],'size':7},customdata=finite[['pixel_latitude','pixel_longitude','match_distance_m','Hylak_id']].to_numpy(),hovertemplate='%{fullData.name}<br>%{x:.1f} nm<br>%{y:.7g}<br>pixel %{customdata[0]:.5f}, %{customdata[1]:.5f}<br>distance %{customdata[2]:.0f} m · Hylak %{customdata[3]}<extra></extra>'))
        fig.update_layout(title=f'Sentinel-3 OLCI coordinate spectra · {len(targets)} point(s)',xaxis_title='Wavelength (nm)',yaxis_title='Rayleigh-corrected rhos',template='plotly_white',height=560,hovermode='closest')
        d=EXPORT_DIR/'spectra'; d.mkdir(parents=True,exist_ok=True); csv=d/f'{scene_id}_coordinate_spectra.csv'; tab.to_csv(csv,index=False)
        return fig,tab.round(8),str(csv),_app_status('Coordinate spectra ready',f'{len(targets)} requested point(s); nearest stored HydroLAKES lake pixels; non-lake pixels are not retained in this efficient archive')
    except Exception as exc: return None,pd.DataFrame(),None,_app_status('Coordinate spectra failed',str(exc),False)


# ---------- NASA OBPG / CyAN same-day comparison using compact native pixels ----------
def _scene_utc_date_global(scene_id,explicit=''):
    if str(explicit or '').strip(): return dt.date.fromisoformat(str(explicit).strip())
    t=registry_table(GLOBAL_STATE.get('config_hash')); r=t[t.scene_id.astype(str)==str(scene_id)]
    if r.empty: raise KeyError(scene_id)
    value=str(r.iloc[0].acquisition_start)
    return pd.to_datetime(value,utc=True).date()


def _obpg_filename_global(acquisition_date):
    return f"L{acquisition_date.year}{acquisition_date.timetuple().tm_yday:03d}.L3m_DAY_CYAN_CI_cyano_CYAN_CONUS_300m.tif"


def _download_obpg_cyano_global(acquisition_date):
    filename=_obpg_filename_global(acquisition_date); destination=PERSISTENT_ROOT/'obpg_cyan'/filename; destination.parent.mkdir(parents=True,exist_ok=True)
    if destination.exists() and destination.stat().st_size>1024: return destination
    netrc_path=Path.home()/'.netrc'
    if not netrc_path.exists(): raise RuntimeError("Earthdata credentials are not configured. Run the notebook's Earthdata Login cell.")
    credentials=netrc_module.netrc(str(netrc_path)).authenticators('urs.earthdata.nasa.gov')
    if not credentials: raise RuntimeError('No urs.earthdata.nasa.gov entry exists in ~/.netrc')
    class EarthdataSession(requests.Session):
        AUTH_HOST='urs.earthdata.nasa.gov'
        def rebuild_auth(self,prepared_request,response):
            headers=prepared_request.headers
            if 'Authorization' not in headers:return
            original=urlparse(response.request.url).hostname; redirect=urlparse(prepared_request.url).hostname
            if original!=redirect and original!=self.AUTH_HOST and redirect!=self.AUTH_HOST: del headers['Authorization']
    url='https://oceandata.sci.gsfc.nasa.gov/getfile/'+filename; temporary=destination.with_suffix('.tif.part'); session=EarthdataSession(); session.auth=(credentials[0],credentials[2])
    with session.get(url,stream=True,timeout=(30,180),allow_redirects=True) as response:
        response.raise_for_status(); content_type=response.headers.get('content-type','').lower(); first=b''
        with temporary.open('wb') as fh:
            for chunk in response.iter_content(1024*1024):
                if not chunk: continue
                if not first:first=chunk[:16]
                fh.write(chunk)
        if 'html' in content_type or first.lstrip().startswith(b'<'):
            temporary.unlink(missing_ok=True); raise RuntimeError('NASA returned an Earthdata sign-in page instead of GeoTIFF')
        if first[:4] not in (b'II*\x00',b'MM\x00*'):
            temporary.unlink(missing_ok=True); raise RuntimeError('OBPG response is not a TIFF file')
    temporary.replace(destination); return destination


def _obpg_ci_from_dn_global(raw_dn,valid_mask):
    raw=np.asarray(raw_dn); science=np.asarray(valid_mask,bool)&np.isfinite(raw)&(raw>=0)&(raw<=250); ci=np.full(raw.shape,np.nan,np.float32); ci[science]=np.power(10.0,(3.0/250.0)*raw[science].astype(float)-4.2); return ci,science


def _selected_target_lake(lake_id):
    lakes=_load_target_lakes_if_needed(); selected=lakes[pd.to_numeric(lakes.Hylak_id,errors='coerce')==int(lake_id)].copy()
    if selected.empty: raise KeyError(f'Hylak_id {lake_id} is not in the current target lake catalogue')
    return selected.to_crs(4326)


def _obpg_lake_window_global(obpg_path,lake_frame):
    with rasterio.open(obpg_path) as source:
        if source.crs is None: raise ValueError('OBPG GeoTIFF has no CRS')
        projected=lake_frame.to_crs(source.crs); west,south,east,north=projected.total_bounds
        window=rasterio.windows.from_bounds(west,south,east,north,source.transform).round_offsets().round_lengths(); full=rasterio.windows.Window(0,0,source.width,source.height)
        try: window=window.intersection(full)
        except rasterio.errors.WindowError as exc: raise ValueError('Selected lake does not overlap this OBPG raster') from exc
        if window.width<1 or window.height<1: raise ValueError('Selected lake does not overlap this OBPG raster')
        raw=source.read(1,window=window,masked=False); valid=source.read_masks(1,window=window)>0; transform=rasterio.windows.transform(window,source.transform); rows,cols=np.indices(raw.shape); xs,ys=rasterio.transform.xy(transform,rows,cols,offset='center'); xs=np.asarray(xs);ys=np.asarray(ys); tr=Transformer.from_crs(source.crs,4326,always_xy=True); lon,lat=tr.transform(xs,ys); lon=np.asarray(lon).reshape(raw.shape);lat=np.asarray(lat).reshape(raw.shape); profile={'crs':source.crs.to_string(),'transform':tuple(transform)}
    geom=lake_frame.geometry.union_all(); inside=contains_xy(geom,lon,lat); return raw,valid,lat,lon,inside,profile


def _cyan_metrics(obpg,local):
    paired=np.isfinite(obpg)&np.isfinite(local)&(obpg>CYAN_CI_DETECTION_LIMIT)&(local>CYAN_CI_DETECTION_LIMIT); x=np.asarray(obpg,float)[paired]; y=np.asarray(local,float)[paired]
    out={'paired_above_detection_limit_pixels':int(len(x)),'pearson_r':np.nan,'spearman_rho':np.nan,'mean_bias_local_minus_obpg':np.nan,'mae':np.nan,'rmse':np.nan}
    if len(x):
        d=y-x; out.update(mean_bias_local_minus_obpg=float(np.mean(d)),mae=float(np.mean(np.abs(d))),rmse=float(np.sqrt(np.mean(d*d))))
    if len(x)>=2 and np.nanstd(x)>0 and np.nanstd(y)>0:
        out['pearson_r']=float(scipy_stats.pearsonr(x,y).statistic); out['spearman_rho']=float(scipy_stats.spearmanr(x,y).statistic)
    return out,paired


def prepare_cyan_source_callback(scene_id,explicit_date,source_mode,uploaded_tif):
    try:
        date=_scene_utc_date_global(scene_id,explicit_date)
        if source_mode=='upload':
            if not uploaded_tif: raise ValueError('Upload an OBPG daily CIcyano GeoTIFF')
            path=Path(str(uploaded_tif))
        else: path=_download_obpg_cyano_global(date)
        with rasterio.open(path) as ds:
            if ds.count<1: raise ValueError('GeoTIFF has no raster band')
        CYAN_STATE.update(source_path=Path(path),source_date=date.isoformat())
        return str(path),_app_status('NASA CyAN source ready',f'{date.isoformat()} · {path.name}')
    except Exception as exc:return None,_app_status('NASA CyAN source failed',str(exc),False)


def compare_cyan_callback(scene_id,lake_id,explicit_date,source_mode,uploaded_tif,max_distance_m):
    try:
        if not scene_id or not lake_id: raise ValueError('Choose a completed scene and lake')
        date=_scene_utc_date_global(scene_id,explicit_date)
        if source_mode=='upload':
            if not uploaded_tif: raise ValueError('Upload the NASA OBPG GeoTIFF')
            obpg_path=Path(str(uploaded_tif))
        elif CYAN_STATE.get('source_path') and CYAN_STATE.get('source_date')==date.isoformat(): obpg_path=Path(CYAN_STATE['source_path'])
        else: obpg_path=_download_obpg_cyano_global(date)
        lake=_selected_target_lake(lake_id); raw,raster_valid,lat,lon,inside,profile=_obpg_lake_window_global(obpg_path,lake); obpg_ci,science=_obpg_ci_from_dn_global(raw,raster_valid)
        local=_load_scene_pixels(scene_id,'lake',lake_id); source=local[_cyan_comparison_selector(local)&np.isfinite(local.latitude)&np.isfinite(local.longitude)].reset_index(drop=True)
        if source.empty: raise ValueError('No compact native pixels exist for selected lake')
        target_lat=lat[inside];target_lon=lon[inside]; tree=cKDTree(_unit_sphere_xyz(source.latitude,source.longitude)); chord,idx=tree.query(_unit_sphere_xyz(target_lat,target_lon),k=1); dist=6371000.0*(2*np.arcsin(np.minimum(1.0,np.asarray(chord,float)/2))); matched=dist<=float(max_distance_m)
        local_val=np.full(len(target_lat),np.nan,np.float32); local_ci_valid=np.zeros(len(target_lat),bool); local_candidate=np.zeros(len(target_lat),bool); local_detected=np.zeros(len(target_lat),bool)
        if matched.any():
            q=source.iloc[np.asarray(idx[matched],int)]; local_val[matched]=pd.to_numeric(q.CI_cyano,errors='coerce').to_numpy(np.float32); local_ci_valid[matched]=pd.to_numeric(q.get('ci_valid_mask',0),errors='coerce').fillna(0).to_numpy(float)>0; local_candidate[matched]=pd.to_numeric(q.get('ci_candidate_mask',0),errors='coerce').fillna(0).to_numpy(float)>0; local_detected[matched]=pd.to_numeric(q.get('ci_detection_mask',0),errors='coerce').fillna(0).to_numpy(float)>0
        raw_inside=raw[inside]; science_inside=science[inside]; obpg_inside=obpg_ci[inside]; obpg_det=science_inside&np.isfinite(obpg_inside)&(obpg_inside>CYAN_CI_DETECTION_LIMIT); local_det=matched&local_detected&np.isfinite(local_val)&(local_val>CYAN_CI_DETECTION_LIMIT); metrics,paired=_cyan_metrics(obpg_inside,np.where(local_det,local_val,np.nan))
        table=pd.DataFrame({'latitude':target_lat,'longitude':target_lon,'obpg_raw_dn':raw_inside.astype(np.int16),'obpg_science_valid':science_inside,'obpg_CI_cyano':obpg_inside,'obpg_detected_gt_0_0001':obpg_det,'local_match_found':matched,'local_match_distance_m':np.where(matched,dist,np.nan),'local_ci_input_valid':local_ci_valid,'local_ci_candidate':local_candidate,'local_ci_cyano_detected':local_det,'local_CI_cyano':local_val,'paired_above_0_0001_for_correlation':paired})
        coverage=pd.DataFrame([{'date_utc':date.isoformat(),'Hylak_id':int(lake_id),'lake_name':str(lake.iloc[0].get('Lake_name','') or ''),'obpg_grid_cells_inside_lake':int(inside.sum()),'obpg_science_valid_cells':int(science_inside.sum()),'obpg_detection_cells_gt_0_0001':int(obpg_det.sum()),'local_nearest_matches':int(matched.sum()),'local_detection_matches_gt_0_0001':int(local_det.sum()),'paired_detection_pixels':metrics['paired_above_detection_limit_pixels'],'max_nearest_distance_m':float(max_distance_m),'comparison_basis':'same-day OBPG merged 300 m cells versus nearest native OLCI pixel passing the stored CyAN-strict QA mask'}]); metric_table=pd.DataFrame([metrics])
        d=EXPORT_DIR/'cyan_comparison'/str(scene_id); d.mkdir(parents=True,exist_ok=True); stem=f'{scene_id}_Hylak{int(lake_id)}_{date.isoformat()}_CyAN'; csv=d/f'{stem}_pairs.csv'; met=d/f'{stem}_metrics.csv'; nc=d/f'{stem}_comparison.nc'; table.to_csv(csv,index=False); pd.concat([coverage,metric_table],axis=1).to_csv(met,index=False)
        shape=raw.shape
        def grid(values,fill=np.nan,dtype=np.float32):
            out=np.full(shape,fill,dtype=dtype);out[inside]=np.asarray(values,dtype=dtype);return out
        ds=xr.Dataset({'latitude':(('y','x'),lat.astype(np.float32)),'longitude':(('y','x'),lon.astype(np.float32)),'inside_lake':(('y','x'),inside.astype(np.uint8)),'obpg_raw_dn':(('y','x'),raw.astype(np.int16)),'obpg_CI_cyano':(('y','x'),np.where(inside,obpg_ci,np.nan).astype(np.float32)),'local_CI_cyano_nearest':(('y','x'),grid(local_val)),'local_match_distance_m':(('y','x'),grid(np.where(matched,dist,np.nan))),'paired_detection_mask':(('y','x'),grid(paired,0,np.uint8))},attrs={'title':'Local L2Gen CyAN-strict CIcyano versus NASA OBPG Merged-S3-CYAN comparison','created_utc':utc_now(),'scene_id':str(scene_id),'acquisition_date_utc':date.isoformat(),'selected_hylak_id':int(lake_id),'source_obpg_geotiff':str(obpg_path),'pairing':'nearest compact native OLCI lake pixel to each in-lake OBPG grid cell centre','max_nearest_distance_m':float(max_distance_m),'ci_cyano_detection_limit':float(CYAN_CI_DETECTION_LIMIT),'note':'OBPG is same-day merged S3A/S3B Level-3; local source is one OLCI scene.'}); ds.to_netcdf(nc,engine='netcdf4',encoding={n:{'zlib':True,'complevel':4} for n in ds.data_vars});ds.close()
        finite=table[table.paired_above_0_0001_for_correlation]; fig=go.Figure(); fig.add_trace(go.Scattergl(x=finite.obpg_CI_cyano,y=finite.local_CI_cyano,mode='markers',name='Paired detections',marker={'size':5,'opacity':.55}));
        if not finite.empty:
            lo=float(min(finite.obpg_CI_cyano.min(),finite.local_CI_cyano.min()));hi=float(max(finite.obpg_CI_cyano.max(),finite.local_CI_cyano.max()));fig.add_trace(go.Scatter(x=[lo,hi],y=[lo,hi],mode='lines',name='1:1',line={'dash':'dash','color':'black'}))
        fig.update_layout(title=f"{coverage.iloc[0].lake_name} · local scene versus same-day NASA CyAN",xaxis={'title':'NASA OBPG CIcyano','type':'log'},yaxis={'title':'Local L2Gen CIcyano','type':'log'},template='plotly_white',height=600)
        return fig,coverage.round(6),metric_table.round(8),str(csv),str(met),str(nc),str(obpg_path),_app_status('NASA CyAN comparison complete',f"Paired detections >0.0001: {metrics['paired_above_detection_limit_pixels']:,}. This is same-day merged Level-3 versus one scene, not exact-overpass independent validation.")
    except Exception as exc:return None,pd.DataFrame(),pd.DataFrame(),None,None,None,None,_app_status('NASA CyAN comparison failed',str(exc),False)


def storage_callback(): return storage_summary(),str(registry_export_csv())
def clear_scratch_callback():
    try: return storage_summary(),_app_status('Scratch cleared',clear_scratch())
    except Exception as exc: return storage_summary(),_app_status('Scratch cleanup failed',str(exc),False)

APP_CSS_GLOBAL = '.gradio-container {max-width: 1550px !important}.small-note{font-size:.9rem;color:#4b5563}'

with gr.Blocks(title='Sentinel-3 OLCI Global CyanoLake Processor v2.6.4') as DEMO_GLOBAL:
    gr.Markdown('# Sentinel-3 OLCI → L2Gen → global HydroLAKES processor · v2.6.4')
    gr.Markdown(
        'No Google Drive permission is requested. Runtime-local, storage-efficient scene deduplication with **CI, CIcyano, NDCI, MPH and '
        'OLCI-adapted FAI/AFAI**, true color, corrected Web-Mercator scene maps, '
        'SNAP/GIS CF export, native orbit/frame archive indexing, and on-demand '
        'share bundles. The share ZIP always includes one consolidated SNAP-ready NetCDF with all production indices and required rhos bands, independent of visualization. FLH and OLH are intentionally excluded from this production branch.'
    )

    with gr.Tab('1 · Plan & process scenes'):
        gr.Markdown(
            '### Recommended order\n'
            '**A.** Load HydroLAKES. For global state/province-by-state/province production, keep **WORLD** so one canonical lake universe is used.  \n'
            '**B.** Choose a planning method: **State / province (ADM1)**, **Bounding box**, or **5° grid**.  \n'
            '**C.** For ADM1, choose a country, click **Load states/provinces**, then choose the first-level administrative unit.  \n'
            '**D.** Click **Plan scene queue** and process a small batch. A state/province is a discovery/work-tracking shard only. If one Sentinel-3 frame crosses two states, its scene ID + science configuration prevents duplicate L2Gen processing.  \n'
            '**E.** All files are written only under `/content` in this Colab runtime. Download share ZIPs / registry exports before the runtime ends. No Google Drive mount is used.'
        )
        with gr.Row():
            region = gr.Dropdown(
                ['WORLD','USA','Africa','Asia','Europe','North America','South America','Oceania'],
                value='WORLD', label='HydroLAKES universe (WORLD recommended)'
            )
            min_area = gr.Number(value=10.0, label='Minimum lake area (km²)')
            existing_hydro = gr.Textbox(value='', label='Optional existing HydroLAKES path')
            load_lakes = gr.Button('Load / prepare HydroLAKES', variant='primary')
        lake_preview = gr.Dataframe(interactive=False)
        lake_status = gr.HTML()
        load_lakes.click(load_lakes_callback, [region,min_area,existing_hydro], [lake_preview,lake_status])

        gr.Markdown('### Choose how to define the next processing shard')
        planning_mode = gr.Radio(
            [('State / province / region (ADM1)','adm1'),('Bounding box','bbox'),('Fixed 5° grid','grid')],
            value='adm1', label='Planning method'
        )
        with gr.Row():
            country_iso3 = gr.Dropdown(COUNTRY_CHOICES, value='USA', label='Country (for ADM1 mode)')
            load_admin = gr.Button('Load states / provinces')
            admin1_choice = gr.Dropdown(label='State / province / first-level region')
        admin_status = gr.HTML()
        load_admin.click(load_adm1_callback, country_iso3, [admin1_choice,admin_status])
        gr.Markdown('<span class="small-note">ADM1 boundaries are fetched on demand from geoBoundaries gbOpen. “ADM1” may be called state, province, region, department, governorate, etc. depending on the country. The boundary is used for scene discovery/tracking; Sentinel-3 scene identity remains the deduplication unit.</span>')

        with gr.Row():
            start_date = gr.Textbox(value='2024-08-01', label='Start date')
            end_date = gr.Textbox(value='2024-08-07', label='End date')
            bbox_text = gr.Textbox(value='', label='BBox west,south,east,north (bbox mode)')
            grid_id = gr.Textbox(value='', placeholder='G05_N40_W085', label='5° grid ID (grid mode)')
            max_products = gr.Number(value=0, precision=0, label='Catalogue cap (0=all)')
            plan_button = gr.Button('Plan scene queue', variant='primary')
        queue_table = gr.Dataframe(interactive=False)
        with gr.Row():
            plan_snapshot = gr.File(label='Current shard plan CSV')
        plan_status = gr.HTML()
        plan_button.click(
            plan_callback,
            [start_date,end_date,planning_mode,country_iso3,admin1_choice,bbox_text,grid_id,max_products],
            [queue_table,plan_snapshot,plan_status]
        )

        with gr.Row():
            cdse_user = gr.Textbox(label='CDSE username')
            cdse_password = gr.Textbox(label='CDSE password', type='password')
            n_scenes = gr.Slider(1,20,value=1,step=1,label='Scenes this batch')
            dl_method = gr.Dropdown(['auto','s3','zip'], value='auto', label='L1 staging')
            retry_failed = gr.Checkbox(False, label='Retry failed')
            run_button = gr.Button('Run next queued scenes', variant='primary')
        batch_status = gr.HTML()
        storage_after = gr.Dataframe(interactive=False)
        run_button.click(
            run_batch_callback,
            [cdse_user,cdse_password,n_scenes,dl_method,retry_failed],
            [queue_table,batch_status,storage_after]
        )
        with gr.Row():
            refresh_queue = gr.Button('Refresh queue')
            registry_file = gr.File(label='Scene registry CSV')
        refresh_queue.click(refresh_queue_callback, [], [queue_table,registry_file])

    with gr.Tab('2 · Lake time series'):
        with gr.Row():
            refresh_lakes = gr.Button('Refresh processed lakes', variant='primary')
            lake_choice = gr.Dropdown(label='Lake')
            plot_metric = gr.Dropdown(PLOT_METRICS, value=PLOT_METRICS[0], label='Statistic')
            plot_button = gr.Button('Plot lake history')
        lakes_status = gr.HTML()
        lake_plot = gr.Plot()
        lake_table = gr.Dataframe(interactive=False)
        refresh_lakes.click(refresh_lakes_callback, [], [lake_choice,lakes_status])
        plot_button.click(lake_timeseries_callback, [lake_choice,plot_metric], [lake_plot,lake_table])

    with gr.Tab('3 · Scene / multi-lake map'):
        gr.Markdown(
            'The live and portable HTML overlays are built directly on **EPSG:3857 Web Mercator**, '
            'matching Leaflet/Esri. The separate CF NetCDF remains EPSG:4326 for SNAP/GIS. '
            'Choose **Entire processed scene** to see all retained HydroLAKES pixels from all windows.'
        )
        with gr.Row():
            refresh_scenes = gr.Button('Refresh completed scenes', variant='primary')
            scene_choice = gr.Dropdown(label='Completed scene')
            map_scope = gr.Radio(
                [('Entire processed scene · all lakes','scene'),('Selected lake','lake')],
                value='scene', label='Map scope'
            )
            scene_lake = gr.Dropdown(label='Lake within scene')
        scene_status = gr.HTML()
        refresh_scenes.click(refresh_scene_choices_callback, [], [scene_choice,scene_lake,scene_status])
        scene_choice.change(scene_changed_callback, scene_choice, scene_lake)

        with gr.Row():
            map_metric = gr.Dropdown(MAP_METRICS, value='CI_cyano', label='Map layer')
            map_resolution = gr.Dropdown([300,500,1000,2000], value=500, label='Display/export grid (m)')
            basemap = gr.Dropdown(BASEMAPS, value='Esri Dark Gray', label='Basemap')
            labels = gr.Checkbox(True, label='Show Esri gray labels')
            show_lakes = gr.Checkbox(True, label='HydroLAKES boundaries')
        with gr.Row():
            cmap = gr.Dropdown(CMAPS, value='cividis', label='Colormap')
            reverse = gr.Checkbox(False, label='Reverse')
            vmin = gr.Textbox(value='', placeholder='Auto', label='Color minimum')
            vmax = gr.Textbox(value='', placeholder='Auto', label='Color maximum')
            render = gr.Button('Render / export map', variant='primary')
        map_status = gr.HTML()
        map_html = gr.HTML()
        with gr.Row():
            html_file = gr.File(label='Portable interactive HTML · hover values')
            snap_nc = gr.File(label='Displayed-layer CF NetCDF for SNAP/GIS (quick export)')
            compact_file = gr.File(label='Compact lake-pixel NetCDF / ZIP')
        render.click(
            map_callback,
            [scene_choice,map_scope,scene_lake,map_metric,map_resolution,basemap,labels,cmap,reverse,vmin,vmax,show_lakes],
            [map_html,html_file,snap_nc,compact_file,map_status]
        )

    with gr.Tab('4 · Spectra & lake statistics'):
        gr.Markdown('Both modes use the **native stored lake pixels** in the compact archive and do not rerun L2Gen. Non-lake pixels are intentionally unavailable in the efficient archive.')
        gr.Markdown('### A · Selected-lake mean / median spectrum')
        with gr.Row():
            spec_scene = gr.Dropdown(label='Completed scene')
            spec_lake = gr.Dropdown(label='Lake within scene')
            spec_stat = gr.Radio(['median','mean'], value='median', label='Spectrum statistic')
            spec_button = gr.Button('Plot lake spectrum', variant='primary')
            spec_refresh = gr.Button('Refresh scenes')
        spec_status = gr.HTML(); spec_plot = gr.Plot(); spec_table = gr.Dataframe(interactive=False, label='Lake spectral summary'); lake_stats_table = gr.Dataframe(interactive=False, label='Stored lake statistics'); spec_csv = gr.File(label='Lake spectrum CSV')
        spec_refresh.click(refresh_scene_choices_callback, [], [spec_scene,spec_lake,spec_status]); spec_scene.change(scene_changed_callback, spec_scene, spec_lake); spec_button.click(scene_spectrum_callback,[spec_scene,spec_lake,spec_stat],[spec_plot,spec_table,lake_stats_table,spec_csv,spec_status])
        gr.Markdown('### B · Coordinate / multi-point native-pixel spectra')
        coordinate_points=gr.Dataframe(headers=['label','latitude','longitude'],datatype=['str','number','number'],value=pd.DataFrame([{'label':'P1','latitude':41.7,'longitude':-82.9}]),row_count=(1,'dynamic'),column_count=(3,'fixed'),type='pandas',interactive=True,label='Edit up to 20 target coordinates')
        with gr.Row():
            coord_label=gr.Textbox(value='P2',label='New point label'); coord_lat=gr.Number(value=41.8,label='Latitude'); coord_lon=gr.Number(value=-83.1,label='Longitude'); coord_add=gr.Button('Add point'); coord_clear=gr.Button('Clear points')
        with gr.Row():
            coord_basis=gr.Radio([('Any stored HydroLAKES pixel','stored_lake'),('Retained valid-water pixel','valid_water'),('CI-input-valid pixel','ci_valid')],value='stored_lake',label='Nearest-pixel basis'); coord_distance=gr.Number(value=2000,minimum=100,maximum=10000,label='Maximum match distance (m)'); coord_plot_button=gr.Button('Plot coordinate spectra',variant='primary')
        coord_plot=gr.Plot(); coord_table=gr.Dataframe(interactive=False,label='Matched spectral values'); coord_csv=gr.File(label='Coordinate spectra CSV'); coord_status=gr.HTML()
        coord_add.click(add_spectrum_point_callback,[coord_label,coord_lat,coord_lon,coordinate_points],[coordinate_points,coord_status]); coord_clear.click(clear_spectrum_points_callback,outputs=[coordinate_points,coord_status]); coord_plot_button.click(coordinate_spectra_callback,[spec_scene,coordinate_points,coord_basis,coord_distance],[coord_plot,coord_table,coord_csv,coord_status])

    with gr.Tab('5 · NASA CyAN comparison'):
        gr.Markdown('Optional U.S./CONUS validation tab. Compare one processed scene/lake with NASA OBPG same-day **Merged-S3-CYAN 300 m CIcyano**. This reuses compact native lake pixels; no dense L2 or L2Gen rerun is required. Because the NASA product is a same-day S3A/S3B Level-3 merge, treat this as implementation/product consistency rather than independent biological validation.')
        with gr.Row():
            cyan_refresh=gr.Button('Refresh completed scenes',variant='primary'); cyan_scene=gr.Dropdown(label='Completed scene'); cyan_lake=gr.Dropdown(label='Lake within scene')
        cyan_refresh_status=gr.HTML(); cyan_refresh.click(refresh_scene_choices_callback,[],[cyan_scene,cyan_lake,cyan_refresh_status]); cyan_scene.change(scene_changed_callback,cyan_scene,cyan_lake)
        with gr.Row():
            cyan_date=gr.Textbox(value='',placeholder='blank = infer from scene',label='UTC date YYYY-MM-DD'); cyan_mode=gr.Radio([('Download from NASA OBPG using configured Earthdata Login','download'),('Upload existing OBPG CIcyano GeoTIFF','upload')],value='download',label='NASA source'); cyan_upload=gr.File(label='Optional OBPG CIcyano GeoTIFF',type='filepath',file_types=['.tif','.tiff']); cyan_distance=gr.Number(value=450,minimum=50,maximum=1500,label='Max nearest-neighbour distance (m)')
        with gr.Row(): cyan_prepare=gr.Button('1 · Download / check NASA GeoTIFF'); cyan_compare=gr.Button('2 · Compare selected lake',variant='primary')
        cyan_source=gr.File(label='NASA source GeoTIFF'); cyan_status=gr.HTML(); cyan_coverage=gr.Dataframe(interactive=False,label='Coverage / pairing audit'); cyan_metrics=gr.Dataframe(interactive=False,label='Comparison metrics'); cyan_plot=gr.Plot(label='Paired detections')
        with gr.Row(): cyan_pairs=gr.File(label='Paired cells CSV'); cyan_metrics_file=gr.File(label='Metrics CSV'); cyan_nc=gr.File(label='Comparison NetCDF'); cyan_tif=gr.File(label='NASA source GeoTIFF')
        cyan_prepare.click(prepare_cyan_source_callback,[cyan_scene,cyan_date,cyan_mode,cyan_upload],[cyan_source,cyan_status]); cyan_compare.click(compare_cyan_callback,[cyan_scene,cyan_lake,cyan_date,cyan_mode,cyan_upload,cyan_distance],[cyan_plot,cyan_coverage,cyan_metrics,cyan_pairs,cyan_metrics_file,cyan_nc,cyan_tif,cyan_status])

    with gr.Tab('6 · Global archive & share'):
        gr.Markdown(
            '### How to use this tab\n'
            'Use it **after processing scenes**. No atmospheric correction happens here.\n\n'
            'Each completed Sentinel-3 frame is catalogued by `platform + relative orbit + frame`, '
            'for example `S3B_R211_F2160`. A fixed 5° grid ID is only a geographic lookup key. '
            '**Do not process the same scene separately for each grid cell.**\n\n'
            '**Refresh archive index** builds a searchable CSV from already completed scenes. '
            '**Create share bundle** packages only files that actually exist from this runtime, avoiding stale-path '
            'download errors. Raw L1 and temporary dense L2 are normally absent by design.'
        )
        with gr.Row():
            archive_grid = gr.Textbox(value='', placeholder='G05_N40_W085', label='Optional grid filter')
            archive_orbit = gr.Textbox(value='', placeholder='211', label='Optional relative-orbit filter')
            archive_refresh = gr.Button('Refresh archive index', variant='primary')
        archive_table = gr.Dataframe(interactive=False)
        archive_csv = gr.File(label='Global archive index CSV')
        archive_status = gr.HTML()
        archive_refresh.click(archive_refresh_callback, [archive_grid,archive_orbit], [archive_table,archive_csv,archive_status])

        with gr.Row():
            bundle_scene = gr.Dropdown(label='Completed scene')
            bundle_resolution = gr.Dropdown([300,500,1000], value=300, label='Complete SNAP science grid requested resolution (m)')
            bundle_refresh = gr.Button('Refresh completed scenes')
            bundle_button = gr.Button('Create complete scene-results ZIP', variant='primary')
        gr.Markdown('The ZIP **always** creates/includes one consolidated CF NetCDF with CI, CIcyano, NDCI, MPH, FAI/AFAI and all rhos bands required for true color and those indices, even if those layers were never rendered. If the scene extent exceeds the safety cell cap the actual regular-grid resolution is increased and recorded in metadata. Compact files retain exact native lake-pixel values.')
        bundle_dummy_lake = gr.Dropdown(visible=False)
        bundle_file = gr.File(label='Shareable scene ZIP')
        bundle_status = gr.HTML()
        bundle_refresh.click(refresh_scene_choices_callback, [], [bundle_scene,bundle_dummy_lake,bundle_status])
        bundle_button.click(bundle_callback, [bundle_scene,bundle_resolution], [bundle_file,bundle_status])

    with gr.Tab('7 · Storage & provenance'):
        with gr.Row():
            refresh_storage = gr.Button('Refresh storage')
            clear_button = gr.Button('Clear scratch only')
        storage_table = gr.Dataframe(interactive=False)
        storage_status = gr.HTML()
        registry_download = gr.File(label='Registry CSV')
        refresh_storage.click(storage_callback, [], [storage_table,registry_download])
        clear_button.click(clear_scratch_callback, [], [storage_table,storage_status])

print('Global-efficient Gradio v2.6 constructed.')


Global-efficient Gradio v2.6 constructed.


## 10.1 Production configuration check

Run this before launching Gradio. It does not download or process a scene.


In [30]:
print("Runtime-local output root:", PERSISTENT_ROOT)
print("Scratch root   :", SCRATCH_ROOT)
print("Google Drive access: DISABLED by design")
print("L1 staging method:", DOWNLOAD_METHOD)
print("Keep staged L1:", KEEP_STAGED_L1)
print("Keep L2 intermediate:", KEEP_L2_INTERMEDIATE)
print("Full rhos spectrum:", SAVE_FULL_REFLECTANCE_SPECTRA)
print("Core rhos bands:", CORE_RHOS_WAVELENGTHS)
print("HydroLAKES default water definition:", DEFAULT_WATER_DEFINITION)
print("Shore buffer (m):", DEFAULT_SHORE_BUFFER_M)
print("Processing strategy:", PROCESSING_STRATEGY)
print("Registry:", REGISTRY_DB)
assert not KEEP_STAGED_L1, "Storage-efficient production expects KEEP_STAGED_L1=False"
assert not KEEP_L2_INTERMEDIATE, "Storage-efficient production expects KEEP_L2_INTERMEDIATE=False"
assert set(CORE_RHOS_WAVELENGTHS) == {620,665,681,709}
print("Configuration check passed.")


Runtime-local output root: /home/myid/mc16405/cyanolake-web/s3_olci_global
Scratch root   : /home/myid/mc16405/cyanolake-web/s3_olci_scratch
Google Drive access: DISABLED by design
L1 staging method: auto
Keep staged L1: False
Keep L2 intermediate: False
Full rhos spectrum: False
Core rhos bands: (620, 665, 681, 709)
HydroLAKES default water definition: hydrolakes
Shore buffer (m): 0.0
Processing strategy: adaptive_lake_windows
Registry: /home/myid/mc16405/cyanolake-web/s3_olci_global/scene_registry.sqlite
Configuration check passed.


## 11. Launch



## v2.6.3 quantitative export / CyAN-comparable production patch

This patch makes two deliberate science changes:

1. **No seam filling or spatial interpolation in the share NetCDF.** Native stored OLCI lake pixels are binned by their actual centres to a ~300 m spherical-sinusoidal grid. Empty bins remain fill/NaN.
2. **CyAN-comparable quantitative QA is the production default.** LAND is excluded through the L2 non-land requirement and `CLDICE` + `HISATZEN` are excluded from `l2_flags`, matching the published OBPG CyAN Level-3 flag set as closely as this custom workflow can.

The SNAP share raster is a custom CyAN-style sinusoidal binned product, **not** a bit-for-bit NASA Integerized Sinusoidal Grid file. Native compact pixels remain the numerical source of truth for lake statistics.


In [31]:

# =============================================================================
# v2.6.3 scientific export/share patch
# - NO seam filling or interpolation
# - CyAN-style metric sinusoidal observation binning for the SNAP/GIS share product
# - minimal share ZIP: complete science NetCDF + all-index lake stats CSV + metadata JSON
# =============================================================================

CYAN_STYLE_GRID_METHOD = "spherical_sinusoidal_native_observation_bin_v1"
CYAN_STYLE_SPHERE_RADIUS_M = 6371007.181


def _compact_qa_metadata(scene_id):
    """Read the actual QA policy embedded in the compact product."""
    files = _scene_compact_files(scene_id)
    meta = {
        "post_l2_mask_profile_inferred": "unknown",
        "qa_exclude_flags": [],
        "l2gen_internal_cloud_mask": False,
        "cloud_ice_flag_CLDICE_excluded": False,
        "high_sensor_zenith_HISATZEN_excluded": False,
        "require_l2_nonland": False,
    }
    if not files:
        return meta
    with netCDF4.Dataset(files[0]) as ds:
        raw = str(getattr(ds, "qa_exclude_flags", "") or "")
        flags = [x.strip().upper() for x in re.split(r"[, ]+", raw) if x.strip()]
        meta.update({
            "qa_exclude_flags": flags,
            "require_l2_nonland": bool(int(getattr(ds, "require_l2_nonland", 0))),
            "shore_buffer_m": float(getattr(ds, "shore_buffer_m", 0.0)),
            "ci_detection_limit": float(getattr(ds, "ci_detection_limit", CYAN_CI_DETECTION_LIMIT)),
            "cloud_ice_flag_CLDICE_excluded": "CLDICE" in flags,
            "high_sensor_zenith_HISATZEN_excluded": "HISATZEN" in flags,
        })
    flagset = set(meta["qa_exclude_flags"])
    for key, preset in MASK_PRESETS.items():
        if flagset == set(str(x).upper() for x in preset.get("qa_flags", ())):
            # LAND is represented separately by require_l2_nonland, so distinguish
            # the CyAN-comparable profile only when LAND is also enforced.
            if key == "cyan_comparable" and not meta.get("require_l2_nonland", False):
                continue
            meta["post_l2_mask_profile_inferred"] = key
            break
    return meta


def _science_sinusoidal_setup(frame, requested_resolution_m=SCIENCE_EXPORT_DEFAULT_RESOLUTION_M):
    """Bin native retained/stored OLCI lake observations to a metric sinusoidal grid.

    This is intentionally observation binning, not interpolation/resampling:
    a target bin receives values only from native OLCI pixel centres that actually
    fall inside that bin. Empty bins remain NetCDF fill/NaN.

    It follows the same *principle* as OBPG CyAN's sinusoidal Level-3 binning,
    but is not claimed to reproduce NASA's Integerized Sinusoidal Grid bit-for-bit.
    """
    from pyproj import CRS, Transformer

    lat = pd.to_numeric(frame["latitude"], errors="coerce").to_numpy(float)
    lon = pd.to_numeric(frame["longitude"], errors="coerce").to_numpy(float)
    finite = np.isfinite(lat) & np.isfinite(lon) & (np.abs(lat) <= 90) & (np.abs(lon) <= 180)
    if not finite.any():
        raise ValueError("Selected scene contains no finite stored geolocated lake pixels")

    radius = float(CYAN_STYLE_SPHERE_RADIUS_M)
    crs = CRS.from_proj4(
        f"+proj=sinu +lon_0=0 +x_0=0 +y_0=0 +R={radius:.3f} +units=m +no_defs"
    )
    transformer = Transformer.from_crs(4326, crs, always_xy=True)
    x_all = np.full(lon.shape, np.nan, dtype=np.float64)
    y_all = np.full(lat.shape, np.nan, dtype=np.float64)
    xx, yy = transformer.transform(lon[finite], lat[finite])
    x_all[finite] = xx
    y_all[finite] = yy

    # Stable global origin. This lets separately processed scenes use aligned cells
    # for a given requested output resolution.
    global_x_min = -math.pi * radius
    global_y_max = 0.5 * math.pi * radius

    requested = max(1.0, float(requested_resolution_m))
    resolution = requested

    def geometry(res):
        cols_global = np.floor((x_all[finite] - global_x_min) / res).astype(np.int64)
        rows_global = np.floor((global_y_max - y_all[finite]) / res).astype(np.int64)
        c0, c1 = int(cols_global.min()), int(cols_global.max())
        r0, r1 = int(rows_global.min()), int(rows_global.max())
        nx, ny = c1 - c0 + 1, r1 - r0 + 1
        return cols_global, rows_global, c0, r0, nx, ny

    cols_g, rows_g, col0, row0, nx, ny = geometry(resolution)
    ncell = int(nx * ny)
    if ncell > int(SCIENCE_EXPORT_MAX_CELLS):
        # Increase the scene export resolution rather than crashing memory.
        scale = math.sqrt(ncell / float(SCIENCE_EXPORT_MAX_CELLS))
        resolution = math.ceil((resolution * scale) / 10.0) * 10.0
        cols_g, rows_g, col0, row0, nx, ny = geometry(resolution)
        ncell = int(nx * ny)

    finite_idx = np.flatnonzero(finite)
    local_c = cols_g - col0
    local_r = rows_g - row0
    inside = np.zeros(len(frame), dtype=bool)
    inside[finite_idx] = (
        (local_c >= 0) & (local_c < nx) & (local_r >= 0) & (local_r < ny)
    )
    kept_local = np.flatnonzero(inside)
    # Recompute local indices in original frame order for the kept points.
    cols_kept_global = np.floor((x_all[inside] - global_x_min) / resolution).astype(np.int64)
    rows_kept_global = np.floor((global_y_max - y_all[inside]) / resolution).astype(np.int64)
    cc = cols_kept_global - col0
    rr = rows_kept_global - row0
    flat = rr * nx + cc

    x_left = global_x_min + col0 * resolution
    y_top = global_y_max - row0 * resolution
    x = x_left + (np.arange(nx, dtype=np.float64) + 0.5) * resolution
    y_desc = y_top - (np.arange(ny, dtype=np.float64) + 0.5) * resolution
    y = y_desc[::-1]

    g = {
        "crs": crs,
        "resolution_m": float(resolution),
        "requested_resolution_m": float(requested),
        "nx": int(nx), "ny": int(ny), "ncell": int(ncell),
        "x": x, "y": y,
        "x_left": float(x_left), "y_top": float(y_top),
        "global_col0": int(col0), "global_row0": int(row0),
        "radius_m": radius,
    }
    return g, inside, flat


def _binned_mask_grid(values, inside, flat, ncell):
    """Binary-mask aggregation with -1 for bins containing no native observations."""
    vals = np.asarray(values, float)[inside]
    support = np.bincount(flat, minlength=int(ncell)) > 0
    out = np.full(int(ncell), -1, dtype=np.int8)
    out[support] = 0
    positive = np.isfinite(vals) & (vals > 0)
    if positive.any():
        np.maximum.at(out, flat[positive], np.int8(1))
    return out


def export_complete_scene_science_netcdf(
    scene_id,
    requested_resolution_m=SCIENCE_EXPORT_DEFAULT_RESOLUTION_M,
    force=False,
):
    """Create one SNAP/GIS NetCDF using only actual native OLCI observations.

    No seam filling, nearest-neighbour hole filling, or spatial interpolation is
    performed. Each output cell is the arithmetic mean of native stored OLCI
    lake-pixel observations whose centres fall within that metric sinusoidal bin.
    Empty bins remain fill/NaN.
    """
    if not scene_id:
        raise ValueError("Choose a completed scene")

    frame = _load_scene_pixels(scene_id, "scene", None)
    g, inside, flat = _science_sinusoidal_setup(frame, float(requested_resolution_m))
    ncell = int(g["ncell"])

    export_dir = EXPORT_DIR / "snap_complete"
    export_dir.mkdir(parents=True, exist_ok=True)
    out = export_dir / (
        f"S3_OLCI_{scene_id}_ALL_SCIENCE_{int(round(g['resolution_m']))}m_CF.nc"
    )

    source_files = _scene_compact_files(scene_id)
    newest_source = max((p.stat().st_mtime_ns for p in source_files), default=0)

    if out.exists() and not force and out.stat().st_mtime_ns >= newest_source:
        with contextlib.suppress(Exception):
            with netCDF4.Dataset(out) as old:
                if getattr(old, "science_grid_method", "") == CYAN_STYLE_GRID_METHOD:
                    return str(out), g

    from pyproj import Transformer

    crs = g["crs"]
    wkt = crs.to_wkt()
    qa_meta = _compact_qa_metadata(scene_id)

    # Geographic metadata is based on actual stored native pixels, not projected corners.
    lat_native = pd.to_numeric(frame["latitude"], errors="coerce").to_numpy(float)
    lon_native = pd.to_numeric(frame["longitude"], errors="coerce").to_numpy(float)
    geo_ok = np.isfinite(lat_native) & np.isfinite(lon_native)

    tmp = out.with_suffix(".nc.tmp")
    tmp.unlink(missing_ok=True)

    with netCDF4.Dataset(tmp, "w", format="NETCDF4_CLASSIC") as ds:
        ds.createDimension("y", int(g["ny"]))
        ds.createDimension("x", int(g["nx"]))
        ds.setncatts({
            "Conventions": "CF-1.8",
            "title": "Sentinel-3 OLCI HydroLAKES all-science CyAN-style binned product",
            "scene_id": str(scene_id),
            "source_representation": (
                "native OLCI lake pixels; observation-centre binning to a metric "
                "spherical sinusoidal grid; no spatial interpolation or seam filling"
            ),
            "science_grid_method": CYAN_STYLE_GRID_METHOD,
            "cyan_comparison_note": (
                "Uses CyAN-style sinusoidal observation binning in principle; this "
                "custom raster is NOT a bit-for-bit NASA OBPG Integerized Sinusoidal Grid product."
            ),
            "requested_grid_resolution_m": float(requested_resolution_m),
            "actual_grid_resolution_m": float(g["resolution_m"]),
            "aggregation": "arithmetic mean of actual native OLCI observations falling in each bin",
            "interpolation": "none",
            "empty_bin_policy": "NetCDF fill/NaN; never gap-filled",
            "geospatial_lat_min": float(np.nanmin(lat_native[geo_ok])),
            "geospatial_lat_max": float(np.nanmax(lat_native[geo_ok])),
            "geospatial_lon_min": float(np.nanmin(lon_native[geo_ok])),
            "geospatial_lon_max": float(np.nanmax(lon_native[geo_ok])),
            "science_variables": (
                "CI, CI_cyano, NDCI, MPH, OLCI-adapted FAI/AFAI plus required "
                "Rayleigh-corrected rhos bands"
            ),
            "true_color_mapping": "red=rhos_665; green=rhos_560; blue=rhos_490",
            "nominal_885_mapping": (
                "compact rhos_885 is sourced from canonical L2Gen rhos_884; "
                "exported here as rhos_884"
            ),
            "CI_cyano_detection_limit": float(CYAN_CI_DETECTION_LIMIT),
            "post_l2_mask_profile_inferred": str(
                qa_meta.get("post_l2_mask_profile_inferred", "unknown")
            ),
            "qa_exclude_flags": " ".join(qa_meta.get("qa_exclude_flags", [])),
            "l2_nonland_required": int(bool(qa_meta.get("require_l2_nonland", False))),
            "cloud_ice_flag_CLDICE_excluded": int(
                bool(qa_meta.get("cloud_ice_flag_CLDICE_excluded", False))
            ),
            "high_sensor_zenith_HISATZEN_excluded": int(
                bool(qa_meta.get("high_sensor_zenith_HISATZEN_excluded", False))
            ),
            "note": (
                "Lake statistics are calculated from native stored OLCI pixels before "
                "this binned interoperability export."
            ),
        })

        yv = ds.createVariable("y", "f8", ("y",))
        yv[:] = g["y"]
        yv.setncatts({
            "standard_name": "projection_y_coordinate",
            "long_name": "y coordinate of projection",
            "units": "m",
            "axis": "Y",
        })

        xv = ds.createVariable("x", "f8", ("x",))
        xv[:] = g["x"]
        xv.setncatts({
            "standard_name": "projection_x_coordinate",
            "long_name": "x coordinate of projection",
            "units": "m",
            "axis": "X",
        })

        crsv = ds.createVariable("crs", "i4")
        crsv.assignValue(0)
        crsv.setncatts({
            "grid_mapping_name": "sinusoidal",
            "longitude_of_central_meridian": 0.0,
            "false_easting": 0.0,
            "false_northing": 0.0,
            "earth_radius": float(CYAN_STYLE_SPHERE_RADIUS_M),
            "spatial_ref": wkt,
            "crs_wkt": wkt,
            "GeoTransform": (
                f"{g['x_left']} {g['resolution_m']} 0 "
                f"{g['y_top']} 0 {-g['resolution_m']}"
            ),
        })

        chunks = (min(512, int(g["ny"])), min(512, int(g["nx"])))
        variable_map = [
            (name, "rhos_884" if name == "rhos_885" else name)
            for name in SCIENCE_EXPORT_VARIABLES
        ]

        for source_name, out_name in variable_map:
            if source_name not in frame.columns:
                continue

            flat_grid = _mean_grid_from_flat(
                frame[source_name].to_numpy(float), inside, flat, ncell
            )
            grid = flat_grid.reshape(int(g["ny"]), int(g["nx"]))[::-1, :]

            var = ds.createVariable(
                out_name, "f4", ("y", "x"),
                zlib=True, complevel=6, shuffle=True,
                chunksizes=chunks, fill_value=np.float32(-9999.0),
            )
            long_names = {
                "CI": "Cyanobacteria Index candidate magnitude",
                "CI_cyano": "CyAN-compatible cyanobacteria index detections",
                "NDCI": "Normalized Difference Chlorophyll Index",
                "MPH": "Maximum Peak Height",
                "FAI": "OLCI-adapted Floating Algae Index (AFAI form)",
            }
            var.setncatts({
                "grid_mapping": "crs",
                "units": "1",
                "long_name": long_names.get(
                    out_name, f"Rayleigh-corrected OLCI reflectance {out_name}"
                ),
            })
            if out_name == "MPH":
                var.setncattr(
                    "formula",
                    "maximum line height at approximately 681/709/754 nm above "
                    "a 665-to-884 nm baseline",
                )
            if out_name == "FAI":
                var.setncattr(
                    "formula",
                    "rhos_754 minus linear baseline between rhos_665 and rhos_865; "
                    "OLCI-adapted AFAI form",
                )
            if out_name == "CI_cyano":
                var.setncattr("detection_limit", float(CYAN_CI_DETECTION_LIMIT))
            if source_name == "rhos_885":
                var.setncattr(
                    "source_compact_variable",
                    "rhos_885 nominal key; canonical L2Gen source rhos_884",
                )
            var[:] = np.ma.masked_invalid(grid)
            del flat_grid, grid

        # QA masks. -1 means there was no native stored observation in the bin.
        for source_name, out_name in (
            ("valid_water_mask", "valid_water_mask"),
            ("ci_valid_mask", "ci_valid_mask"),
            ("ci_candidate_mask", "ci_candidate_mask"),
            ("ci_detection_mask", "ci_detection_mask"),
        ):
            if source_name not in frame.columns:
                continue
            flat_grid = _binned_mask_grid(
                frame[source_name].to_numpy(float), inside, flat, ncell
            )
            grid = flat_grid.reshape(int(g["ny"]), int(g["nx"]))[::-1, :]

            var = ds.createVariable(
                out_name, "i1", ("y", "x"),
                zlib=True, complevel=6, shuffle=True,
                chunksizes=chunks, fill_value=np.int8(-1),
            )
            var.setncatts({
                "grid_mapping": "crs",
                "long_name": out_name.replace("_", " "),
                "flag_values": np.array([0, 1], dtype=np.int8),
                "flag_meanings": "false true",
                "valid_range": np.array([0, 1], dtype=np.int8),
            })
            var[:] = np.ma.masked_equal(grid, np.int8(-1))
            del flat_grid, grid

    tmp.replace(out)
    return str(out), g


def _share_all_index_stats(scene_id, cfg, row):
    """One row per lake with consistent descriptive statistics for every production index."""
    frame = _load_scene_pixels(scene_id, "scene", None).copy()
    if frame.empty:
        raise ValueError("No compact lake pixels found for the selected scene")

    # Avoid double-counting any overlap between adaptive processing windows.
    for c in ("latitude", "longitude", "Hylak_id"):
        frame[c] = pd.to_numeric(frame[c], errors="coerce")
    frame = frame.dropna(subset=["latitude", "longitude", "Hylak_id"]).copy()
    frame["_lat6"] = frame["latitude"].round(6)
    frame["_lon6"] = frame["longitude"].round(6)
    frame = frame.drop_duplicates(["Hylak_id", "_lat6", "_lon6"]).copy()

    lake_meta = {}
    lakes = GLOBAL_STATE.get("lakes")
    if lakes is not None and not lakes.empty and "Hylak_id" in lakes.columns:
        for _, lr in lakes.iterrows():
            try:
                lid = int(lr["Hylak_id"])
            except Exception:
                continue
            lake_meta[lid] = {
                "Lake_name": lr.get("Lake_name", "") if "Lake_name" in lakes.columns else "",
                "Lake_area_km2": float(lr.get("Lake_area", np.nan)) if "Lake_area" in lakes.columns else np.nan,
                "Country": lr.get("Country", "") if "Country" in lakes.columns else "",
                "Continent": lr.get("Continent", "") if "Continent" in lakes.columns else "",
            }

    def mask_col(g, name):
        if name not in g.columns:
            return np.ones(len(g), dtype=bool)
        return pd.to_numeric(g[name], errors="coerce").fillna(0).to_numpy(float) > 0

    def stats(values):
        a = pd.to_numeric(values, errors="coerce").to_numpy(float)
        a = a[np.isfinite(a)]
        if not len(a):
            return {
                "valid_pixels": 0, "mean": np.nan, "median": np.nan,
                "p10": np.nan, "p90": np.nan, "max": np.nan,
            }
        return {
            "valid_pixels": int(len(a)),
            "mean": float(np.nanmean(a)),
            "median": float(np.nanmedian(a)),
            "p10": float(np.nanpercentile(a, 10)),
            "p90": float(np.nanpercentile(a, 90)),
            "max": float(np.nanmax(a)),
        }

    records = []
    for lid, gdf in frame.groupby("Hylak_id", sort=False):
        lid = int(lid)
        retained = mask_col(gdf, "valid_water_mask")
        ci_valid = mask_col(gdf, "ci_valid_mask")
        candidate = mask_col(gdf, "ci_candidate_mask")
        detected = mask_col(gdf, "ci_detection_mask")

        rec = {
            "scene_id": str(scene_id),
            "scene_name": str(row.scene_name),
            "acquisition_start": str(row.acquisition_start),
            "Hylak_id": lid,
            **lake_meta.get(lid, {}),
            "native_lake_pixels": int(len(gdf)),
            "retained_pixels": int(retained.sum()),
            "ci_input_valid_pixels": int(ci_valid.sum()),
            "ci_candidate_pixels": int(candidate.sum()),
            "ci_cyano_detection_pixels": int(detected.sum()),
            "ci_cyano_detection_fraction_of_ci_valid": (
                float(detected.sum() / ci_valid.sum()) if ci_valid.sum() else np.nan
            ),
            "ci_cyano_detection_fraction_of_retained": (
                float(detected.sum() / retained.sum()) if retained.sum() else np.nan
            ),
        }

        bases = {
            "CI": candidate,
            "CI_cyano": detected,
            "NDCI": retained,
            "MPH": retained,
            "FAI": retained,
        }
        for metric, basis in bases.items():
            if metric not in gdf.columns:
                s = stats(pd.Series(dtype=float))
            else:
                s = stats(gdf.loc[basis, metric])
            for stat_name, value in s.items():
                rec[f"{metric}_{stat_name}"] = value
        records.append(rec)

    out = pd.DataFrame(records).sort_values("Hylak_id").reset_index(drop=True)
    d = EXPORT_DIR / "scene_bundles"
    d.mkdir(parents=True, exist_ok=True)
    path = d / f"S3_OLCI_{scene_id}_ALL_INDEX_STATS.csv"
    out.to_csv(path, index=False)
    return path, out



def create_scene_share_bundle(
    scene_id,
    science_resolution_m=SCIENCE_EXPORT_DEFAULT_RESOLUTION_M,
):
    """Create the minimal three-file share ZIP."""
    if not scene_id:
        raise ValueError("Choose a completed scene")

    t = registry_table(GLOBAL_STATE.get("config_hash"))
    r = t[t.scene_id.astype(str) == str(scene_id)]
    if r.empty:
        raise KeyError(scene_id)

    row = r.iloc[0]
    cfg = str(row.config_hash)
    d = EXPORT_DIR / "scene_bundles"
    d.mkdir(parents=True, exist_ok=True)

    complete_nc, g = export_complete_scene_science_netcdf(
        scene_id, float(science_resolution_m), force=False
    )
    stats_csv, stats_table = _share_all_index_stats(scene_id, cfg, row)

    frame_meta = parse_s3_frame_metadata(row.scene_name)
    qa_meta = _compact_qa_metadata(scene_id)

    meta = {
        "schema": "global-efficient-share-v2.6.3",
        "created_utc": utc_now(),
        "scene_id": str(scene_id),
        "scene_name": str(row.scene_name),
        "acquisition_start": str(row.acquisition_start),
        "config_hash": cfg,
        **frame_meta,
        "files": {
            "snap_all_science_netcdf": Path(complete_nc).name,
            "all_index_stats_csv": Path(stats_csv).name,
        },
        "snap_variables": [
            "rhos_490", "rhos_560", "rhos_620", "rhos_665", "rhos_681",
            "rhos_709", "rhos_754", "rhos_865", "rhos_884",
            "CI", "CI_cyano", "NDCI", "MPH", "FAI",
        ],
        "true_color_mapping": {
            "red": "rhos_665",
            "green": "rhos_560",
            "blue": "rhos_490",
        },
        "snap_grid": {
            "projection": "spherical sinusoidal",
            "grid_method": CYAN_STYLE_GRID_METHOD,
            "requested_resolution_m": float(science_resolution_m),
            "actual_resolution_m": float(g["resolution_m"]),
            "aggregation": (
                "native-observation bin mean; no interpolation and no seam/gap filling"
            ),
            "empty_bins": "fill/NaN",
            "comparison_to_OBPG_CyAN": (
                "Same general sinusoidal observation-binning principle, but not a "
                "bit-for-bit implementation of the NASA Integerized Sinusoidal Grid."
            ),
        },
        "statistics_basis": {
            "CI": "finite CI candidate pixels only",
            "CI_cyano": "finite CIcyano detection pixels only",
            "NDCI": "finite retained-water native pixels",
            "MPH": "finite retained-water native pixels",
            "FAI": "finite retained-water native pixels",
        },
        "masking": {
            **qa_meta,
            "l2gen_internal_output_masks": (
                "disabled intentionally; l2_flags are retained and post-L2 exclusions "
                "are applied before quantitative index statistics/storage"
            ),
            "cyan_comparable_target": {
                "LAND": "implemented by require_l2_nonland=True",
                "CLDICE": "post-L2 exclusion",
                "HISATZEN": "post-L2 exclusion",
            },
        },
        "important": (
            "Quantitative CSV statistics are computed from native stored OLCI lake "
            "pixels, not from the mapped/binned SNAP raster."
        ),
    }

    metadata_path = d / f"S3_OLCI_{scene_id}_METADATA.json"
    metadata_path.write_text(json.dumps(meta, indent=2, default=str))

    z = d / f"{str(row.scene_name)}_{cfg}_SHARE_MINIMAL.zip"
    with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED, allowZip64=True) as arc:
        for p in (Path(complete_nc), Path(stats_csv), metadata_path):
            arc.write(p, arcname=p.name)

    return str(z), _app_status(
        "Minimal share bundle created",
        (
            f"3 files only · {g['resolution_m']:.0f} m sinusoidal native-observation bins · "
            "no interpolation/seam filling · all rhos + CI/CIcyano/NDCI/MPH/FAI"
        ),
    )


def bundle_callback(scene_id, science_resolution_m):
    try:
        return create_scene_share_bundle(scene_id, science_resolution_m)
    except Exception as exc:
        return None, _app_status("Bundle failed", str(exc), False)


print("v2.6.3 active:")
print("  default quantitative QA:", DEFAULT_MASK_PROFILE)
print("  QA flags excluded       :", QA_EXCLUDE_FLAGS)
print("  require L2 non-LAND     :", REQUIRE_L2_NONLAND)
print("  SNAP share grid         :", CYAN_STYLE_GRID_METHOD)
print("  gap/interpolation fill  : NONE")


v2.6.3 active:
  default quantitative QA: cyan_comparable
  QA flags excluded       : ('CLDICE', 'HISATZEN')
  require L2 non-LAND     : True
  SNAP share grid         : spherical_sinusoidal_native_observation_bin_v1
  gap/interpolation fill  : NONE


### v2.6.4 science QA and export revision

- Primary quantitative mask is now **bloom-aware**: clouds remain excluded, but `CLDICE` pixels can be restored only when independent bloom spectral evidence is present.
- A separate **CyAN-strict** mask is retained for NASA CyAN comparison.
- Gradio maps and the share NetCDF use the **same primary QA mask**.
- The share NetCDF uses an **exact requested sinusoidal resolution** (300 m by default) with sparse chunk writes; it never silently changes 300 m to 370 m.
- No spatial interpolation or gap filling is performed.


In [32]:

# =============================================================================
# v2.6.4 SCIENCE PATCH
# Bloom-aware cloud recovery + CyAN-strict comparison mask + fixed-resolution
# sparse sinusoidal share export.  This cell intentionally overrides selected
# v2.6.3 helpers at runtime; no Google Drive access is introduced.
# =============================================================================

V264_SCHEMA = "global-efficient-v2.6.4"

# Two explicit QA products are retained:
#   1) CyAN-strict: LAND + CLDICE + HISATZEN, with NAVFAIL removed as a safety QA.
#   2) Primary bloom-aware: same hard exclusions, except CLDICE can be rescued when
#      independent bloom spectral evidence is present.
CYAN_STRICT_QA_FLAGS = ("CLDICE", "HISATZEN", "NAVFAIL")
BLOOM_AWARE_HARD_FLAGS = ("HISATZEN", "NAVFAIL")
BLOOM_RESCUE_REQUIRE_CI_CYANO = True
BLOOM_RESCUE_REQUIRE_POSITIVE_AFAI = True
BLOOM_RESCUE_REQUIRE_NIR_OVER_BLUE = True
BLOOM_RESCUE_AFAI_MIN = 0.0

MASK_PRESETS["bloom_aware_cyan"] = {
    "label": "Bloom-aware quantitative (cloud masked + CLDICE bloom recovery)",
    "qa_flags": BLOOM_AWARE_HARD_FLAGS,
    "land_adjacency_pixels": 0,
    "note": (
        "Primary quantitative mask. LAND, HISATZEN and NAVFAIL are excluded. "
        "CLDICE is excluded unless the pixel independently passes the standard "
        "CIcyano spectral gates, has positive OLCI AFAI, and shows 754-nm "
        "reflectance above blue 490-nm reflectance. The recovery concept is "
        "informed by Wang & Jiang (2025), but is not claimed to reproduce NOAA "
        "MSL12 thresholds exactly."
    ),
}
DEFAULT_MASK_PROFILE = "bloom_aware_cyan"
QA_EXCLUDE_FLAGS = BLOOM_AWARE_HARD_FLAGS
LAND_ADJACENCY_PIXELS = 0
REQUIRE_L2_NONLAND = True
DEFAULT_BRIGHT_SCREEN = False

# Share/export resolution is now exact.  The exporter no longer silently coarsens
# 300 m to 370 m. Large empty regions are represented by unwritten compressed
# NetCDF chunks rather than by allocating a full in-memory raster.
CYAN_STYLE_GRID_METHOD = "spherical_sinusoidal_sparse_native_observation_bin_v2"
SCIENCE_EXPORT_DEFAULT_RESOLUTION_M = 300
SCIENCE_EXPORT_CHUNK = 256


def processing_config_hash(min_area_km2: float, region: str = "WORLD") -> str:
    payload = {
        "l2_mode": L2_MODE,
        "ndci_source": NDCI_SOURCE,
        "analysis_rhos": list(GLOBAL_ANALYSIS_RHOS_WAVELENGTHS),
        "true_color_rhos": list(TRUE_COLOR_RHOS_WAVELENGTHS),
        "indices": ["CI", "CI_cyano", "NDCI", *ADDITIONAL_INDEX_NAMES],
        "full_spectrum": bool(SAVE_FULL_REFLECTANCE_SPECTRA),
        "primary_qa_profile": DEFAULT_MASK_PROFILE,
        "primary_hard_flags": list(BLOOM_AWARE_HARD_FLAGS),
        "cyan_strict_flags": list(CYAN_STRICT_QA_FLAGS),
        "require_l2_nonland": True,
        "bloom_rescue_require_ci_cyano": bool(BLOOM_RESCUE_REQUIRE_CI_CYANO),
        "bloom_rescue_require_positive_afai": bool(BLOOM_RESCUE_REQUIRE_POSITIVE_AFAI),
        "bloom_rescue_afai_min": float(BLOOM_RESCUE_AFAI_MIN),
        "bloom_rescue_require_nir_over_blue": bool(BLOOM_RESCUE_REQUIRE_NIR_OVER_BLUE),
        "detection_limit": float(CI_DETECTION_LIMIT),
        "bright_screen": bool(DEFAULT_BRIGHT_SCREEN),
        "shore_buffer_m": float(DEFAULT_SHORE_BUFFER_M),
        "processing_strategy": PROCESSING_STRATEGY,
        "window_margin_km": float(L2_WINDOW_MARGIN_KM),
        "max_windows": int(MAX_L2_WINDOWS_PER_SCENE),
        "full_scene_fraction": float(FULL_SCENE_FRACTION_THRESHOLD),
        "min_lake_area_km2": float(min_area_km2),
        "region": str(region).upper(),
        "schema": V264_SCHEMA,
    }
    return hashlib.sha1(json.dumps(payload, sort_keys=True).encode()).hexdigest()[:16]


def _open_compact_writer_v264(path: Path, metadata: dict[str, Any]):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.unlink(missing_ok=True)
    ds = netCDF4.Dataset(tmp, "w", format="NETCDF4")
    ds.createDimension("obs", None)
    for key, value in metadata.items():
        if value is not None:
            try:
                ds.setncattr(key, value)
            except Exception:
                ds.setncattr(key, str(value))
    chunk = (COMPACT_CHUNK_OBS,)
    out = {}
    out["Hylak_id"] = ds.createVariable("Hylak_id", "i8", ("obs",), zlib=True, complevel=COMPACT_DEFLATE, shuffle=True, chunksizes=chunk)
    out["source_row"] = ds.createVariable("source_row", "i4", ("obs",), zlib=True, complevel=COMPACT_DEFLATE, shuffle=True, chunksizes=chunk)
    out["source_col"] = ds.createVariable("source_col", "i4", ("obs",), zlib=True, complevel=COMPACT_DEFLATE, shuffle=True, chunksizes=chunk)
    out["l2_flags"] = ds.createVariable("l2_flags", "u4", ("obs",), zlib=True, complevel=COMPACT_DEFLATE, shuffle=True, chunksizes=chunk)
    mask_names = (
        "valid_water_mask",                  # primary bloom-aware mask
        "cyan_strict_valid_mask",            # strict CyAN-comparison mask
        "bloom_rescue_mask",                 # CLDICE pixels restored by bloom evidence
        "cldice_mask", "hisatzen_mask", "navfail_mask",
        "cloud_excluded_mask",
        "ci_valid_mask", "ci_candidate_mask", "ci_detection_mask", "ndci_valid_mask",
        "mph_valid_mask", "fai_valid_mask",
    )
    for name in mask_names:
        out[name] = ds.createVariable(name, "u1", ("obs",), zlib=True, complevel=COMPACT_DEFLATE, shuffle=True, chunksizes=chunk)
    float_names = ["latitude", "longitude"] + [f"rhos_{w}" for w in GLOBAL_ANALYSIS_RHOS_WAVELENGTHS] + [
        "SS_681", "SS_665", "CI", "CI_cyano", "NDCI", "MPH", "MPH_peak_nm", "FAI"
    ]
    for name in float_names:
        out[name] = ds.createVariable(name, "f4", ("obs",), zlib=True, complevel=COMPACT_DEFLATE, shuffle=True, chunksizes=chunk, fill_value=DERIVED_FILL)
    out["latitude"].setncatts({"standard_name":"latitude", "units":"degrees_north"})
    out["longitude"].setncatts({"standard_name":"longitude", "units":"degrees_east"})
    for w in GLOBAL_ANALYSIS_RHOS_WAVELENGTHS:
        out[f"rhos_{w}"].setncatts({"long_name":f"Rayleigh-corrected OLCI reflectance near {w} nm", "units":"1"})
    out["valid_water_mask"].setncattr("long_name", "primary bloom-aware quantitative valid-water mask")
    out["cyan_strict_valid_mask"].setncattr("long_name", "CyAN-strict comparison mask: non-LAND, non-CLDICE, non-HISATZEN, NAVFAIL safety excluded")
    out["bloom_rescue_mask"].setncattr("long_name", "CLDICE pixel restored by conservative bloom spectral evidence")
    out["MPH"].setncatts({"long_name":"Maximum Peak Height reflectance index", "units":"1", "formula":"max line height at 681/709/754 nm above 665-885 nm baseline"})
    out["MPH_peak_nm"].setncatts({"long_name":"wavelength of MPH maximum", "units":"nm"})
    out["FAI"].setncatts({"long_name":"OLCI-adapted Floating Algae Index (AFAI form)", "units":"1", "formula":"rhos754 - linear baseline(rhos665,rhos865)", "note":"OLCI AFAI form used as one independent bloom-rescue criterion."})
    return ds, out, tmp


def extract_lake_only_netcdf(
    l2_path: Path, scene: dict[str, Any], lakes: gpd.GeoDataFrame, output_path: Path,
    window=None, qa_flags: Iterable[str] = QA_EXCLUDE_FLAGS,
    land_adjacency_pixels: int = LAND_ADJACENCY_PIXELS,
    require_l2_nonland: bool = REQUIRE_L2_NONLAND,
    detection_limit: float = CI_DETECTION_LIMIT, ss681_max: float = 0.0, ss665_min: float = 0.0,
    bright_screen: bool = DEFAULT_BRIGHT_SCREEN, bright_rhos865: float = DEFAULT_BRIGHT_RHOS865,
    rows_per_block: int = ROW_BLOCK_SIZE, config_hash: str | None = None,
    shore_buffer_m: float = DEFAULT_SHORE_BUFFER_M,
) -> tuple[Path | None, dict[str, Any]]:
    """Persist HydroLAKES pixels with dual QA masks and bloom-aware CLDICE recovery."""
    output_path = Path(output_path)
    analysis_lakes = erode_lake_geometries(lakes, float(shore_buffer_m)) if float(shore_buffer_m) > 0 else lakes
    if analysis_lakes is None or analysis_lakes.empty:
        return None, {"nobs":0, "missing_flags":[], "reason":"no lake area after shoreline buffer"}
    with netCDF4.Dataset(l2_path) as src:
        geo = src.groups["geophysical_data"]; nav = src.groups["navigation_data"]
        flags_var = locate_l2_flags(src); flag_map = decode_flag_metadata(flags_var)
        if "LAND" not in flag_map:
            raise RuntimeError("L2 l2_flags lacks LAND")
        rhos_names = {w: nearest_product(geo.variables, "rhos", w, tolerance_nm=30) for w in GLOBAL_ANALYSIS_RHOS_WAVELENGTHS}
        actual_waves = {w: product_wavelength(rhos_names[w]) for w in GLOBAL_ANALYSIS_RHOS_WAVELENGTHS}
        ndci_prefix = "rhos" if NDCI_SOURCE.lower() == "rhos" else "Rrs"
        ndci_names = {665: nearest_product(geo.variables, ndci_prefix, 665), 709: nearest_product(geo.variables, ndci_prefix, 709)}
        shape_2d = nav.variables["latitude"].shape
        metadata = {
            "Conventions":"CF-1.8 ACDD-1.3",
            "title":"Sentinel-3 OLCI derived lake-pixel point collection",
            "schema":V264_SCHEMA,
            "processing_config_hash":str(config_hash or ""),
            "scene_id":str(scene.get("id","")), "scene_name":str(scene.get("name","")),
            "acquisition_start":str(scene.get("start","")), "acquisition_end":str(scene.get("end","")),
            "window_id":window_id(window), "window_bbox":"FULL" if window is None else ",".join(map(str,window)),
            "source_l2":str(Path(l2_path).name), "l2gen_version":command_version(L2GEN_BIN),
            "primary_qa_profile":DEFAULT_MASK_PROFILE,
            "primary_hard_exclude_flags":",".join(BLOOM_AWARE_HARD_FLAGS),
            "cyan_strict_exclude_flags":"LAND,CLDICE,HISATZEN,NAVFAIL_safety",
            "conditional_cldice_bloom_recovery":1,
            "bloom_rescue_definition":(
                "CLDICE AND NOT HISATZEN AND NOT NAVFAIL AND standard CIcyano spectral detection "
                "AND OLCI AFAI>0 AND rhos754>rhos490"
            ),
            "bloom_rescue_method_note":(
                "Conservative research implementation informed by the false-cloud recovery concept "
                "of Wang & Jiang 2025 (DOI 10.1016/j.jag.2025.104408); not an exact reproduction "
                "of NOAA MSL12 nAFAI/reflectance-ratio thresholds."
            ),
            "l2_flag_map_json":json.dumps({k:int(v) for k,v in flag_map.items()},sort_keys=True),
            "require_l2_nonland":1, "shore_buffer_m":float(shore_buffer_m),
            "ci_detection_limit":float(detection_limit), "ss681_max":float(ss681_max), "ss665_min":float(ss665_min),
            "non_lake_storage":"omitted entirely; not dense NaN cells",
            "masked_value_semantics":"_FillValue decodes as NaN",
            "true_color_bands":"rhos665,rhos560,rhos490",
            "additional_indices":"MPH, OLCI-adapted FAI/AFAI",
            "l2gen_nominal_band_mapping":"nominal 885 nm -> L2Gen rhos_884",
        }
        ds, out, tmp = _open_compact_writer_v264(output_path, metadata)
        nobs = 0
        missing_flags = [x for x in set(CYAN_STRICT_QA_FLAGS) if x not in flag_map]
        try:
            for start in tqdm(range(0, shape_2d[0], int(rows_per_block)), desc=f"Lake pixels {window_id(window)}"):
                stop=min(shape_2d[0], start+int(rows_per_block)); rs=slice(start,stop)
                lat=to_float(nav.variables["latitude"][rs,:]); lon=to_float(nav.variables["longitude"][rs,:])
                lake_ids=_membership_block(lat,lon,analysis_lakes); inside=lake_ids>0
                if not inside.any():
                    continue
                flags_ma=np.ma.asarray(flags_var[rs,:]); flags_valid=~np.ma.getmaskarray(flags_ma); flags=flags_ma.filled(0).astype(np.uint32)
                geolocated=np.isfinite(lat)&np.isfinite(lon)&flags_valid
                land=flag_is_set(flags,flag_map,"LAND")
                cldice=flag_is_set(flags,flag_map,"CLDICE")
                hisatzen=flag_is_set(flags,flag_map,"HISATZEN")
                navfail=flag_is_set(flags,flag_map,"NAVFAIL")
                base=inside&geolocated&~land

                arrays={w:to_float(geo.variables[rhos_names[w]][rs,:]) for w in GLOBAL_ANALYSIS_RHOS_WAVELENGTHS}
                family=ci_family(
                    arrays[620],arrays[665],arrays[681],arrays[709],
                    wavelengths=tuple(actual_waves[w] for w in CORE_RHOS_WAVELENGTHS),
                    detection_limit=detection_limit,ss681_max=ss681_max,ss665_min=ss665_min,
                )
                extra=additional_olci_indices(arrays,actual_waves)

                bloom_evidence = family["finite"] & family["ci_detection"]
                bloom_evidence &= extra["FAI_valid"] & np.isfinite(extra["FAI"]) & (extra["FAI"] > float(BLOOM_RESCUE_AFAI_MIN))
                bloom_evidence &= np.isfinite(arrays[490]) & np.isfinite(arrays[754]) & (arrays[754] > arrays[490])

                bloom_rescue = base & cldice & ~hisatzen & ~navfail & bloom_evidence
                cyan_strict_valid = base & ~cldice & ~hisatzen & ~navfail
                valid_water = (base & ~hisatzen & ~navfail & ~cldice) | bloom_rescue
                cloud_excluded = base & cldice & ~bloom_rescue

                if int(land_adjacency_pixels)>0:
                    adjacent=binary_dilation(land,iterations=int(land_adjacency_pixels)) & ~land
                    valid_water &= ~adjacent
                    cyan_strict_valid &= ~adjacent
                if bright_screen:
                    bright=np.isfinite(arrays[865]) & (arrays[865] > float(bright_rhos865))
                    valid_water &= ~bright
                    cyan_strict_valid &= ~bright
                    bloom_rescue &= ~bright

                ci_valid=valid_water & family["finite"]
                candidate=ci_valid & family["ci_candidate"]
                detected=ci_valid & family["ci_detection"]
                red=to_float(geo.variables[ndci_names[665]][rs,:]); edge=to_float(geo.variables[ndci_names[709]][rs,:])
                ndci,ndci_finite=ndci_index(red,edge); ndci_valid=valid_water&ndci_finite
                extra_valid={k: valid_water & extra[f"{k}_valid"] for k in ADDITIONAL_INDEX_NAMES}

                take=inside&geolocated; rr,cc=np.where(take)
                if not rr.size:
                    continue
                count=len(rr); sl=slice(nobs,nobs+count)
                out["Hylak_id"][sl]=lake_ids[rr,cc]
                out["source_row"][sl]=rr.astype(np.int32)+int(start)
                out["source_col"][sl]=cc.astype(np.int32)
                out["l2_flags"][sl]=flags[rr,cc]
                mask_map={
                    "valid_water_mask":valid_water,
                    "cyan_strict_valid_mask":cyan_strict_valid,
                    "bloom_rescue_mask":bloom_rescue,
                    "cldice_mask":cldice,
                    "hisatzen_mask":hisatzen,
                    "navfail_mask":navfail,
                    "cloud_excluded_mask":cloud_excluded,
                    "ci_valid_mask":ci_valid,
                    "ci_candidate_mask":candidate,
                    "ci_detection_mask":detected,
                    "ndci_valid_mask":ndci_valid,
                    "mph_valid_mask":extra_valid["MPH"],
                    "fai_valid_mask":extra_valid["FAI"],
                }
                for name,mask in mask_map.items():
                    out[name][sl]=mask[rr,cc].astype(np.uint8)
                _append_float(out["latitude"],nobs,lat[rr,cc]); _append_float(out["longitude"],nobs,lon[rr,cc])
                for w in GLOBAL_ANALYSIS_RHOS_WAVELENGTHS:
                    _append_float(out[f"rhos_{w}"],nobs,arrays[w][rr,cc])
                _append_float(out["SS_681"],nobs,np.where(ci_valid,family["SS_681"],np.nan)[rr,cc])
                _append_float(out["SS_665"],nobs,np.where(ci_valid,family["SS_665"],np.nan)[rr,cc])
                _append_float(out["CI"],nobs,np.where(candidate,family["CI"],np.nan)[rr,cc])
                _append_float(out["CI_cyano"],nobs,np.where(detected,family["CI_cyano"],np.nan)[rr,cc])
                _append_float(out["NDCI"],nobs,np.where(ndci_valid,ndci,np.nan)[rr,cc])
                _append_float(out["MPH"],nobs,np.where(extra_valid["MPH"],extra["MPH"],np.nan)[rr,cc])
                _append_float(out["MPH_peak_nm"],nobs,np.where(extra_valid["MPH"],extra["MPH_peak_nm"],np.nan)[rr,cc])
                _append_float(out["FAI"],nobs,np.where(extra_valid["FAI"],extra["FAI"],np.nan)[rr,cc])
                nobs += count
        finally:
            ds.close()
        if nobs==0:
            tmp.unlink(missing_ok=True)
            return None,{"nobs":0,"missing_flags":missing_flags}
        if tmp.parent==output_path.parent:
            tmp.replace(output_path)
        else:
            transfer=output_path.with_suffix(output_path.suffix+".copying")
            transfer.unlink(missing_ok=True); shutil.copy2(tmp,transfer); transfer.replace(output_path); tmp.unlink(missing_ok=True)
    return output_path,{"nobs":int(nobs),"size_bytes":int(output_path.stat().st_size),"size_human":human_size(output_path.stat().st_size),"missing_flags":missing_flags,"window_id":window_id(window)}


def _mask_values_from_ds(ds, name, n):
    return np.asarray(ds.variables[name][:])==1 if name in ds.variables else np.zeros(n,bool)


def compact_stats(nc_path: Path, lake_metadata: gpd.GeoDataFrame | None = None) -> pd.DataFrame:
    """Primary bloom-aware lake stats plus explicit CyAN-strict/rescue QA counts."""
    with netCDF4.Dataset(nc_path) as ds:
        ids=np.asarray(ds.variables["Hylak_id"][:],dtype=np.int64); n=len(ids)
        data={
            "Hylak_id":ids,
            "valid":_mask_values_from_ds(ds,"valid_water_mask",n),
            "strict":_mask_values_from_ds(ds,"cyan_strict_valid_mask",n),
            "rescue":_mask_values_from_ds(ds,"bloom_rescue_mask",n),
            "cldice":_mask_values_from_ds(ds,"cldice_mask",n),
            "ci_valid":_mask_values_from_ds(ds,"ci_valid_mask",n),
            "candidate":_mask_values_from_ds(ds,"ci_candidate_mask",n),
            "detected":_mask_values_from_ds(ds,"ci_detection_mask",n),
        }
        for name in ("CI","CI_cyano","NDCI","MPH","FAI"):
            data[name]=to_float(ds.variables[name][:]) if name in ds.variables else np.full(n,np.nan)
        frame=pd.DataFrame(data)
        attrs={k:getattr(ds,k,None) for k in ("scene_id","scene_name","acquisition_start","window_id")}
    meta={}
    if lake_metadata is not None and not lake_metadata.empty:
        for _,row in lake_metadata.iterrows():
            meta[int(row["Hylak_id"])]={"Lake_name":row.get("Lake_name","") or "","Lake_area_km2":float(row.get("Lake_area",np.nan)),"Country":row.get("Country","") if "Country" in row else "","Continent":row.get("Continent","") if "Continent" in row else ""}
    def vals(series):
        a=pd.to_numeric(series,errors="coerce").to_numpy(float); return a[np.isfinite(a)]
    def summary(prefix,a):
        if not len(a): return {f"{prefix}_mean_retained":np.nan,f"{prefix}_median_retained":np.nan,f"{prefix}_p10_retained":np.nan,f"{prefix}_p90_retained":np.nan,f"{prefix}_max_retained":np.nan}
        return {f"{prefix}_mean_retained":float(np.nanmean(a)),f"{prefix}_median_retained":float(np.nanmedian(a)),f"{prefix}_p10_retained":float(np.nanpercentile(a,10)),f"{prefix}_p90_retained":float(np.nanpercentile(a,90)),f"{prefix}_max_retained":float(np.nanmax(a))}
    records=[]
    for lake_id,g in frame.groupby("Hylak_id",sort=False):
        ci=vals(g.loc[g["candidate"],"CI"]); cy=vals(g.loc[g["detected"],"CI_cyano"]); nd=vals(g.loc[g["valid"],"NDCI"])
        valid_n=int(g["valid"].sum()); strict_n=int(g["strict"].sum()); ci_valid_n=int(g["ci_valid"].sum()); detected_n=int(g["detected"].sum())
        rec={**attrs,"Hylak_id":int(lake_id),**meta.get(int(lake_id),{}),"native_polygon_pixels":int(len(g)),"retained_pixels":valid_n,"retained_percent":100.0*valid_n/len(g) if len(g) else np.nan,"cyan_strict_pixels":strict_n,"bloom_rescued_pixels":int(g["rescue"].sum()),"CLDICE_pixels":int(g["cldice"].sum()),"ci_input_valid_pixels":ci_valid_n,"ci_candidate_pixels":int(g["candidate"].sum()),"ci_cyano_detection_pixels":detected_n,"ci_cyano_detection_fraction_of_ci_valid":detected_n/ci_valid_n if ci_valid_n else np.nan,"ci_cyano_detection_fraction_of_retained":detected_n/valid_n if valid_n else np.nan,"CI_mean_candidates":float(np.nanmean(ci)) if len(ci) else np.nan,"CI_median_candidates":float(np.nanmedian(ci)) if len(ci) else np.nan,"CI_p10_candidates":float(np.nanpercentile(ci,10)) if len(ci) else np.nan,"CI_p90_candidates":float(np.nanpercentile(ci,90)) if len(ci) else np.nan,"CI_max_candidates":float(np.nanmax(ci)) if len(ci) else np.nan,"CIcyano_mean_detections":float(np.nanmean(cy)) if len(cy) else np.nan,"CIcyano_median_detections":float(np.nanmedian(cy)) if len(cy) else np.nan,"CIcyano_p90_detections":float(np.nanpercentile(cy,90)) if len(cy) else np.nan,"NDCI_mean_retained":float(np.nanmean(nd)) if len(nd) else np.nan,"NDCI_median_retained":float(np.nanmedian(nd)) if len(nd) else np.nan,"NDCI_p90_retained":float(np.nanpercentile(nd,90)) if len(nd) else np.nan,"compact_netcdf":str(nc_path)}
        for k in ADDITIONAL_INDEX_NAMES:
            rec.update(summary(k,vals(g.loc[g["valid"],k])))
        records.append(rec)
    return pd.DataFrame(records)


def _load_scene_pixels(scene_id,scope='scene',lake_id=None):
    frames=[]; wanted=None if scope=='scene' else int(lake_id)
    names=[
        'latitude','longitude','Hylak_id','l2_flags',
        'valid_water_mask','cyan_strict_valid_mask','bloom_rescue_mask','cldice_mask','hisatzen_mask','navfail_mask','cloud_excluded_mask',
        'ci_valid_mask','ci_candidate_mask','ci_detection_mask','mph_valid_mask','fai_valid_mask','MPH_peak_nm',
        *SCALAR_METRICS,
    ]
    for p in _scene_compact_files(scene_id):
        with netCDF4.Dataset(p) as ds:
            ids=np.asarray(ds.variables['Hylak_id'][:],dtype=np.int64); sel=np.ones(len(ids),bool) if wanted is None else ids==wanted
            if not sel.any(): continue
            data={'Hylak_id':ids[sel]}
            for n in names:
                if n=='Hylak_id': continue
                if n=='l2_flags' and n in ds.variables:
                    data[n]=np.asarray(ds.variables[n][:],dtype=np.uint32)[sel]
                elif n in ds.variables:
                    data[n]=to_float(ds.variables[n][:])[sel]
                else:
                    data[n]=np.full(sel.sum(),np.nan)
            frames.append(pd.DataFrame(data))
    if not frames: raise ValueError('No compact lake pixels found for selected scope')
    f=pd.concat(frames,ignore_index=True)
    f['_latr']=f.latitude.round(6); f['_lonr']=f.longitude.round(6)
    f=f.drop_duplicates(['Hylak_id','_latr','_lonr']).drop(columns=['_latr','_lonr'])
    return f


def _primary_selector(frame):
    if 'valid_water_mask' not in frame.columns:
        return np.ones(len(frame),dtype=bool)
    return pd.to_numeric(frame['valid_water_mask'],errors='coerce').fillna(0).to_numpy(float)>0


def _cyan_comparison_selector(frame):
    if 'cyan_strict_valid_mask' in frame.columns and pd.to_numeric(frame['cyan_strict_valid_mask'],errors='coerce').notna().any():
        return pd.to_numeric(frame['cyan_strict_valid_mask'],errors='coerce').fillna(0).to_numpy(float)>0
    return _primary_selector(frame)


def _masked_display_frame(frame, columns):
    out=frame.copy(); keep=_primary_selector(out)
    for c in columns:
        if c in out.columns:
            a=pd.to_numeric(out[c],errors='coerce').to_numpy(float)
            a[~keep]=np.nan; out[c]=a
    return out


def _build_map(scene_id,scope,lake_id,metric,resolution_m,basemap,labels,cmap,reverse,vmin_text,vmax_text,show_lakes=True):
    if not scene_id: raise ValueError('Choose a completed scene')
    scope='scene' if str(scope)=='scene' else 'lake'; raw=_load_scene_pixels(scene_id,scope,lake_id)
    cols=['rhos_665','rhos_560','rhos_490'] if metric=='true_color' else [metric]
    frame=_masked_display_frame(raw,cols)
    grids,g=_regularize_dataframe_mercator(frame,cols,float(resolution_m))
    bounds=[[g['south'],g['west']],[g['north'],g['east']]]
    m=folium.Map(tiles=None,control_scale=True,prefer_canvas=True); _add_basemap(m,basemap,bool(labels))
    if metric=='true_color':
        img=_truecolor_data_url(grids); folium.raster_layers.ImageOverlay(img,bounds=bounds,opacity=.95,name='True color · bloom-aware QA').add_to(m)
    else:
        img,vmin,vmax,cm=_rgba_data_url(grids[metric],metric,cmap,bool(reverse),_optional_float(vmin_text),_optional_float(vmax_text)); folium.raster_layers.ImageOverlay(img,bounds=bounds,opacity=.88,name=f'{metric} · bloom-aware QA').add_to(m); _add_scalar_legend(m,metric,cmap,bool(reverse),vmin,vmax)
    if show_lakes:
        b=_lake_boundaries_for_ids(raw.Hylak_id.unique())
        if not b.empty: folium.GeoJson(json.loads(b.to_crs(4326).to_json()),name='HydroLAKES boundaries',style_function=lambda f:{'color':'#00e5ff','weight':1.2,'fillOpacity':0}).add_to(m)
    folium_plugins.Fullscreen(position='topleft').add_to(m); folium_plugins.MousePosition(position='bottomright',separator=' | ',prefix='Lat / Lon',num_digits=5).add_to(m); folium_plugins.MeasureControl(position='topleft',primary_length_unit='kilometers').add_to(m); folium.LayerControl(collapsed=False).add_to(m); m.fit_bounds(bounds)
    hover_frame=frame.loc[_primary_selector(frame)].copy(); _inject_hover_js(m,hover_frame,metric,g['resolution_m'])
    out=HTML_DIR/f"scene_{scene_id}_{scope}_{lake_id or 'ALL'}_{metric}_{int(g['resolution_m'])}m.html"; m.save(out)
    map_src=f"/gradio_api/file={quote(str(out), safe='/')}?v={out.stat().st_mtime_ns}"
    iframe=("<iframe style='width:100%;height:740px;border:1px solid #ccd5df;border-radius:8px;background:white' loading='eager' src='"+map_src+"'></iframe>")
    return iframe,str(out),frame,grids,g


def export_snap_gridded_netcdf(scene_id, scope, lake_id, metric, frame, resolution_m):
    """Quick per-layer EPSG:4326 export using the same bloom-aware primary QA as the map."""
    SNAP_DIR=EXPORT_DIR/'snap_gridded'; SNAP_DIR.mkdir(parents=True,exist_ok=True)
    names=['rhos_665','rhos_560','rhos_490'] if metric=='true_color' else [metric]
    frame=_masked_display_frame(frame,names)
    grids,g=_regularize_dataframe_geographic(frame,names,float(resolution_m))
    out=SNAP_DIR/f"S3_OLCI_{scene_id}_{scope}_{lake_id or 'ALL'}_{metric}_{int(g['resolution_m'])}m_CF.nc"
    lat_desc=g['north']-(np.arange(g['ny'])+.5)*g['dlat']; lon=g['west']+(np.arange(g['nx'])+.5)*g['dlon']; lat=lat_desc[::-1].astype(np.float64); lon=lon.astype(np.float64)
    from pyproj import CRS; wkt=CRS.from_epsg(4326).to_wkt()
    with netCDF4.Dataset(out,'w',format='NETCDF4_CLASSIC') as ds:
        ds.createDimension('latitude',len(lat)); ds.createDimension('longitude',len(lon)); ds.setncatts({'Conventions':'CF-1.8','title':'Sentinel-3 OLCI bloom-aware HydroLAKES gridded export','scene_id':str(scene_id),'scope':scope,'qa_basis':'primary bloom-aware valid_water_mask','geospatial_lat_min':float(lat.min()),'geospatial_lat_max':float(lat.max()),'geospatial_lon_min':float(lon.min()),'geospatial_lon_max':float(lon.max()),'geospatial_lat_units':'degrees_north','geospatial_lon_units':'degrees_east'})
        y=ds.createVariable('latitude','f8',('latitude',)); y[:]=lat; y.setncatts({'standard_name':'latitude','units':'degrees_north','axis':'Y'}); x=ds.createVariable('longitude','f8',('longitude',)); x[:]=lon; x.setncatts({'standard_name':'longitude','units':'degrees_east','axis':'X'}); crs=ds.createVariable('crs','i4'); crs.assignValue(0); crs.setncatts({'grid_mapping_name':'latitude_longitude','epsg_code':'EPSG:4326','spatial_ref':wkt,'crs_wkt':wkt,'semi_major_axis':6378137.0,'inverse_flattening':298.257223563})
        for name in names:
            v=ds.createVariable(name,'f4',('latitude','longitude'),zlib=True,complevel=6,shuffle=True,fill_value=np.float32(-9999.0)); v.setncatts({'coordinates':'latitude longitude','grid_mapping':'crs','units':'1','long_name':name}); v[:]=np.ma.masked_invalid(np.asarray(grids[name],np.float32)[::-1,:])
        if metric=='true_color': ds.setncattr('true_color_mapping','red=rhos_665; green=rhos_560; blue=rhos_490')
    return str(out)


def _compact_qa_metadata(scene_id):
    files=_scene_compact_files(scene_id)
    meta={"primary_qa_profile":"unknown","primary_hard_exclude_flags":[],"cyan_strict_exclude_flags":[],"conditional_cldice_bloom_recovery":False,"require_l2_nonland":False}
    if not files: return meta
    with netCDF4.Dataset(files[0]) as ds:
        meta.update({
            "primary_qa_profile":str(getattr(ds,"primary_qa_profile",getattr(ds,"schema","unknown"))),
            "primary_hard_exclude_flags":[x for x in str(getattr(ds,"primary_hard_exclude_flags","")).split(',') if x],
            "cyan_strict_exclude_flags":[x for x in str(getattr(ds,"cyan_strict_exclude_flags","")).split(',') if x],
            "conditional_cldice_bloom_recovery":bool(int(getattr(ds,"conditional_cldice_bloom_recovery",0))),
            "bloom_rescue_definition":str(getattr(ds,"bloom_rescue_definition","")),
            "bloom_rescue_method_note":str(getattr(ds,"bloom_rescue_method_note","")),
            "require_l2_nonland":bool(int(getattr(ds,"require_l2_nonland",1))),
            "shore_buffer_m":float(getattr(ds,"shore_buffer_m",0.0)),
            "ci_detection_limit":float(getattr(ds,"ci_detection_limit",CYAN_CI_DETECTION_LIMIT)),
        })
    return meta


def _science_sinusoidal_setup(frame, requested_resolution_m=SCIENCE_EXPORT_DEFAULT_RESOLUTION_M):
    """Exact-resolution globally aligned spherical-sinusoidal grid; never auto-coarsens."""
    from pyproj import CRS, Transformer
    lat=pd.to_numeric(frame['latitude'],errors='coerce').to_numpy(float); lon=pd.to_numeric(frame['longitude'],errors='coerce').to_numpy(float)
    finite=np.isfinite(lat)&np.isfinite(lon)&(np.abs(lat)<=90)&(np.abs(lon)<=180)
    if not finite.any(): raise ValueError('Selected scene contains no finite stored geolocated lake pixels')
    radius=float(CYAN_STYLE_SPHERE_RADIUS_M); crs=CRS.from_proj4(f'+proj=sinu +lon_0=0 +x_0=0 +y_0=0 +R={radius:.3f} +units=m +no_defs'); transformer=Transformer.from_crs(4326,crs,always_xy=True)
    x_all=np.full(lon.shape,np.nan,np.float64); y_all=np.full(lat.shape,np.nan,np.float64); xx,yy=transformer.transform(lon[finite],lat[finite]); x_all[finite]=xx; y_all[finite]=yy
    global_x_min=-math.pi*radius; global_y_max=.5*math.pi*radius; res=max(1.0,float(requested_resolution_m))
    cg=np.floor((x_all[finite]-global_x_min)/res).astype(np.int64); rg=np.floor((global_y_max-y_all[finite])/res).astype(np.int64); c0,c1=int(cg.min()),int(cg.max()); r0,r1=int(rg.min()),int(rg.max()); nx,ny=c1-c0+1,r1-r0+1
    finite_idx=np.flatnonzero(finite); inside=np.zeros(len(frame),bool); inside[finite_idx]=True
    cc=cg-c0; rr=rg-r0; flat=rr*nx+cc
    x_left=global_x_min+c0*res; y_top=global_y_max-r0*res; x=x_left+(np.arange(nx,dtype=np.float64)+.5)*res; y=y_top-(np.arange(ny,dtype=np.float64)+.5)*res
    g={'crs':crs,'resolution_m':float(res),'requested_resolution_m':float(requested_resolution_m),'nx':int(nx),'ny':int(ny),'ncell':int(nx)*int(ny),'x':x,'y':y,'x_left':float(x_left),'y_top':float(y_top),'global_col0':int(c0),'global_row0':int(r0),'radius_m':radius}
    return g,inside,flat


def _unique_bin_index(flat, nx):
    unique, inv=np.unique(np.asarray(flat,dtype=np.int64),return_inverse=True); rows=(unique//int(nx)).astype(np.int64); cols=(unique%int(nx)).astype(np.int64); return unique,inv,rows,cols


def _aggregate_mean_to_unique(values, inside, inv, n_unique, obs_valid=None):
    vals=np.asarray(values,float)[inside]; good=np.isfinite(vals)
    if obs_valid is not None: good &= np.asarray(obs_valid,bool)[inside]
    out=np.full(int(n_unique),np.nan,np.float32)
    if good.any():
        sums=np.bincount(inv[good],weights=vals[good],minlength=int(n_unique)); counts=np.bincount(inv[good],minlength=int(n_unique)); nz=counts>0; out[nz]=(sums[nz]/counts[nz]).astype(np.float32)
    return out


def _aggregate_mask_to_unique(values, inside, inv, n_unique):
    vals=np.asarray(values,float)[inside]; out=np.zeros(int(n_unique),np.int8); pos=np.isfinite(vals)&(vals>0)
    if pos.any(): np.maximum.at(out,inv[pos],np.int8(1))
    return out


def _chunk_groups(rows, cols, ny, nx, chunk=SCIENCE_EXPORT_CHUNK):
    cy=max(1,min(int(chunk),int(ny))); cx=max(1,min(int(chunk),int(nx))); nxc=int(math.ceil(nx/cx)); cid=(rows//cy)*nxc+(cols//cx); order=np.argsort(cid,kind='mergesort'); cid_sorted=cid[order]; cuts=np.flatnonzero(np.r_[True,cid_sorted[1:]!=cid_sorted[:-1],True]); groups=[]
    for a,b in zip(cuts[:-1],cuts[1:]): groups.append(order[a:b])
    return groups,cy,cx


def _write_sparse_float(var, values, rows, cols, groups, cy, cx, ny, nx, fill=-9999.0):
    vals=np.asarray(values,np.float32)
    for pos in groups:
        finite=np.isfinite(vals[pos])
        if not finite.any(): continue
        p=pos[finite]; r0=int((rows[p[0]]//cy)*cy); c0=int((cols[p[0]]//cx)*cx); r1=min(r0+cy,int(ny)); c1=min(c0+cx,int(nx)); block=np.full((r1-r0,c1-c0),np.float32(fill),np.float32); block[rows[p]-r0,cols[p]-c0]=vals[p]; var[r0:r1,c0:c1]=np.ma.masked_equal(block,np.float32(fill))


def _write_sparse_mask(var, values, rows, cols, groups, cy, cx, ny, nx, fill=-1):
    vals=np.asarray(values,np.int8)
    for pos in groups:
        r0=int((rows[pos[0]]//cy)*cy); c0=int((cols[pos[0]]//cx)*cx); r1=min(r0+cy,int(ny)); c1=min(c0+cx,int(nx)); block=np.full((r1-r0,c1-c0),np.int8(fill),np.int8); block[rows[pos]-r0,cols[pos]-c0]=vals[pos]; var[r0:r1,c0:c1]=np.ma.masked_equal(block,np.int8(fill))



def _write_sparse_int(var, values, rows, cols, groups, cy, cx, ny, nx, fill=-1):
    vals=np.asarray(values,np.int32)
    for pos in groups:
        r0=int((rows[pos[0]]//cy)*cy); c0=int((cols[pos[0]]//cx)*cx); r1=min(r0+cy,int(ny)); c1=min(c0+cx,int(nx))
        block=np.full((r1-r0,c1-c0),np.int32(fill),np.int32)
        block[rows[pos]-r0,cols[pos]-c0]=vals[pos]
        var[r0:r1,c0:c1]=np.ma.masked_equal(block,np.int32(fill))


def export_complete_scene_science_netcdf(scene_id,requested_resolution_m=SCIENCE_EXPORT_DEFAULT_RESOLUTION_M,force=False):
    """Fixed-resolution sparse sinusoidal share raster using the primary bloom-aware QA."""
    if not scene_id: raise ValueError('Choose a completed scene')
    frame=_load_scene_pixels(scene_id,'scene',None); g,inside,flat=_science_sinusoidal_setup(frame,float(requested_resolution_m)); unique,inv,rows,cols=_unique_bin_index(flat,g['nx']); n_unique=len(unique); groups,cy,cx=_chunk_groups(rows,cols,g['ny'],g['nx'])
    export_dir=EXPORT_DIR/'snap_complete'; export_dir.mkdir(parents=True,exist_ok=True); out=export_dir/f"S3_OLCI_{scene_id}_ALL_SCIENCE_{int(round(g['resolution_m']))}m_CF.nc"; source_files=_scene_compact_files(scene_id); newest=max((p.stat().st_mtime_ns for p in source_files),default=0)
    if out.exists() and not force and out.stat().st_mtime_ns>=newest:
        with contextlib.suppress(Exception):
            with netCDF4.Dataset(out) as old:
                if getattr(old,'science_grid_method','')==CYAN_STYLE_GRID_METHOD and float(getattr(old,'actual_grid_resolution_m',-1))==float(g['resolution_m']): return str(out),g
    qa=_compact_qa_metadata(scene_id); crs=g['crs']; wkt=crs.to_wkt(); lat_native=pd.to_numeric(frame.latitude,errors='coerce').to_numpy(float); lon_native=pd.to_numeric(frame.longitude,errors='coerce').to_numpy(float); geo=np.isfinite(lat_native)&np.isfinite(lon_native); primary=_primary_selector(frame)
    tmp=out.with_suffix('.nc.tmp'); tmp.unlink(missing_ok=True)
    with netCDF4.Dataset(tmp,'w',format='NETCDF4_CLASSIC') as ds:
        ds.createDimension('y',g['ny']); ds.createDimension('x',g['nx']); ds.setncatts({'Conventions':'CF-1.8','title':'Sentinel-3 OLCI HydroLAKES bloom-aware all-science product','scene_id':str(scene_id),'source_representation':'native OLCI lake observations binned to exact metric spherical-sinusoidal cells; sparse chunk writes; no interpolation/gap filling','science_grid_method':CYAN_STYLE_GRID_METHOD,'requested_grid_resolution_m':float(requested_resolution_m),'actual_grid_resolution_m':float(g['resolution_m']),'aggregation':'arithmetic mean of primary-valid native observations falling in each bin','primary_qa_mask':'valid_water_mask (bloom-aware cloud mask)','interpolation':'none','empty_bin_policy':'NetCDF fill/NaN; never gap-filled','logical_grid_cells':int(g['ncell']),'occupied_native_observation_bins':int(n_unique),'geospatial_lat_min':float(np.nanmin(lat_native[geo])),'geospatial_lat_max':float(np.nanmax(lat_native[geo])),'geospatial_lon_min':float(np.nanmin(lon_native[geo])),'geospatial_lon_max':float(np.nanmax(lon_native[geo])),'true_color_mapping':'red=rhos_665; green=rhos_560; blue=rhos_490','CI_cyano_detection_limit':float(CYAN_CI_DETECTION_LIMIT),'qa_policy_json':json.dumps(qa,default=str),'note':'Science bands and rhos are masked consistently with the primary bloom-aware valid_water_mask. CyAN-strict and bloom-rescue masks are included separately for auditing.'})
        yv=ds.createVariable('y','f8',('y',)); yv[:]=g['y']; yv.setncatts({'standard_name':'projection_y_coordinate','units':'m','axis':'Y'}); xv=ds.createVariable('x','f8',('x',)); xv[:]=g['x']; xv.setncatts({'standard_name':'projection_x_coordinate','units':'m','axis':'X'}); crsv=ds.createVariable('crs','i4'); crsv.assignValue(0); crsv.setncatts({'grid_mapping_name':'sinusoidal','longitude_of_central_meridian':0.0,'false_easting':0.0,'false_northing':0.0,'earth_radius':float(CYAN_STYLE_SPHERE_RADIUS_M),'spatial_ref':wkt,'crs_wkt':wkt,'GeoTransform':f"{g['x_left']} {g['resolution_m']} 0 {g['y_top']} 0 {-g['resolution_m']}"})
        chunks=(min(cy,g['ny']),min(cx,g['nx']))
        for source_name,out_name in [(n,'rhos_884' if n=='rhos_885' else n) for n in SCIENCE_EXPORT_VARIABLES]:
            if source_name not in frame.columns: continue
            agg=_aggregate_mean_to_unique(frame[source_name].to_numpy(float),inside,inv,n_unique,primary); var=ds.createVariable(out_name,'f4',('y','x'),zlib=True,complevel=6,shuffle=True,chunksizes=chunks,fill_value=np.float32(-9999.0)); long_names={'CI':'Cyanobacteria Index candidate magnitude','CI_cyano':'CyAN-compatible cyanobacteria index detections','NDCI':'Normalized Difference Chlorophyll Index','MPH':'Maximum Peak Height','FAI':'OLCI-adapted Floating Algae Index (AFAI form)'}; var.setncatts({'grid_mapping':'crs','units':'1','long_name':long_names.get(out_name,f'Rayleigh-corrected OLCI reflectance {out_name}'),'qa_basis':'primary bloom-aware valid_water_mask'}); _write_sparse_float(var,agg,rows,cols,groups,cy,cx,g['ny'],g['nx'])
        mask_names=['valid_water_mask','cyan_strict_valid_mask','bloom_rescue_mask','cldice_mask','hisatzen_mask','navfail_mask','cloud_excluded_mask','ci_valid_mask','ci_candidate_mask','ci_detection_mask','mph_valid_mask','fai_valid_mask']
        for name in mask_names:
            if name not in frame.columns: continue
            agg=_aggregate_mask_to_unique(frame[name].to_numpy(float),inside,inv,n_unique); var=ds.createVariable(name,'i1',('y','x'),zlib=True,complevel=6,shuffle=True,chunksizes=chunks,fill_value=np.int8(-1)); var.setncatts({'grid_mapping':'crs','long_name':name.replace('_',' '),'flag_values':np.array([0,1],dtype=np.int8),'flag_meanings':'false true','valid_range':np.array([0,1],dtype=np.int8)}); _write_sparse_mask(var,agg,rows,cols,groups,cy,cx,g['ny'],g['nx'])
        counts=np.bincount(inv,minlength=n_unique).astype(np.int32); v=ds.createVariable('native_observation_count','i4',('y','x'),zlib=True,complevel=6,shuffle=True,chunksizes=chunks,fill_value=np.int32(-1)); v.setncatts({'grid_mapping':'crs','long_name':'number of stored native OLCI lake observations assigned to bin'}); _write_sparse_int(v,counts,rows,cols,groups,cy,cx,g['ny'],g['nx'],fill=-1)
    tmp.replace(out); return str(out),g


def _share_all_index_stats(scene_id,cfg,row):
    frame=_load_scene_pixels(scene_id,'scene',None).copy()
    if frame.empty: raise ValueError('No compact lake pixels found for selected scene')
    for c in ('latitude','longitude','Hylak_id'): frame[c]=pd.to_numeric(frame[c],errors='coerce')
    frame=frame.dropna(subset=['latitude','longitude','Hylak_id']).copy(); frame['_lat6']=frame.latitude.round(6); frame['_lon6']=frame.longitude.round(6); frame=frame.drop_duplicates(['Hylak_id','_lat6','_lon6']).drop(columns=['_lat6','_lon6'])
    lakes=GLOBAL_STATE.get('lakes'); lake_meta={}
    if lakes is not None and not lakes.empty:
        for _,lr in lakes[lakes.Hylak_id.astype(int).isin(frame.Hylak_id.astype(int).unique())].iterrows(): lake_meta[int(lr.Hylak_id)]={'Lake_name':lr.get('Lake_name','') or '','Lake_area_km2':float(lr.get('Lake_area',np.nan))}
    def m(g,n): return pd.to_numeric(g.get(n,pd.Series(np.zeros(len(g)),index=g.index)),errors='coerce').fillna(0).to_numpy(float)>0
    def stat(s):
        a=pd.to_numeric(s,errors='coerce').to_numpy(float); a=a[np.isfinite(a)]
        if not len(a): return {'valid_pixels':0,'mean':np.nan,'median':np.nan,'p10':np.nan,'p90':np.nan,'max':np.nan}
        return {'valid_pixels':int(len(a)),'mean':float(np.mean(a)),'median':float(np.median(a)),'p10':float(np.percentile(a,10)),'p90':float(np.percentile(a,90)),'max':float(np.max(a))}
    recs=[]
    for lid,gdf in frame.groupby(frame.Hylak_id.astype(int),sort=True):
        primary=m(gdf,'valid_water_mask'); strict=m(gdf,'cyan_strict_valid_mask'); rescue=m(gdf,'bloom_rescue_mask'); cld=m(gdf,'cldice_mask'); cand=m(gdf,'ci_candidate_mask'); det=m(gdf,'ci_detection_mask'); ci_valid=m(gdf,'ci_valid_mask')
        rec={'scene_id':str(scene_id),'scene_name':str(row.scene_name),'acquisition_start':str(row.acquisition_start),'Hylak_id':int(lid),**lake_meta.get(int(lid),{}),'native_lake_pixels':int(len(gdf)),'primary_retained_pixels':int(primary.sum()),'cyan_strict_retained_pixels':int(strict.sum()),'bloom_rescued_pixels':int(rescue.sum()),'CLDICE_pixels':int(cld.sum()),'bloom_rescue_fraction_of_CLDICE':float(rescue.sum()/cld.sum()) if cld.sum() else np.nan,'ci_input_valid_pixels':int(ci_valid.sum()),'ci_candidate_pixels':int(cand.sum()),'ci_cyano_detection_pixels':int(det.sum())}
        primary_bases={'CI':cand,'CI_cyano':det,'NDCI':primary,'MPH':primary,'FAI':primary}; strict_bases={'CI':cand&strict,'CI_cyano':det&strict,'NDCI':strict,'MPH':strict,'FAI':strict}
        for metric,basis in primary_bases.items():
            s=stat(gdf.loc[basis,metric] if metric in gdf else pd.Series(dtype=float))
            for k,v in s.items(): rec[f'{metric}_{k}']=v
        for metric,basis in strict_bases.items():
            s=stat(gdf.loc[basis,metric] if metric in gdf else pd.Series(dtype=float))
            for k,v in s.items(): rec[f'{metric}_cyan_strict_{k}']=v
        recs.append(rec)
    table=pd.DataFrame(recs).sort_values('Hylak_id').reset_index(drop=True); d=EXPORT_DIR/'scene_bundles'; d.mkdir(parents=True,exist_ok=True); p=d/f'S3_OLCI_{scene_id}_ALL_INDEX_STATS.csv'; table.to_csv(p,index=False); return p,table


def create_scene_share_bundle(scene_id,science_resolution_m=SCIENCE_EXPORT_DEFAULT_RESOLUTION_M):
    """Exactly 3 files: fixed-resolution all-science NetCDF, stats CSV, metadata JSON."""
    if not scene_id: raise ValueError('Choose a completed scene')
    t=registry_table(GLOBAL_STATE.get('config_hash')); r=t[t.scene_id.astype(str)==str(scene_id)]
    if r.empty: raise KeyError(scene_id)
    row=r.iloc[0]; cfg=str(row.config_hash); d=EXPORT_DIR/'scene_bundles'; d.mkdir(parents=True,exist_ok=True)
    complete_nc,g=export_complete_scene_science_netcdf(scene_id,float(science_resolution_m),force=False); stats_csv,stats_table=_share_all_index_stats(scene_id,cfg,row); qa=_compact_qa_metadata(scene_id); frame_meta=parse_s3_frame_metadata(row.scene_name)
    meta={'schema':'global-efficient-share-v2.6.4','created_utc':utc_now(),'scene_id':str(scene_id),'scene_name':str(row.scene_name),'acquisition_start':str(row.acquisition_start),'config_hash':cfg,**frame_meta,'files':{'snap_all_science_netcdf':Path(complete_nc).name,'all_index_stats_csv':Path(stats_csv).name},'snap_variables':['rhos_490','rhos_560','rhos_620','rhos_665','rhos_681','rhos_709','rhos_754','rhos_865','rhos_884','CI','CI_cyano','NDCI','MPH','FAI','valid_water_mask','cyan_strict_valid_mask','bloom_rescue_mask','cldice_mask','hisatzen_mask'],'snap_grid':{'projection':'spherical sinusoidal','grid_method':CYAN_STYLE_GRID_METHOD,'requested_resolution_m':float(science_resolution_m),'actual_resolution_m':float(g['resolution_m']),'resolution_policy':'exact; never silently coarsened','aggregation':'mean of primary-valid native observations in each occupied bin','interpolation':'none','gap_fill':'none','empty_bins':'fill/NaN','sparse_storage':'only chunks touched by native lake observations are written'},'masking':qa,'mask_semantics':{'valid_water_mask':'PRIMARY bloom-aware quantitative mask used by Gradio, SNAP science bands, and unsuffixed CSV statistics','cyan_strict_valid_mask':'strict comparison mask for NASA CyAN comparison','bloom_rescue_mask':'CLDICE pixels restored only with conservative independent bloom spectral evidence'},'statistics':{'unsuffixed_metrics':'primary bloom-aware mask','*_cyan_strict_*':'CyAN-strict subset'},'important':'All mapped science bands/rhos and Gradio map now use the same primary bloom-aware QA. The compact runtime archive retains raw rhos and l2_flags for auditability.'}
    mp=d/f'S3_OLCI_{scene_id}_METADATA.json'; mp.write_text(json.dumps(meta,indent=2,default=str)); z=d/f'{str(row.scene_name)}_{cfg}_SHARE_MINIMAL.zip'
    with zipfile.ZipFile(z,'w',zipfile.ZIP_DEFLATED,allowZip64=True) as arc:
        for p in (Path(complete_nc),Path(stats_csv),mp): arc.write(p,arcname=p.name)
    return str(z),_app_status('Minimal share bundle created',f"3 files · exact {g['resolution_m']:.0f} m sinusoidal grid · primary bloom-aware QA · strict CyAN mask and bloom-rescue mask included")


print('v2.6.4 active:')
print('  primary quantitative QA : bloom-aware CLDICE recovery')
print('  CyAN comparison QA      : LAND + CLDICE + HISATZEN (+ NAVFAIL safety)')
print('  rescue rule             : CLDICE + CIcyano gates + AFAI>0 + rhos754>rhos490')
print('  Gradio/SNAP QA          : SAME primary valid_water_mask')
print('  share grid resolution   : exact requested value; no auto-coarsening')
print('  interpolation/gap fill  : NONE')


v2.6.4 active:
  primary quantitative QA : bloom-aware CLDICE recovery
  CyAN comparison QA      : LAND + CLDICE + HISATZEN (+ NAVFAIL safety)
  rescue rule             : CLDICE + CIcyano gates + AFAI>0 + rhos754>rhos490
  Gradio/SNAP QA          : SAME primary valid_water_mask
  share grid resolution   : exact requested value; no auto-coarsening
  interpolation/gap fill  : NONE


In [33]:
# =============================================================================
# v2.6.5 CORRECTION PATCH
# Exact-resolution map previews, planning-region display, SNAP-friendly stripeless
# native-nearest share export, scene-only lake dropdowns, clickable spectra map,
# and expanded NASA CyAN bias diagnostics.
# =============================================================================

V265_SCHEMA = "global-efficient-v2.6.5"
V265_DISPLAY_MAX_CELLS = 30_000_000
V265_EXPORT_CHUNK = 128
V265_EXPORT_NEAREST_RADIUS_M = 600.0

from PIL import ImageDraw, ImageFont


def _scene_lake_update(scene_id):
    """Dropdown derived from the actual compact scene pixels, never the global HydroLAKES table."""
    if not scene_id:
        return gr.Dropdown(choices=[], value=None)
    try:
        f = _load_scene_pixels(scene_id, "scene", None)
    except Exception:
        return gr.Dropdown(choices=[], value=None)
    ids = np.sort(pd.to_numeric(f["Hylak_id"], errors="coerce").dropna().astype(np.int64).unique())
    name_map = {}
    lakes = GLOBAL_STATE.get("lakes")
    if lakes is not None and not lakes.empty and "Hylak_id" in lakes.columns:
        subset = lakes[lakes.Hylak_id.astype(np.int64).isin(ids)]
        for _, r in subset.iterrows():
            name_map[int(r.Hylak_id)] = str(r.get("Lake_name", "") or "Unnamed lake")
    choices = [(f"{name_map.get(int(lid), 'Unnamed lake')} · Hylak_id {int(lid)}", str(int(lid))) for lid in ids]
    return gr.Dropdown(choices=choices, value=(choices[0][1] if choices else None))


def scene_changed_callback(scene_id):
    return _scene_lake_update(scene_id)


def refresh_scene_choices_callback():
    c = _scene_choices()
    v = c[0][1] if c else None
    n_lakes = 0
    if v:
        with contextlib.suppress(Exception):
            n_lakes = int(_load_scene_pixels(v, "scene", None).Hylak_id.nunique())
    return (
        gr.Dropdown(choices=c, value=v),
        _scene_lake_update(v),
        _app_status("Scene list refreshed", f"{len(c):,} completed scenes; selected scene contains {n_lakes:,} stored lakes."),
    )


def _planning_geometry_frame(frame):
    geom = GLOBAL_STATE.get("planning_geometry")
    if geom is None or getattr(geom, "is_empty", True):
        raise ValueError("No active planning region. Plan a State/Province, bbox, or 5° grid in Tab 1 first.")
    lon = pd.to_numeric(frame.longitude, errors="coerce").to_numpy(float)
    lat = pd.to_numeric(frame.latitude, errors="coerce").to_numpy(float)
    finite = np.isfinite(lon) & np.isfinite(lat)
    keep = np.zeros(len(frame), dtype=bool)
    if finite.any():
        keep[finite] = contains_xy(geom, lon[finite], lat[finite])
    out = frame.loc[keep].reset_index(drop=True)
    if out.empty:
        raise ValueError("The selected processed scene contains no stored lake pixels inside the active planning region.")
    return out


def _grid_geometry_mercator(lat, lon, resolution_m, max_cells=V265_DISPLAY_MAX_CELLS):
    """Exact EPSG:3857 display grid; never silently changes the user-requested resolution."""
    lat = np.asarray(lat, float); lon = np.asarray(lon, float)
    finite = np.isfinite(lat) & np.isfinite(lon) & (np.abs(lat) < 85.0)
    if not finite.any():
        raise ValueError("No finite Web-Mercator-compatible geolocation")
    x, y = _TO_WEBMERC.transform(lon[finite], lat[finite])
    x = np.asarray(x, float); y = np.asarray(y, float)
    res = float(resolution_m)
    if res <= 0:
        raise ValueError("Display resolution must be positive")
    xmin = math.floor(float(np.nanmin(x)) / res) * res - res
    xmax = math.ceil(float(np.nanmax(x)) / res) * res + res
    ymin = math.floor(float(np.nanmin(y)) / res) * res - res
    ymax = math.ceil(float(np.nanmax(y)) / res) * res + res
    nx = max(2, int(round((xmax - xmin) / res)))
    ny = max(2, int(round((ymax - ymin) / res)))
    xmax = xmin + nx * res; ymax = ymin + ny * res
    ncell = int(nx) * int(ny)
    if ncell > int(max_cells):
        raise RuntimeError(
            f"Exact {res:.0f} m display would require {ncell/1e6:.1f} million cells, above the "
            f"{int(max_cells)/1e6:.0f}-million browser safety limit. Choose 'Active planning region' or "
            "'Selected lake', or explicitly choose a coarser preview. The app will not silently change resolution."
        )
    west, south = _FROM_WEBMERC.transform(xmin, ymin)
    east, north = _FROM_WEBMERC.transform(xmax, ymax)
    return {
        "xmin": xmin, "xmax": xmax, "ymin": ymin, "ymax": ymax,
        "nx": nx, "ny": ny, "ncell": ncell, "resolution_m": res,
        "west": float(west), "east": float(east), "south": float(south), "north": float(north),
        "crs": "EPSG:3857",
    }


def _regularize_dataframe_mercator(frame, columns, resolution_m):
    """Nearest-native display remap at the exact requested Web-Mercator resolution.

    The nearest stored observation is selected before QA values are inspected. Therefore an
    invalid/cloudy source pixel remains invalid and is not replaced by a nearby clear pixel.
    """
    lat = pd.to_numeric(frame.latitude, errors="coerce").to_numpy(float)
    lon = pd.to_numeric(frame.longitude, errors="coerce").to_numpy(float)
    g = _grid_geometry_mercator(lat, lon, float(resolution_m), max_cells=V265_DISPLAY_MAX_CELLS)
    finite = np.isfinite(lat) & np.isfinite(lon) & (np.abs(lat) < 85.0)
    x, y = _TO_WEBMERC.transform(lon[finite], lat[finite])
    src_idx = np.flatnonzero(finite)
    tree = cKDTree(np.column_stack([np.asarray(x, float), np.asarray(y, float)]))
    source_values = {c: pd.to_numeric(frame[c], errors="coerce").to_numpy(float) for c in columns}
    out = {c: np.full((g["ny"], g["nx"]), np.nan, dtype=np.float32) for c in columns}
    xs = g["xmin"] + (np.arange(g["nx"], dtype=np.float64) + 0.5) * g["resolution_m"]
    radius = max(650.0, 2.0 * float(g["resolution_m"]))
    row_block = 96
    for r0 in range(0, g["ny"], row_block):
        r1 = min(r0 + row_block, g["ny"])
        ys = g["ymax"] - (np.arange(r0, r1, dtype=np.float64) + 0.5) * g["resolution_m"]
        xx, yy = np.meshgrid(xs, ys)
        pts = np.column_stack([xx.ravel(), yy.ravel()])
        dist, ii = tree.query(pts, k=1, distance_upper_bound=radius, workers=-1)
        matched = np.isfinite(dist) & (ii < len(src_idx))
        if not matched.any():
            continue
        nearest = np.full(len(ii), -1, dtype=np.int64)
        nearest[matched] = src_idx[np.asarray(ii[matched], dtype=np.int64)]
        for c in columns:
            block = np.full(len(ii), np.nan, dtype=np.float32)
            vals = source_values[c]
            good = matched.copy()
            if good.any():
                vv = vals[nearest[good]]
                block[np.flatnonzero(good)] = vv.astype(np.float32)
            out[c][r0:r1, :] = block.reshape(r1-r0, g["nx"])
    return out, g


def _build_map(scene_id, scope, lake_id, metric, resolution_m, basemap, labels, cmap, reverse, vmin_text, vmax_text, show_lakes=True):
    if not scene_id:
        raise ValueError("Choose a completed scene")
    scope = str(scope or "scene")
    if scope == "lake":
        raw = _load_scene_pixels(scene_id, "lake", lake_id)
        scope_label = f"lake {lake_id}"
        planning_geom = None
    else:
        raw = _load_scene_pixels(scene_id, "scene", None)
        planning_geom = GLOBAL_STATE.get("planning_geometry") if scope == "planning" else None
        if scope == "planning":
            raw = _planning_geometry_frame(raw)
            scope_label = GLOBAL_STATE.get("planning_shard") or "active planning region"
        else:
            scope_label = "entire processed scene"
    cols = ["rhos_665", "rhos_560", "rhos_490"] if metric == "true_color" else [metric]
    frame = _masked_display_frame(raw, cols)
    grids, g = _regularize_dataframe_mercator(frame, cols, float(resolution_m))
    bounds = [[g["south"], g["west"]], [g["north"], g["east"]]]
    m = folium.Map(tiles=None, control_scale=True, prefer_canvas=True)
    _add_basemap(m, basemap, bool(labels))
    if metric == "true_color":
        img = _truecolor_data_url(grids)
        folium.raster_layers.ImageOverlay(img, bounds=bounds, opacity=.95, name="True color · bloom-aware QA").add_to(m)
    else:
        img, vmin, vmax, cm = _rgba_data_url(grids[metric], metric, cmap, bool(reverse), _optional_float(vmin_text), _optional_float(vmax_text))
        folium.raster_layers.ImageOverlay(img, bounds=bounds, opacity=.88, name=f"{metric} · bloom-aware QA").add_to(m)
        _add_scalar_legend(m, metric, cmap, bool(reverse), vmin, vmax)
    if show_lakes:
        b = _lake_boundaries_for_ids(raw.Hylak_id.unique())
        if not b.empty:
            folium.GeoJson(json.loads(b.to_crs(4326).to_json()), name="HydroLAKES boundaries", style_function=lambda f:{"color":"#00e5ff","weight":1.2,"fillOpacity":0}).add_to(m)
    if planning_geom is not None and not planning_geom.is_empty:
        reg = gpd.GeoDataFrame({"name":[scope_label]}, geometry=[planning_geom], crs=4326)
        folium.GeoJson(json.loads(reg.to_json()), name="Active planning region", style_function=lambda f:{"color":"#ffcc00","weight":2.2,"fillOpacity":0}).add_to(m)
        west, south, east, north = planning_geom.bounds
        fit_bounds = [[south, west], [north, east]]
    else:
        fit_bounds = bounds
    folium_plugins.Fullscreen(position="topleft").add_to(m)
    folium_plugins.MousePosition(position="bottomright", separator=" | ", prefix="Lat / Lon", num_digits=5).add_to(m)
    folium_plugins.MeasureControl(position="topleft", primary_length_unit="kilometers").add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    m.fit_bounds(fit_bounds)
    hover_frame = frame.loc[_primary_selector(frame)].copy()
    _inject_hover_js(m, hover_frame, metric, g["resolution_m"])
    out = HTML_DIR / f"scene_{scene_id}_{scope}_{lake_id or 'ALL'}_{metric}_{int(round(g['resolution_m']))}m.html"
    m.save(out)
    map_src = f"/gradio_api/file={quote(str(out), safe='/')}?v={out.stat().st_mtime_ns}"
    iframe = "<iframe style='width:100%;height:740px;border:1px solid #ccd5df;border-radius:8px;background:white' loading='eager' src='" + map_src + "'></iframe>"
    return iframe, str(out), frame, grids, g



def export_snap_gridded_netcdf(scene_id, scope, lake_id, metric, frame, resolution_m):
    """Exact nominal-resolution EPSG:4326 quick export using nearest native values.

    This quick export follows the displayed scope/frame. It does not silently coarsen.
    A target cell is populated only from the nearest stored observation within the
    nearest-neighbour radius; already-masked science values remain no-data.
    """
    if frame is None or len(frame) == 0:
        raise ValueError("No displayed pixels are available for quick export")
    names = ["rhos_665", "rhos_560", "rhos_490"] if metric == "true_color" else [metric]
    g = _science_geographic_setup_v265(frame, float(resolution_m))
    ncell = int(g["ncell"])
    if ncell > 35_000_000:
        raise RuntimeError(
            f"Exact {g['resolution_m']:.0f} m quick export would require {ncell/1e6:.1f} million cells. "
            "Use the planning-region or selected-lake map scope, or use the minimal share ZIP exporter, "
            "which writes the full scene in chunks."
        )
    lat_src = pd.to_numeric(frame.latitude, errors="coerce").to_numpy(float)
    lon_src = pd.to_numeric(frame.longitude, errors="coerce").to_numpy(float)
    finite = np.isfinite(lat_src) & np.isfinite(lon_src)
    if not finite.any():
        raise ValueError("No finite source geolocation")
    src_idx = np.flatnonzero(finite)
    tree = cKDTree(_unit_sphere_xyz(lat_src[finite], lon_src[finite]))
    radius_m = max(float(V265_EXPORT_NEAREST_RADIUS_M), 1.75 * float(g["resolution_m"]))
    source_values = {n: pd.to_numeric(frame[n], errors="coerce").to_numpy(float) for n in names if n in frame.columns}
    lake_union = _scene_lake_union(scene_id, frame)
    SNAP_DIR = EXPORT_DIR / "snap_gridded"; SNAP_DIR.mkdir(parents=True, exist_ok=True)
    out = SNAP_DIR / f"S3_OLCI_{scene_id}_{scope}_{lake_id or 'ALL'}_{metric}_{int(round(g['resolution_m']))}m_CF.nc"
    from pyproj import CRS
    wkt = CRS.from_epsg(4326).to_wkt()
    chunk = 128
    with netCDF4.Dataset(out, "w", format="NETCDF4_CLASSIC") as ds:
        ds.createDimension("latitude", g["ny"]); ds.createDimension("longitude", g["nx"])
        ds.setncatts({
            "Conventions":"CF-1.8", "title":"Sentinel-3 OLCI displayed-scope native-nearest quick export",
            "scene_id":str(scene_id), "scope":str(scope), "nominal_grid_resolution_m":float(g["resolution_m"]),
            "science_grid_method":"epsg4326_native_nearest_v1", "nearest_native_radius_m":float(radius_m),
            "resampling":"nearest-native assignment; no averaging/value interpolation or gap filling", "qa_basis":"primary bloom-aware mask already encoded in displayed science values",
        })
        y=ds.createVariable("latitude","f8",("latitude",)); y[:]=g["latitude"]; y.setncatts({"standard_name":"latitude","units":"degrees_north","axis":"Y"})
        x=ds.createVariable("longitude","f8",("longitude",)); x[:]=g["longitude"]; x.setncatts({"standard_name":"longitude","units":"degrees_east","axis":"X"})
        crs=ds.createVariable("crs","i4"); crs.assignValue(0); crs.setncatts({"grid_mapping_name":"latitude_longitude","epsg_code":"EPSG:4326","spatial_ref":wkt,"crs_wkt":wkt,"semi_major_axis":6378137.0,"inverse_flattening":298.257223563})
        vars_={}
        chunks=(min(chunk,g["ny"]),min(chunk,g["nx"]))
        for n in source_values:
            out_n="rhos_884" if n=="rhos_885" else n
            v=ds.createVariable(out_n,"f4",("latitude","longitude"),zlib=True,complevel=5,shuffle=True,chunksizes=chunks,fill_value=np.float32(-9999.0))
            v.setncatts({"coordinates":"latitude longitude","grid_mapping":"crs","units":"1","long_name":out_n})
            vars_[n]=v
        for r0 in range(0,g["ny"],chunk):
            r1=min(r0+chunk,g["ny"]); lats=g["latitude"][r0:r1]
            for c0 in range(0,g["nx"],chunk):
                c1=min(c0+chunk,g["nx"]); lons=g["longitude"][c0:c1]
                lon2,lat2=np.meshgrid(lons,lats)
                if lake_union is not None:
                    allowed=contains_xy(lake_union,lon2,lat2)
                else:
                    allowed=np.ones(lon2.shape,bool)
                if str(scope)=="planning" and GLOBAL_STATE.get("planning_geometry") is not None:
                    allowed &= contains_xy(GLOBAL_STATE["planning_geometry"],lon2,lat2)
                pos=np.flatnonzero(allowed.ravel())
                if not len(pos): continue
                chord,ii=tree.query(_unit_sphere_xyz(lat2.ravel()[pos],lon2.ravel()[pos]),k=1,workers=-1)
                dist=6371000.0*(2*np.arcsin(np.minimum(1.0,np.asarray(chord,float)/2)))
                matched=np.isfinite(dist)&(dist<=radius_m)&(np.asarray(ii)<len(src_idx))
                if not matched.any(): continue
                nearest=src_idx[np.asarray(ii[matched],dtype=np.int64)]
                target_pos=pos[matched]
                rr,cc=np.unravel_index(target_pos,lon2.shape)
                for n,v in vars_.items():
                    vals=source_values[n][nearest]
                    good=np.isfinite(vals)
                    if not good.any(): continue
                    block=np.full(lon2.shape,np.float32(-9999.0),np.float32)
                    block[rr[good],cc[good]]=vals[good].astype(np.float32)
                    v[r0:r1,c0:c1]=np.ma.masked_equal(block,np.float32(-9999.0))
        if metric=="true_color": ds.setncattr("true_color_mapping","red=rhos_665; green=rhos_560; blue=rhos_490")
    return str(out)


def map_callback(scene_id, scope, lake_id, metric, resolution_m, basemap, labels, cmap, reverse, vmin, vmax, show_lakes):
    try:
        iframe, html_path, frame, grids, g = _build_map(scene_id, scope, lake_id, metric, resolution_m, basemap, labels, cmap, reverse, vmin, vmax, show_lakes)
        quick_scope = scope if scope in {"scene", "planning", "lake"} else "scene"
        nc = export_snap_gridded_netcdf(scene_id, quick_scope, lake_id, metric, frame, resolution_m)
        compact = _compact_download(scene_id)
        scope_text = GLOBAL_STATE.get("planning_shard", "active planning region") if scope == "planning" else (f"lake {lake_id}" if scope == "lake" else "full scene")
        return iframe, html_path, nc, compact, _app_status(
            "Map ready",
            f"{len(frame):,} stored lake pixels; {frame.Hylak_id.nunique():,} lakes; {scope_text}; exact Web-Mercator display grid = {g['resolution_m']:.0f} m."
        )
    except Exception as exc:
        return _app_status("Map failed", str(exc), False), None, None, None, _app_status("Map failed", str(exc), False)


def _science_geographic_setup_v265(frame, requested_resolution_m=SCIENCE_EXPORT_DEFAULT_RESOLUTION_M):
    """SNAP-friendly regular EPSG:4326 grid with nominal metric spacing at scene median latitude."""
    lat = pd.to_numeric(frame.latitude, errors="coerce").to_numpy(float)
    lon = pd.to_numeric(frame.longitude, errors="coerce").to_numpy(float)
    finite = np.isfinite(lat) & np.isfinite(lon) & (np.abs(lat) <= 90) & (np.abs(lon) <= 180)
    if not finite.any():
        raise ValueError("No finite stored scene geolocation")
    res = float(requested_resolution_m)
    ref_lat = float(np.nanmedian(lat[finite]))
    dlat = res / 111320.0
    dlon = res / max(111320.0 * math.cos(math.radians(ref_lat)), 1000.0)
    west = math.floor(float(np.nanmin(lon[finite])) / dlon) * dlon - dlon
    east = math.ceil(float(np.nanmax(lon[finite])) / dlon) * dlon + dlon
    south = math.floor(float(np.nanmin(lat[finite])) / dlat) * dlat - dlat
    north = math.ceil(float(np.nanmax(lat[finite])) / dlat) * dlat + dlat
    nx = max(2, int(round((east - west) / dlon)))
    ny = max(2, int(round((north - south) / dlat)))
    east = west + nx * dlon; north = south + ny * dlat
    longitude = west + (np.arange(nx, dtype=np.float64) + 0.5) * dlon
    latitude = south + (np.arange(ny, dtype=np.float64) + 0.5) * dlat
    return {
        "resolution_m": res, "reference_latitude": ref_lat, "dlat": dlat, "dlon": dlon,
        "west": west, "east": east, "south": south, "north": north,
        "nx": nx, "ny": ny, "ncell": int(nx)*int(ny),
        "longitude": longitude, "latitude": latitude, "crs": "EPSG:4326",
    }


def _scene_lake_union(scene_id, frame):
    try:
        b = _lake_boundaries_for_ids(pd.to_numeric(frame.Hylak_id, errors="coerce").dropna().astype(np.int64).unique())
        if b is None or b.empty:
            return None
        b = b.to_crs(4326)
        return b.geometry.union_all()
    except Exception:
        return None


def _candidate_geographic_chunks(frame, g, chunk=V265_EXPORT_CHUNK):
    lat = pd.to_numeric(frame.latitude, errors="coerce").to_numpy(float)
    lon = pd.to_numeric(frame.longitude, errors="coerce").to_numpy(float)
    finite = np.isfinite(lat) & np.isfinite(lon)
    rr = np.floor((lat[finite] - g["south"]) / g["dlat"]).astype(np.int64)
    cc = np.floor((lon[finite] - g["west"]) / g["dlon"]).astype(np.int64)
    good = (rr >= 0) & (rr < g["ny"]) & (cc >= 0) & (cc < g["nx"])
    rr = rr[good]; cc = cc[good]
    nxc = int(math.ceil(g["nx"] / chunk)); nyc = int(math.ceil(g["ny"] / chunk))
    base = np.unique((rr // chunk) * nxc + (cc // chunk))
    keys = set()
    for cid in base.tolist():
        cr = int(cid // nxc); ccol = int(cid % nxc)
        for dr in (-1, 0, 1):
            for dc in (-1, 0, 1):
                ar, ac = cr + dr, ccol + dc
                if 0 <= ar < nyc and 0 <= ac < nxc:
                    keys.add((ar, ac))
    return sorted(keys)


def export_complete_scene_science_netcdf(scene_id, requested_resolution_m=SCIENCE_EXPORT_DEFAULT_RESOLUTION_M, force=False):
    """SNAP-friendly nominal-resolution EPSG:4326 native-nearest remap.

    Each mapped cell inside a HydroLAKES polygon takes the nearest *stored native OLCI
    observation* within the radius. The nearest observation is selected before its QA state
    is inspected, so cloudy/invalid observations remain no-data rather than being replaced by
    a nearby clear pixel. This removes center-binning striping without interpolating values.
    """
    if not scene_id:
        raise ValueError("Choose a completed scene")
    frame = _load_scene_pixels(scene_id, "scene", None).copy()
    g = _science_geographic_setup_v265(frame, float(requested_resolution_m))
    export_dir = EXPORT_DIR / "snap_complete"; export_dir.mkdir(parents=True, exist_ok=True)
    out = export_dir / f"S3_OLCI_{scene_id}_ALL_SCIENCE_{int(round(g['resolution_m']))}m_CF.nc"
    source_files = _scene_compact_files(scene_id)
    newest = max((p.stat().st_mtime_ns for p in source_files), default=0)
    if out.exists() and not force and out.stat().st_mtime_ns >= newest:
        with contextlib.suppress(Exception):
            with netCDF4.Dataset(out) as old:
                if getattr(old, "science_grid_method", "") == "epsg4326_native_nearest_v1" and float(getattr(old, "nominal_grid_resolution_m", -1)) == float(g["resolution_m"]):
                    return str(out), g

    lat_src = pd.to_numeric(frame.latitude, errors="coerce").to_numpy(float)
    lon_src = pd.to_numeric(frame.longitude, errors="coerce").to_numpy(float)
    finite_src = np.isfinite(lat_src) & np.isfinite(lon_src)
    src_global_idx = np.flatnonzero(finite_src)
    tree = cKDTree(_unit_sphere_xyz(lat_src[finite_src], lon_src[finite_src]))
    primary = _primary_selector(frame)
    lake_union = _scene_lake_union(scene_id, frame)
    radius_m = max(float(V265_EXPORT_NEAREST_RADIUS_M), 1.75 * float(g["resolution_m"]))
    chunk = int(V265_EXPORT_CHUNK)
    chunk_keys = _candidate_geographic_chunks(frame, g, chunk=chunk)
    qa = _compact_qa_metadata(scene_id)
    source_values = {n: pd.to_numeric(frame[n], errors="coerce").to_numpy(float) for n in SCIENCE_EXPORT_VARIABLES if n in frame.columns}
    mask_names = ["valid_water_mask","cyan_strict_valid_mask","bloom_rescue_mask","cldice_mask","hisatzen_mask","navfail_mask","cloud_excluded_mask","ci_valid_mask","ci_candidate_mask","ci_detection_mask","mph_valid_mask","fai_valid_mask"]
    source_masks = {n: pd.to_numeric(frame[n], errors="coerce").fillna(0).to_numpy(float) > 0 for n in mask_names if n in frame.columns}
    source_hylak = pd.to_numeric(frame.Hylak_id, errors="coerce").fillna(-1).to_numpy(np.int32)

    tmp = out.with_suffix(".nc.tmp"); tmp.unlink(missing_ok=True)
    with netCDF4.Dataset(tmp, "w", format="NETCDF4_CLASSIC") as ds:
        ds.createDimension("latitude", g["ny"]); ds.createDimension("longitude", g["nx"])
        ds.setncatts({
            "Conventions":"CF-1.8",
            "title":"Sentinel-3 OLCI HydroLAKES bloom-aware all-science native-nearest product",
            "scene_id":str(scene_id),
            "science_grid_method":"epsg4326_native_nearest_v1",
            "nominal_grid_resolution_m":float(g["resolution_m"]),
            "actual_grid_resolution_m":float(g["resolution_m"]),
            "cell_height_m_nominal":float(g["resolution_m"]),
            "cell_width_m_at_reference_latitude":float(g["resolution_m"]),
            "reference_latitude_for_longitude_spacing":float(g["reference_latitude"]),
            "geospatial_lat_min":float(g["south"]), "geospatial_lat_max":float(g["north"]),
            "geospatial_lon_min":float(g["west"]), "geospatial_lon_max":float(g["east"]),
            "latitude_step_degrees":float(g["dlat"]),
            "longitude_step_degrees":float(g["dlon"]),
            "nearest_native_radius_m":float(radius_m),
            "mapping":"nearest stored native OLCI observation; target cells constrained to HydroLAKES polygons when boundaries are available",
            "qa_rule":"nearest source is chosen before QA; if that source is not primary-valid, science value stays fill/NaN",
            "resampling":"nearest-native assignment; no averaging/value interpolation or gap filling",
            "striping_policy":"no center-only binning; nearest-native remap prevents artificial empty scan-line cells",
            "primary_qa_mask":"valid_water_mask (bloom-aware)",
            "true_color_mapping":"red=rhos_665; green=rhos_560; blue=rhos_490",
            "CI_cyano_detection_limit":float(CYAN_CI_DETECTION_LIMIT),
            "qa_policy_json":json.dumps(qa, default=str),
        })
        yv = ds.createVariable("latitude", "f8", ("latitude",)); yv[:] = g["latitude"]; yv.setncatts({"standard_name":"latitude","units":"degrees_north","axis":"Y"})
        xv = ds.createVariable("longitude", "f8", ("longitude",)); xv[:] = g["longitude"]; xv.setncatts({"standard_name":"longitude","units":"degrees_east","axis":"X"})
        from pyproj import CRS
        wkt4326 = CRS.from_epsg(4326).to_wkt()
        crsv = ds.createVariable("crs", "i4"); crsv.assignValue(0); crsv.setncatts({"grid_mapping_name":"latitude_longitude","epsg_code":"EPSG:4326","spatial_ref":wkt4326,"crs_wkt":wkt4326,"semi_major_axis":6378137.0,"inverse_flattening":298.257223563})
        chunks = (min(chunk, g["ny"]), min(chunk, g["nx"]))
        vars_float = {}
        for source_name, out_name in [(n, "rhos_884" if n == "rhos_885" else n) for n in SCIENCE_EXPORT_VARIABLES if n in source_values]:
            var = ds.createVariable(out_name, "f4", ("latitude","longitude"), zlib=True, complevel=6, shuffle=True, chunksizes=chunks, fill_value=np.float32(-9999.0))
            long_names={"CI":"Cyanobacteria Index candidate magnitude","CI_cyano":"CyAN-compatible cyanobacteria index detections","NDCI":"Normalized Difference Chlorophyll Index","MPH":"Maximum Peak Height","FAI":"OLCI-adapted Floating Algae Index (AFAI form)"}
            var.setncatts({"coordinates":"latitude longitude","grid_mapping":"crs","units":"1","long_name":long_names.get(out_name, f"Rayleigh-corrected OLCI reflectance {out_name}"),"qa_basis":"primary bloom-aware valid_water_mask"})
            vars_float[source_name] = var
        vars_mask = {}
        for name in source_masks:
            v = ds.createVariable(name, "i1", ("latitude","longitude"), zlib=True, complevel=6, shuffle=True, chunksizes=chunks, fill_value=np.int8(-1))
            v.setncatts({"coordinates":"latitude longitude","grid_mapping":"crs","long_name":name.replace("_"," "),"flag_values":np.array([0,1],dtype=np.int8),"flag_meanings":"false true","valid_range":np.array([0,1],dtype=np.int8)})
            vars_mask[name] = v
        dist_var = ds.createVariable("native_match_distance_m", "f4", ("latitude","longitude"), zlib=True, complevel=5, shuffle=True, chunksizes=chunks, fill_value=np.float32(-9999.0)); dist_var.setncatts({"coordinates":"latitude longitude","grid_mapping":"crs","units":"m","long_name":"distance from mapped cell centre to nearest stored native OLCI observation"})
        hylak_var = ds.createVariable("nearest_Hylak_id", "i4", ("latitude","longitude"), zlib=True, complevel=5, shuffle=True, chunksizes=chunks, fill_value=np.int32(-1)); hylak_var.setncatts({"coordinates":"latitude longitude","grid_mapping":"crs","long_name":"HydroLAKES identifier of nearest stored native OLCI observation"})

        for cr, cc in chunk_keys:
            r0 = cr * chunk; r1 = min(r0 + chunk, g["ny"]); c0 = cc * chunk; c1 = min(c0 + chunk, g["nx"])
            lats = g["latitude"][r0:r1]; lons = g["longitude"][c0:c1]
            lon2, lat2 = np.meshgrid(lons, lats)
            if lake_union is not None:
                inside_lake = contains_xy(lake_union, lon2, lat2)
            else:
                inside_lake = np.ones(lon2.shape, dtype=bool)
            positions = np.flatnonzero(inside_lake.ravel())
            if not len(positions):
                continue
            xyz = _unit_sphere_xyz(lat2.ravel()[positions], lon2.ravel()[positions])
            chord, ii = tree.query(xyz, k=1, workers=-1)
            dist = 6371000.0 * (2.0 * np.arcsin(np.minimum(1.0, np.asarray(chord, float) / 2.0)))
            matched = np.isfinite(dist) & (dist <= radius_m) & (np.asarray(ii) < len(src_global_idx))
            if not matched.any():
                continue
            nearest = np.full(len(positions), -1, dtype=np.int64)
            nearest[matched] = src_global_idx[np.asarray(ii[matched], dtype=np.int64)]
            base_flat = np.full(lon2.size, -1, dtype=np.int64)
            base_flat[positions[matched]] = nearest[matched]
            nearest_grid = base_flat.reshape(lon2.shape)
            matched_grid = nearest_grid >= 0

            dist_block = np.full(lon2.shape, np.float32(-9999.0), np.float32)
            dflat = dist_block.ravel(); dflat[positions[matched]] = dist[matched].astype(np.float32); dist_var[r0:r1,c0:c1] = np.ma.masked_equal(dist_block, np.float32(-9999.0))
            hblock = np.full(lon2.shape, np.int32(-1), np.int32); hblock[matched_grid] = source_hylak[nearest_grid[matched_grid]]; hylak_var[r0:r1,c0:c1] = np.ma.masked_equal(hblock, np.int32(-1))

            for source_name, var in vars_float.items():
                block = np.full(lon2.shape, np.float32(-9999.0), np.float32)
                if matched_grid.any():
                    src = nearest_grid[matched_grid]
                    vals = source_values[source_name][src]
                    ok = primary[src] & np.isfinite(vals)
                    rr0, cc0 = np.where(matched_grid)
                    block[rr0[ok], cc0[ok]] = vals[ok].astype(np.float32)
                var[r0:r1,c0:c1] = np.ma.masked_equal(block, np.float32(-9999.0))
            for name, var in vars_mask.items():
                block = np.full(lon2.shape, np.int8(-1), np.int8)
                if matched_grid.any():
                    src = nearest_grid[matched_grid]
                    rr0, cc0 = np.where(matched_grid)
                    block[rr0, cc0] = source_masks[name][src].astype(np.int8)
                var[r0:r1,c0:c1] = np.ma.masked_equal(block, np.int8(-1))
    tmp.replace(out)
    return str(out), g


def create_scene_share_bundle(scene_id, science_resolution_m=SCIENCE_EXPORT_DEFAULT_RESOLUTION_M):
    """Exactly three files, using the SNAP-friendly EPSG:4326 native-nearest map product."""
    if not scene_id:
        raise ValueError("Choose a completed scene")
    t = registry_table(GLOBAL_STATE.get("config_hash")); r = t[t.scene_id.astype(str) == str(scene_id)]
    if r.empty:
        raise KeyError(scene_id)
    row = r.iloc[0]; cfg = str(row.config_hash); d = EXPORT_DIR / "scene_bundles"; d.mkdir(parents=True, exist_ok=True)
    complete_nc, g = export_complete_scene_science_netcdf(scene_id, float(science_resolution_m), force=False)
    stats_csv, stats_table = _share_all_index_stats(scene_id, cfg, row)
    qa = _compact_qa_metadata(scene_id); frame_meta = parse_s3_frame_metadata(row.scene_name)
    meta = {
        "schema":"global-efficient-share-v2.6.5", "created_utc":utc_now(), "scene_id":str(scene_id), "scene_name":str(row.scene_name), "acquisition_start":str(row.acquisition_start), "config_hash":cfg, **frame_meta,
        "files":{"snap_all_science_netcdf":Path(complete_nc).name,"all_index_stats_csv":Path(stats_csv).name},
        "snap_variables":["rhos_490","rhos_560","rhos_620","rhos_665","rhos_681","rhos_709","rhos_754","rhos_865","rhos_884","CI","CI_cyano","NDCI","MPH","FAI","valid_water_mask","cyan_strict_valid_mask","bloom_rescue_mask","cldice_mask","hisatzen_mask","native_match_distance_m","nearest_Hylak_id"],
        "snap_grid":{"projection":"EPSG:4326 geographic","grid_method":"epsg4326_native_nearest_v1","nominal_resolution_m":float(g["resolution_m"]),"reference_latitude":float(g["reference_latitude"]),"latitude_step_degrees":float(g["dlat"]),"longitude_step_degrees":float(g["dlon"]),"nearest_radius_m":max(float(V265_EXPORT_NEAREST_RADIUS_M),1.75*float(g["resolution_m"])),"resampling":"nearest-native assignment; no averaging/value interpolation or gap filling","striping":"center-binning stripes avoided by mapping every target lake cell to its nearest stored native observation before QA"},
        "masking":qa,
        "important":"Lake statistics are computed from native compact OLCI observations, not the mapped SNAP raster. The SNAP raster selects the nearest native observation first; cloudy/invalid nearest observations remain no-data."
    }
    mp = d / f"S3_OLCI_{scene_id}_METADATA.json"; mp.write_text(json.dumps(meta, indent=2, default=str))
    z = d / f"{str(row.scene_name)}_{cfg}_SHARE_MINIMAL.zip"
    with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED, allowZip64=True) as arc:
        for p in (Path(complete_nc), Path(stats_csv), mp):
            arc.write(p, arcname=p.name)
    return str(z), _app_status("Minimal share bundle created", f"3 files · nominal {g['resolution_m']:.0f} m EPSG:4326 native-nearest grid · SNAP-friendly · bloom-aware cloud QA")


def _spectrum_map_image(scene_id, points=None, matched_table=None, width=1000, height=560):
    if not scene_id:
        return None, {}
    f = _load_scene_pixels(scene_id, "scene", None)
    finite = np.isfinite(pd.to_numeric(f.latitude, errors="coerce")) & np.isfinite(pd.to_numeric(f.longitude, errors="coerce"))
    f = f.loc[finite].copy()
    if f.empty:
        return None, {}
    # Downsample only for drawing; click-to-spectrum still searches the complete native archive.
    if len(f) > 180000:
        take = np.linspace(0, len(f)-1, 180000, dtype=int); draw_f = f.iloc[take]
    else:
        draw_f = f
    west, east = float(f.longitude.min()), float(f.longitude.max()); south, north = float(f.latitude.min()), float(f.latitude.max())
    dx=max(east-west, 1e-4); dy=max(north-south, 1e-4); west-=0.03*dx; east+=0.03*dx; south-=0.03*dy; north+=0.03*dy
    img = Image.new("RGB", (int(width), int(height)), (244,247,250)); dr = ImageDraw.Draw(img)
    dr.rectangle([0,0,width-1,height-1], outline=(80,90,100), width=1)
    def xy(lon,lat):
        x=(float(lon)-west)/(east-west)*(width-1); y=(north-float(lat))/(north-south)*(height-1); return int(round(x)),int(round(y))
    # Draw HydroLAKES outlines so the clickable image is a geographic map, not only a point cloud.
    with contextlib.suppress(Exception):
        b=_lake_boundaries_for_ids(draw_f.Hylak_id.unique())
        if b is not None and not b.empty:
            b=b.to_crs(4326)
            def draw_geom(geom):
                if geom is None or geom.is_empty: return
                if geom.geom_type=="Polygon":
                    coords=list(geom.exterior.coords)
                    pts=[xy(xx,yy) for xx,yy in coords]
                    if len(pts)>1: dr.line(pts,fill=(0,165,195),width=1)
                elif geom.geom_type in {"MultiPolygon","GeometryCollection"}:
                    for gg in geom.geoms: draw_geom(gg)
            for geom in b.geometry: draw_geom(geom)
    valid = _primary_selector(draw_f)
    lonv=draw_f.longitude.to_numpy(float); latv=draw_f.latitude.to_numpy(float)
    for x0,y0,v in zip(lonv,latv,valid):
        x,y=xy(x0,y0); col=(17,94,140) if v else (175,182,190); dr.point((x,y), fill=col)
    if points is not None:
        with contextlib.suppress(Exception):
            pf=_spectrum_points_frame(points)
            for _,r in pf.iterrows():
                x,y=xy(r.longitude,r.latitude); dr.ellipse([x-5,y-5,x+5,y+5],outline=(220,38,38),width=2); dr.line([x-7,y,x+7,y],fill=(220,38,38),width=1); dr.line([x,y-7,x,y+7],fill=(220,38,38),width=1)
    if matched_table is not None and isinstance(matched_table,pd.DataFrame) and not matched_table.empty:
        mt=matched_table.drop_duplicates("point")
        for _,r in mt.iterrows():
            if np.isfinite(r.get("pixel_longitude",np.nan)) and np.isfinite(r.get("pixel_latitude",np.nan)):
                x,y=xy(r.pixel_longitude,r.pixel_latitude); dr.rectangle([x-4,y-4,x+4,y+4],outline=(0,150,70),width=2)
    dr.rectangle([6,6,365,45],fill=(255,255,255),outline=(180,185,190)); dr.text((12,10),"Click map to add spectrum point",fill=(20,30,40)); dr.text((12,26),"blue=valid native pixels · gray=QA excluded · red=target · green=matched",fill=(55,65,75))
    state={"west":west,"east":east,"south":south,"north":north,"width":int(width),"height":int(height),"scene_id":str(scene_id)}
    return img,state


def spectrum_map_refresh_callback(scene_id, points):
    try:
        img,state=_spectrum_map_image(scene_id,points)
        return img,state,_app_status("Spectrum map ready","Click a location to append a coordinate and immediately plot the nearest native-pixel spectrum.")
    except Exception as exc:
        return None,{},_app_status("Spectrum map failed",str(exc),False)


def coordinate_spectra_with_map_callback(scene_id, points, pixel_basis="stored_lake", max_distance_m=2000):
    fig,tab,csv,status=coordinate_spectra_callback(scene_id,points,pixel_basis,max_distance_m)
    try:
        img,state=_spectrum_map_image(scene_id,points,tab)
    except Exception:
        img,state=None,{}
    return img,state,fig,tab,csv,status


def clear_spectrum_points_and_map_callback(scene_id):
    empty=pd.DataFrame(columns=["label","latitude","longitude"])
    try:
        img,state=_spectrum_map_image(scene_id,empty)
    except Exception:
        img,state=None,{}
    return empty,img,state,_app_status("Spectrum points cleared")


def spectrum_map_click_callback(scene_id, points, pixel_basis, max_distance_m, map_state, evt: gr.SelectData):
    try:
        if not scene_id:
            raise ValueError("Choose a completed scene")
        st=dict(map_state or {})
        if str(st.get("scene_id")) != str(scene_id):
            _,st=_spectrum_map_image(scene_id,points)
        idx=evt.index
        if not isinstance(idx,(tuple,list)) or len(idx)<2:
            raise ValueError("Could not read map click coordinates")
        x=float(idx[0]); y=float(idx[1]); width=float(st["width"]); height=float(st["height"])
        lon=float(st["west"] + x/max(width-1,1)*(st["east"]-st["west"]))
        lat=float(st["north"] - y/max(height-1,1)*(st["north"]-st["south"]))
        current=pd.DataFrame(columns=["label","latitude","longitude"]) if points is None else (points.copy() if isinstance(points,pd.DataFrame) else pd.DataFrame(points,columns=["label","latitude","longitude"]))
        label=f"Click{len(current)+1}"
        current=pd.concat([current,pd.DataFrame([{"label":label,"latitude":lat,"longitude":lon}])],ignore_index=True)
        fig,tab,csv,status=coordinate_spectra_callback(scene_id,current,pixel_basis,max_distance_m)
        img,new_state=_spectrum_map_image(scene_id,current,tab)
        return current,img,new_state,fig,tab,csv,_app_status("Clicked spectrum ready",f"{label}: target {lat:.5f}, {lon:.5f}. The spectrum uses the nearest eligible stored native OLCI pixel.")
    except Exception as exc:
        return points,None,map_state,None,pd.DataFrame(),None,_app_status("Map-click spectrum failed",str(exc),False)


def _cyan_metrics_v265(obpg, local):
    paired=np.isfinite(obpg)&np.isfinite(local)&(obpg>CYAN_CI_DETECTION_LIMIT)&(local>CYAN_CI_DETECTION_LIMIT)
    x=np.asarray(obpg,float)[paired]; y=np.asarray(local,float)[paired]
    out={"paired_above_detection_limit_pixels":int(len(x)),"pearson_r":np.nan,"spearman_rho":np.nan,"mean_bias_local_minus_obpg":np.nan,"mae":np.nan,"rmse":np.nan,"linear_slope":np.nan,"linear_intercept":np.nan,"median_local_over_obpg":np.nan,"median_ratio_obpg_ge_0_01":np.nan,"mean_bias_obpg_ge_0_01":np.nan}
    if len(x):
        d=y-x; out.update(mean_bias_local_minus_obpg=float(np.mean(d)),mae=float(np.mean(np.abs(d))),rmse=float(np.sqrt(np.mean(d*d))),median_local_over_obpg=float(np.nanmedian(y/x)))
        hi=x>=0.01
        if hi.any():
            out["median_ratio_obpg_ge_0_01"]=float(np.nanmedian(y[hi]/x[hi])); out["mean_bias_obpg_ge_0_01"]=float(np.nanmean(y[hi]-x[hi]))
    if len(x)>=2 and np.nanstd(x)>0 and np.nanstd(y)>0:
        out["pearson_r"]=float(scipy_stats.pearsonr(x,y).statistic); out["spearman_rho"]=float(scipy_stats.spearmanr(x,y).statistic)
        slope,intercept=np.polyfit(x,y,1); out["linear_slope"]=float(slope); out["linear_intercept"]=float(intercept)
    return out,paired


def compare_cyan_callback_v265(scene_id,lake_id,explicit_date,source_mode,uploaded_tif,max_distance_m,qa_basis="strict"):
    try:
        if not scene_id or not lake_id: raise ValueError("Choose a completed scene and lake")
        date=_scene_utc_date_global(scene_id,explicit_date)
        if source_mode=="upload":
            if not uploaded_tif: raise ValueError("Upload the NASA OBPG GeoTIFF")
            obpg_path=Path(str(uploaded_tif))
        elif CYAN_STATE.get("source_path") and CYAN_STATE.get("source_date")==date.isoformat(): obpg_path=Path(CYAN_STATE["source_path"])
        else: obpg_path=_download_obpg_cyano_global(date)
        lake=_selected_target_lake(lake_id); raw,raster_valid,lat,lon,inside,profile=_obpg_lake_window_global(obpg_path,lake); obpg_ci,science=_obpg_ci_from_dn_global(raw,raster_valid)
        local=_load_scene_pixels(scene_id,"lake",lake_id)
        selector=_cyan_comparison_selector(local) if str(qa_basis)=="strict" else _primary_selector(local)
        source=local[selector & np.isfinite(local.latitude)&np.isfinite(local.longitude)].reset_index(drop=True)
        if source.empty: raise ValueError("No compact native pixels exist for selected lake and QA basis")
        target_lat=lat[inside]; target_lon=lon[inside]; tree=cKDTree(_unit_sphere_xyz(source.latitude,source.longitude)); chord,idx=tree.query(_unit_sphere_xyz(target_lat,target_lon),k=1,workers=-1); dist=6371000.0*(2*np.arcsin(np.minimum(1.0,np.asarray(chord,float)/2))); matched=dist<=float(max_distance_m)
        local_val=np.full(len(target_lat),np.nan,np.float32); local_ci_valid=np.zeros(len(target_lat),bool); local_candidate=np.zeros(len(target_lat),bool); local_detected=np.zeros(len(target_lat),bool)
        if matched.any():
            q=source.iloc[np.asarray(idx[matched],int)]; local_val[matched]=pd.to_numeric(q.CI_cyano,errors="coerce").to_numpy(np.float32); local_ci_valid[matched]=pd.to_numeric(q.get("ci_valid_mask",0),errors="coerce").fillna(0).to_numpy(float)>0; local_candidate[matched]=pd.to_numeric(q.get("ci_candidate_mask",0),errors="coerce").fillna(0).to_numpy(float)>0; local_detected[matched]=pd.to_numeric(q.get("ci_detection_mask",0),errors="coerce").fillna(0).to_numpy(float)>0
        raw_inside=raw[inside]; science_inside=science[inside]; obpg_inside=obpg_ci[inside]; obpg_det=science_inside&np.isfinite(obpg_inside)&(obpg_inside>CYAN_CI_DETECTION_LIMIT); local_det=matched&local_detected&np.isfinite(local_val)&(local_val>CYAN_CI_DETECTION_LIMIT); metrics,paired=_cyan_metrics_v265(obpg_inside,np.where(local_det,local_val,np.nan))
        table=pd.DataFrame({"latitude":target_lat,"longitude":target_lon,"obpg_raw_dn":raw_inside.astype(np.int16),"obpg_science_valid":science_inside,"obpg_CI_cyano":obpg_inside,"obpg_detected_gt_0_0001":obpg_det,"local_match_found":matched,"local_match_distance_m":np.where(matched,dist,np.nan),"local_ci_input_valid":local_ci_valid,"local_ci_candidate":local_candidate,"local_ci_cyano_detected":local_det,"local_CI_cyano":local_val,"paired_above_0_0001_for_correlation":paired})
        coverage=pd.DataFrame([{"date_utc":date.isoformat(),"Hylak_id":int(lake_id),"lake_name":str(lake.iloc[0].get("Lake_name","") or ""),"obpg_grid_cells_inside_lake":int(inside.sum()),"obpg_science_valid_cells":int(science_inside.sum()),"obpg_detection_cells_gt_0_0001":int(obpg_det.sum()),"local_nearest_matches":int(matched.sum()),"local_detection_matches_gt_0_0001":int(local_det.sum()),"paired_detection_pixels":metrics["paired_above_detection_limit_pixels"],"max_nearest_distance_m":float(max_distance_m),"local_QA_basis":str(qa_basis),"comparison_basis":"same-day Merged-S3-CYAN daily maximum versus one local OLCI scene; nearest native local pixel"}]); metric_table=pd.DataFrame([metrics])
        d=EXPORT_DIR/"cyan_comparison"/str(scene_id); d.mkdir(parents=True,exist_ok=True); stem=f"{scene_id}_Hylak{int(lake_id)}_{date.isoformat()}_CyAN_{qa_basis}"; csv=d/f"{stem}_pairs.csv"; met=d/f"{stem}_metrics.csv"; nc=d/f"{stem}_comparison.nc"; table.to_csv(csv,index=False); pd.concat([coverage,metric_table],axis=1).to_csv(met,index=False)
        shape=raw.shape
        def grid(values,fill=np.nan,dtype=np.float32): out=np.full(shape,fill,dtype=dtype);out[inside]=np.asarray(values,dtype=dtype);return out
        ds=xr.Dataset({"latitude":(("y","x"),lat.astype(np.float32)),"longitude":(("y","x"),lon.astype(np.float32)),"inside_lake":(("y","x"),inside.astype(np.uint8)),"obpg_raw_dn":(("y","x"),raw.astype(np.int16)),"obpg_CI_cyano":(("y","x"),np.where(inside,obpg_ci,np.nan).astype(np.float32)),"local_CI_cyano_nearest":(("y","x"),grid(local_val)),"local_match_distance_m":(("y","x"),grid(np.where(matched,dist,np.nan))),"paired_detection_mask":(("y","x"),grid(paired,0,np.uint8))},attrs={"title":"Local L2Gen CIcyano versus NASA OBPG Merged-S3-CYAN comparison","created_utc":utc_now(),"scene_id":str(scene_id),"acquisition_date_utc":date.isoformat(),"selected_hylak_id":int(lake_id),"source_obpg_geotiff":str(obpg_path),"local_QA_basis":str(qa_basis),"pairing":"nearest compact native OLCI lake pixel to each in-lake OBPG grid cell centre","max_nearest_distance_m":float(max_distance_m),"ci_cyano_detection_limit":float(CYAN_CI_DETECTION_LIMIT),"note":"Merged-S3-CYAN uses the daily per-pixel maximum across S3A/S3B; local source is one scene, so a negative local-minus-OBPG bias is expected when the other platform/time has the daily maximum."}); ds.to_netcdf(nc,engine="netcdf4",encoding={n:{"zlib":True,"complevel":4} for n in ds.data_vars}); ds.close()
        finite=table[table.paired_above_0_0001_for_correlation].copy(); fig=go.Figure(); fig.add_trace(go.Scattergl(x=finite.obpg_CI_cyano,y=finite.local_CI_cyano,mode="markers",name="Paired detections",marker={"size":5,"opacity":.48}))
        if not finite.empty:
            lo=float(min(finite.obpg_CI_cyano.min(),finite.local_CI_cyano.min())); hi=float(max(finite.obpg_CI_cyano.max(),finite.local_CI_cyano.max())); fig.add_trace(go.Scatter(x=[lo,hi],y=[lo,hi],mode="lines",name="1:1",line={"dash":"dash","color":"black"}))
            xmin=max(float(finite.obpg_CI_cyano.min()),CYAN_CI_DETECTION_LIMIT); xmax=float(finite.obpg_CI_cyano.max())
            if xmax>xmin:
                edges=np.geomspace(xmin,xmax,9); medx=[]; medy=[]
                for a,b in zip(edges[:-1],edges[1:]):
                    q=finite[(finite.obpg_CI_cyano>=a)&(finite.obpg_CI_cyano<(b if b<edges[-1] else b*1.000001))]
                    if len(q)>=5: medx.append(float(np.median(q.obpg_CI_cyano))); medy.append(float(np.median(q.local_CI_cyano)))
                if medx: fig.add_trace(go.Scatter(x=medx,y=medy,mode="lines+markers",name="Binned median local",line={"width":3}))
        fig.update_layout(title=f"{coverage.iloc[0].lake_name} · local scene versus same-day NASA CyAN",xaxis={"title":"NASA OBPG Merged-S3-CYAN CIcyano","type":"log"},yaxis={"title":"Local L2Gen CIcyano","type":"log"},template="plotly_white",height=600)
        note=(f"Paired detections >0.0001: {metrics['paired_above_detection_limit_pixels']:,}. NASA Merged-S3-CYAN is the daily per-pixel maximum across S3A/S3B, whereas the local product is one scene. High-end negative bias therefore should not be removed by empirical scaling without a same-platform/same-overpass test.")
        return fig,coverage.round(6),metric_table.round(8),str(csv),str(met),str(nc),str(obpg_path),_app_status("NASA CyAN comparison complete",note)
    except Exception as exc:
        return None,pd.DataFrame(),pd.DataFrame(),None,None,None,None,_app_status("NASA CyAN comparison failed",str(exc),False)


print("v2.6.5 correction patch active:")
print("  map resolution           : exact requested; no silent coarsening")
print("  planning-region map      : enabled")
print("  share SNAP grid          : EPSG:4326 native-nearest nominal resolution")
print("  center-binning striping  : removed from share export")
print("  scene lake dropdown      : compact scene IDs only")
print("  spectra click map        : enabled")
print("  CyAN bias diagnostics    : merged-max caveat + binned median + high-CI metrics")


v2.6.5 correction patch active:
  map resolution           : exact requested; no silent coarsening
  planning-region map      : enabled
  share SNAP grid          : EPSG:4326 native-nearest nominal resolution
  center-binning striping  : removed from share export
  scene lake dropdown      : compact scene IDs only
  spectra click map        : enabled
  CyAN bias diagnostics    : merged-max caveat + binned median + high-CI metrics


In [34]:
# =============================================================================
# v2.6.7 PRODUCTION-EFFICIENT MAPPING PATCH
# Native compact OLCI pixels remain the quantitative source of truth.
# Mapped products use a same-lake nearest-native remap on an exact nominal grid:
# no OCSSW l2bin/l3mapgen area weighting, no value interpolation, no gap filling.
# =============================================================================

V267_SCHEMA = "global-efficient-v2.6.7-efficient-production"
V267_GRID_METHOD = "epsg4326_same_lake_native_nearest_v2"
V267_DISPLAY_METHOD = "webmercator_same_lake_native_nearest_v2"
V267_EXPORT_CHUNK = 128
V267_NEAREST_RADIUS_M = 600.0


def _scene_analysis_lakes_v267(scene_id, frame, clip_geom=None):
    """Return the exact scene lake polygons used as mapped support.

    The shoreline erosion stored in the compact-product metadata is reapplied so the
    mapped product does not expand back into shoreline pixels intentionally removed
    during native-pixel extraction.
    """
    ids = pd.to_numeric(frame.get("Hylak_id", pd.Series(dtype=float)), errors="coerce").dropna().astype(np.int64).unique()
    if not len(ids):
        raise ValueError("No Hylak_id values are available in the selected scene/scope")
    lakes = _lake_boundaries_for_ids(ids)
    if lakes is None or lakes.empty:
        raise RuntimeError("HydroLAKES polygons for the stored scene lakes are unavailable; reload the target HydroLAKES universe before mapping/export.")
    lakes = lakes.to_crs(4326).copy()
    qa = _compact_qa_metadata(scene_id)
    shore = float(qa.get("shore_buffer_m", DEFAULT_SHORE_BUFFER_M) or 0.0)
    if shore > 0:
        lakes = erode_lake_geometries(lakes, shore)
    if clip_geom is not None and not getattr(clip_geom, "is_empty", True):
        cg = clip_geom
        try:
            # Planning geometries are stored in EPSG:4326 in this app.
            lakes["geometry"] = lakes.geometry.intersection(cg)
            lakes = lakes[~lakes.geometry.is_empty].copy()
        except Exception:
            pass
    if lakes.empty:
        raise ValueError("No mapped lake polygons remain in the selected display/export scope")
    return lakes


def _dedupe_target_assignments_v267(rows, cols, src, dist, nx):
    """Keep one source per target cell, preferring a real match and then shortest distance."""
    rows = np.asarray(rows, np.int64); cols = np.asarray(cols, np.int64)
    src = np.asarray(src, np.int64); dist = np.asarray(dist, float)
    if not len(rows):
        return rows, cols, src, dist
    flat = rows * int(nx) + cols
    # Unmatched support cells sort behind matched cells. Among matches choose nearest.
    rank = np.where(src >= 0, dist, np.inf)
    order = np.lexsort((rank, flat))
    flat_o = flat[order]
    keep = np.r_[True, flat_o[1:] != flat_o[:-1]]
    take = order[keep]
    return rows[take], cols[take], src[take], dist[take]


def _native_assignments_geographic_v267(frame, g, scene_id, clip_geom=None, radius_m=None):
    """Generate target lake cells and same-lake nearest-native source assignments.

    A target cell is never allowed to borrow an observation from a different Hylak_id.
    The nearest source is chosen before QA is inspected. Thus a cloud/invalid native
    observation remains no-data rather than being replaced by a farther clear pixel.
    """
    radius_m = max(float(radius_m or V267_NEAREST_RADIUS_M), 2.0 * float(g["resolution_m"]))
    lat = pd.to_numeric(frame.latitude, errors="coerce").to_numpy(float)
    lon = pd.to_numeric(frame.longitude, errors="coerce").to_numpy(float)
    hid = pd.to_numeric(frame.Hylak_id, errors="coerce").fillna(-1).to_numpy(np.int64)
    finite = np.isfinite(lat) & np.isfinite(lon)
    lakes = _scene_analysis_lakes_v267(scene_id, frame, clip_geom=clip_geom)

    all_r=[]; all_c=[]; all_s=[]; all_d=[]
    lats_grid=np.asarray(g["latitude"],float); lons_grid=np.asarray(g["longitude"],float)
    for _, lr in lakes.iterrows():
        lid=int(lr.Hylak_id); geom=lr.geometry
        src_idx=np.flatnonzero(finite & (hid==lid))
        if not len(src_idx) or geom is None or geom.is_empty:
            continue
        tree=cKDTree(_unit_sphere_xyz(lat[src_idx],lon[src_idx]))
        west,south,east,north=map(float,geom.bounds)
        c0=max(0,int(np.searchsorted(lons_grid,west,side="left")-1)); c1=min(g["nx"],int(np.searchsorted(lons_grid,east,side="right")+1))
        r0=max(0,int(np.searchsorted(lats_grid,south,side="left")-1)); r1=min(g["ny"],int(np.searchsorted(lats_grid,north,side="right")+1))
        if c1<=c0 or r1<=r0: continue
        width=max(1,c1-c0); block_rows=max(1,min(r1-r0,int(max(1,350000//width))))
        for rb0 in range(r0,r1,block_rows):
            rb1=min(rb0+block_rows,r1)
            lon2,lat2=np.meshgrid(lons_grid[c0:c1],lats_grid[rb0:rb1])
            support=contains_xy(geom,lon2,lat2)
            pos=np.flatnonzero(support.ravel())
            if not len(pos): continue
            rr_local,cc_local=np.unravel_index(pos,support.shape)
            rr=rr_local.astype(np.int64)+rb0; cc=cc_local.astype(np.int64)+c0
            xyz=_unit_sphere_xyz(lat2.ravel()[pos],lon2.ravel()[pos])
            chord,ii=tree.query(xyz,k=1,workers=-1)
            dist=6371000.0*(2.0*np.arcsin(np.minimum(1.0,np.asarray(chord,float)/2.0)))
            matched=np.isfinite(dist)&(dist<=radius_m)&(np.asarray(ii)<len(src_idx))
            src=np.full(len(pos),-1,np.int64); dd=np.full(len(pos),np.nan,float)
            if matched.any():
                src[matched]=src_idx[np.asarray(ii[matched],np.int64)]
                dd[matched]=dist[matched]
            all_r.append(rr); all_c.append(cc); all_s.append(src); all_d.append(dd)
    if not all_r:
        return (np.empty(0,np.int64),)*3 + (np.empty(0,float),)
    rows=np.concatenate(all_r); cols=np.concatenate(all_c); src=np.concatenate(all_s); dist=np.concatenate(all_d)
    return _dedupe_target_assignments_v267(rows,cols,src,dist,g["nx"])


def _regularize_dataframe_geographic(frame, columns, resolution_m, scene_id=None, clip_geom=None):
    """Exact nominal-resolution EPSG:4326 remap constrained by the same HydroLAKES polygon."""
    if scene_id is None:
        # Infer from current completed-scene context only when called by legacy helpers.
        scene_id = str(frame.get("scene_id", pd.Series([""])).iloc[0]) if isinstance(frame,pd.DataFrame) and "scene_id" in frame.columns else None
    if not scene_id:
        # Existing quick-export paths pass scene_id separately and do not use this fallback.
        raise ValueError("scene_id is required for same-lake geographic remapping")
    g=_science_geographic_setup_v265(frame,float(resolution_m))
    rows,cols,src,dist=_native_assignments_geographic_v267(frame,g,scene_id,clip_geom=clip_geom)
    out={c:np.full((g["ny"],g["nx"]),np.nan,np.float32) for c in columns}
    matched=src>=0
    for c in columns:
        if c not in frame.columns or not matched.any(): continue
        vals=pd.to_numeric(frame[c],errors="coerce").to_numpy(float)
        good=matched & np.isfinite(vals[np.maximum(src,0)])
        if good.any(): out[c][rows[good],cols[good]]=vals[src[good]].astype(np.float32)
    g.update({"mapping_method":V267_GRID_METHOD,"target_lake_cells":int(len(rows)),"matched_target_cells":int(matched.sum()),"nearest_radius_m":max(V267_NEAREST_RADIUS_M,2.0*float(g["resolution_m"]))})
    return out,g


def _native_assignments_mercator_v267(frame, g, scene_id, clip_geom=None, radius_m=None):
    """Same-lake nearest-native assignments for the Leaflet/Web-Mercator display grid."""
    radius_ground=max(float(radius_m or V267_NEAREST_RADIUS_M),2.0*float(g["resolution_m"]))
    lat=pd.to_numeric(frame.latitude,errors="coerce").to_numpy(float)
    lon=pd.to_numeric(frame.longitude,errors="coerce").to_numpy(float)
    hid=pd.to_numeric(frame.Hylak_id,errors="coerce").fillna(-1).to_numpy(np.int64)
    finite=np.isfinite(lat)&np.isfinite(lon)&(np.abs(lat)<85)
    x=np.full(len(frame),np.nan,float); y=np.full(len(frame),np.nan,float)
    if finite.any():
        xx,yy=_TO_WEBMERC.transform(lon[finite],lat[finite]); x[finite]=xx; y[finite]=yy
    lakes=_scene_analysis_lakes_v267(scene_id,frame,clip_geom=clip_geom).to_crs(3857)
    all_r=[];all_c=[];all_s=[];all_d=[]
    xs=g["xmin"]+(np.arange(g["nx"],dtype=float)+0.5)*g["resolution_m"]
    for _,lr in lakes.iterrows():
        lid=int(lr.Hylak_id); geom=lr.geometry; src_idx=np.flatnonzero(finite&(hid==lid))
        if not len(src_idx) or geom is None or geom.is_empty: continue
        tree=cKDTree(np.column_stack([x[src_idx],y[src_idx]]))
        lat_med=float(np.nanmedian(lat[src_idx])); scale=1.0/max(math.cos(math.radians(lat_med)),0.20)
        radius_web=radius_ground*scale
        minx,miny,maxx,maxy=map(float,geom.bounds)
        c0=max(0,int(math.floor((minx-g["xmin"])/g["resolution_m"]))-1); c1=min(g["nx"],int(math.ceil((maxx-g["xmin"])/g["resolution_m"]))+1)
        r0=max(0,int(math.floor((g["ymax"]-maxy)/g["resolution_m"]))-1); r1=min(g["ny"],int(math.ceil((g["ymax"]-miny)/g["resolution_m"]))+1)
        if c1<=c0 or r1<=r0: continue
        width=max(1,c1-c0); block_rows=max(1,min(r1-r0,int(max(1,350000//width))))
        for rb0 in range(r0,r1,block_rows):
            rb1=min(rb0+block_rows,r1)
            ys=g["ymax"]-(np.arange(rb0,rb1,dtype=float)+0.5)*g["resolution_m"]
            xx2,yy2=np.meshgrid(xs[c0:c1],ys)
            support=contains_xy(geom,xx2,yy2); pos=np.flatnonzero(support.ravel())
            if not len(pos): continue
            rr_local,cc_local=np.unravel_index(pos,support.shape); rr=rr_local.astype(np.int64)+rb0; cc=cc_local.astype(np.int64)+c0
            dist,ii=tree.query(np.column_stack([xx2.ravel()[pos],yy2.ravel()[pos]]),k=1,workers=-1)
            matched=np.isfinite(dist)&(dist<=radius_web)&(np.asarray(ii)<len(src_idx))
            src=np.full(len(pos),-1,np.int64); dd=np.full(len(pos),np.nan,float)
            if matched.any():
                src[matched]=src_idx[np.asarray(ii[matched],np.int64)]
                # Convert Web-Mercator distance back to approximate local ground distance.
                dd[matched]=np.asarray(dist,float)[matched]/scale
            all_r.append(rr);all_c.append(cc);all_s.append(src);all_d.append(dd)
    if not all_r:
        return (np.empty(0,np.int64),)*3+(np.empty(0,float),)
    return _dedupe_target_assignments_v267(np.concatenate(all_r),np.concatenate(all_c),np.concatenate(all_s),np.concatenate(all_d),g["nx"])


def _regularize_dataframe_mercator(frame, columns, resolution_m, scene_id=None, clip_geom=None):
    """Exact requested display grid with same-lake nearest-native assignment."""
    if not scene_id:
        raise ValueError("scene_id is required for same-lake Web-Mercator remapping")
    lat=pd.to_numeric(frame.latitude,errors="coerce").to_numpy(float); lon=pd.to_numeric(frame.longitude,errors="coerce").to_numpy(float)
    g=_grid_geometry_mercator(lat,lon,float(resolution_m),max_cells=V265_DISPLAY_MAX_CELLS)
    rows,cols,src,dist=_native_assignments_mercator_v267(frame,g,scene_id,clip_geom=clip_geom)
    out={c:np.full((g["ny"],g["nx"]),np.nan,np.float32) for c in columns}; matched=src>=0
    for c in columns:
        if c not in frame.columns or not matched.any(): continue
        vals=pd.to_numeric(frame[c],errors="coerce").to_numpy(float)
        good=matched & np.isfinite(vals[np.maximum(src,0)])
        if good.any(): out[c][rows[good],cols[good]]=vals[src[good]].astype(np.float32)
    g.update({"mapping_method":V267_DISPLAY_METHOD,"target_lake_cells":int(len(rows)),"matched_target_cells":int(matched.sum()),"nearest_radius_m":max(V267_NEAREST_RADIUS_M,2.0*float(g["resolution_m"]))})
    return out,g


def _build_map(scene_id, scope, lake_id, metric, resolution_m, basemap, labels, cmap, reverse, vmin_text, vmax_text, show_lakes=True):
    if not scene_id: raise ValueError("Choose a completed scene")
    scope=str(scope or "scene"); planning_geom=None
    if scope=="lake":
        raw=_load_scene_pixels(scene_id,"lake",lake_id); scope_label=f"lake {lake_id}"
    else:
        raw=_load_scene_pixels(scene_id,"scene",None)
        if scope=="planning":
            planning_geom=GLOBAL_STATE.get("planning_geometry")
            raw=_planning_geometry_frame(raw); scope_label=GLOBAL_STATE.get("planning_shard") or "active planning region"
        else: scope_label="entire processed scene"
    cols=["rhos_665","rhos_560","rhos_490"] if metric=="true_color" else [metric]
    frame=_masked_display_frame(raw,cols)
    grids,g=_regularize_dataframe_mercator(frame,cols,float(resolution_m),scene_id=scene_id,clip_geom=planning_geom)
    bounds=[[g["south"],g["west"]],[g["north"],g["east"]]]
    m=folium.Map(tiles=None,control_scale=True,prefer_canvas=True); _add_basemap(m,basemap,bool(labels))
    if metric=="true_color":
        folium.raster_layers.ImageOverlay(_truecolor_data_url(grids),bounds=bounds,opacity=.95,name="True color · bloom-aware QA").add_to(m)
    else:
        img,vmin,vmax,cm=_rgba_data_url(grids[metric],metric,cmap,bool(reverse),_optional_float(vmin_text),_optional_float(vmax_text)); folium.raster_layers.ImageOverlay(img,bounds=bounds,opacity=.88,name=f"{metric} · bloom-aware QA").add_to(m); _add_scalar_legend(m,metric,cmap,bool(reverse),vmin,vmax)
    if show_lakes:
        b=_lake_boundaries_for_ids(raw.Hylak_id.unique())
        if not b.empty: folium.GeoJson(json.loads(b.to_crs(4326).to_json()),name="HydroLAKES boundaries",style_function=lambda f:{"color":"#00e5ff","weight":1.2,"fillOpacity":0}).add_to(m)
    if planning_geom is not None and not planning_geom.is_empty:
        reg=gpd.GeoDataFrame({"name":[scope_label]},geometry=[planning_geom],crs=4326); folium.GeoJson(json.loads(reg.to_json()),name="Active planning region",style_function=lambda f:{"color":"#ffcc00","weight":2.2,"fillOpacity":0}).add_to(m); west,south,east,north=planning_geom.bounds; fit_bounds=[[south,west],[north,east]]
    else: fit_bounds=bounds
    folium_plugins.Fullscreen(position="topleft").add_to(m); folium_plugins.MousePosition(position="bottomright",separator=" | ",prefix="Lat / Lon",num_digits=5).add_to(m); folium_plugins.MeasureControl(position="topleft",primary_length_unit="kilometers").add_to(m); folium.LayerControl(collapsed=False).add_to(m); m.fit_bounds(fit_bounds)
    hover_frame=frame.loc[_primary_selector(frame)].copy(); _inject_hover_js(m,hover_frame,metric,g["resolution_m"])
    out=HTML_DIR/f"scene_{scene_id}_{scope}_{lake_id or 'ALL'}_{metric}_{int(round(g['resolution_m']))}m.html"; m.save(out); map_src=f"/gradio_api/file={quote(str(out),safe='/')}?v={out.stat().st_mtime_ns}"; iframe=("<iframe style='width:100%;height:740px;border:1px solid #ccd5df;border-radius:8px;background:white' loading='eager' src='"+map_src+"'></iframe>")
    return iframe,str(out),frame,grids,g


def export_snap_gridded_netcdf(scene_id, scope, lake_id, metric, frame, resolution_m):
    """Fast displayed-scope EPSG:4326 export using same-lake nearest-native values."""
    if frame is None or len(frame)==0: raise ValueError("No displayed pixels are available for quick export")
    names=["rhos_665","rhos_560","rhos_490"] if metric=="true_color" else [metric]
    g=_science_geographic_setup_v265(frame,float(resolution_m)); ncell=int(g["ncell"])
    if ncell>35_000_000: raise RuntimeError(f"Exact {g['resolution_m']:.0f} m quick export would require {ncell/1e6:.1f} million cells. Use the planning-region/selected-lake scope, or the sparse 3-file share exporter.")
    clip_geom=GLOBAL_STATE.get("planning_geometry") if str(scope)=="planning" else None
    rows,cols,src,dist=_native_assignments_geographic_v267(frame,g,scene_id,clip_geom=clip_geom)
    matched=src>=0; source_values={n:pd.to_numeric(frame[n],errors="coerce").to_numpy(float) for n in names if n in frame.columns}
    SNAP_DIR=EXPORT_DIR/"snap_gridded"; SNAP_DIR.mkdir(parents=True,exist_ok=True); out=SNAP_DIR/f"S3_OLCI_{scene_id}_{scope}_{lake_id or 'ALL'}_{metric}_{int(round(g['resolution_m']))}m_CF.nc"
    from pyproj import CRS
    wkt=CRS.from_epsg(4326).to_wkt(); chunk=128; chunks=(min(chunk,g["ny"]),min(chunk,g["nx"]))
    with netCDF4.Dataset(out,"w",format="NETCDF4_CLASSIC") as ds:
        ds.createDimension("latitude",g["ny"]); ds.createDimension("longitude",g["nx"]); ds.setncatts({"Conventions":"CF-1.8","title":"Sentinel-3 OLCI displayed-scope same-lake native-nearest export","scene_id":str(scene_id),"scope":str(scope),"nominal_grid_resolution_m":float(g["resolution_m"]),"science_grid_method":V267_GRID_METHOD,"nearest_native_radius_m":max(V267_NEAREST_RADIUS_M,2.0*float(g["resolution_m"])),"resampling":"same-lake nearest-native assignment; no averaging/value interpolation or gap filling","qa_basis":"primary bloom-aware mask already encoded in displayed science values"})
        y=ds.createVariable("latitude","f8",("latitude",));y[:]=g["latitude"];y.setncatts({"standard_name":"latitude","units":"degrees_north","axis":"Y"});x=ds.createVariable("longitude","f8",("longitude",));x[:]=g["longitude"];x.setncatts({"standard_name":"longitude","units":"degrees_east","axis":"X"});crs=ds.createVariable("crs","i4");crs.assignValue(0);crs.setncatts({"grid_mapping_name":"latitude_longitude","epsg_code":"EPSG:4326","spatial_ref":wkt,"crs_wkt":wkt,"semi_major_axis":6378137.0,"inverse_flattening":298.257223563})
        vars_={}
        for n in source_values:
            on="rhos_884" if n=="rhos_885" else n;v=ds.createVariable(on,"f4",("latitude","longitude"),zlib=True,complevel=5,shuffle=True,chunksizes=chunks,fill_value=np.float32(-9999));v.setncatts({"coordinates":"latitude longitude","grid_mapping":"crs","units":"1","long_name":on});vars_[n]=v
        if len(rows):
            nxc=int(math.ceil(g["nx"]/chunk));cid=(rows//chunk)*nxc+(cols//chunk);order=np.argsort(cid,kind="mergesort");cids=cid[order];cuts=np.flatnonzero(np.r_[True,cids[1:]!=cids[:-1],True])
            for a,b in zip(cuts[:-1],cuts[1:]):
                pos=order[a:b];r0=int((rows[pos[0]]//chunk)*chunk);c0=int((cols[pos[0]]//chunk)*chunk);r1=min(r0+chunk,g["ny"]);c1=min(c0+chunk,g["nx"]);rr=rows[pos]-r0;cc=cols[pos]-c0
                for n,v in vars_.items():
                    block=np.full((r1-r0,c1-c0),np.float32(-9999),np.float32);good=(src[pos]>=0)
                    if good.any():
                        vals=source_values[n][src[pos][good]];ok=np.isfinite(vals);block[rr[good][ok],cc[good][ok]]=vals[ok].astype(np.float32)
                    v[r0:r1,c0:c1]=np.ma.masked_equal(block,np.float32(-9999))
        if metric=="true_color":ds.setncattr("true_color_mapping","red=rhos_665; green=rhos_560; blue=rhos_490")
    return str(out)


def export_complete_scene_science_netcdf(scene_id, requested_resolution_m=SCIENCE_EXPORT_DEFAULT_RESOLUTION_M, force=False):
    """Production share raster: exact nominal grid, same-lake nearest-native remap, sparse writes."""
    if not scene_id: raise ValueError("Choose a completed scene")
    frame=_load_scene_pixels(scene_id,"scene",None).copy();g=_science_geographic_setup_v265(frame,float(requested_resolution_m));export_dir=EXPORT_DIR/"snap_complete";export_dir.mkdir(parents=True,exist_ok=True);out=export_dir/f"S3_OLCI_{scene_id}_ALL_SCIENCE_{int(round(g['resolution_m']))}m_CF.nc";source_files=_scene_compact_files(scene_id);newest=max((p.stat().st_mtime_ns for p in source_files),default=0)
    if out.exists() and not force and out.stat().st_mtime_ns>=newest:
        with contextlib.suppress(Exception):
            with netCDF4.Dataset(out) as old:
                if getattr(old,"science_grid_method","")==V267_GRID_METHOD and float(getattr(old,"nominal_grid_resolution_m",-1))==float(g["resolution_m"]): return str(out),g
    rows,cols,src,dist=_native_assignments_geographic_v267(frame,g,scene_id);matched=src>=0;primary=_primary_selector(frame);qa=_compact_qa_metadata(scene_id);source_values={n:pd.to_numeric(frame[n],errors="coerce").to_numpy(float) for n in SCIENCE_EXPORT_VARIABLES if n in frame.columns};source_hylak=pd.to_numeric(frame.Hylak_id,errors="coerce").fillna(-1).to_numpy(np.int32)
    mask_input_names=["valid_water_mask","cyan_strict_valid_mask","bloom_rescue_mask","cldice_mask","hisatzen_mask","navfail_mask","cloud_excluded_mask","ci_valid_mask","ci_candidate_mask","ci_detection_mask","mph_valid_mask","fai_valid_mask"]
    source_masks={n:pd.to_numeric(frame[n],errors="coerce").fillna(0).to_numpy(float)>0 for n in mask_input_names if n in frame.columns}
    mask_output_name={"cyan_strict_valid_mask":"generic_flag_exclusion_sensitivity_mask"}
    tmp=out.with_suffix(".nc.tmp");tmp.unlink(missing_ok=True);chunk=int(V267_EXPORT_CHUNK)
    with netCDF4.Dataset(tmp,"w",format="NETCDF4_CLASSIC") as ds:
        ds.createDimension("latitude",g["ny"]);ds.createDimension("longitude",g["nx"]);ds.setncatts({"Conventions":"CF-1.8","title":"Sentinel-3 OLCI HydroLAKES bloom-aware all-science efficient 300 m product","scene_id":str(scene_id),"science_grid_method":V267_GRID_METHOD,"nominal_grid_resolution_m":float(g["resolution_m"]),"actual_grid_resolution_m":float(g["resolution_m"]),"cell_height_m_nominal":float(g["resolution_m"]),"cell_width_m_at_reference_latitude":float(g["resolution_m"]),"reference_latitude_for_longitude_spacing":float(g["reference_latitude"]),"geospatial_lat_min":float(g["south"]),"geospatial_lat_max":float(g["north"]),"geospatial_lon_min":float(g["west"]),"geospatial_lon_max":float(g["east"]),"latitude_step_degrees":float(g["dlat"]),"longitude_step_degrees":float(g["dlon"]),"nearest_native_radius_m":max(V267_NEAREST_RADIUS_M,2.0*float(g["resolution_m"])),"mapping":"each target lake cell maps only to the nearest stored native OLCI observation from the same Hylak_id","qa_rule":"nearest same-lake source is chosen before QA; if that native source is not primary-valid, science value remains fill/NaN","resampling":"nearest-native assignment; no averaging/value interpolation or gap filling","striping_policy":"target lake support is populated from same-lake nearest native observations instead of center-only binning","primary_qa_mask":"valid_water_mask (bloom-aware)","generic_CLDICE_sensitivity":"diagnostic only; not claimed equivalent to NASA CyAN upstream cloud processing","NASA_area_weighting":"not used in production v2.6.7; retained as optional validation concept only","statistics_source":"native compact OLCI lake observations, not mapped raster","true_color_mapping":"red=rhos_665; green=rhos_560; blue=rhos_490","CI_cyano_detection_limit":float(CYAN_CI_DETECTION_LIMIT),"qa_policy_json":json.dumps(qa,default=str)})
        yv=ds.createVariable("latitude","f8",("latitude",));yv[:]=g["latitude"];yv.setncatts({"standard_name":"latitude","units":"degrees_north","axis":"Y"});xv=ds.createVariable("longitude","f8",("longitude",));xv[:]=g["longitude"];xv.setncatts({"standard_name":"longitude","units":"degrees_east","axis":"X"});from pyproj import CRS;wkt4326=CRS.from_epsg(4326).to_wkt();crsv=ds.createVariable("crs","i4");crsv.assignValue(0);crsv.setncatts({"grid_mapping_name":"latitude_longitude","epsg_code":"EPSG:4326","spatial_ref":wkt4326,"crs_wkt":wkt4326,"semi_major_axis":6378137.0,"inverse_flattening":298.257223563})
        chunks=(min(chunk,g["ny"]),min(chunk,g["nx"]));vars_float={};long_names={"CI":"Cyanobacteria Index candidate magnitude","CI_cyano":"cyanobacteria index detections","NDCI":"Normalized Difference Chlorophyll Index","MPH":"Maximum Peak Height","FAI":"OLCI-adapted Floating Algae Index (AFAI form)"}
        for sn,on in [(n,"rhos_884" if n=="rhos_885" else n) for n in SCIENCE_EXPORT_VARIABLES if n in source_values]:
            v=ds.createVariable(on,"f4",("latitude","longitude"),zlib=True,complevel=6,shuffle=True,chunksizes=chunks,fill_value=np.float32(-9999));v.setncatts({"coordinates":"latitude longitude","grid_mapping":"crs","units":"1","long_name":long_names.get(on,f"Rayleigh-corrected OLCI reflectance {on}"),"qa_basis":"primary bloom-aware valid_water_mask"});vars_float[sn]=v
        vars_mask={}
        for sn in source_masks:
            on=mask_output_name.get(sn,sn);v=ds.createVariable(on,"i1",("latitude","longitude"),zlib=True,complevel=6,shuffle=True,chunksizes=chunks,fill_value=np.int8(-1));v.setncatts({"coordinates":"latitude longitude","grid_mapping":"crs","long_name":on.replace("_"," "),"flag_values":np.array([0,1],dtype=np.int8),"flag_meanings":"false true","valid_range":np.array([0,1],dtype=np.int8)});vars_mask[sn]=v
        support_var=ds.createVariable("lake_support_mask","i1",("latitude","longitude"),zlib=True,complevel=5,shuffle=True,chunksizes=chunks,fill_value=np.int8(-1));support_var.setncatts({"coordinates":"latitude longitude","grid_mapping":"crs","long_name":"target cell lies inside the analysis HydroLAKES polygon after shoreline buffer","flag_values":np.array([0,1],dtype=np.int8),"flag_meanings":"false true"})
        match_var=ds.createVariable("native_source_match_mask","i1",("latitude","longitude"),zlib=True,complevel=5,shuffle=True,chunksizes=chunks,fill_value=np.int8(-1));match_var.setncatts({"coordinates":"latitude longitude","grid_mapping":"crs","long_name":"target lake cell has a same-lake native OLCI source within search radius","flag_values":np.array([0,1],dtype=np.int8),"flag_meanings":"false true"})
        dist_var=ds.createVariable("native_match_distance_m","f4",("latitude","longitude"),zlib=True,complevel=5,shuffle=True,chunksizes=chunks,fill_value=np.float32(-9999));dist_var.setncatts({"coordinates":"latitude longitude","grid_mapping":"crs","units":"m","long_name":"distance to nearest same-lake stored native OLCI observation"});hylak_var=ds.createVariable("nearest_Hylak_id","i4",("latitude","longitude"),zlib=True,complevel=5,shuffle=True,chunksizes=chunks,fill_value=np.int32(-1));hylak_var.setncatts({"coordinates":"latitude longitude","grid_mapping":"crs","long_name":"HydroLAKES identifier of same-lake native source"})
        if len(rows):
            nxc=int(math.ceil(g["nx"]/chunk));cid=(rows//chunk)*nxc+(cols//chunk);order=np.argsort(cid,kind="mergesort");cids=cid[order];cuts=np.flatnonzero(np.r_[True,cids[1:]!=cids[:-1],True])
            for a,b in zip(cuts[:-1],cuts[1:]):
                pos=order[a:b];r0=int((rows[pos[0]]//chunk)*chunk);c0=int((cols[pos[0]]//chunk)*chunk);r1=min(r0+chunk,g["ny"]);c1=min(c0+chunk,g["nx"]);rr=rows[pos]-r0;cc=cols[pos]-c0;spos=src[pos];m=spos>=0
                sb=np.full((r1-r0,c1-c0),np.int8(-1),np.int8);sb[rr,cc]=1;support_var[r0:r1,c0:c1]=np.ma.masked_equal(sb,np.int8(-1));mb=np.full_like(sb,np.int8(-1));mb[rr,cc]=m.astype(np.int8);match_var[r0:r1,c0:c1]=np.ma.masked_equal(mb,np.int8(-1));db=np.full((r1-r0,c1-c0),np.float32(-9999),np.float32);valid_d=m&np.isfinite(dist[pos]);db[rr[valid_d],cc[valid_d]]=dist[pos][valid_d].astype(np.float32);dist_var[r0:r1,c0:c1]=np.ma.masked_equal(db,np.float32(-9999));hb=np.full((r1-r0,c1-c0),np.int32(-1),np.int32);hb[rr[m],cc[m]]=source_hylak[spos[m]];hylak_var[r0:r1,c0:c1]=np.ma.masked_equal(hb,np.int32(-1))
                for sn,v in vars_float.items():
                    block=np.full((r1-r0,c1-c0),np.float32(-9999),np.float32)
                    if m.any():
                        vv=source_values[sn][spos[m]];ok=np.isfinite(vv)&primary[spos[m]];block[rr[m][ok],cc[m][ok]]=vv[ok].astype(np.float32)
                    v[r0:r1,c0:c1]=np.ma.masked_equal(block,np.float32(-9999))
                for sn,v in vars_mask.items():
                    block=np.full((r1-r0,c1-c0),np.int8(-1),np.int8)
                    if m.any(): block[rr[m],cc[m]]=source_masks[sn][spos[m]].astype(np.int8)
                    v[r0:r1,c0:c1]=np.ma.masked_equal(block,np.int8(-1))
    tmp.replace(out);g.update({"science_grid_method":V267_GRID_METHOD,"target_lake_cells":int(len(rows)),"matched_target_cells":int(matched.sum()),"nearest_radius_m":max(V267_NEAREST_RADIUS_M,2.0*float(g["resolution_m"]))});return str(out),g


def _share_all_index_stats(scene_id,cfg,row):
    """Lean native-observation statistics; generic CLDICE exclusion is counts-only sensitivity QA."""
    frame=_load_scene_pixels(scene_id,"scene",None).copy()
    if frame.empty: raise ValueError("No compact lake pixels found for selected scene")
    for c in ("latitude","longitude","Hylak_id"): frame[c]=pd.to_numeric(frame[c],errors="coerce")
    frame=frame.dropna(subset=["latitude","longitude","Hylak_id"]).copy();frame["_lat6"]=frame.latitude.round(6);frame["_lon6"]=frame.longitude.round(6);frame=frame.drop_duplicates(["Hylak_id","_lat6","_lon6"]).drop(columns=["_lat6","_lon6"])
    lakes=GLOBAL_STATE.get("lakes");lake_meta={}
    if lakes is not None and not lakes.empty:
        for _,lr in lakes[lakes.Hylak_id.astype(int).isin(frame.Hylak_id.astype(int).unique())].iterrows():lake_meta[int(lr.Hylak_id)]={"Lake_name":lr.get("Lake_name","") or "","Lake_area_km2":float(lr.get("Lake_area",np.nan))}
    def mask(g,n):return pd.to_numeric(g.get(n,pd.Series(np.zeros(len(g)),index=g.index)),errors="coerce").fillna(0).to_numpy(float)>0
    def stat(s):
        a=pd.to_numeric(s,errors="coerce").to_numpy(float);a=a[np.isfinite(a)]
        if not len(a):return {"valid_pixels":0,"mean":np.nan,"median":np.nan,"p10":np.nan,"p90":np.nan,"max":np.nan}
        return {"valid_pixels":int(len(a)),"mean":float(np.mean(a)),"median":float(np.median(a)),"p10":float(np.percentile(a,10)),"p90":float(np.percentile(a,90)),"max":float(np.max(a))}
    recs=[]
    for lid,gdf in frame.groupby(frame.Hylak_id.astype(int),sort=True):
        primary=mask(gdf,"valid_water_mask");sens=mask(gdf,"cyan_strict_valid_mask");rescue=mask(gdf,"bloom_rescue_mask");cld=mask(gdf,"cldice_mask");cloud_ex=mask(gdf,"cloud_excluded_mask");cand=mask(gdf,"ci_candidate_mask");det=mask(gdf,"ci_detection_mask");ci_valid=mask(gdf,"ci_valid_mask")
        rec={"scene_id":str(scene_id),"scene_name":str(row.scene_name),"acquisition_start":str(row.acquisition_start),"Hylak_id":int(lid),**lake_meta.get(int(lid),{}),"native_lake_pixels":int(len(gdf)),"primary_retained_pixels":int(primary.sum()),"generic_flag_exclusion_sensitivity_pixels":int(sens.sum()),"bloom_rescued_pixels":int(rescue.sum()),"CLDICE_pixels":int(cld.sum()),"cloud_excluded_pixels":int(cloud_ex.sum()),"bloom_rescue_fraction_of_CLDICE":float(rescue.sum()/cld.sum()) if cld.sum() else np.nan,"ci_input_valid_pixels":int(ci_valid.sum()),"ci_candidate_pixels":int(cand.sum()),"ci_cyano_detection_pixels":int(det.sum())}
        bases={"CI":cand,"CI_cyano":det,"NDCI":primary,"MPH":primary,"FAI":primary}
        for metric,basis in bases.items():
            s=stat(gdf.loc[basis,metric] if metric in gdf else pd.Series(dtype=float))
            for k,v in s.items():rec[f"{metric}_{k}"]=v
        recs.append(rec)
    table=pd.DataFrame(recs).sort_values("Hylak_id").reset_index(drop=True);d=EXPORT_DIR/"scene_bundles";d.mkdir(parents=True,exist_ok=True);p=d/f"S3_OLCI_{scene_id}_ALL_INDEX_STATS.csv";table.to_csv(p,index=False);return p,table


def create_scene_share_bundle(scene_id,science_resolution_m=SCIENCE_EXPORT_DEFAULT_RESOLUTION_M):
    """Exactly three files from the efficient production branch."""
    if not scene_id: raise ValueError("Choose a completed scene")
    t=registry_table(GLOBAL_STATE.get("config_hash"));r=t[t.scene_id.astype(str)==str(scene_id)]
    if r.empty: raise KeyError(scene_id)
    row=r.iloc[0];cfg=str(row.config_hash);d=EXPORT_DIR/"scene_bundles";d.mkdir(parents=True,exist_ok=True);complete_nc,g=export_complete_scene_science_netcdf(scene_id,float(science_resolution_m),force=False);stats_csv,stats_table=_share_all_index_stats(scene_id,cfg,row);qa=_compact_qa_metadata(scene_id);frame_meta=parse_s3_frame_metadata(row.scene_name)
    meta={"schema":"global-efficient-share-v2.6.7","created_utc":utc_now(),"scene_id":str(scene_id),"scene_name":str(row.scene_name),"acquisition_start":str(row.acquisition_start),"config_hash":cfg,**frame_meta,"files":{"snap_all_science_netcdf":Path(complete_nc).name,"all_index_stats_csv":Path(stats_csv).name},"snap_variables":["rhos_490","rhos_560","rhos_620","rhos_665","rhos_681","rhos_709","rhos_754","rhos_865","rhos_884","CI","CI_cyano","NDCI","MPH","FAI","valid_water_mask","generic_flag_exclusion_sensitivity_mask","bloom_rescue_mask","cldice_mask","hisatzen_mask","lake_support_mask","native_source_match_mask","native_match_distance_m","nearest_Hylak_id"],"snap_grid":{"projection":"EPSG:4326 geographic","grid_method":V267_GRID_METHOD,"nominal_resolution_m":float(g["resolution_m"]),"reference_latitude":float(g["reference_latitude"]),"latitude_step_degrees":float(g["dlat"]),"longitude_step_degrees":float(g["dlon"]),"nearest_radius_m":float(g.get("nearest_radius_m",max(V267_NEAREST_RADIUS_M,2.0*float(g["resolution_m"])))),"mapping":"same-lake nearest native observation","interpolation":"none","gap_fill":"none","area_weighting":"not used in production"},"masking":qa,"strict_mask_policy":"Generic LAND+CLDICE+HISATZEN-style exclusion is retained only as a sensitivity mask/count; it is not called CyAN-equivalent because NASA upstream cloud processing differs.","statistics":{"source":"native compact OLCI observations","mapped_raster_not_used_for_statistics":True},"future_daily_composite":"S3A/S3B daily CIcyano maximum is intentionally not implemented yet; add only after this efficient mapping branch is validated.","important":"The mapped raster is for visualization/GIS/SNAP interoperability. Quantitative lake statistics remain native-observation based."}
    mp=d/f"S3_OLCI_{scene_id}_METADATA.json";mp.write_text(json.dumps(meta,indent=2,default=str));z=d/f"{str(row.scene_name)}_{cfg}_SHARE_MINIMAL.zip"
    with zipfile.ZipFile(z,"w",zipfile.ZIP_DEFLATED,allowZip64=True) as arc:
        for p in (Path(complete_nc),Path(stats_csv),mp):arc.write(p,arcname=p.name)
    return str(z),_app_status("Minimal share bundle created",f"3 files · nominal {g['resolution_m']:.0f} m EPSG:4326 · same-lake native-nearest · bloom-aware QA · no area-weighting overhead")


def create_complete_scene_html(scene_id, resolution_m=1000, basemap="Esri Dark Gray", force=False):
    """Portable multi-layer HTML using the efficient same-lake mapper."""
    if not scene_id: raise ValueError("Choose a completed scene")
    frame=_load_scene_pixels(scene_id,"scene",None)
    cols=["rhos_665","rhos_560","rhos_490","CI","CI_cyano","NDCI","MPH","FAI"]
    display=_masked_display_frame(frame,cols)
    grids,g=_regularize_dataframe_mercator(display,cols,float(resolution_m),scene_id=scene_id,clip_geom=None)
    out=HTML_DIR/f"scene_{scene_id}_ALL_SCIENCE_{int(round(g['resolution_m']))}m.html"
    if out.exists() and not force:return str(out),g
    bounds=[[g["south"],g["west"]],[g["north"],g["east"]]];m=folium.Map(tiles=None,control_scale=True,prefer_canvas=True);_add_basemap(m,basemap,True)
    tc=_truecolor_data_url({k:grids[k] for k in ("rhos_665","rhos_560","rhos_490")});folium.raster_layers.ImageOverlay(tc,bounds=bounds,opacity=.95,name="True color rhos 665/560/490",show=False).add_to(m)
    legend_rows=[];palette={"CI":"cividis","CI_cyano":"cividis","NDCI":"BrBG","MPH":"plasma","FAI":"viridis"}
    for metric in ("CI","CI_cyano","NDCI","MPH","FAI"):
        img,vmin,vmax,_=_rgba_data_url(grids[metric],metric,palette[metric],False,None,None);folium.raster_layers.ImageOverlay(img,bounds=bounds,opacity=.88,name=metric,show=(metric=="CI_cyano")).add_to(m);legend_rows.append(f"<tr><td><b>{html_lib.escape(metric)}</b></td><td>{vmin:.4g}</td><td>{vmax:.4g}</td></tr>")
    b=_lake_boundaries_for_ids(frame.Hylak_id.unique())
    if not b.empty:folium.GeoJson(json.loads(b.to_crs(4326).to_json()),name="HydroLAKES boundaries",style_function=lambda f:{"color":"#00e5ff","weight":1.1,"fillOpacity":0}).add_to(m)
    panel="<div style='position:fixed;bottom:28px;left:24px;z-index:9999;background:rgba(255,255,255,.95);padding:8px 10px;border:1px solid #777;border-radius:5px;font:11px sans-serif'><b>Auto display ranges</b><table><tr><th>Layer</th><th>min</th><th>max</th></tr>"+"".join(legend_rows)+"</table><small>Use layer control at upper right.</small></div>";m.get_root().html.add_child(folium.Element(panel));folium_plugins.Fullscreen(position="topleft").add_to(m);folium_plugins.MousePosition(position="bottomright",separator=" | ",prefix="Lat / Lon",num_digits=5).add_to(m);folium_plugins.MeasureControl(position="topleft",primary_length_unit="kilometers").add_to(m);folium.LayerControl(collapsed=False).add_to(m);m.fit_bounds(bounds);_inject_all_science_hover_js(m,display,g["resolution_m"]);m.save(out);return str(out),g


print("v2.6.7 efficient-production patch active:")
print("  native compact pixels      : quantitative source of truth")
print("  mapped grid                : same-lake nearest-native, exact nominal resolution")
print("  gray-striping strategy     : no center-only binning; no seam fill/interpolation")
print("  NASA area weighting        : NOT used in production (validation only)")
print("  primary science QA         : bloom-aware")
print("  generic CLDICE strict mask : sensitivity diagnostic only")


v2.6.7 efficient-production patch active:
  native compact pixels      : quantitative source of truth
  mapped grid                : same-lake nearest-native, exact nominal resolution
  gray-striping strategy     : no center-only binning; no seam fill/interpolation
  NASA area weighting        : NOT used in production (validation only)
  primary science QA         : bloom-aware
  generic CLDICE strict mask : sensitivity diagnostic only


In [35]:

# =============================================================================
# v2.6.8 NASA CyAN COMPARISON PATCH
#
# Key correction:
#   1) For each NASA 300 m cell centre, find the nearest ORIGINAL native OLCI
#      observation from the selected HydroLAKES lake FIRST.
#   2) Only after that native pixel is selected, inspect the requested QA basis.
#   3) If that nearest native pixel fails QA, leave the comparison cell unmatched/
#      no-data. DO NOT search for a farther clear/lower-CI pixel.
#
# Default QA basis in the UI remains the primary bloom-aware science mask.
# Generic CLDICE-excluded screening is retained only as a sensitivity option.
# =============================================================================

V268_SCHEMA = "global-efficient-v2.6.8-native-first-cyan-comparison"


def compare_cyan_callback_v265(
    scene_id, lake_id, explicit_date, source_mode, uploaded_tif,
    max_distance_m, qa_basis="primary"
):
    """Compare NASA Merged-S3-CYAN to one local OLCI scene.

    Pairing semantics are intentionally native-first:
      NASA cell centre -> nearest same-lake native OLCI pixel -> inspect QA.

    The KD-tree is built from all finite stored native observations for the selected
    Hylak_id, not from a QA-prefiltered subset. Therefore a QA-rejected bright/cloudy
    pixel is never replaced by a farther "clear" pixel.
    """
    try:
        if not scene_id or not lake_id:
            raise ValueError("Choose a completed scene and lake")

        date = _scene_utc_date_global(scene_id, explicit_date)

        if source_mode == "upload":
            if not uploaded_tif:
                raise ValueError("Upload the NASA OBPG GeoTIFF")
            obpg_path = Path(str(uploaded_tif))
        elif (
            CYAN_STATE.get("source_path")
            and CYAN_STATE.get("source_date") == date.isoformat()
        ):
            obpg_path = Path(CYAN_STATE["source_path"])
        else:
            obpg_path = _download_obpg_cyano_global(date)

        lake = _selected_target_lake(lake_id)
        raw, raster_valid, lat, lon, inside, profile = _obpg_lake_window_global(
            obpg_path, lake
        )
        obpg_ci, science = _obpg_ci_from_dn_global(raw, raster_valid)

        # SAME-LAKE native source only. No QA prefiltering here.
        local = _load_scene_pixels(scene_id, "lake", lake_id).reset_index(drop=True)
        geo = (
            np.isfinite(pd.to_numeric(local.latitude, errors="coerce").to_numpy(float))
            & np.isfinite(pd.to_numeric(local.longitude, errors="coerce").to_numpy(float))
        )
        source = local.loc[geo].reset_index(drop=True)

        if source.empty:
            raise ValueError(
                "No finite compact native OLCI pixels exist for the selected lake"
            )

        target_lat = lat[inside]
        target_lon = lon[inside]
        if not len(target_lat):
            raise ValueError(
                "No NASA CyAN cell centres fall inside the selected lake polygon"
            )

        # Step 1: nearest ORIGINAL same-lake native observation.
        tree = cKDTree(_unit_sphere_xyz(source.latitude, source.longitude))
        chord, idx = tree.query(
            _unit_sphere_xyz(target_lat, target_lon),
            k=1,
            workers=-1,
        )
        dist = 6371000.0 * (
            2 * np.arcsin(np.minimum(1.0, np.asarray(chord, float) / 2))
        )
        matched = np.isfinite(dist) & (dist <= float(max_distance_m))

        n = len(target_lat)
        local_val = np.full(n, np.nan, np.float32)
        local_ci_valid = np.zeros(n, bool)
        local_candidate = np.zeros(n, bool)
        local_detected_raw = np.zeros(n, bool)

        nearest_primary_valid = np.zeros(n, bool)
        nearest_strict_valid = np.zeros(n, bool)
        nearest_selected_qa_valid = np.zeros(n, bool)
        nearest_bloom_rescued = np.zeros(n, bool)
        nearest_cldice = np.zeros(n, bool)
        nearest_hisatzen = np.zeros(n, bool)
        nearest_navfail = np.zeros(n, bool)

        nearest_lat = np.full(n, np.nan, float)
        nearest_lon = np.full(n, np.nan, float)

        if matched.any():
            pos = np.flatnonzero(matched)
            q = source.iloc[np.asarray(idx[matched], dtype=int)].reset_index(drop=True)

            nearest_lat[pos] = pd.to_numeric(
                q.latitude, errors="coerce"
            ).to_numpy(float)
            nearest_lon[pos] = pd.to_numeric(
                q.longitude, errors="coerce"
            ).to_numpy(float)

            primary_valid_q = _primary_selector(q)
            strict_valid_q = _cyan_comparison_selector(q)

            if str(qa_basis).lower() == "strict":
                selected_valid_q = strict_valid_q
                qa_label = "generic_CLDICE_excluded_sensitivity"
            else:
                selected_valid_q = primary_valid_q
                qa_label = "primary_bloom_aware"

            nearest_primary_valid[pos] = primary_valid_q
            nearest_strict_valid[pos] = strict_valid_q
            nearest_selected_qa_valid[pos] = selected_valid_q

            def _mask_col(frame, name):
                if name not in frame.columns:
                    return np.zeros(len(frame), bool)
                return (
                    pd.to_numeric(frame[name], errors="coerce")
                    .fillna(0)
                    .to_numpy(float)
                    > 0
                )

            nearest_bloom_rescued[pos] = _mask_col(q, "bloom_rescue_mask")
            nearest_cldice[pos] = _mask_col(q, "cldice_mask")
            nearest_hisatzen[pos] = _mask_col(q, "hisatzen_mask")
            nearest_navfail[pos] = _mask_col(q, "navfail_mask")

            q_val = pd.to_numeric(q.CI_cyano, errors="coerce").to_numpy(np.float32)
            q_ci_valid = _mask_col(q, "ci_valid_mask")
            q_candidate = _mask_col(q, "ci_candidate_mask")
            q_detected = _mask_col(q, "ci_detection_mask")

            # Keep native value for audit; QA validity is applied separately below.
            local_val[pos] = q_val
            local_ci_valid[pos] = q_ci_valid
            local_candidate[pos] = q_candidate
            local_detected_raw[pos] = q_detected
        else:
            qa_label = (
                "generic_CLDICE_excluded_sensitivity"
                if str(qa_basis).lower() == "strict"
                else "primary_bloom_aware"
            )

        raw_inside = raw[inside]
        science_inside = science[inside]
        obpg_inside = obpg_ci[inside]

        obpg_det = (
            science_inside
            & np.isfinite(obpg_inside)
            & (obpg_inside > CYAN_CI_DETECTION_LIMIT)
        )

        # Step 2: inspect QA AFTER the nearest native observation was selected.
        # A failed nearest pixel stays invalid; there is no second-nearest substitution.
        local_det = (
            matched
            & nearest_selected_qa_valid
            & local_detected_raw
            & np.isfinite(local_val)
            & (local_val > CYAN_CI_DETECTION_LIMIT)
        )

        metrics, paired = _cyan_metrics_v265(
            obpg_inside,
            np.where(local_det, local_val, np.nan),
        )

        table = pd.DataFrame(
            {
                "latitude": target_lat,
                "longitude": target_lon,
                "obpg_raw_dn": raw_inside.astype(np.int16),
                "obpg_science_valid": science_inside,
                "obpg_CI_cyano": obpg_inside,
                "obpg_detected_gt_0_0001": obpg_det,
                "local_match_found": matched,
                "local_match_distance_m": np.where(matched, dist, np.nan),
                "nearest_native_latitude": nearest_lat,
                "nearest_native_longitude": nearest_lon,
                "nearest_primary_bloom_aware_valid": nearest_primary_valid,
                "nearest_generic_strict_valid": nearest_strict_valid,
                "nearest_selected_QA_valid": nearest_selected_qa_valid,
                "nearest_CLDICE": nearest_cldice,
                "nearest_HISATZEN": nearest_hisatzen,
                "nearest_NAVFAIL": nearest_navfail,
                "nearest_bloom_rescued": nearest_bloom_rescued,
                "local_ci_input_valid": local_ci_valid,
                "local_ci_candidate": local_candidate,
                "local_ci_detection_before_selected_QA": local_detected_raw,
                "local_ci_cyano_detected": local_det,
                "local_CI_cyano": local_val,
                "paired_above_0_0001_for_correlation": paired,
            }
        )

        coverage = pd.DataFrame(
            [
                {
                    "date_utc": date.isoformat(),
                    "Hylak_id": int(lake_id),
                    "lake_name": str(lake.iloc[0].get("Lake_name", "") or ""),
                    "obpg_grid_cells_inside_lake": int(inside.sum()),
                    "obpg_science_valid_cells": int(science_inside.sum()),
                    "obpg_detection_cells_gt_0_0001": int(obpg_det.sum()),
                    "local_nearest_same_lake_matches": int(matched.sum()),
                    "nearest_primary_valid": int(
                        (matched & nearest_primary_valid).sum()
                    ),
                    "nearest_selected_QA_valid": int(
                        (matched & nearest_selected_qa_valid).sum()
                    ),
                    "nearest_CLDICE_pixels": int(
                        (matched & nearest_cldice).sum()
                    ),
                    "nearest_bloom_rescued_pixels": int(
                        (matched & nearest_bloom_rescued).sum()
                    ),
                    "local_detection_matches_gt_0_0001": int(local_det.sum()),
                    "paired_detection_pixels": metrics[
                        "paired_above_detection_limit_pixels"
                    ],
                    "max_nearest_distance_m": float(max_distance_m),
                    "local_QA_basis": qa_label,
                    "pairing_order": (
                        "nearest same-lake native OLCI observation FIRST; "
                        "selected QA inspected SECOND; no farther clear-pixel substitution"
                    ),
                    "comparison_basis": (
                        "same-day NASA Merged-S3-CYAN daily maximum versus one "
                        "local OLCI scene"
                    ),
                }
            ]
        )
        metric_table = pd.DataFrame([metrics])

        d = EXPORT_DIR / "cyan_comparison" / str(scene_id)
        d.mkdir(parents=True, exist_ok=True)
        stem = (
            f"{scene_id}_Hylak{int(lake_id)}_{date.isoformat()}_"
            f"CyAN_native_first_{qa_label}"
        )
        csv = d / f"{stem}_pairs.csv"
        met = d / f"{stem}_metrics.csv"
        nc = d / f"{stem}_comparison.nc"

        table.to_csv(csv, index=False)
        pd.concat([coverage, metric_table], axis=1).to_csv(met, index=False)

        shape = raw.shape

        def grid(values, fill=np.nan, dtype=np.float32):
            out = np.full(shape, fill, dtype=dtype)
            out[inside] = np.asarray(values, dtype=dtype)
            return out

        xds = xr.Dataset(
            {
                "latitude": (("y", "x"), lat.astype(np.float32)),
                "longitude": (("y", "x"), lon.astype(np.float32)),
                "inside_lake": (("y", "x"), inside.astype(np.uint8)),
                "obpg_raw_dn": (("y", "x"), raw.astype(np.int16)),
                "obpg_CI_cyano": (
                    ("y", "x"),
                    np.where(inside, obpg_ci, np.nan).astype(np.float32),
                ),
                "local_CI_cyano_nearest": (
                    ("y", "x"),
                    grid(local_val),
                ),
                "local_match_distance_m": (
                    ("y", "x"),
                    grid(np.where(matched, dist, np.nan)),
                ),
                "nearest_selected_QA_valid": (
                    ("y", "x"),
                    grid(nearest_selected_qa_valid, 0, np.uint8),
                ),
                "nearest_CLDICE": (
                    ("y", "x"),
                    grid(nearest_cldice, 0, np.uint8),
                ),
                "nearest_bloom_rescued": (
                    ("y", "x"),
                    grid(nearest_bloom_rescued, 0, np.uint8),
                ),
                "paired_detection_mask": (
                    ("y", "x"),
                    grid(paired, 0, np.uint8),
                ),
            },
            attrs={
                "title": (
                    "Native-first local L2Gen CIcyano versus NASA OBPG "
                    "Merged-S3-CYAN comparison"
                ),
                "schema": V268_SCHEMA,
                "created_utc": utc_now(),
                "scene_id": str(scene_id),
                "acquisition_date_utc": date.isoformat(),
                "selected_hylak_id": int(lake_id),
                "source_obpg_geotiff": str(obpg_path),
                "local_QA_basis": qa_label,
                "pairing": (
                    "nearest same-lake native OLCI lake pixel selected before QA; "
                    "QA-rejected nearest pixel is not replaced by a farther pixel"
                ),
                "max_nearest_distance_m": float(max_distance_m),
                "ci_cyano_detection_limit": float(CYAN_CI_DETECTION_LIMIT),
                "note": (
                    "NASA Merged-S3-CYAN is the daily per-pixel maximum across "
                    "S3A/S3B; the local source is one scene."
                ),
            },
        )
        xds.to_netcdf(
            nc,
            engine="netcdf4",
            encoding={
                n: {"zlib": True, "complevel": 4}
                for n in xds.data_vars
            },
        )
        xds.close()

        finite_pairs = table[table.paired_above_0_0001_for_correlation].copy()
        fig = go.Figure()
        fig.add_trace(
            go.Scattergl(
                x=finite_pairs.obpg_CI_cyano,
                y=finite_pairs.local_CI_cyano,
                mode="markers",
                name="Paired detections",
                marker={"size": 5, "opacity": 0.48},
            )
        )

        if not finite_pairs.empty:
            lo = float(
                min(
                    finite_pairs.obpg_CI_cyano.min(),
                    finite_pairs.local_CI_cyano.min(),
                )
            )
            hi = float(
                max(
                    finite_pairs.obpg_CI_cyano.max(),
                    finite_pairs.local_CI_cyano.max(),
                )
            )
            fig.add_trace(
                go.Scatter(
                    x=[lo, hi],
                    y=[lo, hi],
                    mode="lines",
                    name="1:1",
                    line={"dash": "dash", "color": "black"},
                )
            )

            xmin = max(
                float(finite_pairs.obpg_CI_cyano.min()),
                CYAN_CI_DETECTION_LIMIT,
            )
            xmax = float(finite_pairs.obpg_CI_cyano.max())
            if xmax > xmin:
                edges = np.geomspace(xmin, xmax, 9)
                medx, medy = [], []
                for a, b in zip(edges[:-1], edges[1:]):
                    upper = b if b < edges[-1] else b * 1.000001
                    qbin = finite_pairs[
                        (finite_pairs.obpg_CI_cyano >= a)
                        & (finite_pairs.obpg_CI_cyano < upper)
                    ]
                    if len(qbin) >= 5:
                        medx.append(float(np.median(qbin.obpg_CI_cyano)))
                        medy.append(float(np.median(qbin.local_CI_cyano)))
                if medx:
                    fig.add_trace(
                        go.Scatter(
                            x=medx,
                            y=medy,
                            mode="lines+markers",
                            name="Binned median local",
                            line={"width": 3},
                        )
                    )

        fig.update_layout(
            title=(
                f"{coverage.iloc[0].lake_name} · native-first local scene "
                "versus same-day NASA CyAN"
            ),
            xaxis={
                "title": "NASA OBPG Merged-S3-CYAN CIcyano",
                "type": "log",
            },
            yaxis={"title": "Local L2Gen CIcyano", "type": "log"},
            template="plotly_white",
            height=600,
        )

        note = (
            f"Paired detections >0.0001: "
            f"{metrics['paired_above_detection_limit_pixels']:,}. "
            "Nearest same-lake native OLCI observation is selected before QA; "
            "a rejected nearest pixel is never replaced by a farther clear pixel. "
            "Default comparison QA is the bloom-aware science mask. "
            "NASA Merged-S3-CYAN is a same-day S3A/S3B daily maximum, while the "
            "local product is one scene."
        )

        return (
            fig,
            coverage.round(6),
            metric_table.round(8),
            str(csv),
            str(met),
            str(nc),
            str(obpg_path),
            _app_status("NASA CyAN comparison complete", note),
        )

    except Exception as exc:
        return (
            None,
            pd.DataFrame(),
            pd.DataFrame(),
            None,
            None,
            None,
            None,
            _app_status("NASA CyAN comparison failed", str(exc), False),
        )


print("v2.6.8 CyAN comparison patch active:")
print("  pairing order            : nearest same-lake native pixel FIRST")
print("  QA application           : AFTER nearest native match")
print("  rejected nearest pixel   : stays no-data; no farther substitution")
print("  default comparison QA    : primary bloom-aware")
print("  strict option            : generic CLDICE-excluded sensitivity only")


v2.6.8 CyAN comparison patch active:
  pairing order            : nearest same-lake native pixel FIRST
  QA application           : AFTER nearest native match
  rejected nearest pixel   : stays no-data; no farther substitution
  default comparison QA    : primary bloom-aware
  strict option            : generic CLDICE-excluded sensitivity only


In [36]:
# =============================================================================
# v2.6.8 GRADIO UI REBUILD
# Uses the v2.6.5 correction helpers defined immediately above.
# =============================================================================

APP_CSS_GLOBAL = '.gradio-container {max-width: 1550px !important}.small-note{font-size:.9rem;color:#4b5563}'

with gr.Blocks(title='Sentinel-3 OLCI Global CyanoLake Processor v2.6.8') as DEMO_GLOBAL:
    gr.Markdown('# Sentinel-3 OLCI → L2Gen → global HydroLAKES processor · v2.6.8')
    gr.Markdown(
        'No Google Drive permission is requested. Production indices are **CI, CIcyano, NDCI, MPH, and OLCI-adapted FAI/AFAI**. '
        'v2.6.5 adds exact-resolution map previews, active State/Province/bbox display filtering, scene-only lake dropdowns, '
        'clickable spectral sampling, expanded NASA CyAN bias diagnostics, and a SNAP-friendly EPSG:4326 share raster that uses '
        'nearest native OLCI observations rather than center-only binning.'
    )

    with gr.Tab('1 · Plan & process scenes'):
        gr.Markdown(
            '### Recommended order\n'
            '**A.** Load HydroLAKES. For country/state-by-state production, keep **WORLD** so one canonical lake universe is used.  \n'
            '**B.** Choose **State / province (ADM1)**, **Bounding box**, or **5° grid** as the work-tracking shard.  \n'
            '**C.** Plan the queue and process a small batch. A shard controls discovery/tracking only; Sentinel-3 scene ID + science configuration prevents duplicate L2Gen processing.  \n'
            '**D.** Files are runtime-local under `/content`. Download share ZIPs/registry files before the Colab runtime ends.'
        )
        with gr.Row():
            region = gr.Dropdown(
                ['WORLD','USA','Africa','Asia','Europe','North America','South America','Oceania'],
                value='WORLD', label='HydroLAKES universe (WORLD recommended)'
            )
            min_area = gr.Number(value=10.0, label='Minimum lake area (km²)')
            existing_hydro = gr.Textbox(value='', label='Optional existing HydroLAKES path')
            load_lakes = gr.Button('Load / prepare HydroLAKES', variant='primary')
        lake_preview = gr.Dataframe(interactive=False)
        lake_status = gr.HTML()
        load_lakes.click(load_lakes_callback, [region,min_area,existing_hydro], [lake_preview,lake_status])

        gr.Markdown('### Choose how to define the next processing shard')
        planning_mode = gr.Radio(
            [('State / province / region (ADM1)','adm1'),('Bounding box','bbox'),('Fixed 5° grid','grid')],
            value='adm1', label='Planning method'
        )
        with gr.Row():
            country_iso3 = gr.Dropdown(COUNTRY_CHOICES, value='USA', label='Country (for ADM1 mode)')
            load_admin = gr.Button('Load states / provinces')
            admin1_choice = gr.Dropdown(label='State / province / first-level region')
        admin_status = gr.HTML()
        load_admin.click(load_adm1_callback, country_iso3, [admin1_choice,admin_status])
        gr.Markdown('<span class="small-note">ADM1 boundaries are fetched on demand from geoBoundaries gbOpen. The selected geometry is retained as the **active planning region** and can be used later to filter the map display without changing the full-scene archive/export.</span>')

        with gr.Row():
            start_date = gr.Textbox(value='2024-08-01', label='Start date')
            end_date = gr.Textbox(value='2024-08-07', label='End date')
            bbox_text = gr.Textbox(value='', label='BBox west,south,east,north (bbox mode)')
            grid_id = gr.Textbox(value='', placeholder='G05_N40_W085', label='5° grid ID (grid mode)')
            max_products = gr.Number(value=0, precision=0, label='Catalogue cap (0=all)')
            plan_button = gr.Button('Plan scene queue', variant='primary')
        queue_table = gr.Dataframe(interactive=False)
        plan_snapshot = gr.File(label='Current shard plan CSV')
        plan_status = gr.HTML()
        plan_button.click(
            plan_callback,
            [start_date,end_date,planning_mode,country_iso3,admin1_choice,bbox_text,grid_id,max_products],
            [queue_table,plan_snapshot,plan_status]
        )

        with gr.Row():
            cdse_user = gr.Textbox(label='CDSE username')
            cdse_password = gr.Textbox(label='CDSE password', type='password')
            n_scenes = gr.Slider(1,20,value=1,step=1,label='Scenes this batch')
            dl_method = gr.Dropdown(['auto','s3','zip'], value='auto', label='L1 staging')
            retry_failed = gr.Checkbox(False, label='Retry failed')
            run_button = gr.Button('Run next queued scenes', variant='primary')
        batch_status = gr.HTML()
        storage_after = gr.Dataframe(interactive=False)
        run_button.click(
            run_batch_callback,
            [cdse_user,cdse_password,n_scenes,dl_method,retry_failed],
            [queue_table,batch_status,storage_after]
        )
        with gr.Row():
            refresh_queue = gr.Button('Refresh queue')
            registry_file = gr.File(label='Scene registry CSV')
        refresh_queue.click(refresh_queue_callback, [], [queue_table,registry_file])

    with gr.Tab('2 · Lake time series'):
        with gr.Row():
            refresh_lakes = gr.Button('Refresh processed lakes', variant='primary')
            lake_choice = gr.Dropdown(label='Lake')
            plot_metric = gr.Dropdown(PLOT_METRICS, value=PLOT_METRICS[0], label='Statistic')
            plot_button = gr.Button('Plot lake history')
        lakes_status = gr.HTML(); lake_plot = gr.Plot(); lake_table = gr.Dataframe(interactive=False)
        refresh_lakes.click(refresh_lakes_callback, [], [lake_choice,lakes_status])
        plot_button.click(lake_timeseries_callback, [lake_choice,plot_metric], [lake_plot,lake_table])

    with gr.Tab('3 · Scene / multi-lake map'):
        gr.Markdown(
            '**Resolution is now literal.** Choosing 300 m requests an exact 300 m Web-Mercator display; the app no longer silently changes it to ~500 m. '
            'If a full-scene 300 m preview is too large for the browser, the app will tell you to use the active planning region, a lake, or explicitly choose a coarser preview. '
            'The **Active planning region** option clips the display to the State/Province/bbox/5° shard selected in Tab 1; the stored scene and share ZIP remain full-scene. Mapped cells use the nearest native observation from the **same HydroLAKES lake only**, preventing cross-lake borrowing and avoiding center-binning striping without interpolation.'
        )
        with gr.Row():
            refresh_scenes = gr.Button('Refresh completed scenes', variant='primary')
            scene_choice = gr.Dropdown(label='Completed scene')
            map_scope = gr.Radio(
                [('Entire processed scene · all lakes','scene'),('Active planning region · State/bbox/grid','planning'),('Selected lake','lake')],
                value='planning', label='Map display scope'
            )
            scene_lake = gr.Dropdown(label='Lake within selected scene')
        scene_status = gr.HTML()
        refresh_scenes.click(refresh_scene_choices_callback, [], [scene_choice,scene_lake,scene_status])
        scene_choice.change(scene_changed_callback, scene_choice, scene_lake)
        with gr.Row():
            map_metric = gr.Dropdown(MAP_METRICS, value='CI_cyano', label='Map layer')
            map_resolution = gr.Dropdown([300,500,1000,2000], value=300, label='Exact display grid (m)')
            basemap = gr.Dropdown(BASEMAPS, value='Esri Dark Gray', label='Basemap')
            labels = gr.Checkbox(True, label='Show Esri gray labels')
            show_lakes = gr.Checkbox(True, label='HydroLAKES boundaries')
        with gr.Row():
            cmap = gr.Dropdown(CMAPS, value='cividis', label='Colormap')
            reverse = gr.Checkbox(False, label='Reverse')
            vmin = gr.Textbox(value='', placeholder='Auto', label='Color minimum')
            vmax = gr.Textbox(value='', placeholder='Auto', label='Color maximum')
            render = gr.Button('Render / export map', variant='primary')
        map_status = gr.HTML(); map_html = gr.HTML()
        with gr.Row():
            html_file = gr.File(label='Portable interactive HTML · hover values')
            snap_nc = gr.File(label='Displayed-scope CF NetCDF · exact nominal resolution')
            compact_file = gr.File(label='Compact native lake-pixel NetCDF / ZIP')
        render.click(
            map_callback,
            [scene_choice,map_scope,scene_lake,map_metric,map_resolution,basemap,labels,cmap,reverse,vmin,vmax,show_lakes],
            [map_html,html_file,snap_nc,compact_file,map_status]
        )

    with gr.Tab('4 · Spectra & lake statistics'):
        gr.Markdown('All spectra use the **native stored OLCI lake pixels**. The coordinate workflow now includes a clickable geographic lake-pixel map; a click is converted to lat/lon and matched to the nearest eligible native OLCI pixel.')
        gr.Markdown('### A · Selected-lake mean / median spectrum')
        with gr.Row():
            spec_scene = gr.Dropdown(label='Completed scene')
            spec_lake = gr.Dropdown(label='Lake within scene')
            spec_stat = gr.Radio(['median','mean'], value='median', label='Spectrum statistic')
            spec_button = gr.Button('Plot lake spectrum', variant='primary')
            spec_refresh = gr.Button('Refresh scenes')
        spec_status = gr.HTML()
        with gr.Row():
            spec_plot = gr.Plot()
            spec_table = gr.Dataframe(interactive=False, label='Lake spectral summary')
        lake_stats_table = gr.Dataframe(interactive=False, label='Stored lake statistics')
        spec_csv = gr.File(label='Lake spectrum CSV')
        spec_refresh.click(refresh_scene_choices_callback, [], [spec_scene,spec_lake,spec_status])
        spec_scene.change(scene_changed_callback, spec_scene, spec_lake)
        spec_button.click(scene_spectrum_callback,[spec_scene,spec_lake,spec_stat],[spec_plot,spec_table,lake_stats_table,spec_csv,spec_status])

        gr.Markdown('### B · Coordinate / map-click native-pixel spectra')
        coordinate_points = gr.Dataframe(
            headers=['label','latitude','longitude'], datatype=['str','number','number'],
            value=pd.DataFrame([{'label':'P1','latitude':41.7,'longitude':-82.9}]),
            row_count=(1,'dynamic'), column_count=(3,'fixed'), type='pandas', interactive=True,
            label='Target coordinates · edit manually or click the map'
        )
        with gr.Row():
            coord_label=gr.Textbox(value='P2',label='New point label')
            coord_lat=gr.Number(value=41.8,label='Latitude')
            coord_lon=gr.Number(value=-83.1,label='Longitude')
            coord_add=gr.Button('Add coordinate')
            coord_clear=gr.Button('Clear points')
            spectrum_map_refresh=gr.Button('Refresh clickable map')
        with gr.Row():
            coord_basis=gr.Radio(
                [('Any stored HydroLAKES pixel','stored_lake'),('Primary bloom-aware valid-water pixel','valid_water'),('CI-input-valid pixel','ci_valid')],
                value='stored_lake',label='Nearest-pixel basis'
            )
            coord_distance=gr.Number(value=2000,minimum=100,maximum=10000,label='Maximum match distance (m)')
            coord_plot_button=gr.Button('Plot coordinate spectra',variant='primary')
        spectrum_map_state = gr.State({})
        with gr.Row():
            spectrum_map = gr.Image(label='Clickable native OLCI lake-pixel map · click to add a spectrum point', type='pil', interactive=True, height=560)
            coord_plot = gr.Plot(label='Native-pixel spectra')
        coord_table=gr.Dataframe(interactive=False,label='Matched spectral values and coordinates')
        coord_csv=gr.File(label='Coordinate spectra CSV')
        coord_status=gr.HTML()
        coord_add.click(add_spectrum_point_callback,[coord_label,coord_lat,coord_lon,coordinate_points],[coordinate_points,coord_status])
        spectrum_map_refresh.click(spectrum_map_refresh_callback,[spec_scene,coordinate_points],[spectrum_map,spectrum_map_state,coord_status])
        coord_clear.click(clear_spectrum_points_and_map_callback,[spec_scene],[coordinate_points,spectrum_map,spectrum_map_state,coord_status])
        coord_plot_button.click(
            coordinate_spectra_with_map_callback,
            [spec_scene,coordinate_points,coord_basis,coord_distance],
            [spectrum_map,spectrum_map_state,coord_plot,coord_table,coord_csv,coord_status]
        )
        spectrum_map.select(
            spectrum_map_click_callback,
            [spec_scene,coordinate_points,coord_basis,coord_distance,spectrum_map_state],
            [coordinate_points,spectrum_map,spectrum_map_state,coord_plot,coord_table,coord_csv,coord_status]
        )
        spec_scene.change(spectrum_map_refresh_callback,[spec_scene,coordinate_points],[spectrum_map,spectrum_map_state,coord_status])

    with gr.Tab('5 · NASA CyAN comparison'):
        gr.Markdown(
            'Compare one processed scene/lake with same-day NASA **Merged-S3-CYAN**. The NASA daily merged product uses the per-pixel maximum across S3A/S3B, while the local result is one OLCI scene; therefore high-end local-minus-NASA negative bias can be physically expected and should not be removed by empirical scaling. '
            'Use **CyAN-strict** QA for the closest product-consistency comparison; bloom-aware QA is provided as a diagnostic sensitivity option.'
        )
        with gr.Row():
            cyan_refresh=gr.Button('Refresh completed scenes',variant='primary')
            cyan_scene=gr.Dropdown(label='Completed scene')
            cyan_lake=gr.Dropdown(label='Lake actually stored in selected scene')
        cyan_refresh_status=gr.HTML()
        cyan_refresh.click(refresh_scene_choices_callback,[],[cyan_scene,cyan_lake,cyan_refresh_status])
        cyan_scene.change(scene_changed_callback,cyan_scene,cyan_lake)
        with gr.Row():
            cyan_date=gr.Textbox(value='',placeholder='blank = infer from scene',label='UTC date YYYY-MM-DD')
            cyan_mode=gr.Radio([('Download from NASA OBPG using configured Earthdata Login','download'),('Upload existing OBPG CIcyano GeoTIFF','upload')],value='download',label='NASA source')
            cyan_upload=gr.File(label='Optional OBPG CIcyano GeoTIFF',type='filepath',file_types=['.tif','.tiff'])
            cyan_distance=gr.Number(value=450,minimum=50,maximum=1500,label='Max nearest-neighbour distance (m)')
            cyan_qa_basis=gr.Radio([('Bloom-aware · recommended science mask','primary'),('Generic CLDICE-excluded · sensitivity only','strict')],value='primary',label='Local QA basis')
        with gr.Row():
            cyan_prepare=gr.Button('1 · Download / check NASA GeoTIFF')
            cyan_compare=gr.Button('2 · Compare selected lake',variant='primary')
        cyan_source=gr.File(label='NASA source GeoTIFF')
        cyan_status=gr.HTML()
        cyan_coverage=gr.Dataframe(interactive=False,label='Coverage / pairing audit')
        cyan_metrics=gr.Dataframe(interactive=False,label='Bias / correlation diagnostics')
        cyan_plot=gr.Plot(label='Paired detections + binned median trend')
        with gr.Row():
            cyan_pairs=gr.File(label='Paired cells CSV')
            cyan_metrics_file=gr.File(label='Metrics CSV')
            cyan_nc=gr.File(label='Comparison NetCDF')
            cyan_tif=gr.File(label='NASA source GeoTIFF')
        cyan_prepare.click(prepare_cyan_source_callback,[cyan_scene,cyan_date,cyan_mode,cyan_upload],[cyan_source,cyan_status])
        cyan_compare.click(
            compare_cyan_callback_v265,
            [cyan_scene,cyan_lake,cyan_date,cyan_mode,cyan_upload,cyan_distance,cyan_qa_basis],
            [cyan_plot,cyan_coverage,cyan_metrics,cyan_pairs,cyan_metrics_file,cyan_nc,cyan_tif,cyan_status]
        )

    with gr.Tab('6 · Global archive & share'):
        gr.Markdown(
            'Use this tab **after processing scenes**. The share ZIP remains deliberately small: **one all-science NetCDF + one all-index per-lake CSV + one metadata JSON**. '\
            'The all-science NetCDF is now **EPSG:4326 for direct SNAP/GIS geolocation** and uses an exact nominal requested grid (300 m by default). '\
            'It maps each target lake cell only to the nearest stored native OLCI observation from the same Hylak_id before applying QA. This avoids artificial center-binning striping while preserving cloud/no-data gaps. NASA-style area weighting is not used in the production branch because native observations remain the quantitative source of truth.'
        )
        with gr.Row():
            archive_grid = gr.Textbox(value='', placeholder='G05_N40_W085', label='Optional grid filter')
            archive_orbit = gr.Textbox(value='', placeholder='211', label='Optional relative-orbit filter')
            archive_refresh = gr.Button('Refresh archive index', variant='primary')
        archive_table = gr.Dataframe(interactive=False); archive_csv = gr.File(label='Global archive index CSV'); archive_status = gr.HTML()
        archive_refresh.click(archive_refresh_callback, [archive_grid,archive_orbit], [archive_table,archive_csv,archive_status])
        with gr.Row():
            bundle_scene = gr.Dropdown(label='Completed scene')
            bundle_resolution = gr.Dropdown([300,500,1000], value=300, label='Share NetCDF nominal grid resolution (m)')
            bundle_refresh = gr.Button('Refresh completed scenes')
            bundle_button = gr.Button('Create 3-file share ZIP', variant='primary')
        bundle_dummy_lake = gr.Dropdown(visible=False)
        bundle_file = gr.File(label='Shareable scene ZIP · exactly 3 files')
        bundle_status = gr.HTML()
        bundle_refresh.click(refresh_scene_choices_callback, [], [bundle_scene,bundle_dummy_lake,bundle_status])
        bundle_button.click(bundle_callback, [bundle_scene,bundle_resolution], [bundle_file,bundle_status])

    with gr.Tab('7 · Storage & provenance'):
        with gr.Row():
            refresh_storage = gr.Button('Refresh storage')
            clear_button = gr.Button('Clear scratch only')
        storage_table = gr.Dataframe(interactive=False); storage_status = gr.HTML(); registry_download = gr.File(label='Registry CSV')
        refresh_storage.click(storage_callback, [], [storage_table,registry_download])
        clear_button.click(clear_scratch_callback, [], [storage_table,storage_status])

print('Global-efficient Gradio v2.6.5 constructed.')


Global-efficient Gradio v2.6.5 constructed.


In [38]:
# Launch only the global-efficient dashboard.
# The share URL is public; a temporary password is generated for Colab.
import secrets
SHARE_APP = bool(IS_COLAB)
APP_LOGIN = ("olci", secrets.token_urlsafe(12)) if IS_COLAB else None
if APP_LOGIN:
    print(f"Temporary Gradio login — username: {APP_LOGIN[0]}  password: {APP_LOGIN[1]}")
GLOBAL_APP_HANDLE = DEMO_GLOBAL.launch(
    server_port=7860,
    share=SHARE_APP, inline=bool(IS_COLAB), prevent_thread_lock=True, show_error=True,
    server_name="0.0.0.0" if IS_COLAB else "127.0.0.1", auth=APP_LOGIN,
    allowed_paths=[str(PERSISTENT_ROOT), str(SCRATCH_ROOT)], css=APP_CSS_GLOBAL,
    theme=gr.themes.Soft(primary_hue="cyan", secondary_hue="emerald"),
)
print("Global-efficient v2.6.5 app launched. Keep this runtime connected while processing.")


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
Global-efficient v2.6.5 app launched. Keep this runtime connected while processing.


HydroLAKES_polys_v10_shp.zip:   0%|          | 0.00/820M [00:00<?, ?B/s]

cb_2025_us_state_5m.zip:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

Cached 10,103 lakes: /home/myid/mc16405/cyanolake-web/s3_olci_global/hydrolakes/HydroLAKES_USA_FULLPOLY_v14_ge1.3km2.gpkg


Planning ADM1 · United States of America · Michigan:   0%|          | 0/4 [00:00<?, ?it/s]

...

Planning ADM1 · United States of America · Michigan:   0%|          | 0/3 [00:00<?, ?it/s]

S3 direct staging failed; using ZIP fallback: 401 Client Error: Unauthorized for url: https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token
Scene failed S3B_OL_1_EFR____20240814T153711_20240814T154011_20240815T084407_0180_096_225_2160_PS2_O_NT_004: 401 Client Error: Unauthorized for url: https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token
S3 direct staging failed; using ZIP fallback: 401 Client Error: Unauthorized for url: https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token
Scene failed S3A_OL_1_EFR____20240814T161556_20240814T161856_20240815T163959_0179_115_368_2160_PS1_O_NT_004: 401 Client Error: Unauthorized for url: https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token


Planning ADM1 · United States of America · Michigan:   0%|          | 0/4 [00:00<?, ?it/s]

Planning ADM1 · United States of America · Michigan:   0%|          | 0/4 [00:00<?, ?it/s]

Planning ADM1 · United States of America · Michigan:   0%|          | 0/4 [00:00<?, ?it/s]

## 12. Production notes and recommended global strategy

1. **Use one canonical WORLD HydroLAKES universe** for production at your chosen minimum lake area. Country/ADM1 selections, date ranges, bboxes, and 5° grid IDs are discovery/work-tracking shards only.
2. **Sentinel-3 scene ID + science configuration is the deduplication key.** Relative orbit + frame organizes the archive; an ADM1 state/province, bbox, or geographic grid never intentionally triggers duplicate atmospheric correction.
3. **Process small batches first.** One day/week or a few scenes at a time is appropriate while validating the workflow.
4. **Persistent products are lake-only.** L1B and dense L2Gen files are temporary unless you explicitly change the retention switches.
5. **Production index set:** CI, CIcyano, NDCI, MPH, and OLCI-adapted FAI/AFAI. FLH and OLH are intentionally excluded.
6. **Map vs GIS export:** live/HTML visualization is exact-resolution Web Mercator for Leaflet/Esri alignment; the share CF NetCDF is nominal-resolution EPSG:4326 native-nearest remapping for SNAP/GIS interoperability.
7. **Archive tab comes after processing.** It creates/searches metadata and share bundles only; it never reruns L2Gen.
8. **For large-scale global production**, move this same registry → scratch worker → compact archive design to persistent VM/HPC/cloud workers once the Colab science is validated.

9. **No Google Drive access is requested.** Runtime-local `/content` files disappear when Colab deletes the runtime, so download share ZIPs and registry/plan CSVs you need to retain.
10. **ADM1 boundaries** are fetched on demand from geoBoundaries gbOpen and cached only in the current runtime.


### v2.6 final additions
- Corrected live-map geolocation by building the displayed raster in EPSG:3857 with exact pixel-edge bounds.
- Esri Light Gray, Dark Gray and World Imagery basemaps require no user API-key entry in the notebook.
- Entire-scene multi-lake visualization remains available.
- Map layers include true color, CI, CIcyano, NDCI, MPH, FAI/AFAI and retained rhos bands.
- Continuous scalar legend and native-pixel hover readout are retained in portable HTML.
- FLH and OLH are removed from production calculations/storage.
- Compact output folders are organized by platform / relative orbit / frame / date.
- Optional 5° grid IDs such as `G05_N40_W085` are available for catalogue planning and archive filtering.
- Added compact lake spectra/statistics tab and on-demand complete persistent-results ZIP.
